# imports

In [4]:
import os, re, csv, time, math, copy, random, hashlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc as sk_auc
import matplotlib.pyplot as plt

import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

torch.set_grad_enabled(True)

# -------------------------
# REPRO
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


# Test paths

In [5]:
# ============================================================
# 2) PATHS (YOUR EXACT KAGGLE PATHS)
# ============================================================
NIH_IMG   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST  = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_TRAIN = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"


# Loading TTDA Case C  ( experiment ) 4 qubits

# Cell1: The code loads a CheXpert-trained frozen ResNet50, prepares NIH images, extracts CNN features, and adds a trainable quantum residual adapter. It performs unlabeled test-time adaptation using TENT, CoTTA, or EATA, averages augmented predictions, computes metrics, compares strategies, and saves results. The frozen ResNet50 converts each X-ray into 2,048 features. A linear layer compresses them into eight rotation angles for a four-qubit circuit. RY/RZ gates encode features, CNOTs entangle qubits, trainable rotations adapt them, and measured Z expectations form a residual correction added to the original disease logits during TTDA adaptation. 


In [5]:
# ============================================================
# CASE C (PAPER-WORTHY, FIXED v3):
# PQC-only Test-Time Adaptation (TTDA) + Continual TTDA + TTA(+AutoAug)
#
# - Load CheX-trained ResNet50 (.pt) and freeze CNN
# - Add PQC residual adapter head (trainable)
# - TTDA strategies (PQC-only):
#     1) PQC_TENT (entropy minimization)
#     2) PQC_CoTTA_head (EMA teacher + stochastic restore)
#     3) PQC_EATA_lite (entropy-min on reliable samples)
# - Eval protocol:
#     "continual": adapt online per test batch (unlabeled), then predict
#     "ttda_once": adapt for N steps on stream first, then predict
# - TTA: identity + flip + optional AutoAugment view
# - Report: per-class ROC-AUC / PR-AUC / F1, macro/micro + timing, save CSV
# ============================================================

import os, math, copy, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# ----------------------------
# CONFIG
# ----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

PI = float(math.pi)

# Your CheX-trained checkpoint
RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"

# NIH paths
NIH_IMG   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST  = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_TRAIN = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"  # unlabeled stream

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

# Runtime knobs
BATCH_SIZE = 16 if DEVICE.type=="cuda" else 8
NUM_WORKERS = 0

# PQC knobs
N_QUBITS = 4
SHOTS = 128                # only used if QNN_ENGINE="sampler"
QNN_ENGINE = "estimator"   # "estimator" (recommended) or "sampler"

# Eval protocol
EVAL_PROTOCOL = "continual"     # "continual" or "ttda_once"
MAX_TEST_BATCHES = None         # set e.g. 200 for quick debug

# Continual update per test batch
ADAPT_STEPS_PER_BATCH = 1
LR_TTA = 5e-4
GRAD_CLIP = 1.0

# TTA (prediction-time augmentation)
USE_TTA = True
USE_AUTOAUG_TTA = True   # expensive (PIL->aug->tensor). Turn off if slow.
TTA_VIEWS = 3            # 1=identity, 2=+flip, 3=+autoaug

# TTDA-once adaptation steps if EVAL_PROTOCOL="ttda_once"
TTDA_ONCE_STEPS = 200

print("DEVICE:", DEVICE)
for p in [RESNET_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_TRAIN]:
    print(("✅" if os.path.exists(p) else "❌"), p)

# ----------------------------
# Install/import Qiskit ML
# ----------------------------
def _pip_install(pkg):

    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import qiskit
    import qiskit_machine_learning
except Exception:
    _pip_install("qiskit")
    _pip_install("qiskit-machine-learning")
    import qiskit
    import qiskit_machine_learning

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.providers.basic_provider import BasicProvider
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

basic_backend = BasicProvider().get_backend("basic_simulator")

# ----------------------------
# NIH utilities
# ----------------------------
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def build_nih_df(split_list_txt, nih_csv_path):
    wanted = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")
    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")  # NIH uses "Effusion"

    pid_col = "Patient ID" if "Patient ID" in df.columns else None
    df["patient_id"] = df[pid_col].astype(str) if pid_col else "unknown"

    df = df.rename(columns={"Image Index":"image"})
    df = df[["image","patient_id"] + UNIFIED_LABELS].reset_index(drop=True)
    return df

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(f"NIH image not found: {p}")
    return p

# ----------------------------
# Datasets returning PIL
# ----------------------------
class CXRDatasetPIL(Dataset):
    def __init__(self, df, img_resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.images = self.df["image"].tolist()
        self.labeled = labeled
        if labeled:
            self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        if self.labeled:
            y = torch.from_numpy(self.labels[idx])
        else:
            y = torch.zeros(len(UNIFIED_LABELS), dtype=torch.float32)
        return img, y

# ✅ FIX: collate function for PIL batches
def pil_collate_fn(batch):
    imgs, ys = zip(*batch)
    return list(imgs), torch.stack(list(ys), dim=0)

nih_test_df  = build_nih_df(NIH_TEST, NIH_CSV)     # labeled for evaluation
nih_train_df = build_nih_df(NIH_TRAIN, NIH_CSV)    # unlabeled stream (labels ignored)

test_ds_pil   = CXRDatasetPIL(nih_test_df, nih_resolver, labeled=True)
stream_ds_pil = CXRDatasetPIL(nih_train_df, nih_resolver, labeled=False)

test_loader = DataLoader(
    test_ds_pil,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,               # PIL lists => pin_memory doesn't help
    collate_fn=pil_collate_fn       # ✅ critical
)

stream_loader = DataLoader(
    stream_ds_pil,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    collate_fn=pil_collate_fn
)

print("NIH test:", len(nih_test_df), "| NIH unlabeled stream:", len(nih_train_df))

# ----------------------------
# Transforms (operate on PIL)
# ----------------------------
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

eval_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    norm
])

weak_tf = eval_tf

strong_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.75, 1.0)),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    norm
])

# AutoAugment (optional)
try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
except Exception:
    autoaug = None
    USE_AUTOAUG_TTA = False
    print("⚠ AutoAugment not available in this torchvision; disabling AutoAug TTA.")

autoaug_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    (autoaug if autoaug is not None else T.Lambda(lambda x: x)),
    T.ToTensor(),
    norm
])

def batch_apply_tf(pil_list, tf):
    xs = [tf(im) for im in pil_list]
    return torch.stack(xs, dim=0)

# ----------------------------
# Load ResNet50 checkpoint (CheX-trained)
# ----------------------------
def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def load_resnet50_full(ckpt_path, n_labels=5):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(2048, n_labels)

    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:
        sd = ckpt["net"]
    else:
        sd = ckpt

    sd = _strip_prefix(sd)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("✅ Loaded checkpoint into ResNet50 (strict=False)")
    print("  missing:", len(missing), "| unexpected:", len(unexpected))
    return m

base_resnet = load_resnet50_full(RESNET_CKPT, n_labels=len(UNIFIED_LABELS)).to(DEVICE).eval()

# Freeze CNN
for p in base_resnet.parameters():
    p.requires_grad_(False)

feat_net = nn.Sequential(*(list(base_resnet.children())[:-1])).to(DEVICE).eval()
fc_w = base_resnet.fc.weight.detach().cpu()
fc_b = base_resnet.fc.bias.detach().cpu()

@torch.no_grad()
def extract_features(x_tensor):
    return feat_net(x_tensor).flatten(1)  # [B,2048] on DEVICE

# ----------------------------
# QNN builder
# ----------------------------
def popcount(x_int: int) -> int:
    return int(bin(x_int).count("1"))

def build_qnn(n_qubits=4, engine="estimator", shots=128):
    in_dim = 2 * n_qubits
    x = ParameterVector("x", in_dim)
    theta = ParameterVector("θ", n_qubits * 2)

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    if n_qubits > 1:
        qc.cx(n_qubits-1, 0)

    t = 0
    for i in range(n_qubits):
        qc.rz(theta[t], i); t += 1
        qc.ry(theta[t], i); t += 1

    qc_t = transpile(qc, basic_backend, optimization_level=0)

    if engine == "sampler":
        from qiskit.primitives import BackendSampler
        sampler = BackendSampler(backend=basic_backend, options={"shots": shots})
        qnn = SamplerQNN(
            circuit=qc_t,
            sampler=sampler,
            input_params=list(x),
            weight_params=list(theta),
            interpret=lambda bitstring: popcount(bitstring),
            output_shape=n_qubits + 1
        )
        return qnn, in_dim, f"SamplerQNN(shots={shots})"

    # estimator
    try:
        from qiskit.primitives import StatevectorEstimator as Est
    except Exception:
        from qiskit.primitives import Estimator as Est

    observables = []
    for i in range(n_qubits):
        z = ["I"] * n_qubits
        z[i] = "Z"
        observables.append(SparsePauliOp.from_list([("".join(z), 1.0)]))

    estimator = Est()
    qnn = EstimatorQNN(
        circuit=qc_t,
        estimator=estimator,
        observables=observables,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=True
    )
    return qnn, in_dim, "EstimatorQNN(Z-exp)"

qnn, Q_IN_DIM, qnn_tag = build_qnn(N_QUBITS, engine=QNN_ENGINE, shots=SHOTS)
qnn_torch = TorchConnector(qnn).to("cpu")  # keep QNN on CPU
print(f"✅ QNN ready: {qnn_tag} | qubits={N_QUBITS} | q_in_dim={Q_IN_DIM}")

# ----------------------------
# PQC Residual Adapter Head (trainable on CPU)
# ----------------------------
class PQCAdapterHead(nn.Module):
    """
    base logits = frozen ResNet fc(feats)  [computed on CPU]
    residual   = q_head(QNN(tanh(project(feats))*pi))
    output     = base + alpha * residual
    """
    def __init__(self, qnn_module, q_in_dim, n_labels=5):
        super().__init__()
        self.project = nn.Linear(2048, q_in_dim)
        self.qnn = qnn_module

        with torch.no_grad():
            dummy = torch.zeros(1, q_in_dim, dtype=torch.float64)
            out = self.qnn(dummy)
            qdim = int(out.shape[-1])

        self.q_head = nn.Linear(qdim, n_labels)
        self.alpha = nn.Parameter(torch.zeros(n_labels))

    def forward_logits_cpu(self, feats_cpu_fp32):
        base = F.linear(feats_cpu_fp32, fc_w, fc_b)
        q_in = torch.tanh(self.project(feats_cpu_fp32)) * PI
        q_in = q_in.to(dtype=torch.float64)
        q_out = self.qnn(q_in).to(dtype=torch.float32)
        res = self.q_head(q_out)
        return base + (self.alpha.unsqueeze(0) * res)

def make_adapter():
    return PQCAdapterHead(qnn_torch, q_in_dim=Q_IN_DIM, n_labels=len(UNIFIED_LABELS)).to("cpu")

# ----------------------------
# Losses (CPU logits)
# ----------------------------
def binary_entropy_from_logits_cpu(logits_cpu):
    p = torch.sigmoid(logits_cpu)
    eps = 1e-6
    ent = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps))
    return ent.mean()

def mse_prob_cpu(logits_a_cpu, logits_b_cpu):
    pa = torch.sigmoid(logits_a_cpu)
    pb = torch.sigmoid(logits_b_cpu)
    return F.mse_loss(pa, pb)

# ----------------------------
# TTA prediction (uses GPU for CNN features, CPU for PQC head)
# ----------------------------
@torch.no_grad()
def predict_tta_with_adapter(adapter, pil_imgs):
    """
    adapter: PQCAdapterHead on CPU
    pil_imgs: list[PIL] length B
    returns probs numpy [B,C]
    """
    views = []

    # view 1: identity
    x1 = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
    feats1 = extract_features(x1).detach().cpu()
    logits1 = adapter.forward_logits_cpu(feats1)
    p1 = torch.sigmoid(logits1).detach()
    views.append(p1)

    if USE_TTA and TTA_VIEWS >= 2:
        # view 2: flip
        pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
        x2 = batch_apply_tf(pil_flip, eval_tf).to(DEVICE)
        feats2 = extract_features(x2).detach().cpu()
        logits2 = adapter.forward_logits_cpu(feats2)
        p2 = torch.sigmoid(logits2).detach()
        views.append(p2)

    if USE_TTA and TTA_VIEWS >= 3 and USE_AUTOAUG_TTA:
        # view 3: AutoAugment
        x3 = batch_apply_tf(pil_imgs, autoaug_tf).to(DEVICE)
        feats3 = extract_features(x3).detach().cpu()
        logits3 = adapter.forward_logits_cpu(feats3)
        p3 = torch.sigmoid(logits3).detach()
        views.append(p3)

    p = torch.stack(views, dim=0).mean(dim=0)  # [B,C]
    return p.numpy()

# ----------------------------
# Metrics
# ----------------------------
def compute_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            auc, ap = np.nan, np.nan
        else:
            auc = roc_auc_score(yt, yp)
            ap  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1 = f1_score(yt.astype(int), yhat, zero_division=0)
        rows.append([lbl, auc, ap, f1])

    df_pc = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])

    out = {}
    out["per_class"] = df_pc
    out["macro_roc_auc"] = float(np.nanmean(df_pc["ROC_AUC"].values))
    out["macro_pr_auc"]  = float(np.nanmean(df_pc["PR_AUC"].values))
    out["macro_f1"]      = float(np.nanmean(df_pc["F1@0.5"].values))

    try:
        out["micro_roc_auc"] = float(roc_auc_score(Y.ravel(), P.ravel()))
        out["micro_pr_auc"]  = float(average_precision_score(Y.ravel(), P.ravel()))
    except Exception:
        out["micro_roc_auc"] = np.nan
        out["micro_pr_auc"]  = np.nan

    return out

# ----------------------------
# TTDA Strategies (PQC-only)
# ----------------------------
class PQC_TENT:
    """Entropy minimization on unlabeled batch (update only PQC adapter params)."""
    def __init__(self, adapter, lr=5e-4):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.parameters(), lr=lr)

    def step(self, pil_imgs):
        self.adapter.train()

        xw = batch_apply_tf(pil_imgs, weak_tf).to(DEVICE)
        feats = extract_features(xw).detach().cpu()
        logits_cpu = self.adapter.forward_logits_cpu(feats)

        loss = binary_entropy_from_logits_cpu(logits_cpu)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.parameters(), GRAD_CLIP)
        self.opt.step()
        return float(loss.detach().cpu())

class PQC_CoTTA_HEAD:
    """
    CoTTA-style for PQC head ONLY:
      teacher = EMA(adapter)
      loss = consistency(student(strong), teacher(weak))
      stochastic restore adapter params
    """
    def __init__(self, adapter, lr=5e-4, ema=0.999, restore_frac=0.01):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.parameters(), lr=lr)
        self.ema = ema
        self.restore_frac = restore_frac

        self.teacher = copy.deepcopy(self.adapter).eval()
        self.source = copy.deepcopy(self.adapter).eval()

    @torch.no_grad()
    def _ema_update(self):
        for pt, ps in zip(self.teacher.parameters(), self.adapter.parameters()):
            pt.data.mul_(self.ema).add_(ps.data, alpha=(1-self.ema))

    @torch.no_grad()
    def _stochastic_restore(self):
        if self.restore_frac <= 0:
            return
        src_params = dict(self.source.named_parameters())
        for name, p in self.adapter.named_parameters():
            if name not in src_params:
                continue
            mask = (torch.rand_like(p) < self.restore_frac)
            p.data[mask] = src_params[name].data[mask]

    def step(self, pil_imgs):
        self.adapter.train()

        # teacher on weak
        xw = batch_apply_tf(pil_imgs, weak_tf).to(DEVICE)
        feats_w = extract_features(xw).detach().cpu()
        with torch.no_grad():
            t_logits = self.teacher.forward_logits_cpu(feats_w)

        # student on strong
        xs = batch_apply_tf(pil_imgs, strong_tf).to(DEVICE)
        feats_s = extract_features(xs).detach().cpu()
        s_logits = self.adapter.forward_logits_cpu(feats_s)

        loss = mse_prob_cpu(s_logits, t_logits)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.parameters(), GRAD_CLIP)
        self.opt.step()

        self._ema_update()
        self._stochastic_restore()
        return float(loss.detach().cpu())

class PQC_EATA_LITE:
    """
    EATA-lite:
      - choose reliable samples with low entropy
      - entropy minimization only on those samples
    """
    def __init__(self, adapter, lr=5e-4, ent_thresh=0.35):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.parameters(), lr=lr)
        self.ent_thresh = ent_thresh

    def step(self, pil_imgs):
        self.adapter.train()

        xw = batch_apply_tf(pil_imgs, weak_tf).to(DEVICE)
        feats = extract_features(xw).detach().cpu()
        logits = self.adapter.forward_logits_cpu(feats)

        p = torch.sigmoid(logits)
        eps = 1e-6
        ent_per = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps)).mean(dim=1)  # [B]
        mask = (ent_per < self.ent_thresh).float()

        loss = (ent_per * mask).sum() / (mask.sum() + 1e-6)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.parameters(), GRAD_CLIP)
        self.opt.step()
        return float(loss.detach().cpu())

# ----------------------------
# Run evaluation
# ----------------------------
def run_eval(strategy_name, make_strategy_fn):
    """
    make_strategy_fn(adapter)-> strategy object with .step(pil_imgs) OR None
    """
    adapter = make_adapter()
    strategy = make_strategy_fn(adapter) if make_strategy_fn is not None else None

    # TTDA-once: adapt on unlabeled stream first, then evaluate
    t_adapt = 0.0
    if EVAL_PROTOCOL == "ttda_once" and strategy is not None:
        t0 = time.time()
        it = iter(stream_loader)
        for k in tqdm(range(TTDA_ONCE_STEPS), desc=f"TTDA-once adapt [{strategy_name}]"):
            try:
                pil_imgs, _ = next(it)
            except StopIteration:
                it = iter(stream_loader)
                pil_imgs, _ = next(it)
            strategy.step(pil_imgs)  # pil_imgs already list[PIL]
        t_adapt = time.time() - t0

    # Evaluate (continual = adapt per test batch; no_adapt = none)
    t0 = time.time()
    Ps, Ys = [], []
    adapt_losses = []

    for bi, (pil_imgs, y) in enumerate(tqdm(test_loader, desc=f"Eval [{strategy_name}]")):
        if MAX_TEST_BATCHES is not None and bi >= MAX_TEST_BATCHES:
            break

        if EVAL_PROTOCOL == "continual" and strategy is not None:
            for _ in range(ADAPT_STEPS_PER_BATCH):
                adapt_losses.append(strategy.step(pil_imgs))

        p_np = predict_tta_with_adapter(adapter, pil_imgs)
        Ps.append(p_np)
        Ys.append(y.numpy())

    t_eval = time.time() - t0

    P = np.vstack(Ps)
    Y = np.vstack(Ys)
    metrics = compute_metrics(Y, P, UNIFIED_LABELS)

    row = {
        "Strategy": strategy_name,
        "Protocol": EVAL_PROTOCOL,
        "QNN": qnn_tag,
        "Macro_ROC_AUC": metrics["macro_roc_auc"],
        "Macro_PR_AUC": metrics["macro_pr_auc"],
        "Macro_F1@0.5": metrics["macro_f1"],
        "Micro_ROC_AUC": metrics["micro_roc_auc"],
        "Micro_PR_AUC": metrics["micro_pr_auc"],
        "Adapt_seconds": t_adapt,
        "Eval_seconds": t_eval,
        "Mean_adapt_loss": float(np.mean(adapt_losses)) if len(adapt_losses) else np.nan
    }
    return row, metrics["per_class"]

# Define strategies
strategies = [
    ("PQC_no_adapt", None),
    ("PQC_TENT",      lambda adapter: PQC_TENT(adapter, lr=LR_TTA)),
    ("PQC_CoTTA_head",lambda adapter: PQC_CoTTA_HEAD(adapter, lr=LR_TTA, ema=0.999, restore_frac=0.01)),
    ("PQC_EATA_lite", lambda adapter: PQC_EATA_LITE(adapter, lr=LR_TTA, ent_thresh=0.35)),
]

summary_rows = []
perclass_dfs = {}

for name, make_strat in strategies:
    row, df_pc = run_eval(name, make_strat)
    summary_rows.append(row)
    perclass_d = df_pc.copy()
    perclass_dfs[name] = R = R = R = df_pc

df_summary = pd.DataFrame(summary_rows).sort_values(["Protocol","Strategy"]).reset_index(drop=True)

print("\n==================== SUMMARY ====================")
print(df_summary)

out_csv = f"CASE_C_PQC_TTDA_{EVAL_PROTOCOL}_{QNN_ENGINE}.csv"
df_summary.to_csv(out_csv, index=False)
print("\n✅ Saved:", out_csv)

# Save per-class for best macro ROC-AUC strategy
best_idx = df_summary["Macro_ROC_AUC"].astype(float).idxmax()
best_name = df_summary.loc[best_idx, "Strategy"]

# recompute best per-class from stored dict
df_best_pc = perclass_dfs[best_name].copy()

pc_csv = f"per_class_best_{best_name}_{EVAL_PROTOCOL}.csv"
df_best_pc.to_csv(pc_csv, index=False)

print(f"\nBest strategy: {best_name} | Macro ROC-AUC={df_summary.loc[best_idx,'Macro_ROC_AUC']:.4f}")
print(df_best_pc)
print("✅ Saved:", pc_csv)


DEVICE: cuda
✅ /kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt
NIH test: 25596 | NIH unlabeled stream: 86524


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


✅ Loaded checkpoint into ResNet50 (strict=False)
  missing: 0 | unexpected: 0
✅ QNN ready: EstimatorQNN(Z-exp) | qubits=4 | q_in_dim=8


Eval [PQC_EATA_lite]: 100%|██████████| 1600/1600 [1:54:22<00:00,  4.29s/it]


==================== SUMMARY ====================
         Strategy   Protocol                  QNN  Macro_ROC_AUC  \
0  PQC_CoTTA_head  continual  EstimatorQNN(Z-exp)       0.772239   
1   PQC_EATA_lite  continual  EstimatorQNN(Z-exp)       0.702807   
2        PQC_TENT  continual  EstimatorQNN(Z-exp)       0.708066   
3    PQC_no_adapt  continual  EstimatorQNN(Z-exp)       0.771299   

   Macro_PR_AUC  Macro_F1@0.5  Micro_ROC_AUC  Micro_PR_AUC  Adapt_seconds  \
0      0.253022      0.291822       0.763764      0.266441            0.0   
1      0.203965      0.238761       0.685316      0.199005            0.0   
2      0.214128      0.243331       0.668508      0.154290            0.0   
3      0.249835      0.290659       0.762963      0.262835            0.0   

   Eval_seconds  Mean_adapt_loss  
0   7199.193209         0.005233  
1   6862.283687         0.152180  
2   6868.801025         0.371807  
3   1104.477454              NaN  

✅ Saved: CASE_C_PQC_TTDA_continual_estimator.c

# Cell2:Prposed architecture
Frozen CheXpert ResNet50 + BiomedBERT Domain Descriptor
LLM-Conditioned 4-Qubit PQC Adapter with Dual-Memory Alignment
Source: CheXpert | Unlabeled Target Adaptation/Evaluation: NIH
Ablations: CNN-Only, QTTA Without Adaptation, Full QTTA
1. ResNet50 extracts frozen chest X-ray features.
2. BiomedBERT creates domain-context embeddings.
3. A four-qubit QNN modifies CNN features.
4. Only 22 quantum parameters are trainable.
5. Source memory stores 480 reference samples.
6. Quantum source statistics are cached.
7. However, `ADAPT_STEPS_PER_BATCH=0`, so no TTDA occurs.
8. Therefore, all adaptation losses remain `NaN`.
9. QTTA improves AUROC only from 0.776187 to 0.776197—effectively zero.
10. It is slower, and the BiomedBERT checkpoint mismatch needs correction. 


In [5]:
# ============================================================
# QTTA / TTDA (PROPOSAL-ALIGNED, YOUR SUPERVISOR CONSTRAINTS)
#
# ✅ Frozen CheXpert-trained ResNet50 checkpoint (.pt)
# ✅ Frozen BiomedBERT checkpoint (.pt) used as "LLM descriptor encoder"
# ✅ NIH is target test domain (evaluate NIH only for now)
# ✅ PQC is NOT trained on source; PQC θ updates ONLY at test-time (TTDA)
#
# Proposal-aligned fixes included:
# (3) L_fid uses stored "source quantum statistics" proxy: q_src_mean
# (4) λ1, λ2 are dt-dependent (LLM-driven) via deterministic dt->λ mapping
# (5) Classifier is a lightweight MLP (fallback to fc if ckpt lacks MLP weights)
# (6) Dual-memory + contrastive + distillation losses implemented
# ============================================================

import os, math, copy, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# ----------------------------
# CONFIG
# ----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
PI = float(math.pi)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ✅ Your pretrained source models (ResNet MUST be CheXpert-only, as you said)
RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"   # you said it's CheXpert-only
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

# NIH paths (target domain)
NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"  # unlabeled stream

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

BATCH_SIZE  = 16 if DEVICE.type=="cuda" else 8
NUM_WORKERS = 0

# LLM prompt config
TEXT_MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
MAX_PROMPT_LEN = 192
INCLUDE_CNN_UNCERT_IN_PROMPT = True

# PQC/QNN config
N_QUBITS = 4
N_ENT_LAYERS = 2
SHOTS = 128   # (not always used by estimator backends)

# TTDA config
EVAL_PROTOCOL = "continual"   # "continual" or "ttda_once"
MAX_TEST_BATCHES = None
ADAPT_STEPS_PER_BATCH =0
TTDA_ONCE_STEPS = 30
LR_TTA = 5e-4
GRAD_CLIP = 1.0

# TTA config
USE_TTA = True
USE_AUTOAUG_TTA = True
TTA_VIEWS = 0  # 1=identity, 2=+flip, 3=+autoaug

# (3) Fidelity stats init
# "Store source-domain quantum statistics for fidelity alignment" (proposal)
# Here: we estimate a stable q_src_mean *once* using the frozen initial PQC on a few stream batches.
INIT_QSTATS_BATCHES = 20
QSTATS_SAVE_PATH = "q_src_stats.npz"  # optional caching

# (6) Memory system config
USE_MEMORY = True
MEM_SIZE = 1024
BETA_CONTRAST = 0.05
BETA_DISTILL  = 0.05
INIT_SOURCE_MEMORY_BATCHES = 30

print("DEVICE:", DEVICE)
for p in [RESNET_CKPT, BIOMED_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM]:
    print(("✅" if os.path.exists(p) else "❌"), p)

# ============================================================
# Utils: pip install (Kaggle-safe)
# ============================================================
def _pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# ============================================================
# NIH utilities + dataset
# ============================================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def build_nih_df(split_list_txt, nih_csv_path):
    wanted = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")
    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")  # NIH uses "Effusion"

    df["patient_id"] = df["Patient ID"].astype(str) if "Patient ID" in df.columns else "unknown"
    df["age"] = df["Patient Age"] if "Patient Age" in df.columns else np.nan
    df["gender"] = df["Patient Gender"] if "Patient Gender" in df.columns else "U"
    df["view"] = df["View Position"] if "View Position" in df.columns else "UNK"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","patient_id","age","gender","view"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(f"NIH image not found: {p}")
    return p

class CXRDatasetPIL(Dataset):
    def __init__(self, df, img_resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.images = self.df["image"].tolist()
        self.labeled = labeled
        if labeled:
            self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "patient_id": str(self.df.loc[idx, "patient_id"]),
            "age": self.df.loc[idx, "age"],
            "gender": str(self.df.loc[idx, "gender"]),
            "view": str(self.df.loc[idx, "view"]),
        }
        if self.labeled:
            y = torch.from_numpy(self.labels[idx])
        else:
            y = torch.zeros(len(UNIFIED_LABELS), dtype=torch.float32)
        return img, y, meta

def pil_collate_fn(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys), dim=0), list(metas)

nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDatasetPIL(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)
stream_loader = DataLoader(
    CXRDatasetPIL(nih_stream_df, nih_resolver, labeled=False),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)

print("NIH test:", len(nih_test_df), "| NIH unlabeled stream:", len(nih_stream_df))

# ============================================================
# Transforms
# ============================================================
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])
weak_tf = eval_tf

try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
except Exception:
    autoaug = None
    USE_AUTOAUG_TTA = False
    print("⚠ AutoAugment not available; disabling AutoAug TTA.")

autoaug_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    (autoaug if autoaug is not None else T.Lambda(lambda x: x)),
    T.ToTensor(),
    norm
])

def batch_apply_tf(pil_list, tf):
    xs = [tf(im) for im in pil_list]
    return torch.stack(xs, dim=0)

# ============================================================
# Load pretrained ResNet50 (frozen feature extractor)
# ============================================================
def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def load_resnet50_full(ckpt_path, n_labels):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(2048, n_labels)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:
        sd = ckpt["net"]
    else:
        sd = ckpt
    sd = _strip_prefix(sd)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("✅ Loaded ResNet checkpoint (strict=False)")
    print("  missing:", len(missing), "| unexpected:", len(unexpected))
    return m, sd

base_resnet, resnet_sd = load_resnet50_full(RESNET_CKPT, n_labels=len(UNIFIED_LABELS))
base_resnet = base_resnet.to(DEVICE).eval()
for p in base_resnet.parameters():
    p.requires_grad_(False)

feat_net = nn.Sequential(*(list(base_resnet.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def extract_features(x_tensor):
    return feat_net(x_tensor).flatten(1)

# ============================================================
# (5) Lightweight MLP classifier (proposal)
# - If checkpoint contains a trained MLP head, load it
# - Else fallback to the trained ResNet fc as a "1-layer MLP"
# ============================================================
class ClassifierMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=0):
        super().__init__()
        if hidden and hidden > 0:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.ReLU(inplace=True),
                nn.Linear(hidden, out_dim),
            )
        else:
            self.net = nn.Sequential(nn.Linear(in_dim, out_dim))

    def forward(self, x):
        return self.net(x)

def try_load_classifier_mlp_from_ckpt(sd, n_labels):
    # Try common key patterns for an MLP head (optional future training)
    # If not found, fallback to ResNet fc weights.
    mlp2 = ClassifierMLP(2048, n_labels, hidden=512)
    mlp1 = ClassifierMLP(2048, n_labels, hidden=0)

    # Try load full mlp2 keys
    cand = {}
    for k, v in sd.items():
        # examples:
        # classifier.net.0.weight, classifier.0.weight, mlp.0.weight, head.0.weight
        if k.startswith("classifier.") or k.startswith("mlp.") or k.startswith("head."):
            cand[k] = v

    # If no classifier keys, fallback to fc from base_resnet
    if len(cand) == 0:
        # load fc weights into 1-layer MLP
        mlp1.net[0].weight.data.copy_(base_resnet.fc.weight.data.cpu())
        mlp1.net[0].bias.data.copy_(base_resnet.fc.bias.data.cpu())
        print("✅ Classifier: fallback to ResNet fc (1-layer MLP) loaded from checkpoint.")
        return mlp1

    # Try best-effort load into mlp2 (will strict=False)
    cand2 = _strip_prefix(cand)
    missing2, unexpected2 = mlp2.load_state_dict(cand2, strict=False)
    if len(missing2) < 4:
        print("✅ Classifier: loaded a 2-layer MLP head from checkpoint keys (strict=False).")
        print("  missing:", len(missing2), "| unexpected:", len(unexpected2))
        return mlp2

    # else fallback to fc
    mlp1.net[0].weight.data.copy_(base_resnet.fc.weight.data.cpu())
    mlp1.net[0].bias.data.copy_(base_resnet.fc.bias.data.cpu())
    print("✅ Classifier: checkpoint had head-like keys but not usable; fallback to ResNet fc (1-layer MLP).")
    return mlp1

classifier = try_load_classifier_mlp_from_ckpt(resnet_sd, len(UNIFIED_LABELS)).to("cpu").eval()
for p in classifier.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def predict_cnn_only(pil_imgs):
    x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
    feats = extract_features(x).detach().cpu().float()
    logits = classifier(feats)
    return torch.sigmoid(logits).cpu().numpy()

# ============================================================
# Artifact features + prompts (LLM domain descriptor)
# ============================================================
def artifact_features(pil_img):
    g = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    mean = float(g.mean())
    std  = float(g.std())
    p1   = float(np.percentile(g, 1))
    p99  = float(np.percentile(g, 99))
    low_clip  = float((g < 0.02).mean())
    high_clip = float((g > 0.98).mean())
    dark_frac = float((g < 0.15).mean())
    bright_frac = float((g > 0.85).mean())
    lap = (
        -4*g
        + np.roll(g, 1, 0) + np.roll(g, -1, 0)
        + np.roll(g, 1, 1) + np.roll(g, -1, 1)
    )
    lap_var = float(lap.var())
    return {
        "mean": mean, "std": std, "p1": p1, "p99": p99,
        "low_clip": low_clip, "high_clip": high_clip,
        "dark_frac": dark_frac, "bright_frac": bright_frac,
        "lap_var": lap_var
    }

def base_probs_and_entropy_from_feats(feats_cpu_fp32):
    logits = classifier(feats_cpu_fp32)
    p = torch.sigmoid(logits).detach().cpu().numpy()
    eps = 1e-6
    pp = np.clip(p, eps, 1-eps)
    ent = -(pp*np.log(pp) + (1-pp)*np.log(1-pp)).mean(axis=1)
    return p, ent

def build_prompts(pil_imgs, metas, base_probs=None, base_ent=None):
    prompts = []
    for i, (im, m) in enumerate(zip(pil_imgs, metas)):
        af = artifact_features(im)
        age = m.get("age", "NA")
        gender = m.get("gender", "U")
        view = m.get("view", "UNK")

        uncert_str = ""
        if INCLUDE_CNN_UNCERT_IN_PROMPT and base_probs is not None and base_ent is not None:
            probs_i = base_probs[i]
            ent_i = base_ent[i]
            uncert_str = (
                f" Classifier_probs={np.round(probs_i,3).tolist()}."
                f" Mean_entropy={float(ent_i):.4f}."
            )

        prompt = (
            "You are a radiology domain-shift oracle. "
            "Describe acquisition differences and imaging artifacts in this CXR and summarize domain context. "
            f"Meta: view={view}, gender={gender}, age={age}. "
            f"Artifacts: mean={af['mean']:.3f}, std={af['std']:.3f}, p1={af['p1']:.3f}, p99={af['p99']:.3f}, "
            f"low_clip={af['low_clip']:.3f}, high_clip={af['high_clip']:.3f}, "
            f"dark_frac={af['dark_frac']:.3f}, bright_frac={af['bright_frac']:.3f}, "
            f"sharpness_lapvar={af['lap_var']:.5f}."
            + uncert_str
        )
        prompts.append(prompt)
    return prompts

# ============================================================
# Frozen BiomedBERT encoder (loads your .pt best-effort)
# ============================================================
try:
    import transformers
except Exception:
    _pip_install("transformers")
    import transformers

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
for p in text_encoder.parameters():
    p.requires_grad_(False)

def load_biomed_pt_into_encoder(pt_path):
    if not os.path.exists(pt_path):
        print("⚠ BiomedBERT .pt not found, using base HF weights only:", pt_path)
        return
    ckpt = torch.load(pt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:
        sd = ckpt["net"]
    else:
        sd = ckpt
    sd = _strip_prefix(sd)
    missing, unexpected = text_encoder.load_state_dict(sd, strict=False)
    print("✅ Loaded BiomedBERT .pt into encoder (strict=False)")
    print("  missing:", len(missing), "| unexpected:", len(unexpected))

load_biomed_pt_into_encoder(BIOMED_CKPT)
DT_DIM = int(text_encoder.config.hidden_size)

@torch.no_grad()
def llm_embed_dt(prompts):
    tok = tokenizer(
        prompts, padding=True, truncation=True,
        max_length=MAX_PROMPT_LEN, return_tensors="pt"
    ).to(DEVICE)
    out = text_encoder(**tok)
    dt = out.last_hidden_state[:, 0, :]
    dt = F.normalize(dt, dim=1)
    return dt

# ============================================================
# Parser: dt -> angles, depth, topology, alpha, lambdas (NO training)
# (4) λ1, λ2 are dt-dependent ("LLM outputs λ1, λ2" in proposal)
# ============================================================
def _fixed_proj(dt_cpu, out_dim, seed, scale=1.0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(dt_cpu.shape[1], out_dim, generator=g) / math.sqrt(dt_cpu.shape[1])
    return (dt_cpu @ W) * scale

def parse_dt(dt_cpu):
    # dt -> modulations
    dt_angles = torch.tanh(_fixed_proj(dt_cpu, N_QUBITS, seed=1001)) * PI
    depth_g   = torch.sigmoid(_fixed_proj(dt_cpu, N_ENT_LAYERS, seed=1002))
    topo      = torch.sigmoid(_fixed_proj(dt_cpu, 1, seed=1003))
    alpha     = torch.sigmoid(_fixed_proj(dt_cpu, 1, seed=1004)) * 0.15

    # (4) dt -> lambdas (bounded, stable)
    lam_raw = _fixed_proj(dt_cpu, 2, seed=1005)
    lam1 = 0.5 + 1.5 * torch.sigmoid(lam_raw[:, 0:1])   # [0.5, 2.0]
    lam2 = 0.05 + 0.45 * torch.sigmoid(lam_raw[:, 1:2]) # [0.05, 0.50]
    return dt_angles.float(), depth_g.float(), topo.float(), alpha.float(), lam1.float(), lam2.float()

@torch.no_grad()
def get_dt_from_batch(pil_imgs, metas, feats_cpu_for_uncert=None):
    base_probs, base_ent = (None, None)
    if INCLUDE_CNN_UNCERT_IN_PROMPT and feats_cpu_for_uncert is not None:
        base_probs, base_ent = base_probs_and_entropy_from_feats(feats_cpu_for_uncert)
    prompts = build_prompts(pil_imgs, metas, base_probs=base_probs, base_ent=base_ent)
    dt = llm_embed_dt(prompts).detach().cpu().float()
    return dt

# ============================================================
# Qiskit QNN
# ============================================================
try:
    import qiskit
    import qiskit_machine_learning
except Exception:
    _pip_install("qiskit")
    _pip_install("qiskit-machine-learning")
    import qiskit
    import qiskit_machine_learning

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.providers.basic_provider import BasicProvider
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

basic_backend = BasicProvider().get_backend("basic_simulator")

Q_IN_DIM = 2*N_QUBITS + N_QUBITS + N_ENT_LAYERS + 1  # feat_angles + dt_angles + depth_g + topo

def build_qnn():
    x = ParameterVector("x", Q_IN_DIM)

    n_var   = 2*N_QUBITS
    n_chain = N_ENT_LAYERS * max(1, (N_QUBITS-1))
    n_ring  = N_ENT_LAYERS * N_QUBITS
    theta = ParameterVector("θ", n_var + n_chain + n_ring)

    qc = QuantumCircuit(N_QUBITS)

    # feature encoding
    for i in range(N_QUBITS):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    dt_off = 2*N_QUBITS
    depth_off = dt_off + N_QUBITS
    topo_idx  = depth_off + N_ENT_LAYERS

    # dt-conditioned controlled-phase gates
    for i in range(N_QUBITS):
        qc.crz(x[dt_off + i], i, (i+1) % N_QUBITS)

    # variational block (trainable θ)
    t = 0
    for i in range(N_QUBITS):
        qc.rz(theta[t], i); t += 1
        qc.ry(theta[t], i); t += 1

    # entanglement layers with dt-gated depth/topology
    for l in range(N_ENT_LAYERS):
        g_depth = x[depth_off + l]
        g_topo  = x[topo_idx]
        g_chain = g_depth * (1 - g_topo)
        g_ring  = g_depth * (g_topo)

        for i in range(max(1, N_QUBITS-1)):
            qc.crz(g_chain * theta[t], i, i+1); t += 1

        for i in range(N_QUBITS):
            qc.crz(g_ring * theta[t], i, (i+1) % N_QUBITS); t += 1

    qc_t = transpile(qc, basic_backend, optimization_level=0)

    # Estimator backend (robust import)
    try:
        from qiskit.primitives import StatevectorEstimator as Est
    except Exception:
        try:
            from qiskit.primitives import Estimator as Est
        except Exception:
            from qiskit.primitives import BaseEstimator as Est

    observables = []
    for i in range(N_QUBITS):
        z = ["I"] * N_QUBITS
        z[i] = "Z"
        observables.append(SparsePauliOp.from_list([("".join(z), 1.0)]))

    estimator = Est()
    qnn = EstimatorQNN(
        circuit=qc_t,
        estimator=estimator,
        observables=observables,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=True
    )
    return qnn

def fixed_q_to_feat_matrix(seed=2026):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(N_QUBITS, 2048, generator=g) / math.sqrt(N_QUBITS)
    return W.float()

Q2F = fixed_q_to_feat_matrix()

# ============================================================
# Adapter: CNN feats + PQC delta -> adapted feats -> MLP classifier
# ============================================================
class QTTAAdapter(nn.Module):
    def __init__(self, qnn_module, classifier_module):
        super().__init__()
        self.qnn = qnn_module
        self.classifier = classifier_module

    def forward_all(self, feats_cpu, dt_cpu):
        """
        Returns:
          logits (B,C),
          q_out (B,N_QUBITS),
          f_adapt (B,2048),
          (lam1, lam2),
          alpha
        """
        # encode small feature chunk into angles
        feat_small = feats_cpu[:, : (2*N_QUBITS)].contiguous()
        feat_angles = torch.tanh(feat_small) * PI

        dt_angles, depth_g, topo, alpha, lam1, lam2 = parse_dt(dt_cpu)

        q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
        q_out = self.qnn(q_in).to(torch.float32)  # (B, N_QUBITS)

        # map quantum outputs back to feature space
        delta_f = (q_out @ Q2F) * 0.05
        f_adapt = feats_cpu + alpha * delta_f

        logits = self.classifier(f_adapt)
        return logits, q_out, f_adapt, (lam1, lam2), alpha

# ============================================================
# Losses
# ============================================================
def binary_entropy_from_logits(logits_cpu):
    p = torch.sigmoid(logits_cpu)
    eps = 1e-6
    ent = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps))
    return ent.mean()

def cosine_fidelity_loss(q_out, q_src_mean):
    """
    (3) Proposal: L_fid = 1 - F(ρsrc, ρadapt)
    We use cosine similarity between q_out and stored q_src_mean as a stable proxy.
    """
    q = F.normalize(q_out, dim=1)
    s = F.normalize(q_src_mean.unsqueeze(0).expand_as(q), dim=1)
    cos = (q * s).sum(dim=1)
    return (1.0 - cos).mean()

def bernoulli_kl(p, q):
    """
    KL(Bern(p) || Bern(q)) averaged over labels.
    p,q in (0,1), shape (B,C)
    """
    eps = 1e-6
    p = torch.clamp(p, eps, 1-eps)
    q = torch.clamp(q, eps, 1-eps)
    kl = p * torch.log(p/q) + (1-p) * torch.log((1-p)/(1-q))
    return kl.mean()

# ============================================================
# (6) Dual-memory system (proposal-aligned)
# - Source Memory: static (stores base CNN features/probs)
# - Target Memory: dynamic (stores adapted features/probs)
# - Losses:
#     L_contrastive ~ 1 - sim(f_target, f_source)
#     L_distill ~ KL(p_source || p_target)
# ============================================================
class FeatureQueue:
    def __init__(self, max_size, feat_dim, prob_dim):
        self.max_size = int(max_size)
        self.feat_dim = int(feat_dim)
        self.prob_dim = int(prob_dim)
        self.feats = torch.zeros((0, feat_dim), dtype=torch.float32)
        self.probs = torch.zeros((0, prob_dim), dtype=torch.float32)

    def __len__(self):
        return int(self.feats.shape[0])

    @torch.no_grad()
    def push(self, feats_b, probs_b):
        feats_b = feats_b.detach().cpu().float()
        probs_b = probs_b.detach().cpu().float()
        self.feats = torch.cat([self.feats, feats_b], dim=0)
        self.probs = torch.cat([self.probs, probs_b], dim=0)
        # keep last max_size
        if len(self) > self.max_size:
            self.feats = self.feats[-self.max_size:]
            self.probs = self.probs[-self.max_size:]

    @torch.no_grad()
    def sample(self, n):
        n = min(int(n), len(self))
        if n <= 0:
            return None, None
        idx = torch.randint(0, len(self), (n,))
        return self.feats[idx], self.probs[idx]

class DualMemory:
    def __init__(self, max_size, feat_dim, prob_dim):
        self.source = FeatureQueue(max_size, feat_dim, prob_dim)  # static after init
        self.target = FeatureQueue(max_size, feat_dim, prob_dim)  # updates online

# ============================================================
# Build adapter + init q_src_mean + init source memory
# ============================================================
def init_q_src_mean(adapter_obj):
    # Try load cached stats
    if os.path.exists(QSTATS_SAVE_PATH):
        try:
            z = np.load(QSTATS_SAVE_PATH)
            v = torch.from_numpy(z["q_src_mean"]).float()
            print(f"✅ Loaded cached q_src_mean from: {QSTATS_SAVE_PATH}")
            return v
        except Exception as e:
            print("⚠ Failed to load cached q_src_mean, will recompute. Reason:", repr(e))

    # Recompute using frozen initial PQC on a few stream batches
    adapter_obj.eval()
    qs = []
    it = iter(stream_loader)
    for _ in tqdm(range(INIT_QSTATS_BATCHES), desc="Init q_src_mean (frozen PQC)"):
        try:
            pil_imgs, _, metas = next(it)
        except StopIteration:
            it = iter(stream_loader)
            pil_imgs, _, metas = next(it)

        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        dt = get_dt_from_batch(pil_imgs, metas, feats_cpu_for_uncert=feats)
        with torch.no_grad():
            _, q_out, _, _, _ = adapter_obj.forward_all(feats, dt)
        qs.append(q_out.detach().cpu().float())

    q_all = torch.cat(qs, dim=0)
    q_mean = q_all.mean(dim=0)
    try:
        np.savez(QSTATS_SAVE_PATH, q_src_mean=q_mean.numpy())
        print(f"✅ Cached q_src_mean to: {QSTATS_SAVE_PATH}")
    except Exception as e:
        print("⚠ Could not save q_src_mean cache. Reason:", repr(e))
    return q_mean

@torch.no_grad()
def init_source_memory(memory_obj, adapter_obj):
    """
    Fill Source Memory with base CNN features/probs (static).
    This does NOT train anything; it just stores reference stats.
    """
    if not USE_MEMORY:
        return
    adapter_obj.eval()
    it = iter(stream_loader)
    filled = 0
    for _ in tqdm(range(INIT_SOURCE_MEMORY_BATCHES), desc="Init Source Memory (static)"):
        try:
            pil_imgs, _, metas = next(it)
        except StopIteration:
            it = iter(stream_loader)
            pil_imgs, _, metas = next(it)

        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats_src = extract_features(x).detach().cpu().float()
        logits_src = classifier(feats_src)
        probs_src  = torch.sigmoid(logits_src)

        memory_obj.source.push(feats_src, probs_src)
        filled += feats_src.shape[0]

    print(f"✅ Source Memory initialized with ~{filled} samples (static).")

# ============================================================
# TTDA Strategy: only QNN θ updates
# ============================================================
def print_trainable_params(title, modules: dict):
    print(f"\n==================== TRAINABLE PARAMS: {title} ====================")
    for name, mod in modules.items():
        total = 0
        train = 0
        sample_trainable = []
        for n, p in mod.named_parameters():
            total += p.numel()
            if p.requires_grad:
                train += p.numel()
                if len(sample_trainable) < 8:
                    sample_trainable.append((n, tuple(p.shape)))
        print(f"- {name}: trainable={train:,} / total={total:,}")
        print("  sample trainable:", sample_trainable if sample_trainable else "[]")

def grad_nonzero_report(mod, max_items=12):
    items = []
    for n, p in mod.named_parameters():
        if p.grad is None:
            continue
        gsum = float(p.grad.detach().abs().sum().cpu())
        if gsum > 0:
            items.append((n, gsum))
    items.sort(key=lambda x: -x[1])
    return items[:max_items], len(items)

class QTTA_Proposal_TTDA:
    def __init__(self, adapter_obj, q_src_mean, dual_memory=None, lr=5e-4, sanity_print_once=True):
        self.adapter = adapter_obj
        self.q_src_mean = q_src_mean.detach().cpu().float()
        self.mem = dual_memory

        # ONLY QNN parameters are updated
        self.opt = torch.optim.Adam(self.adapter.qnn.parameters(), lr=lr)

        self._printed = False
        self._sanity_print_once = sanity_print_once

    def step(self, pil_imgs, metas):
        self.adapter.train()

        # frozen CNN feats
        xw = batch_apply_tf(pil_imgs, weak_tf).to(DEVICE)
        feats_src = extract_features(xw).detach().cpu().float()

        # dt from frozen LLM encoder
        dt = get_dt_from_batch(pil_imgs, metas, feats_cpu_for_uncert=feats_src)

        # forward (gradient flows ONLY through QNN weights)
        logits_t, q_out, f_t, (lam1, lam2), _alpha = self.adapter.forward_all(feats_src, dt)

        # main TTDA losses
        L_ent = binary_entropy_from_logits(logits_t)
        L_fid = cosine_fidelity_loss(q_out, self.q_src_mean)

        lam1m = lam1.mean()
        lam2m = lam2.mean()
        L_main = lam1m * L_ent + lam2m * L_fid

        # (6) memory losses (proposal-aligned)
        L_contrast = torch.tensor(0.0)
        L_distill  = torch.tensor(0.0)

        if USE_MEMORY and self.mem is not None and len(self.mem.source) > 0:
            # Contrastive: encourage similarity between adapted features and source (invariant) features
            # Lcontrastive = (1 - sim(ftarget(xi), fsource(xi))) averaged
            sim = F.cosine_similarity(F.normalize(f_t, dim=1), F.normalize(feats_src, dim=1), dim=1)
            L_contrast = (1.0 - sim).mean()

            # Distillation: KL(p_source || p_target)
            with torch.no_grad():
                p_src = torch.sigmoid(classifier(feats_src))
            p_tgt = torch.sigmoid(logits_t)
            L_distill = bernoulli_kl(p_src, p_tgt)

        L_total = L_main + BETA_CONTRAST * L_contrast + BETA_DISTILL * L_distill

        self.opt.zero_grad(set_to_none=True)
        L_total.backward()

        # sanity gradient check (prints once)
        if self._sanity_print_once and (not self._printed):
            self._printed = True
            print("\n==================== GRADIENT SANITY CHECK (FIRST TTDA STEP) ====================")
            q_items, q_cnt = grad_nonzero_report(self.adapter.qnn)
            print(f"QNN: nonzero_grad_params={q_cnt}")
            print("Top QNN grad sums:", q_items)

            other_items, other_cnt = grad_nonzero_report(self.adapter)
            other_items = [(n, s) for (n, s) in other_items if not n.startswith("qnn.")]
            other_cnt = len(other_items)
            print(f"Adapter NON-QNN: nonzero_grad_params={other_cnt}")
            if other_cnt > 0:
                print("⚠ WARNING: found non-QNN grads (should be 0):", other_items)
            else:
                print("✅ OK: no non-QNN gradients.")
            print("✅ OK: CNN + BiomedBERT + classifier are frozen and never optimized.")

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.qnn.parameters(), GRAD_CLIP)
        self.opt.step()

        # update target memory after step (dynamic)
        if USE_MEMORY and self.mem is not None:
            with torch.no_grad():
                p_tgt = torch.sigmoid(logits_t.detach())
            self.mem.target.push(f_t.detach(), p_tgt.detach())

        return {
            "L_total": float(L_total.detach().cpu()),
            "L_main": float(L_main.detach().cpu()),
            "L_ent": float(L_ent.detach().cpu()),
            "L_fid": float(L_fid.detach().cpu()),
            "L_contrast": float(L_contrast.detach().cpu()) if torch.is_tensor(L_contrast) else 0.0,
            "L_distill": float(L_distill.detach().cpu()) if torch.is_tensor(L_distill) else 0.0,
            "lam1": float(lam1m.detach().cpu()),
            "lam2": float(lam2m.detach().cpu()),
        }

# ============================================================
# Prediction with adapter (+ optional TTA)
# ============================================================
@torch.no_grad()
def predict_with_adapter(adapter_obj, pil_imgs, metas):
    x_id = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
    feats_id = extract_features(x_id).detach().cpu().float()
    dt = get_dt_from_batch(pil_imgs, metas, feats_cpu_for_uncert=feats_id)

    views = []

    logits1, _, _, _, _ = adapter_obj.forward_all(feats_id, dt)
    views.append(torch.sigmoid(logits1).detach())

    if USE_TTA and TTA_VIEWS >= 2:
        pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
        x2 = batch_apply_tf(pil_flip, eval_tf).to(DEVICE)
        feats2 = extract_features(x2).detach().cpu().float()
        logits2, _, _, _, _ = adapter_obj.forward_all(feats2, dt)
        views.append(torch.sigmoid(logits2).detach())

    if USE_TTA and TTA_VIEWS >= 3 and USE_AUTOAUG_TTA:
        x3 = batch_apply_tf(pil_imgs, autoaug_tf).to(DEVICE)
        feats3 = extract_features(x3).detach().cpu().float()
        logits3, _, _, _, _ = adapter_obj.forward_all(feats3, dt)
        views.append(torch.sigmoid(logits3).detach())

    p = torch.stack(views, dim=0).mean(dim=0)
    return p.cpu().numpy()

# ============================================================
# Metrics
# ============================================================
def compute_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            auc, ap = np.nan, np.nan
        else:
            auc = roc_auc_score(yt, yp)
            ap  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1 = f1_score(yt.astype(int), yhat, zero_division=0)
        rows.append([lbl, auc, ap, f1])

    df_pc = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    out = {}
    out["per_class"] = df_pc
    out["macro_roc_auc"] = float(np.nanmean(df_pc["ROC_AUC"].values))
    out["macro_pr_auc"]  = float(np.nanmean(df_pc["PR_AUC"].values))
    out["macro_f1"]      = float(np.nanmean(df_pc["F1@0.5"].values))
    try:
        out["micro_roc_auc"] = float(roc_auc_score(Y.ravel(), P.ravel()))
        out["micro_pr_auc"]  = float(average_precision_score(Y.ravel(), P.ravel()))
    except Exception:
        out["micro_roc_auc"] = np.nan
        out["micro_pr_auc"]  = np.nan
    return out

# ============================================================
# Run evaluation
# ============================================================
def run_eval(strategy_name, make_strategy_fn):
    # Build QNN
    qnn = build_qnn()
    qnn_torch = TorchConnector(qnn).to("cpu")

    # Adapter lives on CPU (qnn runs on CPU); CNN/LLM remain frozen elsewhere
    local_adapter = QTTAAdapter(qnn_torch, classifier).to("cpu")

    # ensure only qnn is trainable
    for p in local_adapter.parameters():
        p.requires_grad_(False)
    for p in local_adapter.qnn.parameters():
        p.requires_grad_(True)

    # Dual memory
    dual_mem = DualMemory(MEM_SIZE, feat_dim=2048, prob_dim=len(UNIFIED_LABELS)) if USE_MEMORY else None
    if USE_MEMORY and dual_mem is not None:
        init_source_memory(dual_mem, local_adapter)

    # (3) init q_src_mean for fidelity (stored stats proxy)
    q_src_mean = init_q_src_mean(local_adapter)

    strategy = make_strategy_fn(local_adapter, q_src_mean, dual_mem) if make_strategy_fn is not None else None

    print_trainable_params(
        title=strategy_name,
        modules={
            "Frozen ResNet (base_resnet)": base_resnet,
            "Frozen BiomedBERT (text_encoder)": text_encoder,
            "Classifier MLP (classifier)": classifier,
            "Adapter (local_adapter)": local_adapter,
        }
    )

    # ttda_once: adapt on unlabeled stream then evaluate
    t_adapt = 0.0
    if EVAL_PROTOCOL == "ttda_once" and strategy is not None:
        t0 = time.time()
        it = iter(stream_loader)
        for _ in tqdm(range(TTDA_ONCE_STEPS), desc=f"TTDA-once adapt [{strategy_name}]"):
            try:
                pil_imgs, _, metas = next(it)
            except StopIteration:
                it = iter(stream_loader)
                pil_imgs, _, metas = next(it)
            _ = strategy.step(pil_imgs, metas)
        t_adapt = time.time() - t0

    # Evaluate
    t0 = time.time()
    Ps, Ys = [], []
    logs = []

    for bi, (pil_imgs, y, metas) in enumerate(tqdm(test_loader, desc=f"Eval [{strategy_name}]")):
        if MAX_TEST_BATCHES is not None and bi >= MAX_TEST_BATCHES:
            break

        if EVAL_PROTOCOL == "continual" and strategy is not None:
            for _ in range(ADAPT_STEPS_PER_BATCH):
                logs.append(strategy.step(pil_imgs, metas))

        if strategy_name == "CNN_only":
            p_np = predict_cnn_only(pil_imgs)
        else:
            p_np = predict_with_adapter(local_adapter, pil_imgs, metas)

        Ps.append(p_np)
        Ys.append(y.numpy())

    t_eval = time.time() - t0

    P = np.vstack(Ps)
    Y = np.vstack(Ys)
    metrics = compute_metrics(Y, P, UNIFIED_LABELS)

    if len(logs):
        df_log = pd.DataFrame(logs)
        mean_L_total = float(df_log["L_total"].mean())
        mean_L_main  = float(df_log["L_main"].mean())
        mean_L_ent   = float(df_log["L_ent"].mean())
        mean_L_fid   = float(df_log["L_fid"].mean())
        mean_L_con   = float(df_log["L_contrast"].mean())
        mean_L_dis   = float(df_log["L_distill"].mean())
        mean_lam1    = float(df_log["lam1"].mean())
        mean_lam2    = float(df_log["lam2"].mean())
    else:
        mean_L_total = mean_L_main = mean_L_ent = mean_L_fid = mean_L_con = mean_L_dis = mean_lam1 = mean_lam2 = np.nan

    row = {
        "Strategy": strategy_name,
        "Protocol": EVAL_PROTOCOL,
        "Macro_ROC_AUC": metrics["macro_roc_auc"],
        "Macro_PR_AUC": metrics["macro_pr_auc"],
        "Macro_F1@0.5": metrics["macro_f1"],
        "Micro_ROC_AUC": metrics["micro_roc_auc"],
        "Micro_PR_AUC": metrics["micro_pr_auc"],
        "Adapt_seconds": t_adapt,
        "Eval_seconds": t_eval,
        "Mean_L_total": mean_L_total,
        "Mean_L_main": mean_L_main,
        "Mean_L_ent": mean_L_ent,
        "Mean_L_fid": mean_L_fid,
        "Mean_L_contrast": mean_L_con,
        "Mean_L_distill": mean_L_dis,
        "Mean_lam1": mean_lam1,
        "Mean_lam2": mean_lam2,
    }
    return row, metrics["per_class"]

# ============================================================
# Strategies
# ============================================================
def make_qtta_strategy(adapter_obj, q_src_mean, dual_mem):
    return QTTA_Proposal_TTDA(
        adapter_obj=adapter_obj,
        q_src_mean=q_src_mean,
        dual_memory=dual_mem,
        lr=LR_TTA,
        sanity_print_once=True
    )

strategies = [
    ("CNN_only", None),
    ("QTTA_no_adapt", None),
    ("QTTA_proposal", make_qtta_strategy),
]

summary_rows = []
perclass_dfs = {}

for name, make_strat in strategies:
    row, df_pc = run_eval(name, make_strat)
    summary_rows.append(row)
    perclass_dfs[name] = df_pc.copy()

df_summary = pd.DataFrame(summary_rows).sort_values(["Protocol","Strategy"]).reset_index(drop=True)

print("\n==================== SUMMARY ====================")
print(df_summary)

print("\n==================== PER-CLASS (ALL STRATEGIES) ====================")
for k, dfpc in perclass_dfs.items():
    print(f"\n--- {k} ---")
    print(dfpc)

out_csv = f"QTTA_TTDA_{EVAL_PROTOCOL}.csv"
df_summary.to_csv(out_csv, index=False)
print("\n✅ Saved:", out_csv)

best_idx = df_summary["Macro_ROC_AUC"].astype(float).idxmax()
best_name = df_summary.loc[best_idx, "Strategy"]
df_best_pc = perclass_dfs[best_name].copy()

pc_csv = f"per_class_best_{best_name}_{EVAL_PROTOCOL}.csv"
df_best_pc.to_csv(pc_csv, index=False)

print(f"\nBest strategy: {best_name} | Macro ROC-AUC={df_summary.loc[best_idx,'Macro_ROC_AUC']:.4f}")
print(df_best_pc)
print("✅ Saved:", pc_csv)


DEVICE: cuda
✅ /kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt
✅ /kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt
NIH test: 25596 | NIH unlabeled stream: 86524
✅ Loaded ResNet checkpoint (strict=False)
  missing: 0 | unexpected: 0
✅ Classifier: fallback to ResNet fc (1-layer MLP) loaded from checkpoint.


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2026-02-05 12:43:17.544486: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770295397.741319      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770295397.794012      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770295398.237162      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770295398.237190      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770295398.237192      55 computation_placer.cc:177] computation placer alr

...

✅ Source Memory initialized with ~480 samples (static).
✅ Loaded cached q_src_mean from: q_src_stats.npz

==================== TRAINABLE PARAMS: QTTA_no_adapt ====================
- Frozen ResNet (base_resnet): trainable=0 / total=23,518,277
  sample trainable: []
- Frozen BiomedBERT (text_encoder): trainable=0 / total=109,482,240
  sample trainable: []
- Classifier MLP (classifier): trainable=0 / total=10,245
  sample trainable: []
- Adapter (local_adapter): trainable=22 / total=10,267
  sample trainable: [('qnn.weight', (22,))]


Init Source Memory (static): 100%|██████████| 30/30 [00:07<00:00,  4.28it/s]


✅ Source Memory initialized with ~480 samples (static).
✅ Loaded cached q_src_mean from: q_src_stats.npz

==================== TRAINABLE PARAMS: QTTA_proposal ====================
- Frozen ResNet (base_resnet): trainable=0 / total=23,518,277
  sample trainable: []
- Frozen BiomedBERT (text_encoder): trainable=0 / total=109,482,240
  sample trainable: []
- Classifier MLP (classifier): trainable=0 / total=10,245
  sample trainable: []
- Adapter (local_adapter): trainable=22 / total=10,267
  sample trainable: [('qnn.weight', (22,))]


Eval [QTTA_proposal]: 100%|██████████| 1600/1600 [12:53<00:00,  2.07it/s]


==================== SUMMARY ====================
        Strategy   Protocol  Macro_ROC_AUC  Macro_PR_AUC  Macro_F1@0.5  \
0       CNN_only  continual       0.776187      0.253673      0.291060   
1  QTTA_no_adapt  continual       0.776195      0.253663      0.291051   
2  QTTA_proposal  continual       0.776197      0.253664      0.291053   

   Micro_ROC_AUC  Micro_PR_AUC  Adapt_seconds  Eval_seconds  Mean_L_total  \
0       0.765719      0.269626            0.0    322.033750           NaN   
1       0.765725      0.269605            0.0    759.836595           NaN   
2       0.765698      0.269549            0.0    773.665680           NaN   

   Mean_L_main  Mean_L_ent  Mean_L_fid  Mean_L_contrast  Mean_L_distill  \
0          NaN         NaN         NaN              NaN             NaN   
1          NaN         NaN         NaN              NaN             NaN   
2          NaN         NaN         NaN              NaN             NaN   

   Mean_lam1  Mean_lam2  
0        NaN    

### Cell 3. Overall Purpose

1. This code implements **Quantum Test-Time Domain Adaptation (QTTA)** for chest X-ray classification.
2. The source model was trained using the CheXpert dataset.
3. The target domain is the NIH Chest X-ray dataset.
4. NIH labels are used only after prediction for evaluation.
5. Target labels are never used during adaptation.
6. The goal is to reduce CheXpert-to-NIH domain shift.
7. It compares classical, quantum, and quantum-adaptation strategies.
8. It also compares six-qubit and eight-qubit quantum circuits.

### 2. Input Data

9. The NIH CSV file provides image names, findings, and patient metadata.
10. Five diseases are evaluated: Atelectasis, Cardiomegaly, Consolidation, Edema, and Pleural Effusion.
11. NIH “Effusion” is renamed “Pleural Effusion” for label consistency.
12. Patient age, gender, and view position are also loaded.
13. The NIH test list supplies images for final evaluation.
14. The NIH train-validation list acts as an unlabeled target stream.
15. Images are loaded as RGB PIL images.
16. A custom collate function preserves images and metadata in each batch.

### 3. Image Preprocessing

17. Images are resized and center-cropped to (224\times224).
18. ImageNet mean and standard deviation are used for normalization.
19. Identity images are used as the standard prediction view.
20. Horizontally flipped images provide a second TTA view.
21. AutoAugment can provide an optional third prediction view.
22. Predictions from available views are averaged.

### 4. Frozen CNN Backbone

23. A CheXpert-trained ResNet50 checkpoint is loaded.
24. ResNet50 converts each X-ray into a 2,048-dimensional feature vector.
25. Every ResNet50 parameter is frozen.
26. Therefore, the CNN is never retrained on NIH.
27. A frozen classifier converts features into five disease logits.
28. The original ResNet fully connected layer is used when no separate MLP exists.

### 5. Domain-Descriptor Generation

29. The code calculates brightness, contrast, clipping, darkness, and sharpness statistics.
30. These statistics describe possible acquisition and scanner differences.
31. Patient metadata and CNN uncertainty are added to a text prompt.
32. The prompt asks for a radiology domain-shift description.
33. Frozen BiomedBERT encodes the prompt into a domain descriptor (d_t).
34. BiomedBERT parameters remain frozen throughout testing.
35. Domain descriptors are cached using each image ID.
36. Caching prevents repeated BiomedBERT computation across strategies.

### 6. Domain-Conditioned Quantum Controls

37. The domain descriptor is deterministically projected into quantum controls.
38. It produces domain-dependent quantum rotation angles.
39. It controls the effective depth of entanglement layers.
40. It controls whether chain or ring entanglement is emphasized.
41. It produces an adaptation strength called alpha.
42. It also produces adaptive loss weights lambda-one and lambda-two.
43. These projections are fixed and are not trained.
44. Thus, BiomedBERT context dynamically configures the quantum circuit.

### 7. Quantum Neural Network

45. Separate quantum circuits are evaluated with six and eight qubits.
46. CNN feature values are converted into (RY) and (RZ) rotation angles.
47. Domain angles are inserted through controlled-(RZ) gates.
48. Trainable (RZ) and (RY) gates form the variational block.
49. Controlled rotations create chain and ring entanglement.
50. Entanglement parameters are shared within each layer for speed.
51. Only the expectation of (Z) on qubit zero is measured.
52. Using one observable greatly reduces quantum-simulation cost.
53. Input gradients are disabled because CNN and text inputs are frozen.
54. Only the quantum circuit’s theta parameters receive gradient updates.

### 8. Quantum Feature Adaptation

55. The quantum measurement is projected back into 2,048-dimensional feature space.
56. This projected signal becomes a small quantum feature correction.
57. Alpha controls how strongly the correction changes CNN features.
58. Adapted features are passed through the frozen disease classifier.
59. Therefore, the quantum circuit modifies features without retraining ResNet50.
60. The unchanged circuit is evaluated as `QTTA_no_adapt`.

### 9. Source Statistics and Memory

61. A source-memory queue stores reference CNN features and probabilities.
62. Approximately 30 batches are used to initialize this memory.
63. A mean quantum output called `q_src_mean` is also calculated.
64. The quantum mean is cached separately for each qubit configuration.
65. Fidelity loss keeps adapted quantum outputs close to this reference.
66. Target memory stores features and predictions observed during testing.

### 10. Test-Time Adaptation Strategies

67. `CNN_only` evaluates the frozen classical model.
68. `QTTA_no_adapt` adds the quantum feature correction without updating theta.
69. `PQC_TENT` updates theta by minimizing prediction entropy.
70. `PQC_CoTTA` uses teacher-student consistency and stochastic restoration.
71. `PQC_EATA` adapts only using sufficiently reliable low-entropy samples.
72. `QTTA_proposal` combines entropy, fidelity, contrastive, and distillation losses.
73. Adaptation occurs once every four test batches.
74. The number of updates is capped at 800 for runtime control.
75. No ground-truth target label is passed into an adaptation loss.
76. Each strategy starts from the same initial quantum weights for fairness.

### 11. Evaluation and Outputs

77. The code reports per-class ROC-AUC, PR-AUC, and F1 scores.
78. It also calculates macro and micro performance across all diseases.
79. ROC curves, precision-recall curves, loss plots, logs, and CSV files are saved.
80. Overall, this is a label-free, LLM-conditioned, multi-qubit QTTA ablation pipeline designed to test whether updating only a small quantum adapter improves NIH generalization efficiently. 


The previous code tests only one QTTA method: a frozen CNN plus a quantum adapter trained with entropy and fidelity losses.

The new code is a full ablation framework. It adds classical methods such as AdaBN, TENT, CoTTA-lite, and Wasserstein alignment, along with several PQC variants. 

Its biggest improvements are:

* It uses LLM/CNN teacher probabilities for distillation.
* It replaces the fixed random quantum output mapping with a trainable zero-initialized head.
* It adds PQC warm-up before full adaptation.
* It adapts on an unlabeled stream before evaluating the test set.
* It produces a ranked ablation table comparing all methods. 

**Conclusion:** The previous code is a simple label-free QTTA prototype. The new code is more stable, comprehensive, and suitable for paper ablation experiments.

Two issues still need fixing: the teacher CSV must match the adaptation-stream images, and the “no-adapt” PQC baseline should not receive warm-up training.


In [ ]:
# ============================================================
# QTTA / TTDA (PROPOSAL-ALIGNED) — FIXED + FAST ENOUGH TO RUN
#
# ✅ Frozen CheXpert ResNet50 checkpoint (.pt)
# ✅ Frozen BiomedBERT (.pt) used as "LLM descriptor encoder" (dt)
# ✅ NIH is target TEST domain (metrics from NIH_TEST labels only)
# ✅ TTDA updates ONLY PQC θ, and ONLY during TEST loop (no labels used)
# ✅ TTA (flip/autoaug) used during TEST prediction
#
# 🔥 BIG FIXES (your 12-hour issue):
#   1) EstimatorQNN: input_gradients=False  (you don't need grads wrt inputs)
#   2) Use ONLY 1 observable (Z on qubit-0)  (was N_qubits observables)
#   3) Share entangling parameters per-layer  (was per-edge => huge #weights)
#   4) Cache dt per image-id  (no repeated BERT per strategy/qubit)
#   5) Avoid recomputing feats/dt twice per batch (reuse inside loop)
#   6) Optional: adapt every K batches + cap total adapt steps
#
# NOTE:
#   ADAPT_STEPS_PER_TEST_BATCH > 0  => TTDA ACTUALLY RUNS (updates PQC θ)
# ============================================================

import os, math, copy, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    roc_curve, precision_recall_curve
)

import matplotlib.pyplot as plt


# ----------------------------
# CONFIG
# ----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
PI = float(math.pi)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# Pretrained checkpoints (FROZEN)
RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

# NIH paths
NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"  # unlabeled (target stream)

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

BATCH_SIZE  = 16 if DEVICE.type=="cuda" else 8
NUM_WORKERS = 0

# LLM descriptor encoder (BiomedBERT)
TEXT_MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
MAX_PROMPT_LEN = 192
INCLUDE_CNN_UNCERT_IN_PROMPT = True

# Multi-qubit sweep
QUBIT_LIST = [6,8]

# QNN depth
N_ENT_LAYERS = 2

# TTDA happens DURING TEST (online)
ADAPT_STEPS_PER_TEST_BATCH = 1   # ✅ set >0 to actually run TTDA updates
ADAPT_EVERY_K_BATCHES = 4        # adapt only every K test batches (still "during test")
MAX_TOTAL_ADAPT_STEPS = 800      # cap total TTDA steps (prevents crazy runtimes)
LR_TTA = 5e-4
GRAD_CLIP = 1.0

# TTA during TEST prediction
USE_TTA = True
USE_AUTOAUG_TTA = True
TTA_VIEWS = 2  # 1=identity, 2=+flip, 3=+autoaug

# Proposal: q_src_mean cache + memory
INIT_QSTATS_BATCHES = 20
USE_MEMORY = True
MEM_SIZE = 1024
BETA_CONTRAST = 0.05
BETA_DISTILL  = 0.05
INIT_SOURCE_MEMORY_BATCHES = 30

# Reliability threshold for EATA-lite
EATA_ENT_THRESH = 0.35

# Speed safety (recommended in Kaggle)
MAX_TEST_BATCHES = None  # set e.g. 300 for quick debug; None = full test

# QNN speed knobs (CRITICAL)
QNN_NUM_OBSERVABLES = 1       # ✅ 1 observable instead of n_qubits observables
QNN_INPUT_GRADS = False       # ✅ do NOT compute gradients wrt inputs
QNN_SHARED_ENT_PARAMS = True  # ✅ share entangling params per-layer (huge speedup)

# dt caching (CRITICAL)
DT_CACHE_MAX = 60000  # cache dt per image id (keeps BERT from repeating)

# Output
OUTDIR = "QTTA_MULTI_QUBIT_RESULTS"
os.makedirs(OUTDIR, exist_ok=True)
QSTATS_DIR = os.path.join(OUTDIR, "qstats_cache")
os.makedirs(QSTATS_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
for p in [RESNET_CKPT, BIOMED_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM]:
    print(("✅" if os.path.exists(p) else "❌"), p)


# ============================================================
# Utils: pip install (Kaggle-safe)
# ============================================================
def _pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# ============================================================
# NIH dataset
# ============================================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def build_nih_df(split_list_txt, nih_csv_path):
    wanted = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")
    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")  # NIH uses "Effusion"

    df["patient_id"] = df["Patient ID"].astype(str) if "Patient ID" in df.columns else "unknown"
    df["age"] = df["Patient Age"] if "Patient Age" in df.columns else np.nan
    df["gender"] = df["Patient Gender"] if "Patient Gender" in df.columns else "U"
    df["view"] = df["View Position"] if "View Position" in df.columns else "UNK"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","patient_id","age","gender","view"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(f"NIH image not found: {p}")
    return p

class CXRDatasetPIL(Dataset):
    def __init__(self, df, img_resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.images = self.df["image"].tolist()
        self.labeled = labeled
        if labeled:
            self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "patient_id": str(self.df.loc[idx, "patient_id"]),
            "age": self.df.loc[idx, "age"],
            "gender": str(self.df.loc[idx, "gender"]),
            "view": str(self.df.loc[idx, "view"]),
        }
        if self.labeled:
            y = torch.from_numpy(self.labels[idx])
        else:
            y = torch.zeros(len(UNIFIED_LABELS), dtype=torch.float32)
        return img, y, meta

def pil_collate_fn(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys), dim=0), list(metas)

nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDatasetPIL(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)
stream_loader = DataLoader(
    CXRDatasetPIL(nih_stream_df, nih_resolver, labeled=False),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)

print("NIH test:", len(nih_test_df), "| NIH unlabeled stream:", len(nih_stream_df))


# ============================================================
# Transforms
# ============================================================
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
except Exception:
    autoaug = None
    USE_AUTOAUG_TTA = False
    print("⚠ AutoAugment not available; disabling AutoAug TTA.")

autoaug_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    (autoaug if autoaug is not None else T.Lambda(lambda x: x)),
    T.ToTensor(),
    norm
])

strong_tf = autoaug_tf if (autoaug is not None) else eval_tf

def batch_apply_tf(pil_list, tf):
    xs = [tf(im) for im in pil_list]
    return torch.stack(xs, dim=0)


# ============================================================
# Load pretrained ResNet50 (frozen feature extractor)
# ============================================================
def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def load_resnet50_full(ckpt_path, n_labels):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(2048, n_labels)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:
        sd = ckpt["net"]
    else:
        sd = ckpt
    sd = _strip_prefix(sd)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("✅ Loaded ResNet checkpoint (strict=False)")
    print("  missing:", len(missing), "| unexpected:", len(unexpected))
    return m, sd

base_resnet, resnet_sd = load_resnet50_full(RESNET_CKPT, n_labels=len(UNIFIED_LABELS))
base_resnet = base_resnet.to(DEVICE).eval()
for p in base_resnet.parameters():
    p.requires_grad_(False)

feat_net = nn.Sequential(*(list(base_resnet.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def extract_features(x_tensor):
    return feat_net(x_tensor).flatten(1)

# ============================================================
# Frozen classifier head (best-effort from ckpt)
# ============================================================
class ClassifierMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=0):
        super().__init__()
        if hidden and hidden > 0:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.ReLU(inplace=True),
                nn.Linear(hidden, out_dim),
            )
        else:
            self.net = nn.Sequential(nn.Linear(in_dim, out_dim))

    def forward(self, x):
        return self.net(x)

def try_load_classifier_mlp_from_ckpt(sd, n_labels):
    mlp2 = ClassifierMLP(2048, n_labels, hidden=512)
    mlp1 = ClassifierMLP(2048, n_labels, hidden=0)

    cand = {}
    for k, v in sd.items():
        if k.startswith("classifier.") or k.startswith("mlp.") or k.startswith("head."):
            cand[k] = v

    if len(cand) == 0:
        mlp1.net[0].weight.data.copy_(base_resnet.fc.weight.data.cpu())
        mlp1.net[0].bias.data.copy_(base_resnet.fc.bias.data.cpu())
        print("✅ Classifier: fallback to ResNet fc (1-layer).")
        return mlp1

    cand2 = _strip_prefix(cand)
    missing2, unexpected2 = mlp2.load_state_dict(cand2, strict=False)
    if len(missing2) < 4:
        print("✅ Classifier: loaded 2-layer head from ckpt keys (strict=False).")
        print("  missing:", len(missing2), "| unexpected:", len(unexpected2))
        return mlp2

    mlp1.net[0].weight.data.copy_(base_resnet.fc.weight.data.cpu())
    mlp1.net[0].bias.data.copy_(base_resnet.fc.bias.data.cpu())
    print("✅ Classifier: fallback to ResNet fc (1-layer).")
    return mlp1

classifier = try_load_classifier_mlp_from_ckpt(resnet_sd, len(UNIFIED_LABELS)).to("cpu").eval()
for p in classifier.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def cnn_probs_from_feats(feats_cpu_fp32):
    logits = classifier(feats_cpu_fp32)
    return torch.sigmoid(logits)

@torch.no_grad()
def base_probs_and_entropy_from_feats(feats_cpu_fp32):
    p = cnn_probs_from_feats(feats_cpu_fp32).cpu().numpy()
    eps = 1e-6
    pp = np.clip(p, eps, 1-eps)
    ent = -(pp*np.log(pp) + (1-pp)*np.log(1-pp)).mean(axis=1)
    return p, ent


# ============================================================
# Artifact features + prompt builder + dt cache
# ============================================================
def artifact_features(pil_img):
    g = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    mean = float(g.mean())
    std  = float(g.std())
    p1   = float(np.percentile(g, 1))
    p99  = float(np.percentile(g, 99))
    low_clip  = float((g < 0.02).mean())
    high_clip = float((g > 0.98).mean())
    dark_frac = float((g < 0.15).mean())
    bright_frac = float((g > 0.85).mean())
    lap = (
        -4*g
        + np.roll(g, 1, 0) + np.roll(g, -1, 0)
        + np.roll(g, 1, 1) + np.roll(g, -1, 1)
    )
    lap_var = float(lap.var())
    return {
        "mean": mean, "std": std, "p1": p1, "p99": p99,
        "low_clip": low_clip, "high_clip": high_clip,
        "dark_frac": dark_frac, "bright_frac": bright_frac,
        "lap_var": lap_var
    }

class LRUCache:
    def __init__(self, max_items=50000):
        self.max_items = int(max_items)
        self.d = OrderedDict()

    def get(self, k):
        if k not in self.d:
            return None
        v = self.d.pop(k)
        self.d[k] = v
        return v

    def put(self, k, v):
        if k in self.d:
            self.d.pop(k)
        self.d[k] = v
        if len(self.d) > self.max_items:
            self.d.popitem(last=False)

DT_CACHE = LRUCache(DT_CACHE_MAX)
AF_CACHE = LRUCache(DT_CACHE_MAX)

def build_prompt_single(meta, af, probs_i=None, ent_i=None):
    age = meta.get("age", "NA")
    gender = meta.get("gender", "U")
    view = meta.get("view", "UNK")
    uncert_str = ""
    if INCLUDE_CNN_UNCERT_IN_PROMPT and (probs_i is not None) and (ent_i is not None):
        uncert_str = (
            f" Classifier_probs={np.round(probs_i,3).tolist()}."
            f" Mean_entropy={float(ent_i):.4f}."
        )
    prompt = (
        "You are a radiology domain-shift oracle. "
        "Describe acquisition differences and imaging artifacts in this CXR and summarize domain context. "
        f"Meta: view={view}, gender={gender}, age={age}. "
        f"Artifacts: mean={af['mean']:.3f}, std={af['std']:.3f}, p1={af['p1']:.3f}, p99={af['p99']:.3f}, "
        f"low_clip={af['low_clip']:.3f}, high_clip={af['high_clip']:.3f}, "
        f"dark_frac={af['dark_frac']:.3f}, bright_frac={af['bright_frac']:.3f}, "
        f"sharpness_lapvar={af['lap_var']:.5f}."
        + uncert_str
    )
    return prompt


# ============================================================
# Frozen BiomedBERT encoder (loads .pt best-effort)
# ============================================================
try:
    import transformers
except Exception:
    _pip_install("transformers")
    import transformers
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
for p in text_encoder.parameters():
    p.requires_grad_(False)

def load_biomed_pt_into_encoder(pt_path):
    if not os.path.exists(pt_path):
        print("⚠ BiomedBERT .pt not found; using HF weights only:", pt_path)
        return
    ckpt = torch.load(pt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:
        sd = ckpt["net"]
    else:
        sd = ckpt
    sd = _strip_prefix(sd)
    missing, unexpected = text_encoder.load_state_dict(sd, strict=False)
    print("✅ Loaded BiomedBERT .pt into encoder (strict=False)")
    print("  missing:", len(missing), "| unexpected:", len(unexpected))

load_biomed_pt_into_encoder(BIOMED_CKPT)
DT_DIM = int(text_encoder.config.hidden_size)

@torch.no_grad()
def llm_embed_dt(prompts):
    tok = tokenizer(
        prompts, padding=True, truncation=True,
        max_length=MAX_PROMPT_LEN, return_tensors="pt"
    ).to(DEVICE)
    out = text_encoder(**tok)
    dt = out.last_hidden_state[:, 0, :]
    dt = F.normalize(dt, dim=1)
    return dt.detach().cpu().float()

@torch.no_grad()
def get_dt_cached(pil_imgs, metas, feats_eval_cpu):
    """
    Returns dt tensor [B, DT_DIM] using LRU cache per image-id.
    dt is computed ONCE per image-id using:
      - artifact_features(pil)
      - meta fields
      - cnn probs/entropy from feats_eval_cpu (eval_tf)
    """
    img_ids = [m["image"] for m in metas]
    dt_list = []
    missing_idx = []
    for i, img_id in enumerate(img_ids):
        v = DT_CACHE.get(img_id)
        if v is None:
            missing_idx.append(i)
            dt_list.append(None)
        else:
            dt_list.append(v)

    if len(missing_idx) > 0:
        base_probs, base_ent = (None, None)
        if INCLUDE_CNN_UNCERT_IN_PROMPT:
            base_probs, base_ent = base_probs_and_entropy_from_feats(feats_eval_cpu)

        prompts = []
        miss_ids = []
        for i in missing_idx:
            img_id = img_ids[i]
            af = AF_CACHE.get(img_id)
            if af is None:
                af = artifact_features(pil_imgs[i])
                AF_CACHE.put(img_id, af)
            probs_i = base_probs[i] if base_probs is not None else None
            ent_i   = base_ent[i]   if base_ent is not None else None
            prompts.append(build_prompt_single(metas[i], af, probs_i=probs_i, ent_i=ent_i))
            miss_ids.append(img_id)

        dt_new = llm_embed_dt(prompts)  # [M, DT_DIM]
        for k, img_id in enumerate(miss_ids):
            DT_CACHE.put(img_id, dt_new[k:k+1])  # store as [1,DT_DIM]

        # fill into dt_list
        ptr = 0
        for i in missing_idx:
            dt_list[i] = DT_CACHE.get(img_ids[i])
            ptr += 1

    dt = torch.cat([x for x in dt_list], dim=0)  # [B, DT_DIM]
    return dt


# ============================================================
# dt -> PQC controls (angles, depth/topology gates, alpha, lambdas)
# deterministic mapping (NO TRAINING)
# ============================================================
def _fixed_proj(dt_cpu, out_dim, seed, scale=1.0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(dt_cpu.shape[1], out_dim, generator=g) / math.sqrt(dt_cpu.shape[1])
    return (dt_cpu @ W) * scale

def parse_dt(dt_cpu, n_qubits, n_ent_layers):
    dt_angles = torch.tanh(_fixed_proj(dt_cpu, n_qubits, seed=1001 + n_qubits)) * PI
    depth_g   = torch.sigmoid(_fixed_proj(dt_cpu, n_ent_layers, seed=1002 + n_qubits))
    topo      = torch.sigmoid(_fixed_proj(dt_cpu, 1, seed=1003 + n_qubits))
    alpha     = torch.sigmoid(_fixed_proj(dt_cpu, 1, seed=1004 + n_qubits)) * 0.15

    lam_raw = _fixed_proj(dt_cpu, 2, seed=1005 + n_qubits)
    lam1 = 0.5 + 1.5 * torch.sigmoid(lam_raw[:, 0:1])   # [0.5, 2.0]
    lam2 = 0.05 + 0.45 * torch.sigmoid(lam_raw[:, 1:2]) # [0.05, 0.50]
    return dt_angles.float(), depth_g.float(), topo.float(), alpha.float(), lam1.float(), lam2.float()


# ============================================================
# Qiskit QNN builder (FAST version)
# ============================================================
try:
    import qiskit
    import qiskit_machine_learning
except Exception:
    _pip_install("qiskit")
    _pip_install("qiskit-machine-learning")
    import qiskit
    import qiskit_machine_learning

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

def _make_estimator():
    # Works across qiskit versions (best-effort)
    try:
        from qiskit.primitives import StatevectorEstimator
        return StatevectorEstimator()
    except Exception:
        try:
            from qiskit.primitives import Estimator
            return Estimator()
        except Exception:
            return None

ESTIMATOR = _make_estimator()

def build_qnn(n_qubits, n_ent_layers):
    """
    FAST QNN:
      - 1 observable only: Z on qubit-0 (QNN_NUM_OBSERVABLES=1)
      - input_gradients=False (QNN_INPUT_GRADS=False)
      - shared entangling params per layer (QNN_SHARED_ENT_PARAMS=True)
    """
    # Inputs: [feat(2*nq), dt_angles(nq), depth_g(nL), topo(1)]
    q_in_dim = 2*n_qubits + n_qubits + n_ent_layers + 1
    x = ParameterVector("x", q_in_dim)

    # Weights:
    #   - per qubit: RZ, RY => 2*nq
    #   - per layer (shared): chain_theta[l], ring_theta[l] => 2*nL
    n_var = 2*n_qubits
    n_ent = 2*n_ent_layers if QNN_SHARED_ENT_PARAMS else (n_ent_layers*(n_qubits-1) + n_ent_layers*n_qubits)
    theta = ParameterVector("θ", n_var + n_ent)

    qc = QuantumCircuit(n_qubits)

    # feature encoding
    for i in range(n_qubits):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    dt_off    = 2*n_qubits
    depth_off = dt_off + n_qubits
    topo_idx  = depth_off + n_ent_layers

    # dt-conditioned controlled-phase
    for i in range(n_qubits):
        qc.crz(x[dt_off + i], i, (i+1) % n_qubits)

    # variational single-qubit block
    t = 0
    for i in range(n_qubits):
        qc.rz(theta[t], i); t += 1
        qc.ry(theta[t], i); t += 1

    # entangling layers gated by depth/topology
    for l in range(n_ent_layers):
        g_depth = x[depth_off + l]
        g_topo  = x[topo_idx]
        g_chain = g_depth * (1 - g_topo)
        g_ring  = g_depth * (g_topo)

        if QNN_SHARED_ENT_PARAMS:
            th_chain = theta[t]; t += 1
            th_ring  = theta[t]; t += 1
            # chain (shared param)
            for i in range(max(1, n_qubits-1)):
                qc.crz(g_chain * th_chain, i, i+1)
            # ring (shared param)
            for i in range(n_qubits):
                qc.crz(g_ring * th_ring, i, (i+1) % n_qubits)
        else:
            # (slow) per-edge params
            for i in range(max(1, n_qubits-1)):
                qc.crz(g_chain * theta[t], i, i+1); t += 1
            for i in range(n_qubits):
                qc.crz(g_ring * theta[t], i, (i+1) % n_qubits); t += 1

    qc_t = transpile(qc, optimization_level=0, seed_transpiler=SEED)

    # observables: just Z on qubit-0 (fast)
    observables = []
    z = ["I"] * n_qubits
    z[0] = "Z"
    observables.append(SparsePauliOp.from_list([("".join(z), 1.0)]))

    qnn = EstimatorQNN(
        circuit=qc_t,
        estimator=ESTIMATOR,
        observables=observables,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=bool(QNN_INPUT_GRADS),
    )
    return qnn, q_in_dim

def fixed_q_to_feat_matrix(q_out_dim, seed=2026):
    g = torch.Generator(device="cpu").manual_seed(seed + 1337*q_out_dim)
    W = torch.randn(q_out_dim, 2048, generator=g) / math.sqrt(q_out_dim)
    return W.float()


# ============================================================
# Adapter: CNN feats + PQC delta -> adapted feats -> frozen classifier
# ============================================================
class QTTAAdapter(nn.Module):
    def __init__(self, qnn_torch, q2f_mat, n_qubits, n_ent_layers, classifier_module):
        super().__init__()
        self.qnn = qnn_torch
        self.q2f = q2f_mat
        self.nq = n_qubits
        self.nl = n_ent_layers
        self.classifier = classifier_module

    def forward_all(self, feats_cpu, dt_cpu):
        # encode small chunk into angles (2*nq)
        feat_small = feats_cpu[:, : (2*self.nq)].contiguous()
        feat_angles = torch.tanh(feat_small) * PI

        dt_angles, depth_g, topo, alpha, lam1, lam2 = parse_dt(dt_cpu, self.nq, self.nl)

        q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
        q_out = self.qnn(q_in).to(torch.float32)  # (B, q_out_dim)   q_out_dim=1 here

        delta_f = (q_out @ self.q2f) * 0.05
        f_adapt = feats_cpu + alpha * delta_f

        logits = self.classifier(f_adapt)
        return logits, q_out, f_adapt, (lam1, lam2), alpha


# ============================================================
# Losses
# ============================================================
def binary_entropy_from_logits(logits_cpu):
    p = torch.sigmoid(logits_cpu)
    eps = 1e-6
    ent = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps))
    return ent.mean()

def mse_prob_logits(s_logits, t_logits):
    sp = torch.sigmoid(s_logits)
    tp = torch.sigmoid(t_logits)
    return ((sp - tp)**2).mean()

def cosine_fidelity_loss(q_out, q_src_mean):
    q = F.normalize(q_out, dim=1)
    s = F.normalize(q_src_mean.unsqueeze(0).expand_as(q), dim=1)
    cos = (q * s).sum(dim=1)
    return (1.0 - cos).mean()

def bernoulli_kl(p, q):
    eps = 1e-6
    p = torch.clamp(p, eps, 1-eps)
    q = torch.clamp(q, eps, 1-eps)
    kl = p * torch.log(p/q) + (1-p) * torch.log((1-p)/(1-q))
    return kl.mean()


# ============================================================
# Dual-memory (proposal)
# ============================================================
class FeatureQueue:
    def __init__(self, max_size, feat_dim, prob_dim):
        self.max_size = int(max_size)
        self.feat_dim = int(feat_dim)
        self.prob_dim = int(prob_dim)
        self.feats = torch.zeros((0, feat_dim), dtype=torch.float32)
        self.probs = torch.zeros((0, prob_dim), dtype=torch.float32)

    def __len__(self): return int(self.feats.shape[0])

    @torch.no_grad()
    def push(self, feats_b, probs_b):
        feats_b = feats_b.detach().cpu().float()
        probs_b = probs_b.detach().cpu().float()
        self.feats = torch.cat([self.feats, feats_b], dim=0)
        self.probs = torch.cat([self.probs, probs_b], dim=0)
        if len(self) > self.max_size:
            self.feats = self.feats[-self.max_size:]
            self.probs = self.probs[-self.max_size:]

class DualMemory:
    def __init__(self, max_size, feat_dim, prob_dim):
        self.source = FeatureQueue(max_size, feat_dim, prob_dim)
        self.target = FeatureQueue(max_size, feat_dim, prob_dim)

@torch.no_grad()
def precompute_source_memory_batches(num_batches):
    it = iter(stream_loader)
    feats_list, probs_list = [], []
    total = 0
    for _ in tqdm(range(num_batches), desc="Precompute Source Memory Data (no training)"):
        try:
            pil_imgs, _, _ = next(it)
        except StopIteration:
            it = iter(stream_loader)
            pil_imgs, _, _ = next(it)
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        probs = cnn_probs_from_feats(feats).detach().cpu().float()
        feats_list.append(feats)
        probs_list.append(probs)
        total += feats.shape[0]
    print(f"✅ Precomputed source-memory tensors for ~{total} samples.")
    return feats_list, probs_list

SOURCE_MEM_FEATS_LIST, SOURCE_MEM_PROBS_LIST = (None, None)
if USE_MEMORY:
    SOURCE_MEM_FEATS_LIST, SOURCE_MEM_PROBS_LIST = precompute_source_memory_batches(INIT_SOURCE_MEMORY_BATCHES)

def init_source_memory(mem_obj):
    if (not USE_MEMORY) or (mem_obj is None): return
    for f, p in zip(SOURCE_MEM_FEATS_LIST, SOURCE_MEM_PROBS_LIST):
        mem_obj.source.push(f, p)


# ============================================================
# q_src_mean init/cache (depends on qubits)
# ============================================================
def init_q_src_mean(adapter_obj, n_qubits, q_out_dim):
    save_path = os.path.join(QSTATS_DIR, f"q_src_stats_Q{n_qubits}_D{q_out_dim}.npz")
    if os.path.exists(save_path):
        try:
            z = np.load(save_path)
            v = torch.from_numpy(z["q_src_mean"]).float()
            print(f"✅ Loaded cached q_src_mean: {save_path}")
            return v
        except Exception as e:
            print("⚠ Failed to load cached q_src_mean; recompute. Reason:", repr(e))

    adapter_obj.eval()
    qs = []
    it = iter(stream_loader)
    for _ in tqdm(range(INIT_QSTATS_BATCHES), desc=f"Init q_src_mean (Q={n_qubits})"):
        try:
            pil_imgs, _, metas = next(it)
        except StopIteration:
            it = iter(stream_loader)
            pil_imgs, _, metas = next(it)

        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        dt = get_dt_cached(pil_imgs, metas, feats)

        with torch.no_grad():
            _, q_out, _, _, _ = adapter_obj.forward_all(feats, dt)
        qs.append(q_out.detach().cpu().float())

    q_all = torch.cat(qs, dim=0)
    q_mean = q_all.mean(dim=0)
    try:
        np.savez(save_path, q_src_mean=q_mean.numpy())
        print(f"✅ Cached q_src_mean: {save_path}")
    except Exception as e:
        print("⚠ Could not save q_src_mean cache. Reason:", repr(e))
    return q_mean


# ============================================================
# TTDA Strategies (PQC-only updates during TEST)
# ============================================================
def grad_nonzero_report(mod, max_items=10):
    items = []
    for n, p in mod.named_parameters():
        if p.grad is None: continue
        gsum = float(p.grad.detach().abs().sum().cpu())
        if gsum > 0: items.append((n, gsum))
    items.sort(key=lambda x: -x[1])
    return items[:max_items], len(items)

class PQC_TENT:
    # Entropy minimization on unlabeled test batch
    def __init__(self, adapter, lr=5e-4, sanity_print_once=True):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.qnn.parameters(), lr=lr)
        self._printed = False
        self._sanity = sanity_print_once

    def step_cached(self, feats_eval_cpu, dt_cpu):
        self.adapter.train()
        logits, _, _, _, _ = self.adapter.forward_all(feats_eval_cpu, dt_cpu)
        loss = binary_entropy_from_logits(logits)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()

        if self._sanity and (not self._printed):
            self._printed = True
            q_items, q_cnt = grad_nonzero_report(self.adapter.qnn)
            print("✅ [TENT] QNN nonzero grads:", q_cnt, "| top:", q_items)

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.qnn.parameters(), GRAD_CLIP)
        self.opt.step()

        return {"L_total": float(loss.detach().cpu()), "L_ent": float(loss.detach().cpu())}

class PQC_CoTTA_HEAD:
    # EMA teacher + consistency + stochastic restore (PQC head only)
    def __init__(self, adapter, lr=5e-4, ema=0.999, restore_frac=0.01, sanity_print_once=True):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.qnn.parameters(), lr=lr)
        self.ema = ema
        self.restore_frac = restore_frac
        self.teacher = copy.deepcopy(self.adapter).eval()
        self.source  = copy.deepcopy(self.adapter).eval()
        for p in self.teacher.parameters(): p.requires_grad_(False)
        for p in self.source.parameters():  p.requires_grad_(False)
        self._printed = False
        self._sanity = sanity_print_once

    @torch.no_grad()
    def _ema_update(self):
        for pt, ps in zip(self.teacher.qnn.parameters(), self.adapter.qnn.parameters()):
            pt.data.mul_(self.ema).add_(ps.data, alpha=(1-self.ema))

    @torch.no_grad()
    def _stochastic_restore(self):
        if self.restore_frac <= 0: return
        src_params = dict(self.source.qnn.named_parameters())
        for name, p in self.adapter.qnn.named_parameters():
            if name not in src_params: continue
            mask = (torch.rand_like(p) < self.restore_frac)
            p.data[mask] = src_params[name].data[mask]

    def step_cached(self, pil_imgs, feats_eval_cpu, dt_cpu):
        self.adapter.train()

        # teacher on weak/eval feats (cached)
        with torch.no_grad():
            t_logits, _, _, _, _ = self.teacher.forward_all(feats_eval_cpu, dt_cpu)

        # student on strong view (needs extra CNN forward once)
        xs = batch_apply_tf(pil_imgs, strong_tf).to(DEVICE)
        feats_s = extract_features(xs).detach().cpu().float()
        s_logits, _, _, _, _ = self.adapter.forward_all(feats_s, dt_cpu)

        loss = mse_prob_logits(s_logits, t_logits)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()

        if self._sanity and (not self._printed):
            self._printed = True
            q_items, q_cnt = grad_nonzero_report(self.adapter.qnn)
            print("✅ [CoTTA] QNN nonzero grads:", q_cnt, "| top:", q_items)

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.qnn.parameters(), GRAD_CLIP)
        self.opt.step()

        self._ema_update()
        self._stochastic_restore()

        return {"L_total": float(loss.detach().cpu()), "L_cons": float(loss.detach().cpu())}

class PQC_EATA_LITE:
    # Reliable-sample entropy minimization
    def __init__(self, adapter, lr=5e-4, ent_thresh=0.35, sanity_print_once=True):
        self.adapter = adapter
        self.opt = torch.optim.Adam(self.adapter.qnn.parameters(), lr=lr)
        self.ent_thresh = ent_thresh
        self._printed = False
        self._sanity = sanity_print_once

    def step_cached(self, feats_eval_cpu, dt_cpu):
        self.adapter.train()
        logits, _, _, _, _ = self.adapter.forward_all(feats_eval_cpu, dt_cpu)
        p = torch.sigmoid(logits)
        eps = 1e-6
        ent_per = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps)).mean(dim=1)  # [B]
        mask = (ent_per < self.ent_thresh).float()
        loss = (ent_per * mask).sum() / (mask.sum() + 1e-6)

        self.opt.zero_grad(set_to_none=True)
        loss.backward()

        if self._sanity and (not self._printed):
            self._printed = True
            q_items, q_cnt = grad_nonzero_report(self.adapter.qnn)
            print("✅ [EATA-lite] QNN nonzero grads:", q_cnt, "| top:", q_items)

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.qnn.parameters(), GRAD_CLIP)
        self.opt.step()

        return {"L_total": float(loss.detach().cpu()),
                "L_ent_rel": float(loss.detach().cpu()),
                "mask_frac": float(mask.mean().detach().cpu())}

class QTTA_Proposal_TTDA:
    # Proposal: dt-driven λ1/λ2, fidelity to q_src_mean, + memory contrast + distill
    def __init__(self, adapter, q_src_mean, dual_memory=None, lr=5e-4, sanity_print_once=True):
        self.adapter = adapter
        self.q_src_mean = q_src_mean.detach().cpu().float()
        self.mem = dual_memory
        self.opt = torch.optim.Adam(self.adapter.qnn.parameters(), lr=lr)
        self._printed = False
        self._sanity = sanity_print_once

    def step_cached(self, feats_eval_cpu, dt_cpu):
        self.adapter.train()
        logits_t, q_out, f_t, (lam1, lam2), _alpha = self.adapter.forward_all(feats_eval_cpu, dt_cpu)

        L_ent = binary_entropy_from_logits(logits_t)
        L_fid = cosine_fidelity_loss(q_out, self.q_src_mean)

        lam1m = lam1.mean()
        lam2m = lam2.mean()
        L_main = lam1m * L_ent + lam2m * L_fid

        L_contrast = torch.tensor(0.0)
        L_distill  = torch.tensor(0.0)

        if USE_MEMORY and (self.mem is not None) and (len(self.mem.source) > 0):
            # align current adapted feats to a small random source-memory sample
            K = min(128, len(self.mem.source))
            idx = torch.randint(0, len(self.mem.source), (K,))
            src_feats = self.mem.source.feats[idx]  # [K, 2048]
            # cosine to mean prototype
            proto = src_feats.mean(dim=0, keepdim=True)
            sim = F.cosine_similarity(F.normalize(f_t, dim=1), F.normalize(proto, dim=1), dim=1)
            L_contrast = (1.0 - sim).mean()

            # distill: keep adapted probs close to base probs
            with torch.no_grad():
                p_src = torch.sigmoid(classifier(feats_eval_cpu))
            p_tgt = torch.sigmoid(logits_t)
            L_distill = bernoulli_kl(p_src, p_tgt)

        L_total = L_main + BETA_CONTRAST * L_contrast + BETA_DISTILL * L_distill

        self.opt.zero_grad(set_to_none=True)
        L_total.backward()

        if self._sanity and (not self._printed):
            self._printed = True
            q_items, q_cnt = grad_nonzero_report(self.adapter.qnn)
            print("✅ [QTTA-proposal] QNN nonzero grads:", q_cnt, "| top:", q_items)

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(self.adapter.qnn.parameters(), GRAD_CLIP)
        self.opt.step()

        if USE_MEMORY and (self.mem is not None):
            with torch.no_grad():
                p_tgt = torch.sigmoid(logits_t.detach())
            self.mem.target.push(f_t.detach(), p_tgt.detach())

        return {
            "L_total": float(L_total.detach().cpu()),
            "L_ent": float(L_ent.detach().cpu()),
            "L_fid": float(L_fid.detach().cpu()),
            "L_contrast": float(L_contrast.detach().cpu()),
            "L_distill": float(L_distill.detach().cpu()),
            "lam1": float(lam1m.detach().cpu()),
            "lam2": float(lam2m.detach().cpu()),
        }


# ============================================================
# Prediction with adapter + TTA (during TEST)
# ============================================================
@torch.no_grad()
def predict_with_adapter_cached(adapter_obj, pil_imgs, feats_eval_cpu, dt_cpu):
    adapter_obj.eval()

    probs_views = []
    logits1, _, _, _, _ = adapter_obj.forward_all(feats_eval_cpu, dt_cpu)
    probs_views.append(torch.sigmoid(logits1).detach())

    # flip
    if USE_TTA and TTA_VIEWS >= 2:
        pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
        x2 = batch_apply_tf(pil_flip, eval_tf).to(DEVICE)
        feats2 = extract_features(x2).detach().cpu().float()
        logits2, _, _, _, _ = adapter_obj.forward_all(feats2, dt_cpu)
        probs_views.append(torch.sigmoid(logits2).detach())

    # autoaug
    if USE_TTA and TTA_VIEWS >= 3 and USE_AUTOAUG_TTA:
        x3 = batch_apply_tf(pil_imgs, autoaug_tf).to(DEVICE)
        feats3 = extract_features(x3).detach().cpu().float()
        logits3, _, _, _, _ = adapter_obj.forward_all(feats3, dt_cpu)
        probs_views.append(torch.sigmoid(logits3).detach())

    p = torch.stack(probs_views, dim=0).mean(dim=0)
    return p.cpu().numpy()

@torch.no_grad()
def predict_cnn_only_from_feats(feats_eval_cpu):
    p = cnn_probs_from_feats(feats_eval_cpu).cpu().numpy()
    return p


# ============================================================
# Metrics + plots
# ============================================================
def compute_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        rows.append([lbl, aucv, apv, f1v])

    df_pc = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    out = {}
    out["per_class"] = df_pc
    out["macro_roc_auc"] = float(np.nanmean(df_pc["ROC_AUC"].values))
    out["macro_pr_auc"]  = float(np.nanmean(df_pc["PR_AUC"].values))
    out["macro_f1"]      = float(np.nanmean(df_pc["F1@0.5"].values))
    try:
        out["micro_roc_auc"] = float(roc_auc_score(Y.ravel(), P.ravel()))
        out["micro_pr_auc"]  = float(average_precision_score(Y.ravel(), P.ravel()))
    except Exception:
        out["micro_roc_auc"] = np.nan
        out["micro_pr_auc"]  = np.nan
    return out

def plot_roc(Y, P, labels, title, save_path):
    plt.figure()
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            continue
        fpr, tpr, _ = roc_curve(yt, yp)
        plt.plot(fpr, tpr, label=f"{lbl} (AUC={roc_auc_score(yt, yp):.3f})")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_pr(Y, P, labels, title, save_path):
    plt.figure()
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            continue
        prec, rec, _ = precision_recall_curve(yt, yp)
        apv = average_precision_score(yt, yp)
        plt.plot(rec, prec, label=f"{lbl} (AP={apv:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_loss(df_log, title, save_path):
    if df_log is None or len(df_log)==0: return
    plt.figure()
    if "L_total" in df_log.columns:
        plt.plot(df_log["L_total"].values, label="L_total")
    if "L_ent" in df_log.columns:
        plt.plot(df_log["L_ent"].values, label="L_ent")
    if "L_fid" in df_log.columns:
        plt.plot(df_log["L_fid"].values, label="L_fid")
    plt.xlabel("TTDA step"); plt.ylabel("loss")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# ============================================================
# Make adapter for a given qubit count
# ============================================================
def make_adapter(n_qubits):
    qnn, q_in_dim = build_qnn(n_qubits, N_ENT_LAYERS)
    qnn_torch = TorchConnector(qnn).to("cpu")   # TorchConnector weight lives on CPU
    q_out_dim = int(qnn.output_shape[0]) if hasattr(qnn, "output_shape") else QNN_NUM_OBSERVABLES
    q2f = fixed_q_to_feat_matrix(q_out_dim).to("cpu")
    adapter = QTTAAdapter(qnn_torch, q2f, n_qubits, N_ENT_LAYERS, classifier).to("cpu").eval()

    # Freeze everything; unfreeze ONLY qnn weights
    for p in adapter.parameters():
        p.requires_grad_(False)
    for p in adapter.qnn.parameters():
        p.requires_grad_(True)

    return adapter, q_in_dim, q_out_dim


# ============================================================
# TEST runner: TTDA happens INSIDE test loop (no labels used)
# ============================================================
def run_test_for_strategy(strategy_name, adapter, q_src_mean, out_dir):
    # memory per strategy (fresh)
    dual_mem = DualMemory(MEM_SIZE, feat_dim=2048, prob_dim=len(UNIFIED_LABELS)) if USE_MEMORY else None
    if USE_MEMORY:
        init_source_memory(dual_mem)

    # Build strategy object
    if strategy_name == "CNN_only":
        strategy = None
    elif strategy_name == "QTTA_no_adapt":
        strategy = None
    elif strategy_name == "PQC_TENT":
        strategy = PQC_TENT(adapter, lr=LR_TTA, sanity_print_once=True)
    elif strategy_name == "PQC_CoTTA":
        strategy = PQC_CoTTA_HEAD(adapter, lr=LR_TTA, ema=0.999, restore_frac=0.01, sanity_print_once=True)
    elif strategy_name == "PQC_EATA":
        strategy = PQC_EATA_LITE(adapter, lr=LR_TTA, ent_thresh=EATA_ENT_THRESH, sanity_print_once=True)
    elif strategy_name == "QTTA_proposal":
        strategy = QTTA_Proposal_TTDA(adapter, q_src_mean=q_src_mean, dual_memory=dual_mem, lr=LR_TTA, sanity_print_once=True)
    else:
        raise ValueError("Unknown strategy: "+strategy_name)

    Ps, Ys = [], []
    logs = []
    total_adapt = 0

    t0 = time.time()
    for bi, (pil_imgs, y, metas) in enumerate(tqdm(test_loader, desc=f"TEST [{strategy_name}]")):
        if (MAX_TEST_BATCHES is not None) and (bi >= int(MAX_TEST_BATCHES)):
            break

        # One CNN forward per batch (reuse)
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats_eval = extract_features(x).detach().cpu().float()

        # Cached dt per image-id (no repeated BERT)
        dt = get_dt_cached(pil_imgs, metas, feats_eval)

        # TTDA during test (unlabeled): DO NOT USE y
        if (strategy is not None) and (ADAPT_STEPS_PER_TEST_BATCH > 0):
            if (bi % max(1, ADAPT_EVERY_K_BATCHES)) == 0 and (total_adapt < MAX_TOTAL_ADAPT_STEPS):
                steps_here = min(ADAPT_STEPS_PER_TEST_BATCH, MAX_TOTAL_ADAPT_STEPS - total_adapt)
                for _ in range(steps_here):
                    if strategy_name == "PQC_CoTTA":
                        logs.append(strategy.step_cached(pil_imgs, feats_eval, dt))
                    else:
                        logs.append(strategy.step_cached(feats_eval, dt))
                    total_adapt += 1

        # prediction (with TTA if enabled)
        if strategy_name == "CNN_only":
            p_np = predict_cnn_only_from_feats(feats_eval)
        else:
            p_np = predict_with_adapter_cached(adapter, pil_imgs, feats_eval, dt)

        Ps.append(p_np)
        Ys.append(y.numpy())

    t_test = time.time() - t0
    P = np.vstack(Ps)
    Y = np.vstack(Ys)

    metrics = compute_metrics(Y, P, UNIFIED_LABELS)
    df_log = pd.DataFrame(logs) if len(logs) else pd.DataFrame()

    # save csvs
    per_class_csv = os.path.join(out_dir, f"per_class_{strategy_name}.csv")
    metrics["per_class"].to_csv(per_class_csv, index=False)

    # plots
    plot_roc(Y, P, UNIFIED_LABELS, title=f"ROC | {strategy_name}", save_path=os.path.join(out_dir, f"ROC_{strategy_name}.png"))
    plot_pr(Y, P, UNIFIED_LABELS, title=f"PR | {strategy_name}", save_path=os.path.join(out_dir, f"PR_{strategy_name}.png"))
    if len(df_log):
        df_log.to_csv(os.path.join(out_dir, f"adapt_logs_{strategy_name}.csv"), index=False)
        plot_loss(df_log, title=f"Loss | {strategy_name}", save_path=os.path.join(out_dir, f"LOSS_{strategy_name}.png"))

    row = {
        "Strategy": strategy_name,
        "Macro_ROC_AUC": metrics["macro_roc_auc"],
        "Macro_PR_AUC": metrics["macro_pr_auc"],
        "Macro_F1@0.5": metrics["macro_f1"],
        "Micro_ROC_AUC": metrics["micro_roc_auc"],
        "Micro_PR_AUC": metrics["micro_pr_auc"],
        "Test_seconds": t_test,
        "TTDA_steps_per_test_batch": ADAPT_STEPS_PER_TEST_BATCH if strategy is not None else 0,
        "Adapt_every_k_batches": (ADAPT_EVERY_K_BATCHES if strategy is not None else 0),
        "Total_adapt_steps": int(total_adapt),
        "Mean_L_total": float(df_log["L_total"].mean()) if ("L_total" in df_log.columns and len(df_log)) else np.nan,
    }
    return row, metrics["per_class"], df_log


# ============================================================
# MAIN: multi-qubit sweep + all TTDA mechanisms
# ============================================================
ALL_SUMMARY = []

STRATEGIES = [
    "CNN_only",
    "QTTA_no_adapt",
    "PQC_TENT",
    "PQC_CoTTA",
    "PQC_EATA",
    "QTTA_proposal",
]

for nq in QUBIT_LIST:
    print("\n" + "="*70)
    print(f"🔥 RUNNING QUBITS = {nq} | TTDA steps/batch = {ADAPT_STEPS_PER_TEST_BATCH} | adapt_every={ADAPT_EVERY_K_BATCHES} | TTA_VIEWS={TTA_VIEWS}")
    print(f"    QNN: obs={QNN_NUM_OBSERVABLES} | input_grads={QNN_INPUT_GRADS} | shared_ent={QNN_SHARED_ENT_PARAMS}")
    print("="*70)

    q_out_dir = os.path.join(OUTDIR, f"Q{nq}")
    os.makedirs(q_out_dir, exist_ok=True)

    # build base adapter once per qubit
    torch.manual_seed(SEED + 100*nq)
    base_adapter, _, q_out_dim = make_adapter(nq)

    # cache initial qnn weight for fair resets across strategies
    init_q_weight = base_adapter.qnn.weight.detach().clone()

    # compute q_src_mean once per qubit
    q_src_mean = init_q_src_mean(base_adapter, nq, q_out_dim)

    qubit_rows = []
    for strat in STRATEGIES:
        # reset adapter fresh with same init weight
        torch.manual_seed(SEED + 100*nq)
        adapter, _, _ = make_adapter(nq)
        with torch.no_grad():
            adapter.qnn.weight.copy_(init_q_weight)

        row, df_pc, df_log = run_test_for_strategy(strat, adapter, q_src_mean, q_out_dir)
        row["Qubits"] = nq
        qubit_rows.append(row)
        ALL_SUMMARY.append(row)

        print(f"\n--- Q{nq} | {strat} ---")
        print(df_pc)

    df_q = pd.DataFrame(qubit_rows).sort_values(["Macro_ROC_AUC"], ascending=False).reset_index(drop=True)
    df_q.to_csv(os.path.join(q_out_dir, f"summary_Q{nq}.csv"), index=False)
    print("\n✅ Saved:", os.path.join(q_out_dir, f"summary_Q{nq}.csv"))
    print(df_q)

# Global summary
df_all = pd.DataFrame(ALL_SUMMARY)
df_all.to_csv(os.path.join(OUTDIR, "summary_ALL_QUBITS.csv"), index=False)
print("\n✅ Saved global summary:", os.path.join(OUTDIR, "summary_ALL_QUBITS.csv"))

# Macro ROC plot vs qubits (all strategies)
plt.figure()
for strat in STRATEGIES:
    sub = df_all[df_all["Strategy"]==strat].sort_values("Qubits")
    plt.plot(sub["Qubits"].values, sub["Macro_ROC_AUC"].values, marker="o", label=strat)
plt.xlabel("Qubits"); plt.ylabel("Macro ROC-AUC")
plt.title("Macro ROC-AUC vs Qubits (all TTDA mechanisms)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "MACRO_ROC_vs_QUBITS.png"), dpi=200)
plt.close()
print("✅ Saved:", os.path.join(OUTDIR, "MACRO_ROC_vs_QUBITS.png"))


DEVICE: cuda
✅ /kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt
✅ /kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt
NIH test: 25596 | NIH unlabeled stream: 86524
✅ Loaded ResNet checkpoint (strict=False)
  missing: 0 | unexpected: 0
✅ Classifier: fallback to ResNet fc (1-layer).
✅ Loaded BiomedBERT .pt into encoder (strict=False)
  missing: 199 | unexpected: 201


Precompute Source Memory Data (no training): 100%|██████████| 30/30 [00:03<00:00,  8.97it/s]


✅ Precomputed source-memory tensors for ~480 samples.

🔥 RUNNING QUBITS = 6 | TTDA steps/batch = 1 | adapt_every=4 | TTA_VIEWS=2
    QNN: obs=1 | input_grads=False | shared_ent=True


Init q_src_mean (Q=6): 100%|██████████| 20/20 [00:10<00:00,  1.95it/s]


✅ Cached q_src_mean: QTTA_MULTI_QUBIT_RESULTS/qstats_cache/q_src_stats_Q6_D1.npz


...

TEST [PQC_CoTTA]: 100%|██████████| 1600/1600 [1:35:36<00:00,  3.59s/it]



--- Q6 | PQC_CoTTA ---
              Label   ROC_AUC    PR_AUC    F1@0.5
0       Atelectasis  0.723284  0.269452  0.338773
1      Cardiomegaly  0.845786  0.275977  0.277275
2     Consolidation  0.697127  0.130515  0.184393
3             Edema  0.773222  0.101352  0.137179
4  Pleural Effusion  0.806250  0.473672  0.494559


TEST [PQC_EATA]:   0%|          | 0/1600 [00:00<?, ?it/s]

✅ [EATA-lite] QNN nonzero grads: 1 | top: [('weight', 9.310092718806118e-05)]


TEST [PQC_EATA]:  24%|██▍       | 384/1600 [22:38<41:55,  2.07s/it]  

# LATEST

# LOW AUC FOR LABEL MISMATCH BUT ALIGNED WITH PDF
## Step-by-step explanation in 50 lines

1. The code builds a **Quantum Test-Time Domain Adaptation (QTTA/TTDA)** pipeline for chest X-ray classification.
2. Its main goal is to adapt a pretrained CheXpert-to-NIH model while testing it on NIH images.
3. It imports PyTorch, torchvision, pandas, NumPy, Qiskit, transformers, and evaluation libraries.
4. A fixed random seed of `42` is used so that experiments are more reproducible.
5. The program automatically uses a GPU when CUDA is available.
6. It defines paths for the ResNet checkpoint, BioMedBERT checkpoint, NIH dataset, and RAG documents.
7. It predicts five diseases: Atelectasis, Cardiomegaly, Consolidation, Edema, and Pleural Effusion.
8. Batch size, quantum-circuit size, adaptation steps, learning rate, TTA views, and RAG settings are configured.
9. The code tests whether all required Kaggle paths exist before continuing. 
10. `read_txt_lines()` loads the NIH train/test image filenames from text files.
11. `nih_resolver()` converts each image filename into its complete image path.
12. `build_nih_df()` selects only images belonging to the requested NIH split.
13. It converts NIH’s text labels into five separate binary label columns.
14. For example, `"Atelectasis|Effusion"` becomes Atelectasis=`1` and Pleural Effusion=`1`.
15. It also keeps patient age, gender, patient ID, and X-ray view information.
16. `CXRDatasetPIL` loads each chest X-ray as an RGB PIL image.
17. The labeled test dataset returns true disease labels for evaluation.
18. The unlabeled stream dataset returns zero labels because it is used only for adaptation.
19. `pil_collate_fn()` keeps images as PIL objects while stacking their label tensors.
20. Two DataLoaders are created: one for testing and one for the unlabeled adaptation stream. 
21. Each image is resized, center-cropped to `224×224`, converted into a tensor, and normalized.
22. Optional AutoAugment transformations are prepared for test-time augmentation.
23. `_strip_prefix()` removes checkpoint prefixes such as `module.`, `model.`, and `backbone.`.
24. `load_ckpt_state_dict()` supports checkpoints stored under `state_dict`, `model`, or `net`.
25. A ResNet-50 is created with five output neurons, one for each disease.
26. The pretrained ResNet weights are loaded using `strict=False` to tolerate unmatched keys.
27. All ResNet parameters are frozen, so the CNN does not change during testing.
28. `feat_net` removes the final classifier and extracts a 2,048-dimensional image representation.
29. The classifier head is recovered from the checkpoint or copied from `ResNet.fc`.
30. Sigmoid converts the five classifier logits into disease probabilities. 
31. `quality_features()` calculates brightness, contrast, entropy, sharpness, edge density, and clipping.
32. These seven measurements describe the visual quality and acquisition properties of an X-ray.
33. The RAG CSV contains source-domain descriptions called `prompt_text`.
34. Numeric quality columns and available `cnn_*` probability columns are selected for retrieval.
35. Every source document is represented by a normalized numeric retrieval vector.
36. For a target image, cosine similarity finds the three most similar source documents.
37. `align_dim()` pads or trims target vectors to prevent the previous RAG dimension-mismatch error.
38. The retrieved source notes are combined with patient metadata and image-quality information. 
39. BioMedBERT converts this generated text prompt into a normalized domain descriptor embedding.
40. An LRU cache stores embeddings so the same image is not repeatedly processed by BioMedBERT.
41. The descriptor controls quantum angles, circuit depth gates, topology, adaptation strength, and loss weights.
42. The quantum circuit applies `RY` and `RZ` rotations to encode CNN and text information.
43. Controlled `RZ` gates create chain or ring entanglement between the qubits.
44. The quantum expectation value is projected into five disease-logit corrections.
45. These corrections are scaled and added to the frozen CNN’s original logits. 
46. Before adaptation, the code estimates the average quantum output from unlabeled NIH stream images.
47. During testing, it minimizes prediction entropy so disease predictions become more confident.
48. A fidelity loss prevents the adapted quantum representation from moving too far from its reference mean.
49. Predictions from original, horizontally flipped, and optionally augmented images are averaged.
50. Finally, it reports per-disease ROC-AUC, PR-AUC, F1, macro AUC, adaptation logs, and quantum-weight changes. 

### Overall data flow

**NIH X-ray → ResNet features and probabilities → quality features → RAG retrieval → BioMedBERT domain descriptor → quantum adapter → corrected disease predictions → entropy/fidelity adaptation → TTA averaging → AUC evaluation.**


In [ ]:
# ============================================================
# QTTA / TTDA (PDF-ALIGNED + HIGHER BASELINE + TTDA MOVES AUC)
# FIXED: RAG retrieval dimension mismatch (auto-align query dims)
# ============================================================

import os, math, copy, time, random, warnings, inspect
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

RAG_SRC_CSV = "/kaggle/input/rag-src/rag_chex_train_documents.csv"

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_PROMPT_LEN  = 192

QUBIT_LIST   = [6, 8]
N_ENT_LAYERS = 2
QNN_NUM_OBS  = 1

ADAPT_STEPS_PER_TEST_BATCH = 1
ADAPT_EVERY_K_BATCHES      = 2
MAX_TOTAL_ADAPT_STEPS      = 800
LR_TTDA                   = 5e-4
GRAD_CLIP                 = 1.0

USE_TTA = True
TTA_VIEWS = 2
USE_AUTOAUG_TTA = True

PQC_LOGIT_SCALE = 0.35

INIT_QSTATS_BATCHES = 20

RAG_DOC_MAX   = 50000
RAG_TOPK      = 3

# If TRUE, we will only include CNN probs in query if the RAG CSV actually contains cnn_* columns
RAG_USE_CNN_PROBS_IN_QUERY = True

OUTDIR = "QTTA_PDF_ALIGNED_HIGHBASE"
os.makedirs(OUTDIR, exist_ok=True)

for p in [RESNET_CKPT, BIOMED_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM, RAG_SRC_CSV]:
    print(("✅" if os.path.exists(p) else "❌"), p)

# ----------------------------
# Helpers
# ----------------------------
def _pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# qiskit-algorithms (optional but reduces noise)
try:
    import qiskit_algorithms  # noqa
except Exception:
    try:
        _pip_install("qiskit-algorithms")
    except Exception as e:
        print("⚠ Could not install qiskit-algorithms (often ok). Reason:", repr(e))

def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(f"NIH image not found: {p}")
    return p

def build_nih_df(split_list_txt, nih_csv_path):
    wanted = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")
    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    df["patient_id"] = df["Patient ID"].astype(str) if "Patient ID" in df.columns else "unknown"
    df["age"]        = df["Patient Age"] if "Patient Age" in df.columns else np.nan
    df["gender"]     = df["Patient Gender"] if "Patient Gender" in df.columns else "U"
    df["view"]       = df["View Position"] if "View Position" in df.columns else "UNK"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","patient_id","age","gender","view"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

class CXRDatasetPIL(Dataset):
    def __init__(self, df, img_resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.images = self.df["image"].tolist()
        self.labeled = labeled
        if labeled:
            self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "patient_id": str(self.df.loc[idx, "patient_id"]),
            "age": self.df.loc[idx, "age"],
            "gender": str(self.df.loc[idx, "gender"]),
            "view": str(self.df.loc[idx, "view"]),
        }
        if self.labeled:
            y = torch.from_numpy(self.labels[idx])
        else:
            y = torch.zeros(C, dtype=torch.float32)
        return img, y, meta

def pil_collate_fn(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys), dim=0), list(metas)

nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDatasetPIL(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)
stream_loader = DataLoader(
    CXRDatasetPIL(nih_stream_df, nih_resolver, labeled=False),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)

print("NIH test:", len(nih_test_df), "| NIH stream(unlabeled):", len(nih_stream_df))

# ----------------------------
# Preprocessing (older-style)
# ----------------------------
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
except Exception:
    autoaug = None
    USE_AUTOAUG_TTA = False
    print("⚠ AutoAugment not available; disabling autoaug TTA.")
autoaug_tf = T.Compose([T.Resize(256), T.CenterCrop(224), (autoaug if autoaug else T.Lambda(lambda x:x)), T.ToTensor(), norm])

def batch_apply_tf(pil_list, tf):
    return torch.stack([tf(im) for im in pil_list], dim=0)

# ----------------------------
# CKPT loading utilities
# ----------------------------
def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def load_ckpt_state_dict(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:     sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:       sd = ckpt["net"]
    else:                                                sd = ckpt
    return _strip_prefix(sd)

class ClassifierMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=0):
        super().__init__()
        if hidden and hidden > 0:
            self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, out_dim))
        else:
            self.net = nn.Sequential(nn.Linear(in_dim, out_dim))
    def forward(self, x): return self.net(x)

def try_load_classifier_from_ckpt(sd, n_labels):
    cand = {}
    for k, v in sd.items():
        if k.startswith("classifier.") or k.startswith("mlp.") or k.startswith("head."):
            cand[k] = v

    mlp2 = ClassifierMLP(2048, n_labels, hidden=512)
    if len(cand) > 0:
        cand2 = _strip_prefix(cand)
        missing2, unexpected2 = mlp2.load_state_dict(cand2, strict=False)
        if len(missing2) <= 2:
            print("✅ Using 2-layer classifier head loaded from ckpt keys.")
            return mlp2.eval()

    mlp1 = ClassifierMLP(2048, n_labels, hidden=0)
    if "fc.weight" in sd and "fc.bias" in sd and sd["fc.weight"].shape[0] == n_labels:
        mlp1.net[0].weight.data.copy_(sd["fc.weight"])
        mlp1.net[0].bias.data.copy_(sd["fc.bias"])
        print("✅ Using 1-layer classifier from ckpt fc.*")
        return mlp1.eval()

    print("⚠ Could not find classifier/head weights; will rely on ResNet.fc fallback.")
    return None

# ----------------------------
# Load ResNet
# ----------------------------
base_resnet = models.resnet50(weights=None)
base_resnet.fc = nn.Linear(2048, C)

resnet_sd = load_ckpt_state_dict(RESNET_CKPT)
missing, unexpected = base_resnet.load_state_dict(resnet_sd, strict=False)
print("✅ Loaded ResNet .pt (strict=False) | missing:", len(missing), "unexpected:", len(unexpected))

base_resnet = base_resnet.to(DEVICE).eval()
for p in base_resnet.parameters():
    p.requires_grad_(False)

feat_net = nn.Sequential(*(list(base_resnet.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def extract_features(x_tensor):
    return feat_net(x_tensor).flatten(1)

classifier = try_load_classifier_from_ckpt(resnet_sd, C)
if classifier is None:
    classifier = ClassifierMLP(2048, C, hidden=0)
    classifier.net[0].weight.data.copy_(base_resnet.fc.weight.detach().cpu())
    classifier.net[0].bias.data.copy_(base_resnet.fc.bias.detach().cpu())
    classifier = classifier.eval()

classifier = classifier.to("cpu").eval()
for p in classifier.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def cnn_logits_from_feats(feats_cpu):
    return classifier(feats_cpu.float())

@torch.no_grad()
def cnn_probs_from_feats(feats_cpu):
    return torch.sigmoid(cnn_logits_from_feats(feats_cpu))

# ----------------------------
# Quality features
# ----------------------------
def quality_features(pil_img):
    g = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    q_mean = float(g.mean())
    q_std  = float(g.std())

    hist, _ = np.histogram(g, bins=64, range=(0,1), density=True)
    hist = hist + 1e-12
    q_entropy = float(-(hist*np.log(hist)).sum())

    lap = (-4*g + np.roll(g,1,0) + np.roll(g,-1,0) + np.roll(g,1,1) + np.roll(g,-1,1))
    q_sharp = float(lap.var())

    gx = np.abs(np.roll(g,-1,1) - np.roll(g,1,1))
    gy = np.abs(np.roll(g,-1,0) - np.roll(g,1,0))
    mag = gx + gy
    lq_edge_d = float((mag > 0.15).mean())

    q_clip_low  = float((g < 0.02).mean())
    q_clip_high = float((g > 0.98).mean())

    return np.array([q_mean, q_std, q_entropy, q_sharp, lq_edge_d, q_clip_low, q_clip_high], dtype=np.float32)

# ============================================================
# ✅ RAG LOADING (FIXED DIM LOGIC)
# ============================================================
rag_df = pd.read_csv(RAG_SRC_CSV)
if "prompt_text" not in rag_df.columns:
    raise ValueError("RAG CSV must contain a 'prompt_text' column.")

rag_df = rag_df.dropna(subset=["prompt_text"]).reset_index(drop=True)
if len(rag_df) > RAG_DOC_MAX:
    rag_df = rag_df.sample(RAG_DOC_MAX, random_state=SEED).reset_index(drop=True)

# 1) Detect q_* feature columns present
q_cols = [c for c in ["q_mean","q_std","q_entropy","q_sharp","lq_edge_d","q_clip_low","q_clip_high"] if c in rag_df.columns]

# 2) Detect cnn_* columns present (any)
cnn_cols_all = [c for c in rag_df.columns if c.startswith("cnn_")]

# decide if we use cnn probs in retrieval feature vector
USE_CNN_IN_RETRIEVAL = (RAG_USE_CNN_PROBS_IN_QUERY and len(cnn_cols_all) > 0)

# 3) Final feature schema used for retrieval = q_cols + (cnn cols if present)
use_cols = q_cols + (cnn_cols_all if USE_CNN_IN_RETRIEVAL else [])

if len(use_cols) == 0:
    # If the CSV doesn't have numeric retrieval columns, we fall back to ONLY q features (7 dims)
    # We'll build rag_X from computed q features of docs? not possible here; so require at least q_cols.
    raise ValueError(
        "RAG CSV has no numeric retrieval columns. Add q_* columns (q_mean,...), "
        "or include at least one numeric feature column to retrieve by."
    )

rag_X = rag_df[use_cols].values.astype(np.float32)
rag_Xn = rag_X / (np.linalg.norm(rag_X, axis=1, keepdims=True) + 1e-12)
rag_prompts = rag_df["prompt_text"].astype(str).tolist()

RAG_DIM = rag_X.shape[1]
print(f"✅ RAG docs loaded: {len(rag_df)} | USE_CNN_IN_RETRIEVAL={USE_CNN_IN_RETRIEVAL} | dims={RAG_DIM}")

def align_dim(vec, D):
    """Pad/trim vec to exactly D dims (safety)."""
    vec = vec.astype(np.float32).reshape(-1)
    if vec.shape[0] == D:
        return vec
    if vec.shape[0] > D:
        return vec[:D]
    pad = np.zeros((D - vec.shape[0],), dtype=np.float32)
    return np.concatenate([vec, pad], axis=0)

def retrieve_topk_prompts(query_vec, topk=3):
    q = align_dim(query_vec, RAG_DIM)
    qn = q / (np.linalg.norm(q) + 1e-12)
    sims = rag_Xn @ qn
    topk = int(min(topk, len(sims)))
    idx = np.argpartition(-sims, topk-1)[:topk]
    idx = idx[np.argsort(-sims[idx])]
    return [rag_prompts[i] for i in idx]

# ============================================================
# BioMedBERT dt encoder + cache
# ============================================================
try:
    import transformers
except Exception:
    _pip_install("transformers")
    import transformers
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
for p in text_encoder.parameters():
    p.requires_grad_(False)

def load_biomed_pt_into_encoder(pt_path):
    if not os.path.exists(pt_path):
        print("⚠ BiomedBERT .pt not found; using HF weights only:", pt_path)
        return
    sd = load_ckpt_state_dict(pt_path)
    sd2 = {k:v for k,v in sd.items() if not k.startswith("cls.")}
    missing, unexpected = text_encoder.load_state_dict(sd2, strict=False)
    print("✅ Loaded BioMedBERT .pt (strict=False) | missing:", len(missing), "unexpected:", len(unexpected))

load_biomed_pt_into_encoder(BIOMED_CKPT)
DT_DIM = int(text_encoder.config.hidden_size)

class LRUCache:
    def __init__(self, max_items=60000):
        self.max_items = int(max_items)
        self.d = OrderedDict()
    def get(self, k):
        if k not in self.d: return None
        v = self.d.pop(k)
        self.d[k] = v
        return v
    def put(self, k, v):
        if k in self.d: self.d.pop(k)
        self.d[k] = v
        if len(self.d) > self.max_items:
            self.d.popitem(last=False)

DT_CACHE = LRUCache(60000)

@torch.no_grad()
def embed_dt(prompts):
    tok = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(DEVICE)
    out = text_encoder(**tok)
    dt = out.last_hidden_state[:, 0, :]
    dt = F.normalize(dt, dim=1)
    return dt.detach().cpu().float()

def build_target_prompt(meta, qfeat, rag_texts, cnn_probs=None):
    rag_block = " ".join([f"[DOC{i+1}] {t}" for i,t in enumerate(rag_texts)])
    base = (
        f"Target CXR meta: view={meta.get('view','UNK')}, sex={meta.get('gender','U')}, age={meta.get('age','NA')}.\n"
        f"Quality: mean={qfeat[0]:.3f}, std={qfeat[1]:.3f}, ent={qfeat[2]:.3f}, sharp={qfeat[3]:.3f}, "
        f"edge_d={qfeat[4]:.3f}, clip_low={qfeat[5]:.3f}, clip_high={qfeat[6]:.3f}.\n"
    )
    if cnn_probs is not None:
        base += f"CNN_probs={np.round(cnn_probs,3).tolist()}.\n"
    base += "Using similar source-domain notes below, produce a domain descriptor embedding for adaptation:\n"
    base += rag_block
    return base

@torch.no_grad()
def get_dt_batch(pil_imgs, metas, feats_eval_cpu):
    # CNN probs used only if retrieval schema includes cnn_* columns
    probs = cnn_probs_from_feats(feats_eval_cpu).cpu().numpy() if USE_CNN_IN_RETRIEVAL else None

    dt_list = []
    miss = []
    ids = [m["image"] for m in metas]

    for i, img_id in enumerate(ids):
        v = DT_CACHE.get(img_id)
        if v is None:
            miss.append(i)
            dt_list.append(None)
        else:
            dt_list.append(v)

    if len(miss) > 0:
        prompts = []
        miss_ids = []
        for i in miss:
            qf = quality_features(pil_imgs[i])

            # Build query vec in EXACT schema: q_cols + cnn_cols_all (if enabled)
            parts = []
            if len(q_cols) > 0:
                # q_cols are ordered, but our qf has fixed order matching those names:
                # [q_mean,q_std,q_entropy,q_sharp,lq_edge_d,q_clip_low,q_clip_high]
                qmap = {
                    "q_mean": qf[0], "q_std": qf[1], "q_entropy": qf[2],
                    "q_sharp": qf[3], "lq_edge_d": qf[4],
                    "q_clip_low": qf[5], "q_clip_high": qf[6]
                }
                parts.append(np.array([qmap[c] for c in q_cols], dtype=np.float32))

            if USE_CNN_IN_RETRIEVAL and probs is not None:
                # RAG expects all cnn_* columns in the same order as in use_cols
                # We don't know their semantic mapping; safest is:
                # - If count matches C, use probs directly (pad/trim anyway)
                parts.append(probs[i].astype(np.float32))

            qvec = np.concatenate(parts, axis=0) if len(parts) > 1 else parts[0]
            qvec = align_dim(qvec, RAG_DIM)  # ✅ FINAL SAFETY

            rag_texts = retrieve_topk_prompts(qvec, topk=RAG_TOPK)
            pr = build_target_prompt(metas[i], qf, rag_texts, cnn_probs=(probs[i] if probs is not None else None))
            prompts.append(pr)
            miss_ids.append(ids[i])

        dt_new = embed_dt(prompts)
        for k, img_id in enumerate(miss_ids):
            DT_CACHE.put(img_id, dt_new[k:k+1])
        for i in miss:
            dt_list[i] = DT_CACHE.get(ids[i])

    dt = torch.cat(dt_list, dim=0)
    return dt

# ============================================================
# Qiskit QNN (EstimatorV2)
# ============================================================
try:
    import qiskit
    import qiskit_machine_learning
except Exception:
    _pip_install("qiskit")
    _pip_install("qiskit-machine-learning")
    import qiskit
    import qiskit_machine_learning

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

def make_estimator_v2():
    try:
        from qiskit_aer.primitives import EstimatorV2
        est = EstimatorV2()
        print("✅ Using V2 Estimator: qiskit_aer.primitives.EstimatorV2")
        return est
    except Exception:
        pass
    try:
        from qiskit.primitives import StatevectorEstimator
        est = StatevectorEstimator()
        sig = inspect.signature(est.run)
        if "precision" in sig.parameters:
            print("✅ Using V2 Estimator: qiskit.primitives.StatevectorEstimator")
            return est
    except Exception:
        pass
    raise RuntimeError("No EstimatorV2 found. Install/enable qiskit-aer.")

ESTIMATOR = make_estimator_v2()
PI = float(math.pi)

def build_qnn(n_qubits, n_ent_layers):
    q_in_dim = 2*n_qubits + n_qubits + n_ent_layers + 1
    x = ParameterVector("x", q_in_dim)

    n_var = 2*n_qubits
    n_ent = 2*n_ent_layers
    theta = ParameterVector("θ", n_var + n_ent)

    qc = QuantumCircuit(n_qubits)

    for i in range(n_qubits):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    dt_off    = 2*n_qubits
    depth_off = dt_off + n_qubits
    topo_idx  = depth_off + n_ent_layers

    for i in range(n_qubits):
        qc.crz(x[dt_off + i], i, (i+1) % n_qubits)

    t = 0
    for i in range(n_qubits):
        qc.rz(theta[t], i); t += 1
        qc.ry(theta[t], i); t += 1

    for l in range(n_ent_layers):
        g_depth = x[depth_off + l]
        g_topo  = x[topo_idx]
        g_chain = g_depth * (1 - g_topo)
        g_ring  = g_depth * (g_topo)

        th_chain = theta[t]; t += 1
        th_ring  = theta[t]; t += 1
        for i in range(max(1, n_qubits-1)):
            qc.crz(g_chain * th_chain, i, i+1)
        for i in range(n_qubits):
            qc.crz(g_ring * th_ring, i, (i+1) % n_qubits)

    qc_t = transpile(qc, optimization_level=0, seed_transpiler=SEED)

    observables = []
    if QNN_NUM_OBS == 1:
        z = ["I"] * n_qubits
        z[0] = "Z"
        observables = [SparsePauliOp.from_list([("".join(z), 1.0)])]
    else:
        for qi in range(min(QNN_NUM_OBS, n_qubits)):
            z = ["I"] * n_qubits
            z[qi] = "Z"
            observables.append(SparsePauliOp.from_list([("".join(z), 1.0)]))

    qnn = EstimatorQNN(
        circuit=qc_t,
        estimator=ESTIMATOR,
        observables=observables,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=False,
    )
    return qnn, q_in_dim, len(observables)

def fixed_proj(dt_cpu, out_dim, seed):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(dt_cpu.shape[1], out_dim, generator=g) / math.sqrt(dt_cpu.shape[1])
    return dt_cpu @ W

def parse_dt(dt_cpu, n_qubits, n_ent_layers):
    dt_angles = torch.tanh(fixed_proj(dt_cpu, n_qubits, seed=1001+n_qubits)) * PI
    depth_g   = torch.sigmoid(fixed_proj(dt_cpu, n_ent_layers, seed=1002+n_qubits))
    topo      = torch.sigmoid(fixed_proj(dt_cpu, 1, seed=1003+n_qubits))
    alpha     = torch.sigmoid(fixed_proj(dt_cpu, 1, seed=1004+n_qubits))

    lam_raw = fixed_proj(dt_cpu, 2, seed=1005+n_qubits)
    lam1 = 0.5 + 1.5 * torch.sigmoid(lam_raw[:, 0:1])
    lam2 = 0.05 + 0.45 * torch.sigmoid(lam_raw[:, 1:2])
    return dt_angles.float(), depth_g.float(), topo.float(), alpha.float(), lam1.float(), lam2.float()

def make_q2logit(q_out_dim, C, seed=2026):
    g = torch.Generator(device="cpu").manual_seed(seed + 997*q_out_dim + 17*C)
    W = torch.randn(q_out_dim, C, generator=g) / math.sqrt(q_out_dim)
    return W.float()

class QTTA_LogitAdapter(nn.Module):
    def __init__(self, qnn_torch, q2logit, n_qubits, n_ent_layers, q_src_mean):
        super().__init__()
        self.qnn = qnn_torch
        self.W   = q2logit
        self.nq  = n_qubits
        self.nl  = n_ent_layers
        self.q_src_mean = q_src_mean

    def forward_logits(self, feats_cpu, dt_cpu):
        base_logits = cnn_logits_from_feats(feats_cpu)

        feat_small = feats_cpu[:, : (2*self.nq)].contiguous()
        feat_angles = torch.tanh(feat_small) * PI

        dt_angles, depth_g, topo, alpha, lam1, lam2 = parse_dt(dt_cpu, self.nq, self.nl)

        q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
        q_out = self.qnn(q_in).to(torch.float32)

        q_center = q_out - self.q_src_mean.unsqueeze(0)
        delta_logits = (q_center @ self.W)
        logits = base_logits + (alpha * PQC_LOGIT_SCALE) * delta_logits
        return logits, base_logits, q_out, (lam1, lam2), alpha

def binary_entropy_from_logits(logits):
    p = torch.sigmoid(logits)
    eps = 1e-6
    ent = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps))
    return ent.mean()

def cosine_fidelity_loss(q_out, q_src_mean):
    q = F.normalize(q_out, dim=1)
    s = F.normalize(q_src_mean.unsqueeze(0).expand_as(q), dim=1)
    cos = (q * s).sum(dim=1)
    return (1.0 - cos).mean()

@torch.no_grad()
def init_q_src_mean(adapter_tmp, n_batches=20):
    qs = []
    it = iter(stream_loader)
    for _ in tqdm(range(n_batches), desc="Init q_src_mean (unlabeled stream)"):
        try:
            pil_imgs, _, metas = next(it)
        except StopIteration:
            it = iter(stream_loader)
            pil_imgs, _, metas = next(it)

        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        dt = get_dt_batch(pil_imgs, metas, feats)
        _, _, q_out, _, _ = adapter_tmp.forward_logits(feats, dt)
        qs.append(q_out.detach().cpu())

    q_all = torch.cat(qs, dim=0)
    return q_all.mean(dim=0).float()

def compute_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        rows.append([lbl, aucv, apv, f1v])
    df_pc = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    return {"per_class": df_pc, "macro_roc_auc": float(np.nanmean(df_pc["ROC_AUC"].values))}

def make_adapter(nq):
    qnn, q_in_dim, q_out_dim = build_qnn(nq, N_ENT_LAYERS)
    qnn_torch = TorchConnector(qnn).to("cpu")
    q2logit = make_q2logit(q_out_dim, C).to("cpu")
    tmp_mean = torch.zeros((q_out_dim,), dtype=torch.float32)
    adapter = QTTA_LogitAdapter(qnn_torch, q2logit, nq, N_ENT_LAYERS, tmp_mean).to("cpu")
    for p in adapter.parameters(): p.requires_grad_(False)
    for p in adapter.qnn.parameters(): p.requires_grad_(True)
    return adapter, q_in_dim, q_out_dim

@torch.no_grad()
def eval_cnn_only():
    Ps, Ys = [], []
    t0 = time.time()
    nimg = 0
    for pil_imgs, y, _ in tqdm(test_loader, desc="EVAL CNN-only"):
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        p = cnn_probs_from_feats(feats).cpu().numpy()
        Ps.append(p); Ys.append(y.numpy())
        nimg += len(pil_imgs)
    dt = time.time() - t0
    P = np.vstack(Ps); Y = np.vstack(Ys)
    m = compute_metrics(Y, P, UNIFIED_LABELS)
    print(f"✅ CNN-only done | images={nimg} | seconds={dt:.2f} | imgs/s={nimg/dt:.2f}")
    print(m["per_class"])
    print("Macro AUC:", m["macro_roc_auc"])
    return m

def run_ttda(adapter, nq, strategy_name="QTTA_TENT+FID"):
    adapter.eval()
    q_src_mean = init_q_src_mean(adapter, n_batches=INIT_QSTATS_BATCHES).to("cpu")
    adapter.q_src_mean = q_src_mean

    opt = torch.optim.Adam(adapter.qnn.parameters(), lr=LR_TTDA)

    Ps, Ys = [], []
    logs = []
    total_adapt = 0
    nimg = 0

    t0 = time.time()
    for bi, (pil_imgs, y, metas) in enumerate(tqdm(test_loader, desc=f"TEST [{strategy_name}] Q{nq}")):
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        feats = extract_features(x).detach().cpu().float()
        dt_batch = get_dt_batch(pil_imgs, metas, feats)

        with torch.no_grad():
            logits_before, _, _, _, _ = adapter.forward_logits(feats, dt_batch)
            p_before = torch.sigmoid(logits_before).detach()

        if ADAPT_STEPS_PER_TEST_BATCH > 0 and (bi % max(1, ADAPT_EVERY_K_BATCHES) == 0) and (total_adapt < MAX_TOTAL_ADAPT_STEPS):
            steps_here = min(ADAPT_STEPS_PER_TEST_BATCH, MAX_TOTAL_ADAPT_STEPS - total_adapt)
            for _ in range(steps_here):
                adapter.train()
                logits, _, q_out, (lam1, lam2), _ = adapter.forward_logits(feats, dt_batch)

                L_ent = binary_entropy_from_logits(logits)
                L_fid = cosine_fidelity_loss(q_out, q_src_mean)
                L = lam1.mean()*L_ent + lam2.mean()*L_fid

                opt.zero_grad(set_to_none=True)
                L.backward()
                if GRAD_CLIP is not None:
                    nn.utils.clip_grad_norm_(adapter.qnn.parameters(), GRAD_CLIP)
                opt.step()

                logs.append({
                    "step": total_adapt,
                    "L_total": float(L.detach().cpu()),
                    "L_ent": float(L_ent.detach().cpu()),
                    "L_fid": float(L_fid.detach().cpu()),
                    "lam1": float(lam1.mean().detach().cpu()),
                    "lam2": float(lam2.mean().detach().cpu()),
                })
                total_adapt += 1

        adapter.eval()
        with torch.no_grad():
            logits1, _, _, _, _ = adapter.forward_logits(feats, dt_batch)
            probs_views = [torch.sigmoid(logits1).detach()]

            if USE_TTA and TTA_VIEWS >= 2:
                pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
                x2 = batch_apply_tf(pil_flip, eval_tf).to(DEVICE)
                feats2 = extract_features(x2).detach().cpu().float()
                logits2, _, _, _, _ = adapter.forward_logits(feats2, dt_batch)
                probs_views.append(torch.sigmoid(logits2).detach())

            if USE_TTA and TTA_VIEWS >= 3 and USE_AUTOAUG_TTA and autoaug is not None:
                x3 = batch_apply_tf(pil_imgs, autoaug_tf).to(DEVICE)
                feats3 = extract_features(x3).detach().cpu().float()
                logits3, _, _, _, _ = adapter.forward_logits(feats3, dt_batch)
                probs_views.append(torch.sigmoid(logits3).detach())

            p_after = torch.stack(probs_views, dim=0).mean(dim=0)

        if bi in [0, 10, 50] and total_adapt > 0:
            delta = float((p_after - p_before).abs().mean().cpu())
            print(f"🔎 pred-change@batch{bi}: mean|Δp|={delta:.6f}")

        Ps.append(p_after.cpu().numpy())
        Ys.append(y.numpy())
        nimg += len(pil_imgs)

    dt = time.time() - t0
    P = np.vstack(Ps); Y = np.vstack(Ys)
    m = compute_metrics(Y, P, UNIFIED_LABELS)

    print(f"\n✅ {strategy_name} done | Q={nq} | images={nimg} | seconds={dt:.2f} | imgs/s={nimg/dt:.2f}")
    print(m["per_class"])
    print("Macro AUC:", m["macro_roc_auc"])
    print("Total TTDA steps:", total_adapt)

    out_dir = os.path.join(OUTDIR, f"Q{nq}")
    os.makedirs(out_dir, exist_ok=True)
    m["per_class"].to_csv(os.path.join(out_dir, f"per_class_{strategy_name}.csv"), index=False)
    if len(logs) > 0:
        pd.DataFrame(logs).to_csv(os.path.join(out_dir, f"adapt_logs_{strategy_name}.csv"), index=False)

    return m, logs

# ----------------------------
# MAIN
# ----------------------------
print("\n==============================")
print("1) CNN-only baseline")
print("==============================")
base_m = eval_cnn_only()

ALL_SUM = []
for nq in QUBIT_LIST:
    print("\n" + "="*70)
    print(f"2) QTTA/TTDA with PQC adapter | Q={nq}")
    print("="*70)

    adapter, q_in_dim, q_out_dim = make_adapter(nq)
    w0 = adapter.qnn.weight.detach().clone()

    m_ttda, logs = run_ttda(adapter, nq, strategy_name="QTTA_TENT+FID")

    w1 = adapter.qnn.weight.detach().clone()
    wchg = float((w1 - w0).abs().mean().cpu())
    print(f"🔧 mean|Δθ| after TTDA (Q{nq}): {wchg:.8f}")

    ALL_SUM.append({
        "Qubits": nq,
        "CNN_macro_auc": base_m["macro_roc_auc"],
        "QTTA_macro_auc": m_ttda["macro_roc_auc"],
        "TTDA_steps": len(logs),
        "mean_abs_dtheta": wchg,
        "PQC_LOGIT_SCALE": PQC_LOGIT_SCALE,
        "RAG_dim": RAG_DIM,
        "USE_CNN_IN_RETRIEVAL": USE_CNN_IN_RETRIEVAL,
    })

df = pd.DataFrame(ALL_SUM)
df.to_csv(os.path.join(OUTDIR, "summary.csv"), index=False)
print("\n✅ Saved summary:", os.path.join(OUTDIR, "summary.csv"))
print(df)


DEVICE: cuda
✅ /kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt
✅ /kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt
✅ /kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt
✅ /kaggle/input/rag-src/rag_chex_train_documents.csv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.0 MB/s eta 0:00:00
NIH test: 25596 | NIH stream(unlabeled): 86524
✅ Loaded ResNet .pt (strict=False) | missing: 0 unexpected: 0
✅ Using 1-layer classifier from ckpt fc.*
✅ RAG docs loaded: 30000 | USE_CNN_IN_RETRIEVAL=T

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2026-02-06 11:20:54.152644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770376854.352631      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770376854.406601      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770376854.891454      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770376854.891481      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770376854.891483      55 computation_placer.cc:177] computation placer alr

...

TEST [QTTA_TENT+FID] Q8:   0%|          | 1/1600 [00:23<10:23:07, 23.38s/it]

🔎 pred-change@batch0: mean|Δp|=0.045072


TEST [QTTA_TENT+FID] Q8:   1%|          | 11/1600 [02:20<6:04:36, 13.77s/it]

🔎 pred-change@batch10: mean|Δp|=0.052600


TEST [QTTA_TENT+FID] Q8:   1%|▏         | 20/1600 [03:56<4:22:21,  9.96s/it]

#The code adapts a CheXpert-trained chest X-ray model to NIH images without labels. ResNet-50 gives initial predictions, while AdaBN updates image statistics, TENT reduces uncertainty, CoTTA-lite enforces stable predictions, Wasserstein aligns domains, and a quantum adapter uses RAG, BioMedBERT, teacher probabilities, distillation, fidelity, and warm-up to improve AUC safely overall.


In [3]:
# ============================================================
# TEST-ONLY DOMAIN ADAPTATION (CHEXPERT -> NIH)
# ✅ Classical TTDA ablation: AdaBN / TENT / CoTTA-lite / Wasserstein(+Ent) (if source-ref provided)
# ✅ PQC TTDA ablation: PQC(no-adapt) / PQC+LLM-distill / PQC+LLM-distill+Fidelity / PQC+LLM-distill+CoTTA-lite / PQC+LLM-distill+Wass (if source-ref)
# ✅ Uses your CSV (like screenshot) to pull per-image targets:
#     - y_*   (LLM label / pseudo-label / soft label)
#     - cnn_* (teacher probs)
# ✅ The key AUC fix: distillation loss drives ranking, PQC head starts neutral (zero-init)
# ============================================================

import os, math, copy, time, random, warnings, re, inspect
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------
# CONFIG (EDIT)
# ----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# ---- Your checkpoints (trained on CheXpert) ----
RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

# ---- NIH target ----
NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

# ---- Your CSVs like screenshot ----
# 1) Source doc CSV for RAG prompts (CheXpert)
RAG_SRC_CSV = "/kaggle/input/rag-src/rag_chex_train_documents.csv"

# 2) Target CSV (NIH) containing per-image y_* and cnn_* columns (the one you said you also have)
#    MUST contain at least: image column + cnn_* OR y_* columns
TARGET_TEACHER_CSV = "/kaggle/input/text-descriptions/nih_test_features.csv"  # <-- CHANGE THIS

# Optional: Source reference for Wasserstein (CheXpert subset paths)
CHEX_REF_IMG   = ""   # e.g. "/kaggle/input/chexpert-small/CheXpert-v1.0-small"
CHEX_REF_LIST  = ""   # e.g. "/kaggle/input/chexpert-ref/paths.txt"
CHEX_REF_MAX   = 6000

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

# ---- LLM DT encoder (BioMedBERT) ----
TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_PROMPT_LEN  = 192
USE_LLM_DT = True

# ---- Teacher blend (what actually moves AUC) ----
USE_CSV_TEACHER = True   # must be True to use your y_* / cnn_* columns
TEACH_W_LLM      = 0.65  # blend weight for y_* (LLM) vs cnn_*
LLM_CONF_THR     = 0.20  # only trust LLM y_* when |y-0.5| > thr
BETA_DISTILL     = 1.0   # weight of distillation BCE loss

# ---- Classical TTDA ----
ADAPT_PROTOCOL = "stream_then_test"  # "stream_then_test" recommended
MAX_ADAPT_BATCHES = None             # set e.g. 400 for faster dev
MAX_TEST_BATCHES  = None             # set e.g. 200 for faster dev

TENT_LR = 1e-4
COTTA_LR = 1e-4
TENT_STEPS_PER_BATCH = 1
COTTA_STEPS_PER_BATCH = 1
ENT_THR = 0.55
USE_RELIABLE_FILTER = True

WASS_LR = 1e-4
WASS_WEIGHT_CLASSICAL = 0.02  # only if source stats exist

USE_TTA = True  # flip + (optional) autoaug

# ---- PQC ----
RUN_PQC = True
QUBIT_LIST = [6, 8]
N_ENT_LAYERS = 2
QNN_NUM_OBS = 1
PQC_LOGIT_SCALE = 0.20          # reduced for stability
LR_PQC = 3e-4
GRAD_CLIP = 1.0
MAX_TOTAL_PQC_STEPS = 600       # PQC is slow
PQC_WARMUP_STEPS = 250          # distill head to mimic base CNN first (important)
PQC_FID_WEIGHT = 0.15
PQC_WASS_WEIGHT = 0.02          # if source stats exist

OUTDIR = "ABLAT_PDF_MATCH_TEACHER"
os.makedirs(OUTDIR, exist_ok=True)

for p in [RESNET_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM, RAG_SRC_CSV, TARGET_TEACHER_CSV]:
    print(("✅" if os.path.exists(p) else "❌"), p)

# ----------------------------
# Helpers
# ----------------------------
def _pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(f"NIH image not found: {p}")
    return p

def build_nih_df(split_list_txt, nih_csv_path):
    wanted = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")
    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    df["patient_id"] = df["Patient ID"].astype(str) if "Patient ID" in df.columns else "unknown"
    df["age"]        = df["Patient Age"] if "Patient Age" in df.columns else np.nan
    df["gender"]     = df["Patient Gender"] if "Patient Gender" in df.columns else "U"
    df["view"]       = df["View Position"] if "View Position" in df.columns else "UNK"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","patient_id","age","gender","view"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

class CXRDatasetPIL(Dataset):
    def __init__(self, df, img_resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.images = self.df["image"].tolist()
        self.labeled = labeled
        if labeled:
            self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "image_key": os.path.basename(str(self.images[idx])),
            "patient_id": str(self.df.loc[idx, "patient_id"]),
            "age": self.df.loc[idx, "age"],
            "gender": str(self.df.loc[idx, "gender"]),
            "view": str(self.df.loc[idx, "view"]),
        }
        if self.labeled:
            y = torch.from_numpy(self.labels[idx])
        else:
            y = torch.zeros(C, dtype=torch.float32)
        return img, y, meta

def pil_collate_fn(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys), dim=0), list(metas)

nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDatasetPIL(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)
stream_loader = DataLoader(
    CXRDatasetPIL(nih_stream_df, nih_resolver, labeled=False),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=False, collate_fn=pil_collate_fn
)

print("NIH test:", len(nih_test_df), "| NIH stream(unlabeled):", len(nih_stream_df))

# ----------------------------
# Preprocessing
# ----------------------------
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
    autoaug_tf = T.Compose([T.Resize(256), T.CenterCrop(224), autoaug, T.ToTensor(), norm])
except Exception:
    autoaug_tf = None
    print("⚠ AutoAugment not available; TTA will use flip only.")

def batch_apply_tf(pil_list, tf):
    return torch.stack([tf(im) for im in pil_list], dim=0)

# ----------------------------
# CKPT loading utilities
# ----------------------------
def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def load_ckpt_state_dict(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:     sd = ckpt["model"]
    elif isinstance(ckpt, dict) and "net" in ckpt:       sd = ckpt["net"]
    else:                                                sd = ckpt
    return _strip_prefix(sd)

# ----------------------------
# ResNet wrapper (logits + feats)
# ----------------------------
class ResNet50WithFeat(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*(list(m.children())[:-1]))
        self.fc = nn.Linear(2048, num_classes)
    def forward(self, x, return_feats=False):
        feats = self.backbone(x).flatten(1)
        logits = self.fc(feats)
        if return_feats:
            return logits, feats
        return logits

base = ResNet50WithFeat(C)
resnet_sd = load_ckpt_state_dict(RESNET_CKPT)
missing, unexpected = base.load_state_dict(resnet_sd, strict=False)
print("✅ Loaded ResNet .pt (strict=False) | missing:", len(missing), "unexpected:", len(unexpected))

base = base.to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)

@torch.no_grad()
def forward_logits_feats(model, pil_imgs, tf):
    x = batch_apply_tf(pil_imgs, tf).to(DEVICE)
    logits, feats = model(x, return_feats=True)
    return logits, feats

# ----------------------------
# Metrics
# ----------------------------
def compute_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        rows.append([lbl, aucv, apv, f1v])
    df_pc = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    return {"per_class": df_pc, "macro_roc_auc": float(np.nanmean(df_pc["ROC_AUC"].values))}

@torch.no_grad()
def eval_logits_fn(name, logits_fn, use_tta=True):
    Ps, Ys = [], []
    nimg = 0
    t0 = time.time()
    for bi, (pil_imgs, y, metas) in enumerate(tqdm(test_loader, desc=f"EVAL [{name}]")):
        if MAX_TEST_BATCHES is not None and bi >= MAX_TEST_BATCHES: break

        logits1 = logits_fn(pil_imgs, metas, view="orig")
        probs = [torch.sigmoid(logits1).detach().cpu()]

        if use_tta:
            pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
            logits2 = logits_fn(pil_flip, metas, view="flip")
            probs.append(torch.sigmoid(logits2).detach().cpu())

            if autoaug_tf is not None:
                logits3 = logits_fn(pil_imgs, metas, view="autoaug")
                probs.append(torch.sigmoid(logits3).detach().cpu())

        p = torch.stack(probs, dim=0).mean(dim=0).numpy()
        Ps.append(p); Ys.append(y.numpy())
        nimg += len(pil_imgs)

    dt = time.time() - t0
    P = np.vstack(Ps); Y = np.vstack(Ys)
    m = compute_metrics(Y, P, UNIFIED_LABELS)
    print(f"\n✅ {name} | images={nimg} | sec={dt:.2f} | imgs/s={nimg/dt:.2f} | MacroAUC={m['macro_roc_auc']:.6f}")
    print(m["per_class"])
    return m

# ============================================================
# A) LOAD YOUR TARGET TEACHER CSV (y_* and cnn_*)
# ============================================================
teacher_y = {}
teacher_cnn = {}
HAS_TEACHER = False

def _norm_col(s): return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def find_best_col(cols, prefix, label):
    """
    Find column like:
      y_Atelectasis, y_Atelect, y_atelectasis
      cnn_Atelectasis, cnn_Atele, etc.
    """
    cols_norm = [_norm_col(c) for c in cols]
    pref = _norm_col(prefix)
    labn = _norm_col(label).replace("pleuraleffusion","effusion")  # allow effusion naming
    # rank by contains both prefix and label token
    best = None
    best_score = -1
    for c, cn in zip(cols, cols_norm):
        score = 0
        if cn.startswith(pref): score += 2
        if labn in cn: score += 3
        if "effusion" in labn and "effusion" in cn: score += 3
        if score > best_score:
            best_score = score
            best = c
    return best if best_score >= 3 else None

if USE_CSV_TEACHER and os.path.exists(TARGET_TEACHER_CSV):
    tdf = pd.read_csv(TARGET_TEACHER_CSV)

    # robust image key column
    # prefer 'image' if exists else any column containing 'image'
    img_col = "image" if "image" in tdf.columns else None
    if img_col is None:
        for c in tdf.columns:
            if "image" in str(c).lower():
                img_col = c; break
    if img_col is None:
        raise ValueError("TARGET_TEACHER_CSV must contain an image column (e.g., 'image').")

    cols = list(tdf.columns)

    y_cols_map = {}
    cnn_cols_map = {}
    for lbl in UNIFIED_LABELS:
        yc = find_best_col(cols, "y_", lbl)
        cc = find_best_col(cols, "cnn_", lbl)
        y_cols_map[lbl] = yc
        cnn_cols_map[lbl] = cc

    print("✅ Teacher column mapping:")
    print("y_ map:", y_cols_map)
    print("cnn map:", cnn_cols_map)

    have_any = any(v is not None for v in y_cols_map.values()) or any(v is not None for v in cnn_cols_map.values())
    if not have_any:
        raise ValueError("Could not find any y_* or cnn_* columns in TARGET_TEACHER_CSV.")

    # build lookups
    tdf["image_key"] = tdf[img_col].astype(str).apply(lambda s: os.path.basename(s))

    for _, r in tqdm(tdf.iterrows(), total=len(tdf), desc="Building teacher lookup"):
        k = r["image_key"]
        yv = []
        cv = []
        y_ok = True
        c_ok = True
        for lbl in UNIFIED_LABELS:
            yc = y_cols_map[lbl]
            cc = cnn_cols_map[lbl]
            if yc is None:
                y_ok = False
            else:
                yv.append(float(r[yc]))
            if cc is None:
                c_ok = False
            else:
                cv.append(float(r[cc]))

        if y_ok:
            teacher_y[k] = np.array(yv, dtype=np.float32)
        if c_ok:
            teacher_cnn[k] = np.array(cv, dtype=np.float32)

    HAS_TEACHER = (len(teacher_y) > 0) or (len(teacher_cnn) > 0)
    print(f"✅ Teacher ready | y_entries={len(teacher_y)} | cnn_entries={len(teacher_cnn)} | HAS_TEACHER={HAS_TEACHER}")
else:
    print("⚠ No TARGET_TEACHER_CSV found or disabled. Distillation will be OFF.")

def get_teacher_batch(metas, device="cpu"):
    """
    Returns:
      t_final: (B,C) tensor in [0,1] if available, else None
    """
    if not HAS_TEACHER:
        return None

    ys, cs = [], []
    has_y = True
    has_c = True

    for m in metas:
        k = m.get("image_key", os.path.basename(str(m["image"])))
        yv = teacher_y.get(k, None)
        cv = teacher_cnn.get(k, None)
        if yv is None: has_y = False
        if cv is None: has_c = False
        ys.append(yv)
        cs.append(cv)

    # build per-sample with fallback
    out = []
    for i in range(len(metas)):
        yv = ys[i]
        cv = cs[i]
        if (yv is None) and (cv is None):
            return None  # no teacher for this batch
        if yv is None:
            out.append(cv)
        elif cv is None:
            out.append(yv)
        else:
            # confidence-gated blend
            y_t = yv
            c_t = cv
            conf = np.abs(y_t - 0.5)
            mask = (conf > LLM_CONF_THR).astype(np.float32)
            blend = TEACH_W_LLM * y_t + (1 - TEACH_W_LLM) * c_t
            final = mask * blend + (1 - mask) * c_t
            out.append(final)

    t = torch.from_numpy(np.stack(out, axis=0)).to(device).float().clamp(1e-4, 1-1e-4)
    return t

# ============================================================
# B) OPTIONAL: SOURCE REF STATS FOR WASSERSTEIN
# ============================================================
src_mu = src_var = None
if CHEX_REF_IMG and CHEX_REF_LIST and os.path.exists(CHEX_REF_IMG) and os.path.exists(CHEX_REF_LIST):
    def chex_resolver(rel):
        p = os.path.join(CHEX_REF_IMG, rel)
        if not os.path.exists(p): raise FileNotFoundError(p)
        return p

    ref_paths = read_txt_lines(CHEX_REF_LIST)[:CHEX_REF_MAX]
    ref_df = pd.DataFrame({"image": ref_paths})
    for lbl in UNIFIED_LABELS: ref_df[lbl] = 0
    ref_df["patient_id"] = "src"; ref_df["age"] = np.nan; ref_df["gender"] = "U"; ref_df["view"] = "UNK"

    src_loader = DataLoader(
        CXRDatasetPIL(ref_df, chex_resolver, labeled=False),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=False, collate_fn=pil_collate_fn
    )

    feats_all = []
    for bi, (pil_imgs, _, _) in enumerate(tqdm(src_loader, desc="Compute source feature stats")):
        if bi >= 200: break
        _, feats = forward_logits_feats(base, pil_imgs, eval_tf)
        feats_all.append(feats.detach().cpu())
    Fcat = torch.cat(feats_all, dim=0)
    src_mu = Fcat.mean(dim=0).float()
    src_var = (Fcat.var(dim=0, unbiased=False) + 1e-6).float()
    print("✅ Source stats computed for Wasserstein.")
else:
    print("ℹ️ No source ref provided: Wasserstein methods will be skipped.")

def wasserstein_diag_gaussian(mu_s, var_s, mu_t, var_t):
    sig_s = torch.sqrt(var_s); sig_t = torch.sqrt(var_t)
    return ((mu_s - mu_t)**2).mean() + ((sig_s - sig_t)**2).mean()

# ============================================================
# C) CLASSICAL TTDA (AdaBN / TENT / CoTTA / WASS) + DISTILL
# ============================================================
def is_bn(m): return isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d))

def collect_bn_affine_params(model):
    params = []
    for m in model.modules():
        if is_bn(m) and m.affine:
            if m.weight is not None: params.append(m.weight)
            if m.bias is not None:   params.append(m.bias)
    return params

def set_bn_mode(model, track_running_stats=True):
    for m in model.modules():
        if is_bn(m):
            m.train()
            m.track_running_stats = track_running_stats
            if not track_running_stats:
                m.running_mean = None
                m.running_var  = None

def entropy_multilabel_from_logits(logits):
    p = torch.sigmoid(logits)
    eps = 1e-6
    ent = -(p*torch.log(p+eps) + (1-p)*torch.log(1-p+eps))  # (B,C)
    return ent.mean(dim=1)  # (B,)

class FeatureBuf:
    def __init__(self, max_items=256):
        self.max_items = max_items
        self.buf = []
    def too_similar(self, feats, thr=0.98):
        if len(self.buf) == 0:
            return torch.zeros((feats.size(0),), dtype=torch.bool)
        mem = torch.stack(self.buf, dim=0)
        mem = F.normalize(mem, dim=1)
        f = F.normalize(feats.detach().cpu(), dim=1)
        sims = f @ mem.t()
        return (sims.max(dim=1).values > thr)
    def add(self, feats):
        feats = feats.detach().cpu()
        for i in range(feats.size(0)):
            self.buf.append(feats[i])
        if len(self.buf) > self.max_items:
            self.buf = self.buf[-self.max_items:]

def adabn_adapt(model, loader, max_batches=None):
    model = model.to(DEVICE)
    set_bn_mode(model, track_running_stats=True)
    with torch.no_grad():
        for bi, (pil_imgs, _, _) in enumerate(tqdm(loader, desc="AdaBN adapt")):
            if max_batches is not None and bi >= max_batches: break
            x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
            _ = model(x)  # updates BN stats
    model.eval()
    return model

def tent_adapt(model, loader, lr=1e-4, steps_per_batch=1, max_batches=None):
    model = model.to(DEVICE)
    set_bn_mode(model, track_running_stats=False)
    params = collect_bn_affine_params(model)
    opt = torch.optim.Adam(params, lr=lr) if len(params) else None
    buf = FeatureBuf(256)

    for bi, (pil_imgs, _, metas) in enumerate(tqdm(loader, desc="TENT(+distill) adapt")):
        if max_batches is not None and bi >= max_batches: break
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)

        for _ in range(steps_per_batch):
            logits, feats = model(x, return_feats=True)
            ent = entropy_multilabel_from_logits(logits)  # (B,)

            mask = torch.ones_like(ent, dtype=torch.bool)
            if USE_RELIABLE_FILTER:
                mask = mask & (ent < ENT_THR)
                mask = mask & (~buf.too_similar(feats, thr=0.98).to(ent.device))

            # distill teacher (if available)
            t = get_teacher_batch(metas, device=logits.device)
            distill = None
            if t is not None:
                p = torch.sigmoid(logits).clamp(1e-4, 1-1e-4)
                distill = F.binary_cross_entropy(p, t)

            if mask.sum() < 2 and distill is None:
                buf.add(feats)
                continue

            loss = 0.0
            if mask.sum() >= 2:
                loss = loss + ent[mask].mean()
            if distill is not None:
                loss = loss + (BETA_DISTILL * distill)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            buf.add(feats)

    model.eval()
    return model

def cotta_adapt(model, loader, lr=1e-4, steps_per_batch=1, max_batches=None, ema=0.999, restore_prob=0.01):
    model = model.to(DEVICE)
    set_bn_mode(model, track_running_stats=False)

    init_sd = copy.deepcopy(model.state_dict())
    teacher = copy.deepcopy(model).eval()

    params = collect_bn_affine_params(model)
    opt = torch.optim.Adam(params, lr=lr) if len(params) else None

    for bi, (pil_imgs, _, metas) in enumerate(tqdm(loader, desc="CoTTA-lite(+distill) adapt")):
        if max_batches is not None and bi >= max_batches: break

        xw = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)
        pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
        xs = batch_apply_tf(pil_flip, eval_tf).to(DEVICE)

        with torch.no_grad():
            t_logits = teacher(xw)
            t_prob = torch.sigmoid(t_logits)

        for _ in range(steps_per_batch):
            s_logits = model(xs)
            s_prob = torch.sigmoid(s_logits)

            cons = F.mse_loss(s_prob, t_prob)
            ent = entropy_multilabel_from_logits(s_logits).mean()

            # distill teacher (CSV)
            t_csv = get_teacher_batch(metas, device=s_logits.device)
            distill = None
            if t_csv is not None:
                distill = F.binary_cross_entropy(s_prob.clamp(1e-4, 1-1e-4), t_csv)

            loss = cons + ent
            if distill is not None:
                loss = loss + (BETA_DISTILL * distill)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            # stochastic restore
            if random.random() < restore_prob:
                cur = model.state_dict()
                for k in cur.keys():
                    if ("bn" in k or "backbone" in k) and (k.endswith("weight") or k.endswith("bias")):
                        cur[k] = init_sd[k]
                model.load_state_dict(cur, strict=False)

            # EMA teacher
            with torch.no_grad():
                msd = model.state_dict()
                tsd = teacher.state_dict()
                for k in tsd.keys():
                    tsd[k].mul_(ema).add_(msd[k].detach() * (1 - ema))
                teacher.load_state_dict(tsd, strict=True)

    model.eval()
    return model

def wass_adapt(model, loader, src_mu, src_var, lr=1e-4, steps_per_batch=1, max_batches=None, w_weight=0.02):
    model = model.to(DEVICE)
    set_bn_mode(model, track_running_stats=False)
    params = collect_bn_affine_params(model)
    opt = torch.optim.Adam(params, lr=lr) if len(params) else None

    mu_s = src_mu.to(DEVICE)
    var_s = src_var.to(DEVICE)

    for bi, (pil_imgs, _, metas) in enumerate(tqdm(loader, desc="Wass(+distill) adapt")):
        if max_batches is not None and bi >= max_batches: break
        x = batch_apply_tf(pil_imgs, eval_tf).to(DEVICE)

        for _ in range(steps_per_batch):
            logits, feats = model(x, return_feats=True)
            ent = entropy_multilabel_from_logits(logits).mean()

            mu_t = feats.mean(dim=0)
            var_t = feats.var(dim=0, unbiased=False) + 1e-6
            wass = wasserstein_diag_gaussian(mu_s, var_s, mu_t, var_t)

            t_csv = get_teacher_batch(metas, device=logits.device)
            distill = None
            if t_csv is not None:
                p = torch.sigmoid(logits).clamp(1e-4, 1-1e-4)
                distill = F.binary_cross_entropy(p, t_csv)

            loss = ent + w_weight * wass
            if distill is not None:
                loss = loss + (BETA_DISTILL * distill)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    model.eval()
    return model

# ============================================================
# D) RAG + BioMedBERT DT (used to gate PQC and set lam weights)
# ============================================================
def quality_features(pil_img):
    g = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    q_mean = float(g.mean())
    q_std  = float(g.std())

    hist, _ = np.histogram(g, bins=64, range=(0,1), density=True)
    hist = hist + 1e-12
    q_entropy = float(-(hist*np.log(hist)).sum())

    lap = (-4*g + np.roll(g,1,0) + np.roll(g,-1,0) + np.roll(g,1,1) + np.roll(g,-1,1))
    q_sharp = float(lap.var())

    gx = np.abs(np.roll(g,-1,1) - np.roll(g,1,1))
    gy = np.abs(np.roll(g,-1,0) - np.roll(g,1,0))
    mag = gx + gy
    lq_edge_d = float((mag > 0.15).mean())

    q_clip_low  = float((g < 0.02).mean())
    q_clip_high = float((g > 0.98).mean())

    return np.array([q_mean, q_std, q_entropy, q_sharp, lq_edge_d, q_clip_low, q_clip_high], dtype=np.float32)

rag_df = pd.read_csv(RAG_SRC_CSV)
if "prompt_text" not in rag_df.columns:
    raise ValueError("RAG CSV must contain 'prompt_text'.")
rag_df = rag_df.dropna(subset=["prompt_text"]).reset_index(drop=True)

q_cols = [c for c in ["q_mean","q_std","q_entropy","q_sharp","lq_edge_d","q_clip_low","q_clip_high"] if c in rag_df.columns]
cnn_cols_all = [c for c in rag_df.columns if c.startswith("cnn_")]
USE_CNN_IN_RETRIEVAL = (len(cnn_cols_all) > 0)

use_cols = q_cols + (cnn_cols_all if USE_CNN_IN_RETRIEVAL else [])
if len(use_cols) == 0:
    raise ValueError("RAG CSV must contain q_* and/or cnn_* numeric columns.")

rag_X = rag_df[use_cols].values.astype(np.float32)
rag_Xn = rag_X / (np.linalg.norm(rag_X, axis=1, keepdims=True) + 1e-12)
rag_prompts = rag_df["prompt_text"].astype(str).tolist()
RAG_DIM = rag_X.shape[1]
print(f"✅ RAG loaded | docs={len(rag_df)} | dim={RAG_DIM} | USE_CNN_IN_RETRIEVAL={USE_CNN_IN_RETRIEVAL}")

def align_dim(vec, D):
    vec = vec.astype(np.float32).reshape(-1)
    if vec.shape[0] == D: return vec
    if vec.shape[0] > D: return vec[:D]
    return np.concatenate([vec, np.zeros((D-vec.shape[0],), np.float32)], axis=0)

def retrieve_topk_prompts(query_vec, topk=3):
    q = align_dim(query_vec, RAG_DIM)
    qn = q / (np.linalg.norm(q) + 1e-12)
    sims = rag_Xn @ qn
    topk = int(min(topk, len(sims)))
    idx = np.argpartition(-sims, topk-1)[:topk]
    idx = idx[np.argsort(-sims[idx])]
    return [rag_prompts[i] for i in idx]

# ---- BioMedBERT encoder ----
if USE_LLM_DT:
    try:
        import transformers
    except Exception:
        _pip_install("transformers")
        import transformers
    from transformers import AutoTokenizer, AutoModel

    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
    text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
    for p in text_encoder.parameters(): p.requires_grad_(False)

    # optional load pt
    if BIOMED_CKPT and os.path.exists(BIOMED_CKPT):
        try:
            sd = load_ckpt_state_dict(BIOMED_CKPT)
            sd2 = {k:v for k,v in sd.items() if not k.startswith("cls.")}
            missing, unexpected = text_encoder.load_state_dict(sd2, strict=False)
            print("✅ Loaded BioMedBERT .pt | missing:", len(missing), "unexpected:", len(unexpected))
        except Exception as e:
            print("⚠ Could not load BIOMED_CKPT into HF encoder. Using HF weights only. Reason:", repr(e))

    DT_DIM = int(text_encoder.config.hidden_size)

    class LRUCache:
        def __init__(self, max_items=60000):
            self.max_items = int(max_items)
            self.d = OrderedDict()
        def get(self, k):
            if k not in self.d: return None
            v = self.d.pop(k); self.d[k] = v; return v
        def put(self, k, v):
            if k in self.d: self.d.pop(k)
            self.d[k] = v
            if len(self.d) > self.max_items:
                self.d.popitem(last=False)

    DT_CACHE = LRUCache(60000)

    @torch.no_grad()
    def embed_dt(prompts):
        tok = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(DEVICE)
        out = text_encoder(**tok)
        dt = out.last_hidden_state[:, 0, :]
        dt = F.normalize(dt, dim=1)
        return dt.detach().cpu().float()

    def build_target_prompt(meta, qfeat, rag_texts, cnn_probs=None):
        rag_block = " ".join([f"[DOC{i+1}] {t}" for i,t in enumerate(rag_texts)])
        base_txt = (
            f"Target CXR meta: view={meta.get('view','UNK')}, sex={meta.get('gender','U')}, age={meta.get('age','NA')}.\n"
            f"Quality: mean={qfeat[0]:.3f}, std={qfeat[1]:.3f}, ent={qfeat[2]:.3f}, sharp={qfeat[3]:.3f}, "
            f"edge_d={qfeat[4]:.3f}, clip_low={qfeat[5]:.3f}, clip_high={qfeat[6]:.3f}.\n"
        )
        if cnn_probs is not None:
            base_txt += f"CNN_probs={np.round(cnn_probs,3).tolist()}.\n"
        base_txt += "Using similar source-domain notes below, produce a domain descriptor embedding for adaptation:\n"
        base_txt += rag_block
        return base_txt

    @torch.no_grad()
    def get_dt_batch(pil_imgs, metas, probs=None, topk=3):
        ids = [m["image_key"] for m in metas]
        dt_list, miss = [], []
        for i, img_id in enumerate(ids):
            v = DT_CACHE.get(img_id)
            if v is None:
                miss.append(i); dt_list.append(None)
            else:
                dt_list.append(v)

        if len(miss) > 0:
            prompts, miss_ids = [], []
            for i in miss:
                qf = quality_features(pil_imgs[i])
                parts = []
                if len(q_cols) > 0:
                    qmap = {"q_mean":qf[0],"q_std":qf[1],"q_entropy":qf[2],"q_sharp":qf[3],
                            "lq_edge_d":qf[4],"q_clip_low":qf[5],"q_clip_high":qf[6]}
                    parts.append(np.array([qmap[c] for c in q_cols], dtype=np.float32))
                if USE_CNN_IN_RETRIEVAL and probs is not None:
                    parts.append(np.array(probs[i], dtype=np.float32))
                qvec = np.concatenate(parts, axis=0) if len(parts) > 1 else parts[0]
                qvec = align_dim(qvec, RAG_DIM)

                rag_texts = retrieve_topk_prompts(qvec, topk=topk)
                pr = build_target_prompt(metas[i], qf, rag_texts, cnn_probs=(probs[i] if probs is not None else None))
                prompts.append(pr)
                miss_ids.append(ids[i])

            dt_new = embed_dt(prompts)
            for k, img_id in enumerate(miss_ids):
                DT_CACHE.put(img_id, dt_new[k:k+1])
            for i in miss:
                dt_list[i] = DT_CACHE.get(ids[i])

        return torch.cat(dt_list, dim=0)  # (B,DT_DIM)
else:
    DT_DIM = 768
    def get_dt_batch(pil_imgs, metas, probs=None, topk=3):
        return torch.zeros((len(pil_imgs), DT_DIM), dtype=torch.float32)

# ============================================================
# E) PQC / QNN + Adapter (distill drives AUC)
# ============================================================
RUNS = []
summary_rows = []

def add_summary(method, macro_auc, extra=None):
    row = {"Method": method, "MacroAUC": float(macro_auc)}
    if extra:
        row.update(extra)
    summary_rows.append(row)

# BASE logits fn
def logits_fn_base(pil_imgs, metas, view="orig"):
    tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
    logits, _ = forward_logits_feats(base, pil_imgs, tf)
    return logits.detach().cpu().float()

# ============================================================
# 1) BASELINES
# ============================================================
print("\n==============================")
print("1) BASE (CNN) and BASE+TTA")
print("==============================")
m_base = eval_logits_fn("BASE", logits_fn_base, use_tta=False)
add_summary("BASE", m_base["macro_roc_auc"])

m_base_tta = eval_logits_fn("BASE+TTA", logits_fn_base, use_tta=USE_TTA)
add_summary("BASE+TTA", m_base_tta["macro_roc_auc"])

adapt_loader = stream_loader if ADAPT_PROTOCOL == "stream_then_test" else test_loader

# ============================================================
# 2) CLASSICAL TTDA ABLATION
# ============================================================
print("\n==============================")
print("2) CLASSICAL TTDA ablation (with teacher distill if available)")
print("==============================")

# AdaBN
model_adabn = copy.deepcopy(base)
model_adabn = adabn_adapt(model_adabn, adapt_loader, max_batches=MAX_ADAPT_BATCHES)

def logits_fn_adabn(pil_imgs, metas, view="orig"):
    tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
    logits, _ = forward_logits_feats(model_adabn, pil_imgs, tf)
    return logits.detach().cpu().float()

m_adabn = eval_logits_fn("AdaBN", logits_fn_adabn, use_tta=USE_TTA)
add_summary("AdaBN", m_adabn["macro_roc_auc"])

# TENT (+distill)
model_tent = copy.deepcopy(base)
model_tent = tent_adapt(model_tent, adapt_loader, lr=TENT_LR, steps_per_batch=TENT_STEPS_PER_BATCH, max_batches=MAX_ADAPT_BATCHES)

def logits_fn_tent(pil_imgs, metas, view="orig"):
    tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
    logits, _ = forward_logits_feats(model_tent, pil_imgs, tf)
    return logits.detach().cpu().float()

m_tent = eval_logits_fn("TENT(+distill)", logits_fn_tent, use_tta=USE_TTA)
add_summary("TENT(+distill)", m_tent["macro_roc_auc"])

# CoTTA-lite (+distill)
model_cotta = copy.deepcopy(base)
model_cotta = cotta_adapt(model_cotta, adapt_loader, lr=COTTA_LR, steps_per_batch=COTTA_STEPS_PER_BATCH, max_batches=MAX_ADAPT_BATCHES)

def logits_fn_cotta(pil_imgs, metas, view="orig"):
    tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
    logits, _ = forward_logits_feats(model_cotta, pil_imgs, tf)
    return logits.detach().cpu().float()

m_cotta = eval_logits_fn("CoTTA-lite(+distill)", logits_fn_cotta, use_tta=USE_TTA)
add_summary("CoTTA-lite(+distill)", m_cotta["macro_roc_auc"])

# Wasserstein (+Ent +distill) if src stats
if src_mu is not None and src_var is not None:
    model_wass = copy.deepcopy(base)
    model_wass = wass_adapt(model_wass, adapt_loader, src_mu, src_var, lr=WASS_LR, steps_per_batch=1,
                            max_batches=MAX_ADAPT_BATCHES, w_weight=WASS_WEIGHT_CLASSICAL)

    def logits_fn_wass(pil_imgs, metas, view="orig"):
        tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
        logits, _ = forward_logits_feats(model_wass, pil_imgs, tf)
        return logits.detach().cpu().float()

    m_wass = eval_logits_fn("Wass+Ent(+distill)", logits_fn_wass, use_tta=USE_TTA)
    add_summary("Wass+Ent(+distill)", m_wass["macro_roc_auc"])
else:
    print("ℹ️ Skipping Wasserstein(classical): no source ref stats.")

# ============================================================
# 3) PQC TTDA ABLATION (slow)
# ============================================================
if RUN_PQC:
    print("\n==============================")
    print("3) PQC TTDA ablation (teacher distill is the main AUC driver)")
    print("==============================")

    # Install qiskit if needed
    try:
        import qiskit
        import qiskit_machine_learning
    except Exception:
        _pip_install("qiskit")
        _pip_install("qiskit-machine-learning")
        import qiskit
        import qiskit_machine_learning

    from qiskit import transpile
    from qiskit.circuit import QuantumCircuit, ParameterVector
    from qiskit.quantum_info import SparsePauliOp
    from qiskit_machine_learning.neural_networks import EstimatorQNN
    from qiskit_machine_learning.connectors import TorchConnector

    def make_estimator_v2():
        try:
            from qiskit_aer.primitives import EstimatorV2
            est = EstimatorV2()
            print("✅ Using EstimatorV2 (Aer)")
            return est
        except Exception:
            from qiskit.primitives import StatevectorEstimator
            est = StatevectorEstimator()
            print("✅ Using StatevectorEstimator (fallback)")
            return est

    ESTIMATOR = make_estimator_v2()
    PI = float(math.pi)

    def build_qnn(n_qubits, n_ent_layers):
        q_in_dim = 2*n_qubits + n_qubits + n_ent_layers + 1
        x = ParameterVector("x", q_in_dim)

        n_var = 2*n_qubits
        n_ent = 2*n_ent_layers
        theta = ParameterVector("θ", n_var + n_ent)

        qc = QuantumCircuit(n_qubits)

        # feature angles
        for i in range(n_qubits):
            qc.ry(x[2*i], i)
            qc.rz(x[2*i+1], i)

        dt_off    = 2*n_qubits
        depth_off = dt_off + n_qubits
        topo_idx  = depth_off + n_ent_layers

        # dt-conditioned entanglers
        for i in range(n_qubits):
            qc.crz(x[dt_off + i], i, (i+1) % n_qubits)

        # trainable local layer
        t = 0
        for i in range(n_qubits):
            qc.rz(theta[t], i); t += 1
            qc.ry(theta[t], i); t += 1

        # dt-gated topology mixer
        for l in range(n_ent_layers):
            g_depth = x[depth_off + l]
            g_topo  = x[topo_idx]
            g_chain = g_depth * (1 - g_topo)
            g_ring  = g_depth * (g_topo)

            th_chain = theta[t]; t += 1
            th_ring  = theta[t]; t += 1
            for i in range(max(1, n_qubits-1)):
                qc.crz(g_chain * th_chain, i, i+1)
            for i in range(n_qubits):
                qc.crz(g_ring * th_ring, i, (i+1) % n_qubits)

        qc_t = transpile(qc, optimization_level=0, seed_transpiler=SEED)

        observables = []
        if QNN_NUM_OBS == 1:
            z = ["I"] * n_qubits
            z[0] = "Z"
            observables = [SparsePauliOp.from_list([("".join(z), 1.0)])]
        else:
            for qi in range(min(QNN_NUM_OBS, n_qubits)):
                z = ["I"] * n_qubits
                z[qi] = "Z"
                observables.append(SparsePauliOp.from_list([("".join(z), 1.0)]))

        qnn = EstimatorQNN(
            circuit=qc_t,
            estimator=ESTIMATOR,
            observables=observables,
            input_params=list(x),
            weight_params=list(theta),
            input_gradients=False,
        )
        return qnn, q_in_dim, len(observables)

    def fixed_proj(dt_cpu, out_dim, seed):
        g = torch.Generator(device="cpu").manual_seed(seed)
        W = torch.randn(dt_cpu.shape[1], out_dim, generator=g) / math.sqrt(dt_cpu.shape[1])
        return dt_cpu @ W

    def parse_dt(dt_cpu, n_qubits, n_ent_layers):
        dt_angles = torch.tanh(fixed_proj(dt_cpu, n_qubits, seed=1001+n_qubits)) * PI
        depth_g   = torch.sigmoid(fixed_proj(dt_cpu, n_ent_layers, seed=1002+n_qubits))
        topo      = torch.sigmoid(fixed_proj(dt_cpu, 1, seed=1003+n_qubits))
        alpha     = torch.sigmoid(fixed_proj(dt_cpu, 1, seed=1004+n_qubits))

        lam_raw = fixed_proj(dt_cpu, 2, seed=1005+n_qubits)
        lam1 = 0.75 + 0.75 * torch.sigmoid(lam_raw[:, 0:1])  # [0.75,1.5]
        lam2 = 0.05 + 0.25 * torch.sigmoid(lam_raw[:, 1:2])  # [0.05,0.30]
        return dt_angles.float(), depth_g.float(), topo.float(), alpha.float(), lam1.float(), lam2.float()

    def cosine_fidelity_loss(q_out, q_src_mean):
        q = F.normalize(q_out, dim=1)
        s = F.normalize(q_src_mean.unsqueeze(0).expand_as(q), dim=1)
        cos = (q * s).sum(dim=1)
        return (1.0 - cos).mean()

    class PQCAdapter(nn.Module):
        """
        PQC delta head is TRAINABLE + zero-init => starts neutral (prevents AUC drop)
        """
        def __init__(self, qnn_torch, n_qubits, n_ent_layers, q_out_dim, n_classes):
            super().__init__()
            self.qnn = qnn_torch
            self.nq  = n_qubits
            self.nl  = n_ent_layers
            self.q_head = nn.Linear(q_out_dim, n_classes, bias=False)
            nn.init.zeros_(self.q_head.weight)
            self.register_buffer("q_src_mean", torch.zeros((q_out_dim,), dtype=torch.float32))

        def forward_delta(self, feats_cpu, dt_cpu):
            feat_small = feats_cpu[:, : (2*self.nq)].contiguous()
            feat_angles = torch.tanh(feat_small) * PI

            dt_angles, depth_g, topo, alpha, lam1, lam2 = parse_dt(dt_cpu, self.nq, self.nl)

            q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
            q_out = self.qnn(q_in).to(torch.float32)

            q_center = q_out - self.q_src_mean.unsqueeze(0)
            delta_logits = self.q_head(q_center)
            return delta_logits, q_out, alpha, lam1, lam2

    def make_adapter(nq):
        qnn, q_in_dim, q_out_dim = build_qnn(nq, N_ENT_LAYERS)
        qnn_torch = TorchConnector(qnn).to("cpu")
        adapter = PQCAdapter(qnn_torch, nq, N_ENT_LAYERS, q_out_dim, C).to("cpu")
        return adapter, q_in_dim, q_out_dim

    @torch.no_grad()
    def init_q_src_mean(adapter, n_batches=30):
        """
        If you provided CheX source stats, you can also provide a CheX loader,
        but here we just use target stream (safe) to get a stable center.
        """
        qs = []
        it = iter(stream_loader)
        for _ in tqdm(range(n_batches), desc="Init q_src_mean (stream)"):
            pil_imgs, _, metas = next(it)
            logits, feats = forward_logits_feats(base, pil_imgs, eval_tf)
            feats_cpu = feats.detach().cpu().float()

            probs = torch.sigmoid(logits).detach().cpu().numpy() if USE_CNN_IN_RETRIEVAL else None
            dt = get_dt_batch(pil_imgs, metas, probs=probs, topk=3)

            _, q_out, _, _, _ = adapter.forward_delta(feats_cpu, dt)
            qs.append(q_out.detach().cpu())
        q_all = torch.cat(qs, dim=0)
        return q_all.mean(dim=0).float()

    def pqc_logits_fn(adapter, model_for_feats):
        @torch.no_grad()
        def _fn(pil_imgs, metas, view="orig"):
            tf = eval_tf if (view != "autoaug" or autoaug_tf is None) else autoaug_tf
            logits, feats = forward_logits_feats(model_for_feats, pil_imgs, tf)
            feats_cpu = feats.detach().cpu().float()
            base_logits_cpu = logits.detach().cpu().float()

            probs = torch.sigmoid(logits).detach().cpu().numpy() if USE_CNN_IN_RETRIEVAL else None
            dt = get_dt_batch(pil_imgs, metas, probs=probs, topk=3)

            delta, q_out, alpha, lam1, lam2 = adapter.forward_delta(feats_cpu, dt)
            out = base_logits_cpu + (alpha * PQC_LOGIT_SCALE) * delta
            return out
        return _fn

    def pqc_warmup_distill(adapter, model_for_feats, n_steps=200):
        """
        Warmup to mimic base CNN on stream so PQC starts aligned.
        """
        adapter.train()
        for p in adapter.qnn.parameters(): p.requires_grad_(False)
        for p in adapter.q_head.parameters(): p.requires_grad_(True)

        opt = torch.optim.Adam(adapter.q_head.parameters(), lr=5e-3)
        it = iter(stream_loader)

        for _ in tqdm(range(n_steps), desc="PQC warmup distill (q_head only)"):
            pil_imgs, _, metas = next(it)
            logits, feats = forward_logits_feats(model_for_feats, pil_imgs, eval_tf)
            feats_cpu = feats.detach().cpu().float()
            base_logits_cpu = logits.detach().cpu().float()

            probs = torch.sigmoid(logits).detach().cpu().numpy() if USE_CNN_IN_RETRIEVAL else None
            dt = get_dt_batch(pil_imgs, metas, probs=probs, topk=3)

            delta, q_out, alpha, lam1, lam2 = adapter.forward_delta(feats_cpu, dt)
            student = base_logits_cpu + (alpha * PQC_LOGIT_SCALE) * delta

            # target = teacher from CSV if available else base CNN
            t_csv = get_teacher_batch(metas, device=student.device)
            if t_csv is None:
                t = torch.sigmoid(base_logits_cpu).clamp(1e-4, 1-1e-4)
            else:
                t = t_csv

            s = torch.sigmoid(student).clamp(1e-4, 1-1e-4)
            loss = F.binary_cross_entropy(s, t)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        for p in adapter.qnn.parameters(): p.requires_grad_(True)
        adapter.eval()
        return adapter

    def pqc_adapt(adapter, model_for_feats, mode="distill_fid", max_steps=600):
        """
        mode:
          - "distill"        : distillation only (best for AUC)
          - "distill_fid"    : distill + fidelity
          - "distill_cotta"  : distill + cotta teacher consistency + fidelity
          - "distill_wass"   : distill + wass(feature) + fidelity (if src stats exist)
        """
        adapter = adapter.to("cpu")
        adapter.train()

        # train both qnn and q_head
        for p in adapter.qnn.parameters(): p.requires_grad_(True)
        for p in adapter.q_head.parameters(): p.requires_grad_(True)

        opt = torch.optim.Adam(list(adapter.qnn.parameters()) + list(adapter.q_head.parameters()), lr=LR_PQC)
        teacher = copy.deepcopy(adapter).eval() if "cotta" in mode else None

        it = iter(stream_loader)
        step = 0
        while step < max_steps:
            pil_imgs, _, metas = next(it)
            logits, feats = forward_logits_feats(model_for_feats, pil_imgs, eval_tf)
            feats_cpu = feats.detach().cpu().float()
            base_logits_cpu = logits.detach().cpu().float()

            probs = torch.sigmoid(logits).detach().cpu().numpy() if USE_CNN_IN_RETRIEVAL else None
            dt = get_dt_batch(pil_imgs, metas, probs=probs, topk=3)

            delta, q_out, alpha, lam1, lam2 = adapter.forward_delta(feats_cpu, dt)
            student_logits = base_logits_cpu + (alpha * PQC_LOGIT_SCALE) * delta
            s_prob = torch.sigmoid(student_logits).clamp(1e-4, 1-1e-4)

            # teacher from CSV (preferred), else base CNN
            t_csv = get_teacher_batch(metas, device=s_prob.device)
            if t_csv is None:
                t_prob = torch.sigmoid(base_logits_cpu).clamp(1e-4, 1-1e-4)
            else:
                t_prob = t_csv

            L_distill = F.binary_cross_entropy(s_prob, t_prob)

            loss = BETA_DISTILL * L_distill

            if "fid" in mode:
                L_fid = cosine_fidelity_loss(q_out, adapter.q_src_mean)
                loss = loss + lam2.mean() * PQC_FID_WEIGHT * L_fid

            if "wass" in mode and (src_mu is not None and src_var is not None):
                mu_t = feats_cpu.mean(dim=0)
                var_t = feats_cpu.var(dim=0, unbiased=False) + 1e-6
                wass = wasserstein_diag_gaussian(src_mu, src_var, mu_t.cpu(), var_t.cpu())
                loss = loss + PQC_WASS_WEIGHT * wass

            if "cotta" in mode and teacher is not None:
                with torch.no_grad():
                    t_delta, _, t_alpha, _, _ = teacher.forward_delta(feats_cpu, dt)
                    t_logits = base_logits_cpu + (t_alpha * PQC_LOGIT_SCALE) * t_delta
                    t_p = torch.sigmoid(t_logits).clamp(1e-4, 1-1e-4)

                pil_flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pil_imgs]
                logits2, feats2 = forward_logits_feats(model_for_feats, pil_flip, eval_tf)
                feats2_cpu = feats2.detach().cpu().float()
                base2 = logits2.detach().cpu().float()

                s_delta2, _, s_alpha2, _, _ = adapter.forward_delta(feats2_cpu, dt)
                s_logits2 = base2 + (s_alpha2 * PQC_LOGIT_SCALE) * s_delta2
                s_p2 = torch.sigmoid(s_logits2).clamp(1e-4, 1-1e-4)

                cons = F.mse_loss(s_p2, t_p)
                loss = loss + cons

            opt.zero_grad(set_to_none=True)
            loss.backward()
            if GRAD_CLIP is not None:
                nn.utils.clip_grad_norm_(list(adapter.qnn.parameters()) + list(adapter.q_head.parameters()), GRAD_CLIP)
            opt.step()

            # EMA teacher update
            if teacher is not None:
                with torch.no_grad():
                    msd = adapter.state_dict()
                    tsd = teacher.state_dict()
                    for k in tsd.keys():
                        tsd[k].mul_(0.999).add_(msd[k] * 0.001)
                    teacher.load_state_dict(tsd, strict=True)

            step += 1

        adapter.eval()
        return adapter

    # ---- PQC runs ----
    # We use base (or AdaBN) as the feature provider backbone; TTDA itself acts on PQC params.
    BACKBONES_FOR_PQC = [
        ("base", base),
        ("AdaBN", model_adabn),
    ]

    PQC_MODES = [
        ("no_adapt", None),
        ("distill", "distill"),
        ("distill_fid", "distill_fid"),
        ("distill_cotta", "distill_cotta"),
        ("distill_wass", "distill_wass"),
    ]

    for nq in QUBIT_LIST:
        print("\n" + "="*80)
        print(f"PQC | Q={nq}")
        print("="*80)

        adapter, q_in_dim, q_out_dim = make_adapter(nq)
        adapter.q_src_mean.copy_(init_q_src_mean(adapter, n_batches=20))

        for bname, bmodel in BACKBONES_FOR_PQC:
            # warmup
            adapter0 = copy.deepcopy(adapter)
            if PQC_WARMUP_STEPS and PQC_WARMUP_STEPS > 0:
                adapter0 = pqc_warmup_distill(adapter0, bmodel, n_steps=PQC_WARMUP_STEPS)

            for tag, mode in PQC_MODES:
                if tag == "distill_wass" and (src_mu is None or src_var is None):
                    print("ℹ️ Skipping PQC distill_wass: no source stats.")
                    continue

                if tag == "no_adapt":
                    adapter_use = adapter0
                else:
                    adapter_use = pqc_adapt(copy.deepcopy(adapter0), bmodel, mode=mode, max_steps=MAX_TOTAL_PQC_STEPS)

                fn = pqc_logits_fn(adapter_use, bmodel)
                name = f"PQC[{tag}]_{bname}_Q{nq}"
                m = eval_logits_fn(name, fn, use_tta=USE_TTA)
                add_summary(name, m["macro_roc_auc"], extra={"Q": nq, "Backbone": bname, "PQC_mode": tag})

# ============================================================
# SAVE SUMMARY
# ============================================================
df_sum = pd.DataFrame(summary_rows).sort_values("MacroAUC", ascending=False).reset_index(drop=True)
df_sum.to_csv(os.path.join(OUTDIR, "ablation_summary.csv"), index=False)
print("\n✅ Saved:", os.path.join(OUTDIR, "ablation_summary.csv"))
print(df_sum.head(30))


SystemError: () method: bad call flags

# 18th February Jpdate 

In [2]:
def _pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import qiskit
    import qiskit_machine_learning
except Exception:
    _pip_install("qiskit")
    _pip_install("qiskit-machine-learning")
    import qiskit
    import qiskit_machine_learning

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.providers.basic_provider import BasicProvider
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

basic_backend = BasicProvider().get_backend("basic_simulator")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 5.4 MB/s eta 0:00:00


In [3]:
# --- Estimator import that works across Qiskit versions ---
try:
    # Fast simulator primitive (if Aer is installed)
    from qiskit_aer.primitives import Estimator as AerEstimator
    ESTIMATOR = AerEstimator()
    EST_BACKEND = "aer"
except Exception:
    # Reference primitive in newer Qiskit
    from qiskit.primitives import StatevectorEstimator as Estimator
    ESTIMATOR = Estimator()
    EST_BACKEND = "statevector"

print("Estimator backend:", EST_BACKEND)


Estimator backend: statevector


# Domain invariant increment Quantum

In [4]:
import qiskit, qiskit_machine_learning
print("qiskit:", qiskit.__version__)
print("qiskit-machine-learning:", qiskit_machine_learning.__version__)
try:
    import qiskit_aer
    print("qiskit-aer:", qiskit_aer.__version__)
except Exception as e:
    print("qiskit-aer not available:", e)


qiskit: 2.3.1
qiskit-machine-learning: 0.9.0
qiskit-aer not available: No module named 'qiskit_aer'


In [5]:
!pip -q install faiss-cpu
!pip -q install "qiskit>=1.0" qiskit-aer qiskit-machine-learning
!pip -q install transformers

# Fix fastai conflict if you need fastai
!pip -q install "fastcore>=1.8.0,<1.9"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 84.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
execnb 0.1.18 requires fastcore>=1.10.4, but you have fastcore 1.8.18 which is incompatible.
nbdev 2.4.10 requires fastcore>=1.11.0, but you have fastcore 1.8.18 which is incompatible.


In [6]:
import faiss, qiskit, qiskit_aer, qiskit_machine_learning
print("faiss OK:", faiss.__version__)
print("qiskit OK:", qiskit.__version__)


faiss OK: 1.13.2
qiskit OK: 2.3.1


In [7]:
!pip -q install -U "qiskit>=1.0" qiskit-aer "qiskit-machine-learning>=0.8"


In [8]:
!pip -q install faiss-cpu --no-deps


# FAISS and ALL

# CORRECTED QC ADAPTER
## Simple comparison

The **previous code** is a **test-time domain adaptation ablation**. It compares AdaBN, TENT, CoTTA, Wasserstein, and several teacher-distilled PQC methods. It is mainly designed to test different adaptation strategies. 

The **new code** focuses on a **quantum adapter with memory retrieval**. It retrieves similar stored image/text embeddings and labels, then uses them to guide the quantum model. It compares only:

1. CNN baseline
2. Quantum adapter without memory
3. Quantum adapter with memory



### Important differences

* **Previous:** 6 and 8 qubits; new code uses a smaller **4-qubit ZZFeatureMap + RealAmplitudes** circuit.
* **Previous:** RAG text documents and teacher CSV; new code uses a structured memory containing embeddings, labels, text descriptors, prototypes, confidence values, and retrieval indexes.
* **Previous:** broad classical and quantum ablation; new code directly measures whether memory improves the quantum adapter.
* **Previous:** mainly pseudo-label/test-time adaptation; new code uses actual NIH labels during calibration and stream adaptation. 
* **New:** adds a nonlinear MLP, text-conditioned quantum angles, a learned gate, and memory-neighbor probability regularization.
* **New:** reports AUROC, PR-AUC, F1, ECE, and Brier score, so it evaluates both classification and calibration.

## Which is better?

The **new code is better for proving that memory retrieval improves the quantum adapter**, because it directly compares CNN, adapter-only, and adapter-plus-memory. 

The **previous code is better for a complete TTDA ablation paper**, because it compares the quantum method against several classical adaptation baselines.

### Critical warning

The new code is **not label-free test-time adaptation** because it trains using NIH labels during calibration and incremental adaptation. Therefore, describe it as **supervised domain-incremental adaptation**, not unsupervised TTDA.


In [5]:
# ============================================================
# NIH Domain-Incremental Adaptation (PDF-strategy aligned)
# ✅ Fixes: baseline AUROC drop + logit mismatch
# ✅ Keeps: ZZFeatureMap encoding + tiny NON-LINEAR quantum adapter
# ✅ Uses: ALL memory files (embeddings/labels/t_desc/v_img/prototypes/base_conf/metadata/index_sklearn)
# ✅ Reports: per-class ROC_AUC for
#   (A) No-adapter / No-memory (CNN-only strong baseline)
#   (B) Adapter but No-memory (dt=0, no retrieval)
#   (C) Quantum Adapter + Memory retrieval
# ✅ Prints: sanity checks after EACH stage
# ============================================================

import os, json, math, time, random, inspect
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# -------------------------
# (0) Repro
# -------------------------
SEED = 7
def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# (1) Paths (yours)
# -------------------------
MEM_DIR = "/kaggle/input/datasets/zarinn/memory/MEMORY"

RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

OUT_DIR = "/kaggle/working/outputs_fixed_auc"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# (2) Speed knobs (keep qb / keep quantum, but faster)
# -------------------------
BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

CALIBRATION_N = 512          # onboarding set size
EPOCHS_CALIB  = 1

MAX_STEPS     = 50           # adaptation steps
LOG_EVERY     = 1            # print every step to avoid "stuck UI"
TEST_MAX_BATCHES = None      # set 30 for quick debug

K_RETRIEVE     = 8           # neighbor retrieval
TAU_MEM        = 10.0        # softmax temperature for neighbor weighting
LAMBDA_MEMPROB = 0.5         # weight of memory-prob loss

# Quantum dimensions (tiny)
DQ_MAIN   = 4                # qubits = dq
VQC_REPS  = 1                # keep tiny
N_OBS     = 2                # reduce observables for speed (<= dq). if you want, set = dq

LR_CLASSICAL = 1e-3
LR_QUANTUM   = 1e-2

# Residual scaling (prevents AUROC crash)
DELTA_LOGIT_SCALE = 0.35

print("\n[Sanity] Config:",
      {"BATCH_SIZE":BATCH_SIZE,"DQ_MAIN":DQ_MAIN,"VQC_REPS":VQC_REPS,"N_OBS":N_OBS,
       "K_RETRIEVE":K_RETRIEVE,"MAX_STEPS":MAX_STEPS,"DELTA_LOGIT_SCALE":DELTA_LOGIT_SCALE})

# ============================================================
# (3) Qiskit (EstimatorV2 + EstimatorQNN)
# ============================================================
import qiskit
import qiskit_machine_learning

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

print("Qiskit version:", getattr(qiskit, "__version__", "unknown"))
print("Qiskit-ML version:", getattr(qiskit_machine_learning, "__version__", "unknown"))

ESTIMATOR = None
EST_BACKEND = None
try:
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
    ESTIMATOR = AerEstimatorV2()
    EST_BACKEND = "qiskit_aer.primitives.EstimatorV2"
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator
        ESTIMATOR = StatevectorEstimator()
        EST_BACKEND = "qiskit.primitives.StatevectorEstimator"
    except Exception as e:
        raise ImportError("No compatible V2 Estimator found. Install qiskit-aer.") from e

sig = inspect.signature(ESTIMATOR.run)
params = list(sig.parameters.keys())
if ("observables" in params) and ("circuits" in params):
    raise RuntimeError(f"Estimator looks like V1 ({EST_BACKEND}). Need V2 for EstimatorQNN path.")

print("Using estimator:", EST_BACKEND)

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

_AER_BACKEND = AerSimulator()
_PM = generate_preset_pass_manager(optimization_level=1, backend=_AER_BACKEND)

def build_qnn(n_qubits, reps=1, entanglement="circular", n_obs=2):
    """
    ZZFeatureMap (encoding) + RealAmplitudes (trainable).
    We decompose + run a preset pass manager for Aer compatibility.
    """
    fm = zz_feature_map(feature_dimension=n_qubits, reps=2, entanglement=entanglement)
    ans = real_amplitudes(num_qubits=n_qubits, reps=reps, entanglement=entanglement)

    qc = QuantumCircuit(n_qubits)
    qc = qc.compose(fm, inplace=False, wrap=False)
    qc = qc.compose(ans, inplace=False, wrap=False)

    qc = qc.decompose(reps=15)
    qc = _PM.run(qc)

    # fewer observables -> faster
    obs = []
    n_obs = int(min(max(1, n_obs), n_qubits))
    for i in range(n_obs):
        pauli = ["I"] * n_qubits
        pauli[i] = "Z"
        obs.append(SparsePauliOp("".join(pauli)))

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm.parameters,
        weight_params=ans.parameters,
        observables=obs,
        input_gradients=True,
        estimator=ESTIMATOR,
    )
    return qnn, len(fm.parameters), qnn.num_weights, len(obs)

# ============================================================
# (4) Encoders: BiomedBERT + ResNet50 (frozen)
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torchvision
from torchvision import transforms

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

def _strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p): kk = kk[len(p):]
        out[kk] = v
    return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu")
    if isinstance(ck, dict) and "state_dict" in ck: sd = ck["state_dict"]
    elif isinstance(ck, dict) and "model" in ck: sd = ck["model"]
    elif isinstance(ck, dict) and "net" in ck: sd = ck["net"]
    else: sd = ck
    return _strip_prefix(sd)

def load_biomedbert():
    tok_dir = os.path.dirname(BIOMED_CKPT)
    try:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

    model = AutoModel.from_pretrained(TEXT_MODEL_NAME)
    if os.path.exists(BIOMED_CKPT):
        sd = load_state_dict_any(BIOMED_CKPT)
        # ignore classifier heads if present
        sd = {k:v for k,v in sd.items() if not k.startswith("cls.")}
        model.load_state_dict(sd, strict=False)

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return tokenizer, model

tokenizer, text_model = load_biomedbert()
DT_DIM = int(text_model.config.hidden_size)

@torch.no_grad()
def text_embed(text_list, max_len=64):
    enc = tokenizer(text_list, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    out = text_model(**enc, return_dict=True)
    t = out.last_hidden_state[:, 0, :]  # CLS
    return F.normalize(t, dim=1)

# ---- ResNet feature extractor + STRONG CNN head from checkpoint
def load_resnet50_with_head():
    m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(2048, C)
    sd = load_state_dict_any(RESNET_CKPT)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("[Sanity] ResNet load strict=False | missing:", len(missing), "| unexpected:", len(unexpected))
    m.eval().to(DEVICE)
    for p in m.parameters():
        p.requires_grad = False
    feat = nn.Sequential(*(list(m.children())[:-1])).eval().to(DEVICE)
    return m, feat

resnet_full, resnet_feat = load_resnet50_with_head()

img_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def img_embed(x):
    v = resnet_feat(x).flatten(1)  # [B,2048]
    return F.normalize(v, dim=1)

@torch.no_grad()
def cnn_logits_from_resnet(x):
    # uses the ckpt head (strong baseline)
    return resnet_full(x)

# ============================================================
# (5) NIH dataset
# ============================================================
def read_list(path):
    with open(path, "r") as f:
        return [x.strip() for x in f if x.strip()]

nih_df = pd.read_csv(NIH_CSV)
nih_df = nih_df.rename(columns={"Image Index":"image", "Finding Labels":"labels"})
nih_row = {r["image"]: r for _, r in nih_df.iterrows()}

def labels_to_multi_hot(lbl_str):
    parts = set([p.strip() for p in str(lbl_str).split("|")])
    y = np.zeros((C,), dtype=np.float32)
    for i, name in enumerate(UNIFIED_LABELS):
        if name == "Pleural Effusion":
            if "Effusion" in parts: y[i] = 1.0
        else:
            if name in parts: y[i] = 1.0
    return y

def make_text_descriptor(r):
    view = r.get("View Position", r.get("ViewPosition", ""))
    age  = r.get("Patient Age", r.get("PatientAge", ""))
    sex  = r.get("Patient Gender", r.get("PatientGender", ""))
    return f"Chest X-ray. View: {view}. Patient age: {age}. Sex: {sex}."

class NIHDataset(Dataset):
    def __init__(self, list_file, max_n=None):
        self.items = read_list(list_file)
        if max_n is not None:
            self.items = self.items[:max_n]

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        name = self.items[idx]
        img = Image.open(os.path.join(NIH_IMG, name)).convert("RGB")
        x = img_tfms(img)

        r = nih_row.get(name, {})
        y = labels_to_multi_hot(r.get("labels", ""))
        txt = make_text_descriptor(r)
        return x, torch.tensor(y, dtype=torch.float32), txt, name

calib_ds   = NIHDataset(NIH_STREAM, max_n=CALIBRATION_N)
stream_ds  = NIHDataset(NIH_STREAM, max_n=None)
test_ds    = NIHDataset(NIH_TEST, max_n=None)

calib_loader  = DataLoader(calib_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
stream_loader = DataLoader(stream_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader   = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("\n[Sanity] NIH sizes:", {"calib":len(calib_ds),"stream":len(stream_ds),"test":len(test_ds)})

# ============================================================
# (6) Memory bundle (ALL files) + retrieval (sklearn index if possible, else FAISS)
# ============================================================
import faiss
try:
    import joblib
except Exception:
    joblib = None

class MemoryBundle:
    def __init__(self, mem_dir):
        self.mem_dir = mem_dir

        self.path_embeddings = os.path.join(mem_dir, "embeddings.npy")
        self.path_labels     = os.path.join(mem_dir, "labels.npy")
        self.path_tdesc      = os.path.join(mem_dir, "t_desc.npy")
        self.path_vimg       = os.path.join(mem_dir, "v_img.npy")
        self.path_protos     = os.path.join(mem_dir, "prototypes.npy")
        self.path_protolbl   = os.path.join(mem_dir, "prototypes_labels.json")
        self.path_baseconf   = os.path.join(mem_dir, "base_conf.npy")
        self.path_meta       = os.path.join(mem_dir, "metadata.jsonl")
        self.path_idx_skl    = os.path.join(mem_dir, "index_sklearn.joblib")

        assert os.path.exists(self.path_embeddings), "Missing embeddings.npy"
        assert os.path.exists(self.path_labels), "Missing labels.npy"

        E = np.load(self.path_embeddings)
        Y = np.load(self.path_labels)

        if E.dtype != np.float32: E = E.astype(np.float32, copy=False)
        E = np.ascontiguousarray(E)
        self.E = E
        self.N, self.D = self.E.shape

        # labels -> [N,C]
        if Y.ndim == 1:
            Y_oh = np.zeros((Y.shape[0], C), dtype=np.float32)
            Y_oh[np.arange(Y.shape[0]), Y.astype(np.int64)] = 1.0
            Y = Y_oh
        if Y.dtype != np.float32: Y = Y.astype(np.float32, copy=False)
        self.Y = np.ascontiguousarray(Y)

        # t_desc
        self.t_desc = None
        if os.path.exists(self.path_tdesc):
            tdesc = np.load(self.path_tdesc)
            if tdesc.dtype != np.float32: tdesc = tdesc.astype(np.float32, copy=False)
            self.t_desc = np.ascontiguousarray(tdesc)
        self.t_desc_dim = int(self.t_desc.shape[1]) if self.t_desc is not None else 0

        # v_img
        self.v_img = None
        if os.path.exists(self.path_vimg):
            vimg = np.load(self.path_vimg)
            if vimg.dtype != np.float32: vimg = vimg.astype(np.float32, copy=False)
            self.v_img = np.ascontiguousarray(vimg)

        # prototypes
        self.prototypes = None
        self.prototypes_labels = None
        if os.path.exists(self.path_protos):
            P = np.load(self.path_protos)
            if P.dtype != np.float32: P = P.astype(np.float32, copy=False)
            self.prototypes = np.ascontiguousarray(P)
        if os.path.exists(self.path_protolbl):
            with open(self.path_protolbl, "r") as f:
                self.prototypes_labels = json.load(f)

        # base_conf
        self.base_conf = None
        if os.path.exists(self.path_baseconf):
            bc = np.load(self.path_baseconf)
            bc = bc.astype(np.float32, copy=False).reshape(-1)
            if bc.shape[0] == self.N:
                self.base_conf = np.clip(bc, 0.0, 1.0)
            else:
                print("[Warn] base_conf.npy has wrong length; ignoring.")

        # metadata (lazy)
        self.metadata_available = os.path.exists(self.path_meta)

        # retrieval index
        self.index_sklearn = None
        if os.path.exists(self.path_idx_skl) and joblib is not None:
            try:
                obj = joblib.load(self.path_idx_skl)
                # must have kneighbors
                if hasattr(obj, "kneighbors"):
                    self.index_sklearn = obj
                    print("[Sanity] Loaded index_sklearn.joblib (kneighbors available).")
            except Exception as e:
                print("[Warn] Could not load index_sklearn.joblib; will use FAISS. Reason:", repr(e))

        # FAISS fallback (cosine via normalized IP)
        self.faiss_index = None
        E2 = self.E.copy()
        faiss.normalize_L2(E2)
        self.faiss_index = faiss.IndexFlatIP(self.D)
        self.faiss_index.add(E2)
        self._E_norm = E2  # for self-check

        # torch tensors for fast gather
        self._Y_t = torch.tensor(self.Y, device=DEVICE, dtype=torch.float32)
        self._td_t = torch.tensor(self.t_desc, device=DEVICE, dtype=torch.float32) if self.t_desc is not None else None

    def memory_gb(self):
        total = 0
        for fn in ["embeddings.npy","labels.npy","t_desc.npy","v_img.npy","prototypes.npy","base_conf.npy","metadata.jsonl","index_sklearn.joblib"]:
            p = os.path.join(self.mem_dir, fn)
            if os.path.exists(p):
                total += os.path.getsize(p)
        return total / 1e9

    @torch.no_grad()
    def query_key(self, v, t):
        """
        Build query matching embeddings.npy dimension.
        If embeddings are concat([v,t]) use that.
        Else fall back to v-only / t-only / projection.
        """
        dv, dt = v.shape[1], t.shape[1]
        if self.D == dv + dt:
            q = torch.cat([v, t], dim=1)
        elif self.D == dv:
            q = v
        elif self.D == dt:
            q = t
        else:
            # deterministic projection
            torch.manual_seed(SEED)
            W = torch.randn(dv + dt, self.D, device=v.device) / math.sqrt(dv + dt)
            q = torch.cat([v, t], dim=1) @ W
        return F.normalize(q, dim=1)

    @torch.no_grad()
    def retrieve(self, q, k=16):
        """
        Returns idxs [B,k], sims [B,k]
        sims should be cosine-ish if vectors are normalized.
        """
        qn = F.normalize(q, dim=1)
        q_np = np.ascontiguousarray(qn.detach().cpu().numpy().astype(np.float32, copy=False))

        if self.index_sklearn is not None:
            try:
                # sklearn returns distances by default; often Euclidean on normalized vectors
                dists, idxs = self.index_sklearn.kneighbors(q_np, n_neighbors=min(k, self.N), return_distance=True)
                # convert to similarity proxy (negative dist)
                sims = -dists.astype(np.float32, copy=False)
                return idxs.astype(np.int64, copy=False), sims
            except Exception as e:
                print("[Warn] sklearn index kneighbors failed; using FAISS. Reason:", repr(e))

        # FAISS cosine: normalize + inner product
        sims, idxs = self.faiss_index.search(q_np, min(k, self.N))
        return idxs.astype(np.int64, copy=False), sims.astype(np.float32, copy=False)

    @torch.no_grad()
    def retrieve_dt(self, idxs):
        if self._td_t is None:
            return torch.zeros((idxs.shape[0], 1), device=DEVICE, dtype=torch.float32)
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        dt = self._td_t[idxs_t].mean(dim=1)
        return F.normalize(dt, dim=1)

    @torch.no_grad()
    def pmem_from_neighbors(self, idxs, sims, tau=10.0, eps=1e-8):
        """
        Soft neighbor vote for label distribution.
        If base_conf exists, we multiply weights by conf (reliability weighting).
        """
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        sims_t = torch.tensor(sims, device=DEVICE, dtype=torch.float32)

        neigh_y = self._Y_t[idxs_t]  # [B,k,C]
        w = torch.softmax(tau * sims_t, dim=1)  # [B,k]
        if self.base_conf is not None:
            conf = torch.tensor(self.base_conf, device=DEVICE, dtype=torch.float32)[idxs_t]  # [B,k]
            w = w * conf
            w = w / (w.sum(dim=1, keepdim=True) + 1e-12)

        w = w.unsqueeze(-1)  # [B,k,1]
        p = (w * neigh_y).sum(dim=1)
        return torch.clamp(p, eps, 1 - eps)

    def stage3_self_retrieval(self, n_tests=200):
        hits = 0
        E = self._E_norm  # normalized
        for _ in range(n_tests):
            i = random.randrange(self.N)
            q = np.ascontiguousarray(E[i:i+1].astype(np.float32, copy=False))
            sims, idxs = self.faiss_index.search(q, 1)
            if idxs[0,0] == i:
                hits += 1
        return hits / n_tests

    def stage3_label_agreement(self, n_tests=200, k=5):
        arg_ok = 0
        jac = []
        def jacc(a, b, eps=1e-9):
            a = (a > 0.5); b = (b > 0.5)
            inter = np.logical_and(a,b).sum()
            uni   = np.logical_or(a,b).sum()
            return inter / (uni + eps)

        for _ in range(n_tests):
            i = random.randrange(self.N)
            q = np.ascontiguousarray(self._E_norm[i:i+1].astype(np.float32, copy=False))
            sims, idxs = self.faiss_index.search(q, min(k, self.N))
            nn = idxs[0]
            y_true = self.Y[i]
            if int(np.argmax(self.Y[nn].mean(axis=0))) == int(np.argmax(y_true)):
                arg_ok += 1
            jac.append(np.mean([jacc(y_true, self.Y[j]) for j in nn]))
        return {"argmax_agreement": arg_ok / n_tests, "mean_jaccard": float(np.mean(jac))}

mem = MemoryBundle(MEM_DIR)
print("\n[Sanity] Memory loaded:",
      {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
       "has_v_img":mem.v_img is not None, "has_prototypes":mem.prototypes is not None,
       "has_base_conf":mem.base_conf is not None, "has_metadata":mem.metadata_available,
       "sklearn_index":mem.index_sklearn is not None})

print("[Stage3] self-retrieval top1:", mem.stage3_self_retrieval(200))
print("[Stage3] label agreement:", mem.stage3_label_agreement(200, k=K_RETRIEVE))

# ============================================================
# (7) Models
# ============================================================
crit_sup = nn.BCEWithLogitsLoss()
crit_mem = nn.BCELoss()

class NonLinearProjector(nn.Module):
    """
    Non-linear tiny projection (MLP) to dq.
    This is your "Stage 1: projection + nonlinearity".
    """
    def __init__(self, in_dim, dq):
        super().__init__()
        h = max(64, 2*dq)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Linear(h, dq),
        )
    def forward(self, z):
        return self.net(z)

class QuantumResidualAdapter(nn.Module):
    """
    Residual quantum adapter:
      base_logits (from strong CNN head) + gate(dt)*delta_logits
    Non-linear adapter ensured via MLP + tanh + dt-FiLM.
    """
    def __init__(self, dv=2048, dt=DT_DIM, dt_cond_dim=1, dq=4, n_classes=5, reps=1, n_obs=2):
        super().__init__()
        self.dq = dq
        self.dt_cond_dim = max(int(dt_cond_dim), 1)

        self.proj = NonLinearProjector(dv + dt, dq)  # non-linear
        self.dt_to_angles = nn.Sequential(
            nn.Linear(self.dt_cond_dim, max(8, 2*dq)),
            nn.GELU(),
            nn.Linear(max(8, 2*dq), dq),
        )
        self.dt_gate = nn.Sequential(
            nn.Linear(self.dt_cond_dim, 8),
            nn.GELU(),
            nn.Linear(8, 1),
        )

        qnn, n_inp, n_w, nobs = build_qnn(dq, reps=reps, entanglement="circular", n_obs=n_obs)
        init_w = 0.01 * np.random.randn(n_w).astype(np.float32)
        self.q = TorchConnector(qnn, initial_weights=init_w)

        # map obs -> logits delta
        self.delta_head = nn.Sequential(
            nn.Linear(nobs, max(16, 2*dq)),
            nn.GELU(),
            nn.Linear(max(16, 2*dq), n_classes),
        )

        self._nobs = nobs

    def forward_delta(self, v, t, dt):
        z = torch.cat([v, t], dim=1)
        u = self.proj(z)
        u = math.pi * torch.tanh(u)  # [-pi, pi]

        mod = math.pi * torch.tanh(self.dt_to_angles(dt))
        u_mod = u + 0.25 * mod

        g = torch.sigmoid(self.dt_gate(dt))  # [B,1]
        q_out = self.q(u_mod)                # [B, nobs]
        delta = self.delta_head(q_out)       # [B,C]
        return delta, g, q_out

    def forward_logits(self, x, txt, dt, base_logits=None):
        # v,t
        v = img_embed(x)
        t = text_embed(list(txt))
        if base_logits is None:
            base_logits = cnn_logits_from_resnet(x)
        delta, g, q_out = self.forward_delta(v, t, dt)
        logits = base_logits + (g * DELTA_LOGIT_SCALE) * delta
        return logits, base_logits, delta, g, q_out

def split_params_for_opt(model):
    q_params, c_params = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "q." in n:  # TorchConnector params
            q_params.append(p)
        else:
            c_params.append(p)
    return c_params, q_params

def make_opt(model):
    c_params, q_params = split_params_for_opt(model)
    return torch.optim.Adam([
        {"params": c_params, "lr": LR_CLASSICAL},
        {"params": q_params, "lr": LR_QUANTUM},
    ])

def trainable_param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# dt_cond_dim from memory
dt_cond_dim = mem.t_desc_dim if mem.t_desc_dim > 0 else 1

qadapter = QuantumResidualAdapter(
    dq=DQ_MAIN, n_classes=C, dt_cond_dim=dt_cond_dim, reps=VQC_REPS, n_obs=N_OBS
).to(DEVICE)

opt_q = make_opt(qadapter)

print("\n[Sanity] Adapter params:",
      {"trainable": trainable_param_count(qadapter), "dt_cond_dim": dt_cond_dim,
       "dq":DQ_MAIN, "n_obs":qadapter._nobs})

# ============================================================
# (8) Metrics (per-class + calibration)
# ============================================================
def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p_pred >= lo) & (p_pred < hi) if i < n_bins-1 else (p_pred >= lo) & (p_pred <= hi)
        if m.sum() == 0:
            continue
        conf = p_pred[m].mean()
        acc  = y_true[m].mean()
        ece += (m.sum()/len(y_true)) * abs(acc - conf)
    return float(ece)

def brier_binary(y_true, p_pred):
    y_true = y_true.astype(np.float32)
    p_pred = p_pred.astype(np.float32)
    return float(np.mean((p_pred - y_true)**2))

def per_class_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        ecev = ece_binary(yt.astype(np.float32), yp.astype(np.float32), n_bins=15)
        brv  = brier_binary(yt, yp)
        rows.append([lbl, aucv, apv, f1v, ecev, brv])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5","ECE","Brier"])
    macro_auc = float(np.nanmean(df["ROC_AUC"].values))
    macro_ap  = float(np.nanmean(df["PR_AUC"].values))
    macro_f1  = float(np.nanmean(df["F1@0.5"].values))
    macro_ece = float(np.nanmean(df["ECE"].values))
    macro_br  = float(np.nanmean(df["Brier"].values))
    return df, {"macro_auc":macro_auc,"macro_ap":macro_ap,"macro_f1":macro_f1,"macro_ece":macro_ece,"macro_brier":macro_br}

@torch.no_grad()
def collect_preds_cnn_only(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = cnn_logits_from_resnet(x)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_adapter(loader, use_memory=True, max_batches=None):
    """
    If use_memory=False: dt=zeros, no retrieval.
    If use_memory=True: retrieve dt from memory (embeddings/t_desc).
    """
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)

        if use_memory:
            v = img_embed(x)
            t = text_embed(list(txt))
            qkey = mem.query_key(v,t)
            idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
            dt = mem.retrieve_dt(idxs)
        else:
            dt = torch.zeros((x.size(0), dt_cond_dim), device=DEVICE, dtype=torch.float32)

        logits, _, _, _, _ = qadapter.forward_logits(x, txt, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

# ============================================================
# (9) Stage sanity checks (PDF-style steps)
# ============================================================
def sanity_check_stage_inputs():
    x,y,txt,_ = next(iter(calib_loader))
    x = x.to(DEVICE)
    v = img_embed(x)
    t = text_embed(list(txt))
    print("\n[Stage Sanity] Encoders:",
          {"v_shape":tuple(v.shape),"t_shape":tuple(t.shape),
           "v_norm_mean":float(v.norm(dim=1).mean().cpu()),
           "t_norm_mean":float(t.norm(dim=1).mean().cpu())})

    q = mem.query_key(v,t)
    idxs, sims = mem.retrieve(q, k=K_RETRIEVE)
    dt = mem.retrieve_dt(idxs)
    print("[Stage Sanity] Memory retrieval:",
          {"q_shape":tuple(q.shape),"idxs_shape":idxs.shape,"sims_minmax":(float(np.min(sims)), float(np.max(sims))),
           "dt_shape":tuple(dt.shape), "dt_norm_mean":float(dt.norm(dim=1).mean().cpu())})

    base_logits = cnn_logits_from_resnet(x)
    logits, base, delta, g, q_out = qadapter.forward_logits(x, txt, dt, base_logits=base_logits)
    print("[Stage Sanity] Adapter forward:",
          {"base_logits_shape":tuple(base.shape),"delta_shape":tuple(delta.shape),"g_minmax":(float(g.min().cpu()), float(g.max().cpu())),
           "q_out_shape":tuple(q_out.shape), "logits_shape":tuple(logits.shape)})

sanity_check_stage_inputs()

# ============================================================
# (10) Calibration (onboarding): train ONLY adapter parts (not encoders)
# ============================================================
def calibrate_adapter_onboarding():
    qadapter.train()
    t0 = time.time()
    seen = 0

    for ep in range(EPOCHS_CALIB):
        for bi, (x,y,txt,_) in enumerate(calib_loader, start=1):
            x = x.to(DEVICE); y = y.to(DEVICE)

            # use memory dt during onboarding (matches paper)
            v = img_embed(x); t = text_embed(list(txt))
            qkey = mem.query_key(v,t)
            idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
            dt = mem.retrieve_dt(idxs)

            base_logits = cnn_logits_from_resnet(x)  # strong baseline
            logits, _, _, _, _ = qadapter.forward_logits(x, txt, dt, base_logits=base_logits)

            loss = crit_sup(logits, y)

            opt_q.zero_grad(set_to_none=True)
            loss.backward()
            opt_q.step()

            seen += x.size(0)
            if bi % 20 == 0:
                print(f"[calib-adapter] batch {bi} loss={loss.item():.4f} seen={seen}/{len(calib_ds)} elapsed={time.time()-t0:.1f}s")

    return time.time() - t0

print("\n[Stage] Onboarding calibration (adapter)...")
calib_time = calibrate_adapter_onboarding()
print("[Stage] Calibration time (sec):", calib_time)

# ============================================================
# (11) Incremental adaptation (stream): supervised + memory-prob regularizer
# ============================================================
def incremental_adapt_stream():
    qadapter.train()
    logs = []
    t_start = time.time()

    step = 0
    avg_step_time = None

    for x,y,txt,_ in stream_loader:
        st0 = time.time()

        x = x.to(DEVICE); y = y.to(DEVICE)

        v = img_embed(x)
        t = text_embed(list(txt))

        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)
        p_mem = mem.pmem_from_neighbors(idxs, sims, tau=TAU_MEM)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _, g, _ = qadapter.forward_logits(x, txt, dt, base_logits=base_logits)

        p_hat = torch.sigmoid(logits)
        loss_sup = crit_sup(logits, y)
        loss_mem = crit_mem(p_hat, p_mem)
        loss = loss_sup + LAMBDA_MEMPROB * loss_mem

        opt_q.zero_grad(set_to_none=True)
        loss.backward()
        opt_q.step()

        step += 1
        step_time = time.time() - st0
        avg_step_time = step_time if avg_step_time is None else (0.9*avg_step_time + 0.1*step_time)

        if step % LOG_EVERY == 0:
            elapsed = time.time() - t_start
            eta = (MAX_STEPS - step) * (avg_step_time if avg_step_time is not None else 0.0)
            print(f"[adapt] step {step}/{MAX_STEPS}  loss={loss.item():.4f} sup={loss_sup.item():.4f} mem={loss_mem.item():.4f} "
                  f"g_mean={float(g.mean().detach().cpu()):.3f}  step_time={step_time:.2f}s  avg_step≈{avg_step_time:.2f}s  ETA≈{eta/60:.1f}min")

            logs.append({
                "step": step,
                "loss": float(loss.item()),
                "loss_sup": float(loss_sup.item()),
                "loss_memprob": float(loss_mem.item()),
                "g_mean": float(g.mean().detach().cpu()),
                "step_time_sec": float(step_time),
                "avg_step_time_sec": float(avg_step_time),
                "elapsed_sec": float(elapsed),
            })

        if step >= MAX_STEPS:
            break

    return pd.DataFrame(logs)

print("\n[Stage] Incremental adaptation on stream...")
log_df = incremental_adapt_stream()
log_df.to_csv(os.path.join(OUT_DIR, "adapt_logs.csv"), index=False)
print("[Stage] Saved adapt logs:", os.path.join(OUT_DIR, "adapt_logs.csv"))

# ============================================================
# (12) Final evaluation: three modes + per-class ROC-AUC
# ============================================================
print("\n==============================")
print("(A) CNN-only (no adapter, no memory)  ✅ strong baseline")
print("==============================")
Y0, P0 = collect_preds_cnn_only(test_loader, max_batches=TEST_MAX_BATCHES)
df0, m0 = per_class_metrics(Y0, P0, UNIFIED_LABELS)
print(df0)
print("[A] Macro:", m0)
df0.to_csv(os.path.join(OUT_DIR, "per_class_A_cnn_only.csv"), index=False)

print("\n==============================")
print("(B) Adapter BUT no-memory (dt=0, no retrieval)")
print("==============================")
Y1, P1 = collect_preds_adapter(test_loader, use_memory=False, max_batches=TEST_MAX_BATCHES)
df1, m1 = per_class_metrics(Y1, P1, UNIFIED_LABELS)
print(df1)
print("[B] Macro:", m1)
df1.to_csv(os.path.join(OUT_DIR, "per_class_B_adapter_no_memory.csv"), index=False)

print("\n==============================")
print("(C) Quantum Adapter + Memory (retrieval-conditioned)")
print("==============================")
Y2, P2 = collect_preds_adapter(test_loader, use_memory=True, max_batches=TEST_MAX_BATCHES)
df2, m2 = per_class_metrics(Y2, P2, UNIFIED_LABELS)
print(df2)
print("[C] Macro:", m2)
df2.to_csv(os.path.join(OUT_DIR, "per_class_C_quantum_memory.csv"), index=False)

summary = {
    "estimator_backend": EST_BACKEND,
    "config": {
        "DQ_MAIN":DQ_MAIN,"VQC_REPS":VQC_REPS,"N_OBS":N_OBS,"DELTA_LOGIT_SCALE":DELTA_LOGIT_SCALE,
        "CALIBRATION_N":CALIBRATION_N,"MAX_STEPS":MAX_STEPS,"K_RETRIEVE":K_RETRIEVE,"TAU_MEM":TAU_MEM,"LAMBDA_MEMPROB":LAMBDA_MEMPROB
    },
    "memory": {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
               "has_v_img":mem.v_img is not None,"has_prototypes":mem.prototypes is not None,"has_base_conf":mem.base_conf is not None,
               "has_metadata":mem.metadata_available,"sklearn_index":mem.index_sklearn is not None},
    "metrics_A_cnn_only": m0,
    "metrics_B_adapter_no_memory": m1,
    "metrics_C_quantum_memory": m2
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Saved outputs to:", OUT_DIR)
print("  - per_class_A_cnn_only.csv")
print("  - per_class_B_adapter_no_memory.csv")
print("  - per_class_C_quantum_memory.csv")
print("  - adapt_logs.csv")
print("  - summary.json")


Device: cuda

[Sanity] Config: {'BATCH_SIZE': 16, 'DQ_MAIN': 4, 'VQC_REPS': 1, 'N_OBS': 2, 'K_RETRIEVE': 8, 'MAX_STEPS': 50, 'DELTA_LOGIT_SCALE': 0.35}
Qiskit version: 2.3.0
Qiskit-ML version: 0.9.0
Using estimator: qiskit_aer.primitives.EstimatorV2
[Sanity] ResNet load strict=False | missing: 0 | unexpected: 0

[Sanity] NIH sizes: {'calib': 512, 'stream': 86524, 'test': 25596}


[Sanity] Loaded index_sklearn.joblib (kneighbors available).

[Sanity] Memory loaded: {'N': 30000, 'D': 12, 't_desc_dim': 7, 'sizeGB': 0.01169471, 'has_v_img': True, 'has_prototypes': True, 'has_base_conf': True, 'has_metadata': True, 'sklearn_index': True}
[Stage3] self-retrieval top1: 1.0
[Stage3] label agreement: {'argmax_agreement': 0.465, 'mean_jaccard': 0.29791666645717885}

[Sanity] Adapter params: {'trainable': 180862, 'dt_cond_dim': 7, 'dq': 4, 'n_obs': 2}

[Stage Sanity] Encoders: {'v_shape': (16, 2048), 't_shape': (16, 768), 'v_norm_mean': 1.0, 't_norm_mean': 1.0}
[Stage Sanity] Memory retrieval: {'q_shape': (16, 12), 'idxs_shape': (16, 8), 'sims_minmax': (-0.9661256074905396, -0.6904600858688354), 'dt_shape': (16, 7), 'dt_norm_mean': 1.0}


/tmp/ipykernel_55/1368691595.py:750: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  {"base_logits_shape":tuple(base.shape),"delta_shape":tuple(delta.shape),"g_minmax":(float(g.min().cpu()), float(g.max().cpu())),


[Stage Sanity] Adapter forward: {'base_logits_shape': (16, 5), 'delta_shape': (16, 5), 'g_minmax': (0.5367990136146545, 0.5378332734107971), 'q_out_shape': (16, 2), 'logits_shape': (16, 5)}

[Stage] Onboarding calibration (adapter)...
[calib-adapter] batch 20 loss=0.3830 seen=320/512 elapsed=185.6s
[Stage] Calibration time (sec): 296.0384349822998

[Stage] Incremental adaptation on stream...
[adapt] step 1/50  loss=0.7258 sup=0.3482 mem=0.7551 g_mean=0.549  step_time=9.18s  avg_step≈9.18s  ETA≈7.5min
[adapt] step 2/50  loss=0.7188 sup=0.3654 mem=0.7067 g_mean=0.550  step_time=9.27s  avg_step≈9.19s  ETA≈7.4min
[adapt] step 3/50  loss=0.7014 sup=0.3359 mem=0.7310 g_mean=0.551  step_time=9.13s  avg_step≈9.19s  ETA≈7.2min
[adapt] step 4/50  loss=0.7118 sup=0.3646 mem=0.6945 g_mean=0.552  step_time=9.25s  avg_step≈9.19s  ETA≈7.0min
[adapt] step 5/50  loss=0.7288 sup=0.3594 mem=0.7387 g_mean=0.553  step_time=9.22s  avg_step≈9.20s  ETA≈6.9min
[adapt] step 6/50  loss=0.6194 sup=0.2653 mem=0.70

#This code extends the earlier quantum-memory model by adding CaD adapters, session-specific learning, domain routing, contrastive loss, and shared knowledge absorption, then compares four progressively stronger prediction settings on NIH. 


In [9]:
# ============================================================
# 100% aligned to CaD equations / flow:
#   C-adapter: Eq (2)-(4) + Absorb Eq (5)-(8)
#   D-adapter: Eq (9)-(15)
#   Domain-id: Eq (16)
#   Loss: BCE + λ * MLCL (Eq (17)-(19))
# Plus your: ZZFeatureMap encoding + tiny nonlinear quantum residual adapter
# ============================================================

import os, json, math, time, random, inspect
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# -------------------------
# (0) Repro
# -------------------------
SEED = 7
def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# (1) Paths (yours)
# -------------------------
MEM_DIR = "/kaggle/input/datasets/zarinn/memory/MEMORY"

RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

OUT_DIR = "/kaggle/working/outputs_cad_aligned"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# (2) Speed knobs
# -------------------------
BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

CALIBRATION_N = 512
EPOCHS_CALIB  = 1

# Domain sessions (simulate T sessions on NIH stream)
N_SESSIONS    = 3     # session count T (paper has multiple domains; here we simulate)
EPOCHS_PER_SESSION = 1
MAX_STEPS_PER_SESSION = 80  # cap updates per session for speed
LOG_EVERY     = 10

TEST_MAX_BATCHES = None

# Memory knobs
K_RETRIEVE     = 8
TAU_MEM        = 10.0
LAMBDA_MEMPROB = 0.5

# CaD loss knob (Eq.17)
LAMBDA_MLCL    = 0.1
TEMP_MLCL      = 0.2

# Quantum residual adapter
DQ_MAIN   = 4
VQC_REPS  = 1
N_OBS     = 2
LR_CLASSICAL = 1e-3
LR_QUANTUM   = 1e-2
DELTA_LOGIT_SCALE = 0.35

# CaD bottleneck d'
CAD_DPRIME = 256

print("\n[Sanity] Config:",
      {"BATCH_SIZE":BATCH_SIZE, "N_SESSIONS":N_SESSIONS,
       "CAD_DPRIME":CAD_DPRIME, "DQ_MAIN":DQ_MAIN,
       "K_RETRIEVE":K_RETRIEVE, "LAMBDA_MLCL":LAMBDA_MLCL, "TEMP_MLCL":TEMP_MLCL})

# ============================================================
# (3) Qiskit (EstimatorV2 + EstimatorQNN)
# ============================================================
import qiskit
import qiskit_machine_learning

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

print("Qiskit version:", getattr(qiskit, "__version__", "unknown"))
print("Qiskit-ML version:", getattr(qiskit_machine_learning, "__version__", "unknown"))

ESTIMATOR = None
EST_BACKEND = None
try:
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
    ESTIMATOR = AerEstimatorV2()
    EST_BACKEND = "qiskit_aer.primitives.EstimatorV2"
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator
        ESTIMATOR = StatevectorEstimator()
        EST_BACKEND = "qiskit.primitives.StatevectorEstimator"
    except Exception as e:
        raise ImportError("No compatible V2 Estimator found. Install qiskit-aer.") from e

sig = inspect.signature(ESTIMATOR.run)
params = list(sig.parameters.keys())
if ("observables" in params) and ("circuits" in params):
    raise RuntimeError(f"Estimator looks like V1 ({EST_BACKEND}). Need V2 for EstimatorQNN path.")
print("Using estimator:", EST_BACKEND)

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

_AER_BACKEND = AerSimulator()
_PM = generate_preset_pass_manager(optimization_level=1, backend=_AER_BACKEND)

def build_qnn(n_qubits, reps=1, entanglement="circular", n_obs=2):
    fm = zz_feature_map(feature_dimension=n_qubits, reps=2, entanglement=entanglement)
    ans = real_amplitudes(num_qubits=n_qubits, reps=reps, entanglement=entanglement)

    qc = QuantumCircuit(n_qubits)
    qc = qc.compose(fm, inplace=False, wrap=False)
    qc = qc.compose(ans, inplace=False, wrap=False)

    qc = qc.decompose(reps=15)
    qc = _PM.run(qc)

    obs = []
    n_obs = int(min(max(1, n_obs), n_qubits))
    for i in range(n_obs):
        pauli = ["I"] * n_qubits
        pauli[i] = "Z"
        obs.append(SparsePauliOp("".join(pauli)))

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm.parameters,
        weight_params=ans.parameters,
        observables=obs,
        input_gradients=True,  # REQUIRED with TorchConnector
        estimator=ESTIMATOR,
    )
    return qnn, len(fm.parameters), qnn.num_weights, len(obs)

# ============================================================
# (4) Encoders: BiomedBERT + ResNet50 (frozen)
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torchvision
from torchvision import transforms

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

def _strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p): kk = kk[len(p):]
        out[kk] = v
    return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu")
    if isinstance(ck, dict) and "state_dict" in ck: sd = ck["state_dict"]
    elif isinstance(ck, dict) and "model" in ck: sd = ck["model"]
    elif isinstance(ck, dict) and "net" in ck: sd = ck["net"]
    else: sd = ck
    return _strip_prefix(sd)

def load_biomedbert():
    tok_dir = os.path.dirname(BIOMED_CKPT)
    try:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

    model = AutoModel.from_pretrained(TEXT_MODEL_NAME)
    if os.path.exists(BIOMED_CKPT):
        sd = load_state_dict_any(BIOMED_CKPT)
        sd = {k:v for k,v in sd.items() if not k.startswith("cls.")}
        model.load_state_dict(sd, strict=False)

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return tokenizer, model

tokenizer, text_model = load_biomedbert()
DT_DIM = int(text_model.config.hidden_size)

@torch.no_grad()
def text_embed(text_list, max_len=64):
    enc = tokenizer(text_list, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    out = text_model(**enc, return_dict=True)
    t = out.last_hidden_state[:, 0, :]  # CLS
    return F.normalize(t, dim=1)

# ============================================================
# (4) ResNet50 (frozen) — SAME loader as your high-AUC cell
#     + spatial fmap extractor for CaD
# ============================================================
import torchvision
from torchvision import transforms

def load_resnet50_full_and_spatial():
    m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(2048, C)

    sd = load_state_dict_any(RESNET_CKPT)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("[Sanity] ResNet load strict=False | missing:", len(missing), "| unexpected:", len(unexpected))

    m.eval().to(DEVICE)
    for p in m.parameters():
        p.requires_grad = False

    @torch.no_grad()
    def spatial(x):
        b = m
        x = b.conv1(x); x = b.bn1(x); x = b.relu(x); x = b.maxpool(x)
        x = b.layer1(x); x = b.layer2(x); x = b.layer3(x); x = b.layer4(x)  # [B,2048,7,7]
        return x

    @torch.no_grad()
    def logits(x):
        return m(x)  # uses the pretrained head from ckpt

    return m, spatial, logits

resnet_full, resnet_spatial, cnn_logits_from_resnet = load_resnet50_full_and_spatial()

img_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def pool2048_from_fmap(fmap):
    v = F.adaptive_avg_pool2d(fmap, (1,1)).flatten(1)  # [B,2048]
    return F.normalize(v, dim=1)
def cnn_logits_from_resnet(x):
    return resnet.forward_logits(x)

@torch.no_grad()
def resnet_spatial(x):
    return resnet.forward_spatial(x)

@torch.no_grad()
def pool2048_from_fmap(fmap):
    v = F.adaptive_avg_pool2d(fmap, (1,1)).flatten(1)  # [B,2048]
    return F.normalize(v, dim=1)

# ============================================================
# (5) NIH dataset + session split
# ============================================================
def read_list(path):
    with open(path, "r") as f:
        return [x.strip() for x in f if x.strip()]

nih_df = pd.read_csv(NIH_CSV)
nih_df = nih_df.rename(columns={"Image Index":"image", "Finding Labels":"labels"})
nih_row = {r["image"]: r for _, r in nih_df.iterrows()}

def labels_to_multi_hot(lbl_str):
    parts = set([p.strip() for p in str(lbl_str).split("|")])
    y = np.zeros((C,), dtype=np.float32)
    for i, name in enumerate(UNIFIED_LABELS):
        if name == "Pleural Effusion":
            if "Effusion" in parts: y[i] = 1.0
        else:
            if name in parts: y[i] = 1.0
    return y

def make_text_descriptor(r):
    view = r.get("View Position", r.get("ViewPosition", ""))
    age  = r.get("Patient Age", r.get("PatientAge", ""))
    sex  = r.get("Patient Gender", r.get("PatientGender", ""))
    return f"Chest X-ray. View: {view}. Patient age: {age}. Sex: {sex}."

class NIHListDataset(Dataset):
    def __init__(self, names):
        self.items = list(names)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        name = self.items[idx]
        img = Image.open(os.path.join(NIH_IMG, name)).convert("RGB")
        x = img_tfms(img)
        r = nih_row.get(name, {})
        y = labels_to_multi_hot(r.get("labels", ""))
        txt = make_text_descriptor(r)
        return x, torch.tensor(y, dtype=torch.float32), txt, name

stream_list = read_list(NIH_STREAM)
test_list   = read_list(NIH_TEST)

# Calibration subset from stream
calib_list = stream_list[:CALIBRATION_N]

# Create sessions: contiguous split (deterministic)
def split_sessions(lst, n_sessions):
    n = len(lst)
    splits = []
    for s in range(n_sessions):
        a = (n*s)//n_sessions
        b = (n*(s+1))//n_sessions
        splits.append(lst[a:b])
    return splits

session_lists = split_sessions(stream_list, N_SESSIONS)

calib_loader = DataLoader(NIHListDataset(calib_list), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(NIHListDataset(test_list),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("\n[Sanity] NIH sizes:", {"stream_total":len(stream_list),
                             "calib":len(calib_list),
                             "sessions":[len(x) for x in session_lists],
                             "test":len(test_list)})

# ============================================================
# (6) Memory bundle (ALL files) + retrieval
# ============================================================
import faiss
try:
    import joblib
except Exception:
    joblib = None

class MemoryBundle:
    def __init__(self, mem_dir):
        self.mem_dir = mem_dir
        self.path_embeddings = os.path.join(mem_dir, "embeddings.npy")
        self.path_labels     = os.path.join(mem_dir, "labels.npy")
        self.path_tdesc      = os.path.join(mem_dir, "t_desc.npy")
        self.path_vimg       = os.path.join(mem_dir, "v_img.npy")
        self.path_protos     = os.path.join(mem_dir, "prototypes.npy")
        self.path_protolbl   = os.path.join(mem_dir, "prototypes_labels.json")
        self.path_baseconf   = os.path.join(mem_dir, "base_conf.npy")
        self.path_meta       = os.path.join(mem_dir, "metadata.jsonl")
        self.path_idx_skl    = os.path.join(mem_dir, "index_sklearn.joblib")

        assert os.path.exists(self.path_embeddings), "Missing embeddings.npy"
        assert os.path.exists(self.path_labels), "Missing labels.npy"

        E = np.load(self.path_embeddings)
        Y = np.load(self.path_labels)

        if E.dtype != np.float32: E = E.astype(np.float32, copy=False)
        E = np.ascontiguousarray(E)
        self.E = E
        self.N, self.D = self.E.shape

        if Y.ndim == 1:
            Y_oh = np.zeros((Y.shape[0], C), dtype=np.float32)
            Y_oh[np.arange(Y.shape[0]), Y.astype(np.int64)] = 1.0
            Y = Y_oh
        if Y.dtype != np.float32: Y = Y.astype(np.float32, copy=False)
        self.Y = np.ascontiguousarray(Y)

        self.t_desc = None
        if os.path.exists(self.path_tdesc):
            tdesc = np.load(self.path_tdesc)
            if tdesc.dtype != np.float32: tdesc = tdesc.astype(np.float32, copy=False)
            self.t_desc = np.ascontiguousarray(tdesc)
        self.t_desc_dim = int(self.t_desc.shape[1]) if self.t_desc is not None else 0

        self.v_img = None
        if os.path.exists(self.path_vimg):
            vimg = np.load(self.path_vimg)
            if vimg.dtype != np.float32: vimg = vimg.astype(np.float32, copy=False)
            self.v_img = np.ascontiguousarray(vimg)

        self.prototypes = None
        self.prototypes_labels = None
        if os.path.exists(self.path_protos):
            P = np.load(self.path_protos)
            if P.dtype != np.float32: P = P.astype(np.float32, copy=False)
            self.prototypes = np.ascontiguousarray(P)
        if os.path.exists(self.path_protolbl):
            with open(self.path_protolbl, "r") as f:
                self.prototypes_labels = json.load(f)

        self.base_conf = None
        if os.path.exists(self.path_baseconf):
            bc = np.load(self.path_baseconf).astype(np.float32, copy=False).reshape(-1)
            if bc.shape[0] == self.N:
                self.base_conf = np.clip(bc, 0.0, 1.0)

        self.metadata_available = os.path.exists(self.path_meta)

        self.index_sklearn = None
        if os.path.exists(self.path_idx_skl) and joblib is not None:
            try:
                obj = joblib.load(self.path_idx_skl)
                if hasattr(obj, "kneighbors"):
                    self.index_sklearn = obj
                    print("[Sanity] Loaded index_sklearn.joblib (kneighbors available).")
            except Exception as e:
                print("[Warn] Could not load index_sklearn.joblib; using FAISS. Reason:", repr(e))

        # FAISS cosine via normalized IP
        E2 = self.E.copy()
        faiss.normalize_L2(E2)
        self.faiss_index = faiss.IndexFlatIP(self.D)
        self.faiss_index.add(E2)
        self._E_norm = E2

        self._Y_t = torch.tensor(self.Y, device=DEVICE, dtype=torch.float32)
        self._td_t = torch.tensor(self.t_desc, device=DEVICE, dtype=torch.float32) if self.t_desc is not None else None

    def memory_gb(self):
        total = 0
        for fn in ["embeddings.npy","labels.npy","t_desc.npy","v_img.npy","prototypes.npy","base_conf.npy","metadata.jsonl","index_sklearn.joblib"]:
            p = os.path.join(self.mem_dir, fn)
            if os.path.exists(p):
                total += os.path.getsize(p)
        return total / 1e9

    @torch.no_grad()
    def query_key(self, v, t):
        dv, dt = v.shape[1], t.shape[1]
        if self.D == dv + dt:
            q = torch.cat([v, t], dim=1)
        elif self.D == dv:
            q = v
        elif self.D == dt:
            q = t
        else:
            torch.manual_seed(SEED)
            W = torch.randn(dv + dt, self.D, device=v.device) / math.sqrt(dv + dt)
            q = torch.cat([v, t], dim=1) @ W
        return F.normalize(q, dim=1)

    @torch.no_grad()
    def retrieve(self, q, k=16):
        qn = F.normalize(q, dim=1)
        q_np = np.ascontiguousarray(qn.detach().cpu().numpy().astype(np.float32, copy=False))

        if self.index_sklearn is not None:
            try:
                dists, idxs = self.index_sklearn.kneighbors(q_np, n_neighbors=min(k, self.N), return_distance=True)
                sims = -dists.astype(np.float32, copy=False)
                return idxs.astype(np.int64, copy=False), sims
            except Exception:
                pass

        sims, idxs = self.faiss_index.search(q_np, min(k, self.N))
        return idxs.astype(np.int64, copy=False), sims.astype(np.float32, copy=False)

    @torch.no_grad()
    def retrieve_dt(self, idxs):
        if self._td_t is None:
            return torch.zeros((idxs.shape[0], 1), device=DEVICE, dtype=torch.float32)
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        dt = self._td_t[idxs_t].mean(dim=1)
        return F.normalize(dt, dim=1)

    @torch.no_grad()
    def pmem_from_neighbors(self, idxs, sims, tau=10.0, eps=1e-8):
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        sims_t = torch.tensor(sims, device=DEVICE, dtype=torch.float32)

        neigh_y = self._Y_t[idxs_t]  # [B,k,C]
        w = torch.softmax(tau * sims_t, dim=1)  # [B,k]

        if self.base_conf is not None:
            conf = torch.tensor(self.base_conf, device=DEVICE, dtype=torch.float32)[idxs_t]  # [B,k]
            w = w * conf
            w = w / (w.sum(dim=1, keepdim=True) + 1e-12)

        w = w.unsqueeze(-1)
        p = (w * neigh_y).sum(dim=1)
        return torch.clamp(p, eps, 1 - eps)

mem = MemoryBundle(MEM_DIR)
dt_cond_dim = mem.t_desc_dim if mem.t_desc_dim > 0 else 1

print("\n[Sanity] Memory loaded:",
      {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
       "has_v_img":mem.v_img is not None, "has_prototypes":mem.prototypes is not None,
       "has_base_conf":mem.base_conf is not None, "has_metadata":mem.metadata_available,
       "sklearn_index":mem.index_sklearn is not None})

# ============================================================
# (7) CaD Adapters EXACT shapes (token-based on resnet fmap)
#    Treat fmap [B,2048,7,7] -> tokens x_img [B,N=49,d=2048]
#    CLS token = pooled [B,1,d]
# ============================================================
def fmap_to_tokens(fmap):
    # fmap: [B,2048,7,7] -> img tokens [B,49,2048]
    B, d, H, W = fmap.shape
    x_img = fmap.permute(0,2,3,1).reshape(B, H*W, d)  # [B,N,d]
    x_cls = fmap.mean(dim=(2,3), keepdim=False).unsqueeze(1)  # [B,1,d]
    return x_cls, x_img, H, W

def tokens_to_fmap(x_img, H, W):
    # x_img: [B,N,d] -> [B,d,H,W]
    B, N, d = x_img.shape
    return x_img.reshape(B, H, W, d).permute(0,3,1,2).contiguous()

class CAdapter(nn.Module):
    """
    C-adapter EXACT Eq (2)-(4):
      M_C,mid = ReLU(LN(M_C,in) W_down)
      M~_C    = LN(M_C,mid) W_up
      M_C,out = ReLU(M~_C + shortcut(M_C,in))
    """
    def __init__(self, d=2048, dprime=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.up   = nn.Linear(dprime, d)

    def forward(self, M_in):
        # M_in: [B, N+1, d]
        M_mid = F.relu(self.down(self.ln1(M_in)))
        M_til = self.up(self.ln2(M_mid))
        M_out = F.relu(M_til + M_in)
        return M_out

@torch.no_grad()
def absorb_cadapter_(cadapter_running: CAdapter, cadapter_current: CAdapter, t: int):
    """
    Absorb strategy EXACT Eq (5)-(8): running <- (1/t)*((t-1)*running + current)
    """
    for (n1,p1), (n2,p2) in zip(cadapter_running.named_parameters(), cadapter_current.named_parameters()):
        assert n1 == n2
        p1.data.mul_((t-1)/t).add_(p2.data, alpha=(1.0/t))

class DAdapter(nn.Module):
    """
    D-adapter EXACT Eq (9)-(15) (adapted to our tokens):
      M_D,mid = ReLU(LN(M_D,in) W_D,down)                 (9)
      split [CLS] and img tokens                          (10)
      reshape img tokens to HxW grid                       (11)
      x_img = Reshape(Conv2D(x_img_grid))                 (12)
      M~_D,mid = ReLU(Concat([x_img, x_cls]))             (13)
      M~_D = LN(M~_D,mid) W_D,up                           (14)
      M_D,out = ReLU(M~_D + shortcut(M_D,in))             (15)
    """
    def __init__(self, d=2048, dprime=256, k=3):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.conv = nn.Conv2d(dprime, dprime, kernel_size=k, padding=k//2)
        self.up   = nn.Linear(dprime, d)

    def forward(self, M_in, H=7, W=7):
        # M_in: [B, N+1, d]
        M_mid = F.relu(self.down(self.ln1(M_in)))  # (9)
        x_cls = M_mid[:, :1, :]                    # [B,1,d']
        x_img = M_mid[:, 1:, :]                    # [B,N,d'] (10)
        x_grid = x_img.reshape(x_img.size(0), H, W, x_img.size(2)).permute(0,3,1,2).contiguous()  # (11)
        x_grid2 = self.conv(x_grid)                # conv2d
        x_img2 = x_grid2.permute(0,2,3,1).reshape(x_img.size(0), H*W, x_img.size(2)).contiguous()  # (12)
        M_til_mid = F.relu(torch.cat([x_cls, x_img2], dim=1))  # (13)
        M_til = self.up(self.ln2(M_til_mid))       # (14)
        M_out = F.relu(M_til + M_in)               # (15)
        return M_out

class DAdapterBank(nn.Module):
    def __init__(self, n_domains, d=2048, dprime=256):
        super().__init__()
        self.adapters = nn.ModuleList([DAdapter(d=d, dprime=dprime) for _ in range(n_domains)])

    def forward(self, M_in, domain_id, H=7, W=7):
        return self.adapters[domain_id](M_in, H=H, W=W)

    def freeze_domain(self, domain_id):
        for p in self.adapters[domain_id].parameters():
            p.requires_grad = False

# Domain-specific classifier heads (paper stores fc per domain)
class DomainFC(nn.Module):
    def __init__(self, d=2048, n_classes=5):
        super().__init__()
        self.fc = nn.Linear(d, n_classes)
    def forward(self, cls_token):
        return self.fc(cls_token)

# ============================================================
# (8) Quantum residual adapter (your style, unchanged)
# ============================================================
class NonLinearProjector(nn.Module):
    def __init__(self, in_dim, dq):
        super().__init__()
        h = max(64, 2*dq)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Linear(h, dq),
        )
    def forward(self, z):
        return self.net(z)

class QuantumResidualAdapter(nn.Module):
    """
    logits = base_logits + sigmoid(g(dt)) * scale * delta_logits
    """
    def __init__(self, dv=2048, dt=DT_DIM, dt_cond_dim=1, dq=4, n_classes=5, reps=1, n_obs=2):
        super().__init__()
        self.dq = dq
        self.dt_cond_dim = max(int(dt_cond_dim), 1)

        self.proj = NonLinearProjector(dv + dt, dq)
        self.dt_to_angles = nn.Sequential(
            nn.Linear(self.dt_cond_dim, max(8, 2*dq)),
            nn.GELU(),
            nn.Linear(max(8, 2*dq), dq),
        )
        self.dt_gate = nn.Sequential(
            nn.Linear(self.dt_cond_dim, 8),
            nn.GELU(),
            nn.Linear(8, 1),
        )

        qnn, _, n_w, nobs = build_qnn(dq, reps=reps, entanglement="circular", n_obs=n_obs)
        init_w = 0.01 * np.random.randn(n_w).astype(np.float32)
        self.q = TorchConnector(qnn, initial_weights=init_w)

        self.delta_head = nn.Sequential(
            nn.Linear(nobs, max(16, 2*dq)),
            nn.GELU(),
            nn.Linear(max(16, 2*dq), n_classes),
        )
        self._nobs = nobs

    def forward_delta(self, v, t, dt):
        z = torch.cat([v, t], dim=1)
        u = self.proj(z)
        u = math.pi * torch.tanh(u)

        mod = math.pi * torch.tanh(self.dt_to_angles(dt))
        u_mod = u + 0.25 * mod

        g = torch.sigmoid(self.dt_gate(dt))
        q_out = self.q(u_mod)
        delta = self.delta_head(q_out)
        return delta, g, q_out

    def forward_logits(self, base_logits, v, t, dt):
        delta, g, _ = self.forward_delta(v, t, dt)
        logits = base_logits + (g * DELTA_LOGIT_SCALE) * delta
        return logits, delta, g

qadapter = QuantumResidualAdapter(dq=DQ_MAIN, n_classes=C, dt_cond_dim=dt_cond_dim, reps=VQC_REPS, n_obs=N_OBS).to(DEVICE)

def split_params_for_opt(model):
    q_params, c_params = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "q." in n:
            q_params.append(p)
        else:
            c_params.append(p)
    return c_params, q_params

def make_opt(model, extra_params=None):
    c_params, q_params = split_params_for_opt(model)
    if extra_params is not None:
        c_params = list(c_params) + list(extra_params)
    return torch.optim.Adam([
        {"params": c_params, "lr": LR_CLASSICAL},
        {"params": q_params, "lr": LR_QUANTUM},
    ])

# ============================================================
# (9) Loss: BCE + Multi-label contrastive (Eq.17-19)
# ============================================================
crit_bce = nn.BCEWithLogitsLoss()
crit_mem = nn.BCELoss()

def label_similarity_r(y_i, y_j, eps=1e-9):
    # multi-label similarity (Jaccard on binary)
    yi = (y_i > 0.5).float()
    yj = (y_j > 0.5).float()
    inter = (yi*yj).sum(dim=-1)
    uni   = ((yi + yj) > 0.0).float().sum(dim=-1)
    return inter / (uni + eps)

def mlcl_loss(features, labels, temp=0.2, eps=1e-9):
    """
    Implements Eq.(19) style:
      L = - sum_{j!=i} r_ij log ( exp(<f_i,f_j>)/sum_{k!=i} exp(<f_i,f_k>) )
    features: [B,d] (normalized)
    labels:   [B,C] multi-hot
    """
    B = features.size(0)
    f = F.normalize(features, dim=1)
    sim = (f @ f.t()) / temp  # cosine / temp
    sim = sim - torch.eye(B, device=sim.device) * 1e9  # exclude i==i

    # r_ij matrix
    yi = labels.unsqueeze(1).expand(B,B,-1)
    yj = labels.unsqueeze(0).expand(B,B,-1)
    r = label_similarity_r(yi, yj)  # [B,B]
    r = r * (1.0 - torch.eye(B, device=r.device))  # zero diagonal

    logp = F.log_softmax(sim, dim=1)  # over j
    loss_i = -(r * logp).sum(dim=1) / (r.sum(dim=1) + eps)
    return loss_i.mean()

# ============================================================
# (10) CaD Model wrapper
# ============================================================
class CaDModel(nn.Module):
    """
    Shared C-adapter is absorbed across sessions (Eq.5-8).
    D-adapter + FC are per session (stored).
    """
    def __init__(self, n_sessions, d=2048, dprime=256, n_classes=5):
        super().__init__()
        self.c_running = CAdapter(d=d, dprime=dprime)   # absorbed/shared
        self.c_current = CAdapter(d=d, dprime=dprime)   # trained each session then absorbed into c_running

        self.d_bank = DAdapterBank(n_domains=n_sessions, d=d, dprime=dprime)
        self.fc_bank = nn.ModuleList([DomainFC(d=d, n_classes=n_classes) for _ in range(n_sessions)])

        self.domain_centers = [None for _ in range(n_sessions)]  # store 1 center per session (Eq.16)

    def forward_tokens(self, fmap, domain_id):
        # fmap -> tokens -> adapters -> tokens out
        x_cls, x_img, H, W = fmap_to_tokens(fmap)        # [B,1,d] and [B,N,d]
        M_in = torch.cat([x_cls, x_img], dim=1)          # [B,N+1,d]
        # shared C-adapter (running)
        M_c = self.c_running(M_in)                       # Eq.2-4
        # domain D-adapter (parallel, independent)
        M_d = self.d_bank(M_in, domain_id=domain_id, H=H, W=W)  # Eq.9-15
        # paper DA output adds MHSA out too; here M_in plays that role:
        M_out = M_c + M_d + M_in                         # Eq.(1) analog
        cls = M_out[:, :1, :].squeeze(1)                 # [B,d]
        img_tokens = M_out[:, 1:, :]                     # [B,N,d]
        return cls, img_tokens

    def logits_for_domain(self, fmap, domain_id):
        cls, img_tokens = self.forward_tokens(fmap, domain_id)
        logits = self.fc_bank[domain_id](cls)            # per-domain classifier
        return logits, cls, img_tokens

cad = CaDModel(n_sessions=N_SESSIONS, d=2048, dprime=CAD_DPRIME, n_classes=C).to(DEVICE)

# optimizer: train c_current + d_adapter(session) + fc(session) + qadapter
def make_session_optimizer(session_id):
    # enable grads for c_current and this session's D+FC (freeze others)
    for p in cad.c_running.parameters():
        p.requires_grad = False
    for p in cad.c_current.parameters():
        p.requires_grad = True
    for di in range(N_SESSIONS):
        for p in cad.d_bank.adapters[di].parameters():
            p.requires_grad = (di == session_id)
        for p in cad.fc_bank[di].parameters():
            p.requires_grad = (di == session_id)

    # IMPORTANT: during training, use cad.c_current (not running), then absorb into running
    # We'll swap forward by temporarily using c_current in a helper below.
    train_params = list(cad.c_current.parameters()) + \
                   list(cad.d_bank.adapters[session_id].parameters()) + \
                   list(cad.fc_bank[session_id].parameters())
    opt = make_opt(qadapter, extra_params=train_params)
    return opt

# helper: forward with c_current during session training
def forward_cad_train(fmap, session_id):
    x_cls, x_img, H, W = fmap_to_tokens(fmap)
    M_in = torch.cat([x_cls, x_img], dim=1)

    M_c = cad.c_current(M_in)  # train current C-adapter
    M_d = cad.d_bank(M_in, domain_id=session_id, H=H, W=W)
    M_out = M_c + M_d + M_in
    cls = M_out[:, :1, :].squeeze(1)
    img_tokens = M_out[:, 1:, :]
    logits = cad.fc_bank[session_id](cls)
    return logits, cls, img_tokens

# ============================================================
# (11) Sanity check
# ============================================================
@torch.no_grad()
def sanity_check_shapes():
    x,y,txt,_ = next(iter(calib_loader))
    x = x.to(DEVICE)
    fmap = resnet_spatial(x)
    xcls, ximg, H, W = fmap_to_tokens(fmap)
    print("\n[Sanity] fmap/tokens:",
          {"fmap":tuple(fmap.shape),"xcls":tuple(xcls.shape),"ximg":tuple(ximg.shape),"H":H,"W":W})

    # memory retrieval dt
    v = pool2048_from_fmap(fmap)
    t = text_embed(list(txt))
    qkey = mem.query_key(v,t)
    idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
    dt = mem.retrieve_dt(idxs)
    print("[Sanity] memory dt:", {"dt":tuple(dt.shape), "sims_minmax":(float(np.min(sims)), float(np.max(sims)))})

    # CaD forward per domain
    for d_id in range(min(2, N_SESSIONS)):
        logits, cls, toks = cad.logits_for_domain(fmap, d_id)
        print(f"[Sanity] CaD domain {d_id}:", {"logits":tuple(logits.shape), "cls":tuple(cls.shape), "tokens":tuple(toks.shape)})

    # quantum delta
    base_logits = cnn_logits_from_resnet(x)
    qlogits, delta, g = qadapter.forward_logits(base_logits, v, t, dt)
    print("[Sanity] Quantum:", {"base":tuple(base_logits.shape),"delta":tuple(delta.shape),"g_minmax":(float(g.min()), float(g.max()))})

sanity_check_shapes()

# ============================================================
# (12) Training loop: sessions (simulate domains) + absorb + centers
# ============================================================
def compute_domain_center(session_id, loader):
    """
    Paper uses KMeans per domain to store domain center (Eq.16).
    Since it is 1 center per domain, mean(feature) == 1-cluster KMeans center.
    Use image tokens feature map WITHOUT CLS (paper), after adapters.
    """
    cad.eval()
    feats = []
    with torch.no_grad():
        for x,y,txt,_ in loader:
            x = x.to(DEVICE)
            fmap = resnet_spatial(x)
            _, img_tokens = cad.forward_tokens(fmap, domain_id=session_id)  # [B,N,d]
            f = img_tokens.mean(dim=1)  # [B,d] mean over tokens
            feats.append(f.cpu())
    Fm = torch.cat(feats, dim=0).mean(dim=0).numpy().astype(np.float32)  # [d]
    return Fm

def train_sessions():
    history = []
    for s in range(N_SESSIONS):
        print(f"\n==============================")
        print(f"Session {s+1}/{N_SESSIONS} (domain {s}) TRAIN")
        print("==============================")

        ds = NIHListDataset(session_lists[s])
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

        opt = make_session_optimizer(s)
        cad.train()
        qadapter.train()

        step = 0
        t0 = time.time()

        for ep in range(EPOCHS_PER_SESSION):
            for x,y,txt,_ in loader:
                x = x.to(DEVICE); y = y.to(DEVICE)
                fmap = resnet_spatial(x)

                # memory dt + pmem (for your regularizer)
                v = pool2048_from_fmap(fmap)
                t = text_embed(list(txt))
                qkey = mem.query_key(v,t)
                idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
                dt = mem.retrieve_dt(idxs)
                p_mem = mem.pmem_from_neighbors(idxs, sims, tau=TAU_MEM)

                # CaD logits (train c_current + d_s + fc_s)
                cad_logits, cls_feat, _ = forward_cad_train(fmap, session_id=s)

                # add quantum residual on top of CaD logits (hybrid)
                logits, _, g = qadapter.forward_logits(cad_logits, v, t, dt)

                # losses: BCE + λ*MLCL + memprob (your)
                loss_bce = crit_bce(logits, y)
                loss_mlcl = mlcl_loss(cls_feat, y, temp=TEMP_MLCL)
                p_hat = torch.sigmoid(logits)
                loss_mem = crit_mem(p_hat, p_mem)

                loss = loss_bce + LAMBDA_MLCL * loss_mlcl + LAMBDA_MEMPROB * loss_mem

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

                step += 1
                if step % LOG_EVERY == 0:
                    print(f"[sess{s}] step {step:4d}  loss={loss.item():.4f} "
                          f"bce={loss_bce.item():.4f} mlcl={loss_mlcl.item():.4f} mem={loss_mem.item():.4f} "
                          f"g_mean={float(g.mean().detach().cpu()):.3f}  elapsed={time.time()-t0:.1f}s")

                if step >= MAX_STEPS_PER_SESSION:
                    break
            if step >= MAX_STEPS_PER_SESSION:
                break

        # ---- Absorb current C-adapter into running (Eq.5-8)
        cad.eval()
        with torch.no_grad():
            absorb_cadapter_(cad.c_running, cad.c_current, t=s+1)
            # reset c_current to running weights (so next session starts from absorbed)
            cad.c_current.load_state_dict(cad.c_running.state_dict())

        # ---- Freeze this session's D-adapter + FC (paper freezes after training)
        cad.d_bank.freeze_domain(s)
        for p in cad.fc_bank[s].parameters():
            p.requires_grad = False

        # ---- Store domain center (Eq.16) using session data
        center_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        center = compute_domain_center(s, center_loader)
        cad.domain_centers[s] = center
        print(f"[sess{s}] stored domain center shape:", center.shape)

        history.append({"session": s, "steps": step, "train_sec": time.time()-t0})

    return pd.DataFrame(history)

print("\n[Stage] Train sessions + absorb + centers ...")
train_log = train_sessions()
train_log.to_csv(os.path.join(OUT_DIR, "train_sessions_log.csv"), index=False)
print("[Stage] Saved:", os.path.join(OUT_DIR, "train_sessions_log.csv"))

# ============================================================
# (13) Evaluation helpers (A)(B)(C)(D)
# ============================================================
def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p_pred >= lo) & (p_pred < hi) if i < n_bins-1 else (p_pred >= lo) & (p_pred <= hi)
        if m.sum() == 0:
            continue
        conf = p_pred[m].mean()
        acc  = y_true[m].mean()
        ece += (m.sum()/len(y_true)) * abs(acc - conf)
    return float(ece)

def brier_binary(y_true, p_pred):
    y_true = y_true.astype(np.float32)
    p_pred = p_pred.astype(np.float32)
    return float(np.mean((p_pred - y_true)**2))

def per_class_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        ecev = ece_binary(yt.astype(np.float32), yp.astype(np.float32), n_bins=15)
        brv  = brier_binary(yt, yp)
        rows.append([lbl, aucv, apv, f1v, ecev, brv])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5","ECE","Brier"])
    macro_auc = float(np.nanmean(df["ROC_AUC"].values))
    macro_ap  = float(np.nanmean(df["PR_AUC"].values))
    macro_f1  = float(np.nanmean(df["F1@0.5"].values))
    macro_ece = float(np.nanmean(df["ECE"].values))
    macro_br  = float(np.nanmean(df["Brier"].values))
    return df, {"macro_auc":macro_auc,"macro_ap":macro_ap,"macro_f1":macro_f1,"macro_ece":macro_ece,"macro_brier":macro_br}

@torch.no_grad()
def collect_preds_A_cnn_only(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = cnn_logits_from_resnet(x)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_B_quantum_no_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        dt = torch.zeros((x.size(0), dt_cond_dim), device=DEVICE, dtype=torch.float32)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_C_quantum_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))

        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_D_cad_quantum_memory(loader, max_batches=None):
    """
    CaD inference EXACT idea:
      For each test sample, compute feature Fi for each domain-i adapter set,
      compute distance to saved center Ai, pick argmin (Eq.16),
      then use the corresponding (D-adapter_i + FC_i) together with shared C-adapter.
    Here we also add quantum residual on top (your hybrid).
    """
    cad.eval(); qadapter.eval()
    centers = [c for c in cad.domain_centers if c is not None]
    T = len(centers)
    A = torch.tensor(np.stack(centers, axis=0), device=DEVICE, dtype=torch.float32)  # [T,d]

    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)

        fmap = resnet_spatial(x)
        # memory dt
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        # compute Fi and logits for each domain branch
        logits_all = []
        feat_all = []
        for di in range(T):
            logits_di, cls_di, img_tokens_di = cad.logits_for_domain(fmap, di)
            # feature Fi uses token map WITHOUT CLS (paper); take mean over tokens
            fi = img_tokens_di.mean(dim=1)  # [B,d]
            logits_all.append(logits_di.unsqueeze(1))  # [B,1,C]
            feat_all.append(fi.unsqueeze(1))           # [B,1,d]
        logits_all = torch.cat(logits_all, dim=1)  # [B,T,C]
        feat_all   = torch.cat(feat_all, dim=1)    # [B,T,d]

        # distances to centers: ||A_i - F_i||^2 (Eq.16 analog)
        # A: [T,d], feat_all: [B,T,d]
        dist = ((feat_all - A.unsqueeze(0))**2).sum(dim=2)  # [B,T]
        dom_id = torch.argmin(dist, dim=1)                  # [B]

        # pick logits by dom_id
        Bsz = x.size(0)
        picked = logits_all[torch.arange(Bsz, device=DEVICE), dom_id, :]  # [B,C]

        # add quantum residual on top of picked CaD logits
        logits, _, _ = qadapter.forward_logits(picked, v, t, dt)
        p = torch.sigmoid(logits)

        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())

    return np.concatenate(Ys,0), np.concatenate(Ps,0)

# ============================================================
# (14) Final evaluation
# ============================================================
print("\n==============================")
print("(A) CNN-only (no CaD, no quantum, no memory)")
print("==============================")
Y0, P0 = collect_preds_A_cnn_only(test_loader, max_batches=TEST_MAX_BATCHES)
df0, m0 = per_class_metrics(Y0, P0, UNIFIED_LABELS)
print(df0); print("[A] Macro:", m0)
df0.to_csv(os.path.join(OUT_DIR, "per_class_A_cnn_only.csv"), index=False)

print("\n==============================")
print("(B) Quantum adapter (dt=0, no retrieval)")
print("==============================")
Y1, P1 = collect_preds_B_quantum_no_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df1, m1 = per_class_metrics(Y1, P1, UNIFIED_LABELS)
print(df1); print("[B] Macro:", m1)
df1.to_csv(os.path.join(OUT_DIR, "per_class_B_quantum_no_memory.csv"), index=False)

print("\n==============================")
print("(C) Quantum adapter + Memory retrieval")
print("==============================")
Y2, P2 = collect_preds_C_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df2, m2 = per_class_metrics(Y2, P2, UNIFIED_LABELS)
print(df2); print("[C] Macro:", m2)
df2.to_csv(os.path.join(OUT_DIR, "per_class_C_quantum_memory.csv"), index=False)

print("\n==============================")
print("(D) CaD (C-adapter absorbed + per-domain D-adapter/FC) + Domain-ID routing + Quantum + Memory")
print("==============================")
Y3, P3 = collect_preds_D_cad_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df3, m3 = per_class_metrics(Y3, P3, UNIFIED_LABELS)
print(df3); print("[D] Macro:", m3)
df3.to_csv(os.path.join(OUT_DIR, "per_class_D_cad_quantum_memory.csv"), index=False)

summary = {
    "cad_paper_alignment": {
        "C_adapter_eq_2_4": True,
        "Absorb_eq_5_8": True,
        "D_adapter_eq_9_15": True,
        "Domain_id_eq_16": True,
        "Loss_eq_17_19": True
    },
    "estimator_backend": EST_BACKEND,
    "config": {
        "N_SESSIONS": N_SESSIONS,
        "CAD_DPRIME": CAD_DPRIME,
        "DQ_MAIN":DQ_MAIN, "VQC_REPS":VQC_REPS, "N_OBS":N_OBS,
        "DELTA_LOGIT_SCALE":DELTA_LOGIT_SCALE,
        "K_RETRIEVE":K_RETRIEVE,"TAU_MEM":TAU_MEM,"LAMBDA_MEMPROB":LAMBDA_MEMPROB,
        "LAMBDA_MLCL":LAMBDA_MLCL,"TEMP_MLCL":TEMP_MLCL
    },
    "memory": {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
               "has_v_img":mem.v_img is not None,"has_prototypes":mem.prototypes is not None,
               "has_base_conf":mem.base_conf is not None,"has_metadata":mem.metadata_available,
               "sklearn_index":mem.index_sklearn is not None},
    "metrics_A_cnn_only": m0,
    "metrics_B_quantum_no_memory": m1,
    "metrics_C_quantum_memory": m2,
    "metrics_D_cad_quantum_memory": m3
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Saved outputs to:", OUT_DIR)
print("  - train_sessions_log.csv")
print("  - per_class_A_cnn_only.csv")
print("  - per_class_B_quantum_no_memory.csv")
print("  - per_class_C_quantum_memory.csv")
print("  - per_class_D_cad_quantum_memory.csv")
print("  - summary.json")

Device: cuda

[Sanity] Config: {'BATCH_SIZE': 16, 'N_SESSIONS': 3, 'CAD_DPRIME': 256, 'DQ_MAIN': 4, 'K_RETRIEVE': 8, 'LAMBDA_MLCL': 0.1, 'TEMP_MLCL': 0.2}
Qiskit version: 2.3.0
Qiskit-ML version: 0.9.0
Using estimator: qiskit_aer.primitives.EstimatorV2


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2026-02-27 11:34:55.000389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772192095.227896      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772192095.291313      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772192095.833392      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772192095.833438      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772192095.833441      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[Sanity] ResNet load strict=False | missing: 267 | unexpected: 320

[Sanity] NIH sizes: {'stream_total': 86524, 'calib': 512, 'sessions': [28841, 28841, 28842], 'test': 25596}


[Sanity] Loaded index_sklearn.joblib (kneighbors available).

[Sanity] Memory loaded: {'N': 30000, 'D': 12, 't_desc_dim': 7, 'sizeGB': 0.01169471, 'has_v_img': True, 'has_prototypes': True, 'has_base_conf': True, 'has_metadata': True, 'sklearn_index': True}

[Sanity] fmap/tokens: {'fmap': (16, 2048, 7, 7), 'xcls': (16, 1, 2048), 'ximg': (16, 49, 2048), 'H': 7, 'W': 7}
[Sanity] memory dt: {'dt': (16, 7), 'sims_minmax': (-0.7192796468734741, -0.681080162525177)}
[Sanity] CaD domain 0: {'logits': (16, 5), 'cls': (16, 2048), 'tokens': (16, 49, 2048)}
[Sanity] CaD domain 1: {'logits': (16, 5), 'cls': (16, 2048), 'tokens': (16, 49, 2048)}
[Sanity] Quantum: {'base': (16, 5), 'delta': (16, 5), 'g_minmax': (0.5378372073173523, 0.5378385782241821)}

[Stage] Train sessions + absorb + centers ...

Session 1/3 (domain 0) TRAIN
[sess0] step   10  loss=14.0380 bce=5.2447 mlcl=0.0000 mem=17.5866 g_mean=0.538  elapsed=89.8s
[sess0] step   20  loss=12.8673 bce=4.9192 mlcl=0.3384 mem=15.8286 g_mean=0.538

# **1st March**

In [9]:
!pip -q install qiskit-aer

import qiskit_aer
print("qiskit_aer version:", qiskit_aer.__version__)

qiskit_aer version: 0.17.2


# This version fixes the earlier low-AUC problem. It keeps the strong ResNet predictions, starts CaD adapters neutrally, avoids double-counting features, and adds memory-guided quantum correction plus domain-specific learning without immediately damaging the baseline. 


In [ ]:
# ============================================================
# NIH Domain-Incremental Learning (CaD-aligned) + Quantum + Memory
# ✅ FIXED to preserve high AUROC baseline
# Key fixes vs your low-AUROC CaD cell:
#   1) ResNet load + logits EXACTLY like your high-AUROC baseline (uses ckpt head)
#   2) CaD DomainFC heads are initialized from pretrained resnet_full.fc
#   3) C/D adapters start as identity (W_up=0, conv=0) so CaD starts == baseline
#   4) Remove double-counting shortcut: M_out = M_c + M_d - M_in
# ============================================================

import os, json, math, time, random, inspect
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# -------------------------
# (0) Repro
# -------------------------
SEED = 7
def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# (1) Paths (yours)
# -------------------------
MEM_DIR = "/kaggle/input/datasets/zarinn/memory/MEMORY"

RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

OUT_DIR = "/kaggle/working/outputs_cad_aligned_fixed_auc"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# (2) Speed knobs
# -------------------------
BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

CALIBRATION_N = 512
EPOCHS_CALIB  = 1

# Domain sessions (simulate T sessions on NIH stream)
N_SESSIONS    = 3
EPOCHS_PER_SESSION = 1
MAX_STEPS_PER_SESSION = 80
LOG_EVERY     = 10

TEST_MAX_BATCHES = None

# Memory knobs
K_RETRIEVE     = 8
TAU_MEM        = 10.0
LAMBDA_MEMPROB = 0.5

# CaD loss knob (Eq.17)
LAMBDA_MLCL    = 0.1
TEMP_MLCL      = 0.2

# Quantum residual adapter
DQ_MAIN   = 4
VQC_REPS  = 1
N_OBS     = 2
LR_CLASSICAL = 1e-3
LR_QUANTUM   = 1e-2
DELTA_LOGIT_SCALE = 0.35

# CaD bottleneck d'
CAD_DPRIME = 256

# Whether to fine-tune domain FC heads (usually keep False to preserve baseline)
TRAIN_DOMAIN_FC = False

print("\n[Sanity] Config:",
      {"BATCH_SIZE":BATCH_SIZE, "N_SESSIONS":N_SESSIONS,
       "CAD_DPRIME":CAD_DPRIME, "DQ_MAIN":DQ_MAIN,
       "K_RETRIEVE":K_RETRIEVE, "LAMBDA_MLCL":LAMBDA_MLCL, "TEMP_MLCL":TEMP_MLCL,
       "TRAIN_DOMAIN_FC": TRAIN_DOMAIN_FC})

# ============================================================
# (3) Qiskit (EstimatorV2 + EstimatorQNN)
# ============================================================
import qiskit
import qiskit_machine_learning

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

print("Qiskit version:", getattr(qiskit, "__version__", "unknown"))
print("Qiskit-ML version:", getattr(qiskit_machine_learning, "__version__", "unknown"))

ESTIMATOR = None
EST_BACKEND = None
try:
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
    ESTIMATOR = AerEstimatorV2()
    EST_BACKEND = "qiskit_aer.primitives.EstimatorV2"
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator
        ESTIMATOR = StatevectorEstimator()
        EST_BACKEND = "qiskit.primitives.StatevectorEstimator"
    except Exception as e:
        raise ImportError("No compatible V2 Estimator found. Install qiskit-aer.") from e

sig = inspect.signature(ESTIMATOR.run)
params = list(sig.parameters.keys())
if ("observables" in params) and ("circuits" in params):
    raise RuntimeError(f"Estimator looks like V1 ({EST_BACKEND}). Need V2 for EstimatorQNN path.")
print("Using estimator:", EST_BACKEND)

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

_AER_BACKEND = AerSimulator()
_PM = generate_preset_pass_manager(optimization_level=1, backend=_AER_BACKEND)

def build_qnn(n_qubits, reps=1, entanglement="circular", n_obs=2):
    fm = zz_feature_map(feature_dimension=n_qubits, reps=2, entanglement=entanglement)
    ans = real_amplitudes(num_qubits=n_qubits, reps=reps, entanglement=entanglement)

    qc = QuantumCircuit(n_qubits)
    qc = qc.compose(fm, inplace=False, wrap=False)
    qc = qc.compose(ans, inplace=False, wrap=False)

    qc = qc.decompose(reps=15)
    qc = _PM.run(qc)

    obs = []
    n_obs = int(min(max(1, n_obs), n_qubits))
    for i in range(n_obs):
        pauli = ["I"] * n_qubits
        pauli[i] = "Z"
        obs.append(SparsePauliOp("".join(pauli)))

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm.parameters,
        weight_params=ans.parameters,
        observables=obs,
        input_gradients=True,
        estimator=ESTIMATOR,
    )
    return qnn, len(fm.parameters), qnn.num_weights, len(obs)

# ============================================================
# (4) Encoders: BiomedBERT + ResNet50 (frozen)
#     ✅ ResNet loader fixed to match your high-AUROC baseline cell
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torchvision
from torchvision import transforms

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

def _strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p): kk = kk[len(p):]
        out[kk] = v
    return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu")
    if isinstance(ck, dict) and "state_dict" in ck: sd = ck["state_dict"]
    elif isinstance(ck, dict) and "model" in ck: sd = ck["model"]
    elif isinstance(ck, dict) and "net" in ck: sd = ck["net"]
    else: sd = ck
    return _strip_prefix(sd)

def load_biomedbert():
    tok_dir = os.path.dirname(BIOMED_CKPT)
    try:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

    model = AutoModel.from_pretrained(TEXT_MODEL_NAME)
    if os.path.exists(BIOMED_CKPT):
        sd = load_state_dict_any(BIOMED_CKPT)
        sd = {k:v for k,v in sd.items() if not k.startswith("cls.")}
        model.load_state_dict(sd, strict=False)

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return tokenizer, model

tokenizer, text_model = load_biomedbert()
DT_DIM = int(text_model.config.hidden_size)

@torch.no_grad()
def text_embed(text_list, max_len=64):
    enc = tokenizer(text_list, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    out = text_model(**enc, return_dict=True)
    t = out.last_hidden_state[:, 0, :]  # CLS
    return F.normalize(t, dim=1)

def load_resnet50_full_and_spatial():
    m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(2048, C)

    sd = load_state_dict_any(RESNET_CKPT)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("[Sanity] ResNet load strict=False | missing:", len(missing), "| unexpected:", len(unexpected))

    m.eval().to(DEVICE)
    for p in m.parameters():
        p.requires_grad = False

    @torch.no_grad()
    def spatial(x):
        b = m
        x = b.conv1(x); x = b.bn1(x); x = b.relu(x); x = b.maxpool(x)
        x = b.layer1(x); x = b.layer2(x); x = b.layer3(x); x = b.layer4(x)  # [B,2048,7,7]
        return x

    @torch.no_grad()
    def logits(x):
        return m(x)  # uses ckpt head

    return m, spatial, logits

resnet_full, resnet_spatial, cnn_logits_from_resnet = load_resnet50_full_and_spatial()

img_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def pool2048_from_fmap(fmap):
    v = F.adaptive_avg_pool2d(fmap, (1,1)).flatten(1)
    return F.normalize(v, dim=1)

# ============================================================
# (5) NIH dataset + session split
# ============================================================
def read_list(path):
    with open(path, "r") as f:
        return [x.strip() for x in f if x.strip()]

nih_df = pd.read_csv(NIH_CSV)
nih_df = nih_df.rename(columns={"Image Index":"image", "Finding Labels":"labels"})
nih_row = {r["image"]: r for _, r in nih_df.iterrows()}

def labels_to_multi_hot(lbl_str):
    parts = set([p.strip() for p in str(lbl_str).split("|")])
    y = np.zeros((C,), dtype=np.float32)
    for i, name in enumerate(UNIFIED_LABELS):
        if name == "Pleural Effusion":
            if "Effusion" in parts: y[i] = 1.0
        else:
            if name in parts: y[i] = 1.0
    return y

def make_text_descriptor(r):
    view = r.get("View Position", r.get("ViewPosition", ""))
    age  = r.get("Patient Age", r.get("PatientAge", ""))
    sex  = r.get("Patient Gender", r.get("PatientGender", ""))
    return f"Chest X-ray. View: {view}. Patient age: {age}. Sex: {sex}."

class NIHListDataset(Dataset):
    def __init__(self, names):
        self.items = list(names)
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        name = self.items[idx]
        img = Image.open(os.path.join(NIH_IMG, name)).convert("RGB")
        x = img_tfms(img)
        r = nih_row.get(name, {})
        y = labels_to_multi_hot(r.get("labels", ""))
        txt = make_text_descriptor(r)
        return x, torch.tensor(y, dtype=torch.float32), txt, name

stream_list = read_list(NIH_STREAM)
test_list   = read_list(NIH_TEST)

calib_list = stream_list[:CALIBRATION_N]

def split_sessions(lst, n_sessions):
    n = len(lst)
    splits = []
    for s in range(n_sessions):
        a = (n*s)//n_sessions
        b = (n*(s+1))//n_sessions
        splits.append(lst[a:b])
    return splits

session_lists = split_sessions(stream_list, N_SESSIONS)

calib_loader = DataLoader(NIHListDataset(calib_list), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(NIHListDataset(test_list),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("\n[Sanity] NIH sizes:", {"stream_total":len(stream_list),
                             "calib":len(calib_list),
                             "sessions":[len(x) for x in session_lists],
                             "test":len(test_list)})

# ============================================================
# (6) Memory bundle (ALL files) + retrieval
# ============================================================
import faiss
try:
    import joblib
except Exception:
    joblib = None

class MemoryBundle:
    def __init__(self, mem_dir):
        self.mem_dir = mem_dir
        self.path_embeddings = os.path.join(mem_dir, "embeddings.npy")
        self.path_labels     = os.path.join(mem_dir, "labels.npy")
        self.path_tdesc      = os.path.join(mem_dir, "t_desc.npy")
        self.path_vimg       = os.path.join(mem_dir, "v_img.npy")
        self.path_protos     = os.path.join(mem_dir, "prototypes.npy")
        self.path_protolbl   = os.path.join(mem_dir, "prototypes_labels.json")
        self.path_baseconf   = os.path.join(mem_dir, "base_conf.npy")
        self.path_meta       = os.path.join(mem_dir, "metadata.jsonl")
        self.path_idx_skl    = os.path.join(mem_dir, "index_sklearn.joblib")

        assert os.path.exists(self.path_embeddings), "Missing embeddings.npy"
        assert os.path.exists(self.path_labels), "Missing labels.npy"

        E = np.load(self.path_embeddings)
        Y = np.load(self.path_labels)

        if E.dtype != np.float32: E = E.astype(np.float32, copy=False)
        E = np.ascontiguousarray(E)
        self.E = E
        self.N, self.D = self.E.shape

        if Y.ndim == 1:
            Y_oh = np.zeros((Y.shape[0], C), dtype=np.float32)
            Y_oh[np.arange(Y.shape[0]), Y.astype(np.int64)] = 1.0
            Y = Y_oh
        if Y.dtype != np.float32: Y = Y.astype(np.float32, copy=False)
        self.Y = np.ascontiguousarray(Y)

        self.t_desc = None
        if os.path.exists(self.path_tdesc):
            tdesc = np.load(self.path_tdesc)
            if tdesc.dtype != np.float32: tdesc = tdesc.astype(np.float32, copy=False)
            self.t_desc = np.ascontiguousarray(tdesc)
        self.t_desc_dim = int(self.t_desc.shape[1]) if self.t_desc is not None else 0

        self.v_img = None
        if os.path.exists(self.path_vimg):
            vimg = np.load(self.path_vimg)
            if vimg.dtype != np.float32: vimg = vimg.astype(np.float32, copy=False)
            self.v_img = np.ascontiguousarray(vimg)

        self.prototypes = None
        self.prototypes_labels = None
        if os.path.exists(self.path_protos):
            P = np.load(self.path_protos)
            if P.dtype != np.float32: P = P.astype(np.float32, copy=False)
            self.prototypes = np.ascontiguousarray(P)
        if os.path.exists(self.path_protolbl):
            with open(self.path_protolbl, "r") as f:
                self.prototypes_labels = json.load(f)

        self.base_conf = None
        if os.path.exists(self.path_baseconf):
            bc = np.load(self.path_baseconf).astype(np.float32, copy=False).reshape(-1)
            if bc.shape[0] == self.N:
                self.base_conf = np.clip(bc, 0.0, 1.0)

        self.metadata_available = os.path.exists(self.path_meta)

        self.index_sklearn = None
        if os.path.exists(self.path_idx_skl) and joblib is not None:
            try:
                obj = joblib.load(self.path_idx_skl)
                if hasattr(obj, "kneighbors"):
                    self.index_sklearn = obj
                    print("[Sanity] Loaded index_sklearn.joblib (kneighbors available).")
            except Exception as e:
                print("[Warn] Could not load index_sklearn.joblib; using FAISS. Reason:", repr(e))

        # FAISS cosine via normalized IP
        E2 = self.E.copy()
        faiss.normalize_L2(E2)
        self.faiss_index = faiss.IndexFlatIP(self.D)
        self.faiss_index.add(E2)
        self._E_norm = E2

        self._Y_t = torch.tensor(self.Y, device=DEVICE, dtype=torch.float32)
        self._td_t = torch.tensor(self.t_desc, device=DEVICE, dtype=torch.float32) if self.t_desc is not None else None

    def memory_gb(self):
        total = 0
        for fn in ["embeddings.npy","labels.npy","t_desc.npy","v_img.npy","prototypes.npy","base_conf.npy","metadata.jsonl","index_sklearn.joblib"]:
            p = os.path.join(self.mem_dir, fn)
            if os.path.exists(p):
                total += os.path.getsize(p)
        return total / 1e9

    @torch.no_grad()
    def query_key(self, v, t):
        dv, dt = v.shape[1], t.shape[1]
        if self.D == dv + dt:
            q = torch.cat([v, t], dim=1)
        elif self.D == dv:
            q = v
        elif self.D == dt:
            q = t
        else:
            torch.manual_seed(SEED)
            W = torch.randn(dv + dt, self.D, device=v.device) / math.sqrt(dv + dt)
            q = torch.cat([v, t], dim=1) @ W
        return F.normalize(q, dim=1)

    @torch.no_grad()
    def retrieve(self, q, k=16):
        qn = F.normalize(q, dim=1)
        q_np = np.ascontiguousarray(qn.detach().cpu().numpy().astype(np.float32, copy=False))

        if self.index_sklearn is not None:
            try:
                dists, idxs = self.index_sklearn.kneighbors(q_np, n_neighbors=min(k, self.N), return_distance=True)
                sims = -dists.astype(np.float32, copy=False)
                return idxs.astype(np.int64, copy=False), sims
            except Exception:
                pass

        sims, idxs = self.faiss_index.search(q_np, min(k, self.N))
        return idxs.astype(np.int64, copy=False), sims.astype(np.float32, copy=False)

    @torch.no_grad()
    def retrieve_dt(self, idxs):
        if self._td_t is None:
            return torch.zeros((idxs.shape[0], 1), device=DEVICE, dtype=torch.float32)
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        dt = self._td_t[idxs_t].mean(dim=1)
        return F.normalize(dt, dim=1)

    @torch.no_grad()
    def pmem_from_neighbors(self, idxs, sims, tau=10.0, eps=1e-8):
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        sims_t = torch.tensor(sims, device=DEVICE, dtype=torch.float32)

        neigh_y = self._Y_t[idxs_t]  # [B,k,C]
        w = torch.softmax(tau * sims_t, dim=1)  # [B,k]

        if self.base_conf is not None:
            conf = torch.tensor(self.base_conf, device=DEVICE, dtype=torch.float32)[idxs_t]  # [B,k]
            w = w * conf
            w = w / (w.sum(dim=1, keepdim=True) + 1e-12)

        w = w.unsqueeze(-1)
        p = (w * neigh_y).sum(dim=1)
        return torch.clamp(p, eps, 1 - eps)

mem = MemoryBundle(MEM_DIR)
dt_cond_dim = mem.t_desc_dim if mem.t_desc_dim > 0 else 1

print("\n[Sanity] Memory loaded:",
      {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
       "has_v_img":mem.v_img is not None, "has_prototypes":mem.prototypes is not None,
       "has_base_conf":mem.base_conf is not None, "has_metadata":mem.metadata_available,
       "sklearn_index":mem.index_sklearn is not None})

# ============================================================
# (7) CaD tokenization + C/D adapters (FIXED)
# ============================================================
def fmap_to_tokens(fmap):
    B, d, H, W = fmap.shape
    x_img = fmap.permute(0,2,3,1).reshape(B, H*W, d)  # [B,49,2048]
    x_cls = fmap.mean(dim=(2,3), keepdim=False).unsqueeze(1)  # [B,1,2048]
    return x_cls, x_img, H, W

class CAdapter(nn.Module):
    """
    Eq (2)-(4), identity-start by zeroing W_up.
    """
    def __init__(self, d=2048, dprime=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in):
        M_mid = F.relu(self.down(self.ln1(M_in)))
        M_til = self.up(self.ln2(M_mid))
        M_out = F.relu(M_til + M_in)
        return M_out

@torch.no_grad()
def absorb_cadapter_(cadapter_running: CAdapter, cadapter_current: CAdapter, t: int):
    """
    Eq (5)-(8): running <- (1/t)*((t-1)*running + current)
    """
    for (n1,p1), (n2,p2) in zip(cadapter_running.named_parameters(), cadapter_current.named_parameters()):
        assert n1 == n2
        p1.data.mul_((t-1)/t).add_(p2.data, alpha=(1.0/t))

class DAdapter(nn.Module):
    """
    Eq (9)-(15), identity-start by zeroing conv and W_up.
    """
    def __init__(self, d=2048, dprime=256, k=3):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.conv = nn.Conv2d(dprime, dprime, kernel_size=k, padding=k//2)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.conv.weight)
        if self.conv.bias is not None: nn.init.zeros_(self.conv.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in, H=7, W=7):
        M_mid = F.relu(self.down(self.ln1(M_in)))  # (9)
        x_cls = M_mid[:, :1, :]
        x_img = M_mid[:, 1:, :]
        x_grid = x_img.reshape(x_img.size(0), H, W, x_img.size(2)).permute(0,3,1,2).contiguous()
        x_grid2 = self.conv(x_grid)
        x_img2 = x_grid2.permute(0,2,3,1).reshape(x_img.size(0), H*W, x_img.size(2)).contiguous()
        M_til_mid = F.relu(torch.cat([x_cls, x_img2], dim=1))
        M_til = self.up(self.ln2(M_til_mid))
        M_out = F.relu(M_til + M_in)
        return M_out

class DAdapterBank(nn.Module):
    def __init__(self, n_domains, d=2048, dprime=256):
        super().__init__()
        self.adapters = nn.ModuleList([DAdapter(d=d, dprime=dprime) for _ in range(n_domains)])

    def forward(self, M_in, domain_id, H=7, W=7):
        return self.adapters[domain_id](M_in, H=H, W=W)

    def freeze_domain(self, domain_id):
        for p in self.adapters[domain_id].parameters():
            p.requires_grad = False

class DomainFC(nn.Module):
    def __init__(self, d=2048, n_classes=5):
        super().__init__()
        self.fc = nn.Linear(d, n_classes)
    def forward(self, cls_token):
        return self.fc(cls_token)

class CaDModel(nn.Module):
    """
    Shared C-adapter absorbed across sessions.
    D-adapter + FC per session.
    ✅ FIX: combine without double-counting M_in:
        M_out = M_c + M_d - M_in
    ✅ FIX: init all FC from pretrained resnet_full.fc so CaD starts == baseline.
    """
    def __init__(self, n_sessions, d=2048, dprime=256, n_classes=5, init_fc_from_resnet=True):
        super().__init__()
        self.c_running = CAdapter(d=d, dprime=dprime)
        self.c_current = CAdapter(d=d, dprime=dprime)

        self.d_bank = DAdapterBank(n_domains=n_sessions, d=d, dprime=dprime)
        self.fc_bank = nn.ModuleList([DomainFC(d=d, n_classes=n_classes) for _ in range(n_sessions)])

        if init_fc_from_resnet:
            base_sd = resnet_full.fc.state_dict()
            for di in range(n_sessions):
                self.fc_bank[di].fc.load_state_dict(base_sd, strict=True)

        self.domain_centers = [None for _ in range(n_sessions)]

    def forward_tokens(self, fmap, domain_id):
        x_cls, x_img, H, W = fmap_to_tokens(fmap)
        M_in = torch.cat([x_cls, x_img], dim=1)
        M_c = self.c_running(M_in)
        M_d = self.d_bank(M_in, domain_id=domain_id, H=H, W=W)
        M_out = M_c + M_d - M_in   # ✅ critical fix
        cls = M_out[:, :1, :].squeeze(1)
        img_tokens = M_out[:, 1:, :]
        return cls, img_tokens

    def logits_for_domain(self, fmap, domain_id):
        cls, img_tokens = self.forward_tokens(fmap, domain_id)
        logits = self.fc_bank[domain_id](cls)
        return logits, cls, img_tokens

cad = CaDModel(n_sessions=N_SESSIONS, d=2048, dprime=CAD_DPRIME, n_classes=C, init_fc_from_resnet=True).to(DEVICE)

# Freeze FC by default to retain baseline
for di in range(N_SESSIONS):
    for p in cad.fc_bank[di].parameters():
        p.requires_grad = TRAIN_DOMAIN_FC

# ============================================================
# (8) Quantum residual adapter (unchanged)
# ============================================================
class NonLinearProjector(nn.Module):
    def __init__(self, in_dim, dq):
        super().__init__()
        h = max(64, 2*dq)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Linear(h, dq),
        )
    def forward(self, z):
        return self.net(z)

class QuantumResidualAdapter(nn.Module):
    """
    logits = base_logits + sigmoid(g(dt)) * scale * delta_logits
    """
    def __init__(self, dv=2048, dt=DT_DIM, dt_cond_dim=1, dq=4, n_classes=5, reps=1, n_obs=2):
        super().__init__()
        self.dq = dq
        self.dt_cond_dim = max(int(dt_cond_dim), 1)

        self.proj = NonLinearProjector(dv + dt, dq)
        self.dt_to_angles = nn.Sequential(
            nn.Linear(self.dt_cond_dim, max(8, 2*dq)),
            nn.GELU(),
            nn.Linear(max(8, 2*dq), dq),
        )
        self.dt_gate = nn.Sequential(
            nn.Linear(self.dt_cond_dim, 8),
            nn.GELU(),
            nn.Linear(8, 1),
        )

        qnn, _, n_w, nobs = build_qnn(dq, reps=reps, entanglement="circular", n_obs=n_obs)
        init_w = 0.01 * np.random.randn(n_w).astype(np.float32)
        self.q = TorchConnector(qnn, initial_weights=init_w)

        self.delta_head = nn.Sequential(
            nn.Linear(nobs, max(16, 2*dq)),
            nn.GELU(),
            nn.Linear(max(16, 2*dq), n_classes),
        )
        self._nobs = nobs

    def forward_delta(self, v, t, dt):
        z = torch.cat([v, t], dim=1)
        u = self.proj(z)
        u = math.pi * torch.tanh(u)

        mod = math.pi * torch.tanh(self.dt_to_angles(dt))
        u_mod = u + 0.25 * mod

        g = torch.sigmoid(self.dt_gate(dt))
        q_out = self.q(u_mod)
        delta = self.delta_head(q_out)
        return delta, g, q_out

    def forward_logits(self, base_logits, v, t, dt):
        delta, g, _ = self.forward_delta(v, t, dt)
        logits = base_logits + (g * DELTA_LOGIT_SCALE) * delta
        return logits, delta, g

qadapter = QuantumResidualAdapter(dq=DQ_MAIN, n_classes=C, dt_cond_dim=dt_cond_dim, reps=VQC_REPS, n_obs=N_OBS).to(DEVICE)

def split_params_for_opt(model):
    q_params, c_params = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "q." in n:
            q_params.append(p)
        else:
            c_params.append(p)
    return c_params, q_params

def make_opt(model, extra_params=None):
    c_params, q_params = split_params_for_opt(model)
    if extra_params is not None:
        c_params = list(c_params) + list(extra_params)
    return torch.optim.Adam([
        {"params": c_params, "lr": LR_CLASSICAL},
        {"params": q_params, "lr": LR_QUANTUM},
    ])

# ============================================================
# (9) Loss: BCE + Multi-label contrastive (Eq.17-19) + memprob
# ============================================================
crit_bce = nn.BCEWithLogitsLoss()
crit_mem = nn.BCELoss()

def label_similarity_r(y_i, y_j, eps=1e-9):
    yi = (y_i > 0.5).float()
    yj = (y_j > 0.5).float()
    inter = (yi*yj).sum(dim=-1)
    uni   = ((yi + yj) > 0.0).float().sum(dim=-1)
    return inter / (uni + eps)

def mlcl_loss(features, labels, temp=0.2, eps=1e-9):
    B = features.size(0)
    f = F.normalize(features, dim=1)
    sim = (f @ f.t()) / temp
    sim = sim - torch.eye(B, device=sim.device) * 1e9

    yi = labels.unsqueeze(1).expand(B,B,-1)
    yj = labels.unsqueeze(0).expand(B,B,-1)
    r = label_similarity_r(yi, yj)
    r = r * (1.0 - torch.eye(B, device=r.device))

    logp = F.log_softmax(sim, dim=1)
    loss_i = -(r * logp).sum(dim=1) / (r.sum(dim=1) + eps)
    return loss_i.mean()

# ============================================================
# (10) Optimizer per session (train c_current + D_s (+ FC_s if enabled) + qadapter)
# ============================================================
def make_session_optimizer(session_id):
    # running C frozen; current C trainable
    for p in cad.c_running.parameters():
        p.requires_grad = False
    for p in cad.c_current.parameters():
        p.requires_grad = True

    # only train this session's D (+ FC if enabled)
    for di in range(N_SESSIONS):
        for p in cad.d_bank.adapters[di].parameters():
            p.requires_grad = (di == session_id)
        for p in cad.fc_bank[di].parameters():
            p.requires_grad = (TRAIN_DOMAIN_FC and (di == session_id))

    train_params = (
        list(cad.c_current.parameters()) +
        list(cad.d_bank.adapters[session_id].parameters()) +
        (list(cad.fc_bank[session_id].parameters()) if TRAIN_DOMAIN_FC else [])
    )
    opt = make_opt(qadapter, extra_params=train_params)
    return opt

def forward_cad_train(fmap, session_id):
    x_cls, x_img, H, W = fmap_to_tokens(fmap)
    M_in = torch.cat([x_cls, x_img], dim=1)

    M_c = cad.c_current(M_in)
    M_d = cad.d_bank(M_in, domain_id=session_id, H=H, W=W)
    M_out = M_c + M_d - M_in   # ✅ critical fix

    cls = M_out[:, :1, :].squeeze(1)
    img_tokens = M_out[:, 1:, :]
    logits = cad.fc_bank[session_id](cls)
    return logits, cls, img_tokens

# ============================================================
# (11) Sanity checks (includes baseline-equality check)
# ============================================================
@torch.no_grad()
def sanity_check_shapes():
    x,y,txt,_ = next(iter(calib_loader))
    x = x.to(DEVICE)
    fmap = resnet_spatial(x)

    xcls, ximg, H, W = fmap_to_tokens(fmap)
    print("\n[Sanity] fmap/tokens:",
          {"fmap":tuple(fmap.shape),"xcls":tuple(xcls.shape),"ximg":tuple(ximg.shape),"H":H,"W":W})

    # baseline-equality check at init:
    # resnet_full(x) should match cad logits (any domain) because adapters are identity + FC cloned
    base_logits = cnn_logits_from_resnet(x)  # strong baseline
    cad_logits0, _, _ = cad.logits_for_domain(fmap, 0)
    diff = (base_logits - cad_logits0).abs().max().item()
    print("[Sanity] Baseline check max|resnet - cad(domain0)| =", diff)

    # memory retrieval dt
    v = pool2048_from_fmap(fmap)
    t = text_embed(list(txt))
    qkey = mem.query_key(v,t)
    idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
    dt = mem.retrieve_dt(idxs)
    print("[Sanity] memory dt:", {"dt":tuple(dt.shape), "sims_minmax":(float(np.min(sims)), float(np.max(sims)))})

    # quantum delta
    qlogits, delta, g = qadapter.forward_logits(base_logits, v, t, dt)
    print("[Sanity] Quantum:", {"base":tuple(base_logits.shape),"delta":tuple(delta.shape),"g_minmax":(float(g.min()), float(g.max()))})

sanity_check_shapes()

# ============================================================
# (12) Training loop: sessions + absorb + centers
# ============================================================
def compute_domain_center(session_id, loader):
    cad.eval()
    feats = []
    with torch.no_grad():
        for x,y,txt,_ in loader:
            x = x.to(DEVICE)
            fmap = resnet_spatial(x)
            _, img_tokens = cad.forward_tokens(fmap, domain_id=session_id)  # [B,N,d]
            f = img_tokens.mean(dim=1)  # [B,d]
            feats.append(f.cpu())
    Fm = torch.cat(feats, dim=0).mean(dim=0).numpy().astype(np.float32)
    return Fm

def train_sessions():
    history = []
    for s in range(N_SESSIONS):
        print(f"\n==============================")
        print(f"Session {s+1}/{N_SESSIONS} (domain {s}) TRAIN")
        print("==============================")

        ds = NIHListDataset(session_lists[s])
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

        opt = make_session_optimizer(s)
        cad.train()
        qadapter.train()

        step = 0
        t0 = time.time()

        for ep in range(EPOCHS_PER_SESSION):
            for x,y,txt,_ in loader:
                x = x.to(DEVICE); y = y.to(DEVICE)
                fmap = resnet_spatial(x)

                # memory dt + pmem
                v = pool2048_from_fmap(fmap)
                t = text_embed(list(txt))
                qkey = mem.query_key(v,t)
                idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
                dt = mem.retrieve_dt(idxs)
                p_mem = mem.pmem_from_neighbors(idxs, sims, tau=TAU_MEM)

                # CaD logits (train c_current + d_s (+ fc_s if enabled))
                cad_logits, cls_feat, _ = forward_cad_train(fmap, session_id=s)

                # quantum residual
                logits, _, g = qadapter.forward_logits(cad_logits, v, t, dt)

                # losses
                loss_bce = crit_bce(logits, y)
                loss_mlcl = mlcl_loss(cls_feat, y, temp=TEMP_MLCL)
                p_hat = torch.sigmoid(logits)
                loss_mem = crit_mem(p_hat, p_mem)

                loss = loss_bce + LAMBDA_MLCL * loss_mlcl + LAMBDA_MEMPROB * loss_mem

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

                step += 1
                if step % LOG_EVERY == 0:
                    print(f"[sess{s}] step {step:4d}  loss={loss.item():.4f} "
                          f"bce={loss_bce.item():.4f} mlcl={loss_mlcl.item():.4f} mem={loss_mem.item():.4f} "
                          f"g_mean={float(g.mean().detach().cpu()):.3f}  elapsed={time.time()-t0:.1f}s")

                if step >= MAX_STEPS_PER_SESSION:
                    break
            if step >= MAX_STEPS_PER_SESSION:
                break

        # ---- Absorb current C-adapter into running (Eq.5-8)
        cad.eval()
        with torch.no_grad():
            absorb_cadapter_(cad.c_running, cad.c_current, t=s+1)
            cad.c_current.load_state_dict(cad.c_running.state_dict())

        # ---- Freeze this session's D-adapter (+ FC if it was trainable)
        cad.d_bank.freeze_domain(s)
        if TRAIN_DOMAIN_FC:
            for p in cad.fc_bank[s].parameters():
                p.requires_grad = False

        # ---- Store domain center (Eq.16)
        center_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        center = compute_domain_center(s, center_loader)
        cad.domain_centers[s] = center
        print(f"[sess{s}] stored domain center shape:", center.shape)

        history.append({"session": s, "steps": step, "train_sec": time.time()-t0})

    return pd.DataFrame(history)

print("\n[Stage] Train sessions + absorb + centers ...")
train_log = train_sessions()
train_log.to_csv(os.path.join(OUT_DIR, "train_sessions_log.csv"), index=False)
print("[Stage] Saved:", os.path.join(OUT_DIR, "train_sessions_log.csv"))

# ============================================================
# (13) Evaluation helpers (A)(B)(C)(D)
# ============================================================
def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p_pred >= lo) & (p_pred < hi) if i < n_bins-1 else (p_pred >= lo) & (p_pred <= hi)
        if m.sum() == 0:
            continue
        conf = p_pred[m].mean()
        acc  = y_true[m].mean()
        ece += (m.sum()/len(y_true)) * abs(acc - conf)
    return float(ece)

def brier_binary(y_true, p_pred):
    y_true = y_true.astype(np.float32)
    p_pred = p_pred.astype(np.float32)
    return float(np.mean((p_pred - y_true)**2))

def per_class_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        ecev = ece_binary(yt.astype(np.float32), yp.astype(np.float32), n_bins=15)
        brv  = brier_binary(yt, yp)
        rows.append([lbl, aucv, apv, f1v, ecev, brv])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5","ECE","Brier"])
    macro_auc = float(np.nanmean(df["ROC_AUC"].values))
    macro_ap  = float(np.nanmean(df["PR_AUC"].values))
    macro_f1  = float(np.nanmean(df["F1@0.5"].values))
    macro_ece = float(np.nanmean(df["ECE"].values))
    macro_br  = float(np.nanmean(df["Brier"].values))
    return df, {"macro_auc":macro_auc,"macro_ap":macro_ap,"macro_f1":macro_f1,"macro_ece":macro_ece,"macro_brier":macro_br}

@torch.no_grad()
def collect_preds_A_cnn_only(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = cnn_logits_from_resnet(x)  # ✅ strong ckpt head
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_B_quantum_no_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        dt = torch.zeros((x.size(0), dt_cond_dim), device=DEVICE, dtype=torch.float32)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_C_quantum_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))

        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_D_cad_quantum_memory(loader, max_batches=None):
    cad.eval(); qadapter.eval()
    centers = [c for c in cad.domain_centers if c is not None]
    T = len(centers)
    A = torch.tensor(np.stack(centers, axis=0), device=DEVICE, dtype=torch.float32)  # [T,d]

    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)

        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        logits_all = []
        feat_all = []
        for di in range(T):
            logits_di, cls_di, img_tokens_di = cad.logits_for_domain(fmap, di)
            fi = img_tokens_di.mean(dim=1)  # [B,d]
            logits_all.append(logits_di.unsqueeze(1))  # [B,1,C]
            feat_all.append(fi.unsqueeze(1))           # [B,1,d]
        logits_all = torch.cat(logits_all, dim=1)  # [B,T,C]
        feat_all   = torch.cat(feat_all, dim=1)    # [B,T,d]

        dist = ((feat_all - A.unsqueeze(0))**2).sum(dim=2)  # [B,T]
        dom_id = torch.argmin(dist, dim=1)

        Bsz = x.size(0)
        picked = logits_all[torch.arange(Bsz, device=DEVICE), dom_id, :]  # [B,C]

        logits, _, _ = qadapter.forward_logits(picked, v, t, dt)
        p = torch.sigmoid(logits)

        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())

    return np.concatenate(Ys,0), np.concatenate(Ps,0)

# ============================================================
# (14) Final evaluation
# ============================================================
print("\n==============================")
print("(A) CNN-only (no CaD, no quantum, no memory)  ✅ should be HIGH again")
print("==============================")
Y0, P0 = collect_preds_A_cnn_only(test_loader, max_batches=TEST_MAX_BATCHES)
df0, m0 = per_class_metrics(Y0, P0, UNIFIED_LABELS)
print(df0); print("[A] Macro:", m0)
df0.to_csv(os.path.join(OUT_DIR, "per_class_A_cnn_only.csv"), index=False)

print("\n==============================")
print("(B) Quantum adapter (dt=0, no retrieval)")
print("==============================")
Y1, P1 = collect_preds_B_quantum_no_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df1, m1 = per_class_metrics(Y1, P1, UNIFIED_LABELS)
print(df1); print("[B] Macro:", m1)
df1.to_csv(os.path.join(OUT_DIR, "per_class_B_quantum_no_memory.csv"), index=False)

print("\n==============================")
print("(C) Quantum adapter + Memory retrieval")
print("==============================")
Y2, P2 = collect_preds_C_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df2, m2 = per_class_metrics(Y2, P2, UNIFIED_LABELS)
print(df2); print("[C] Macro:", m2)
df2.to_csv(os.path.join(OUT_DIR, "per_class_C_quantum_memory.csv"), index=False)

print("\n==============================")
print("(D) CaD (absorbed C + per-domain D/FC) + Domain-ID routing + Quantum + Memory  ✅ should NOT collapse")
print("==============================")
Y3, P3 = collect_preds_D_cad_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df3, m3 = per_class_metrics(Y3, P3, UNIFIED_LABELS)
print(df3); print("[D] Macro:", m3)
df3.to_csv(os.path.join(OUT_DIR, "per_class_D_cad_quantum_memory.csv"), index=False)

summary = {
    "cad_fixed_preserve_baseline": {
        "resnet_ckpt_head_used": True,
        "fc_initialized_from_resnet": True,
        "adapters_identity_init": True,
        "no_double_shortcut": True,
        "train_domain_fc": TRAIN_DOMAIN_FC
    },
    "estimator_backend": EST_BACKEND,
    "config": {
        "N_SESSIONS": N_SESSIONS,
        "CAD_DPRIME": CAD_DPRIME,
        "DQ_MAIN":DQ_MAIN, "VQC_REPS":VQC_REPS, "N_OBS":N_OBS,
        "DELTA_LOGIT_SCALE":DELTA_LOGIT_SCALE,
        "K_RETRIEVE":K_RETRIEVE,"TAU_MEM":TAU_MEM,"LAMBDA_MEMPROB":LAMBDA_MEMPROB,
        "LAMBDA_MLCL":LAMBDA_MLCL,"TEMP_MLCL":TEMP_MLCL
    },
    "memory": {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
               "has_v_img":mem.v_img is not None,"has_prototypes":mem.prototypes is not None,
               "has_base_conf":mem.base_conf is not None,"has_metadata":mem.metadata_available,
               "sklearn_index":mem.index_sklearn is not None},
    "metrics_A_cnn_only": m0,
    "metrics_B_quantum_no_memory": m1,
    "metrics_C_quantum_memory": m2,
    "metrics_D_cad_quantum_memory": m3
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Saved outputs to:", OUT_DIR)
print("  - train_sessions_log.csv")
print("  - per_class_A_cnn_only.csv")
print("  - per_class_B_quantum_no_memory.csv")
print("  - per_class_C_quantum_memory.csv")
print("  - per_class_D_cad_quantum_memory.csv")
print("  - summary.json")

Device: cuda

[Sanity] Config: {'BATCH_SIZE': 16, 'N_SESSIONS': 3, 'CAD_DPRIME': 256, 'DQ_MAIN': 4, 'K_RETRIEVE': 8, 'LAMBDA_MLCL': 0.1, 'TEMP_MLCL': 0.2, 'TRAIN_DOMAIN_FC': False}
Qiskit version: 2.3.0
Qiskit-ML version: 0.9.0
Using estimator: qiskit_aer.primitives.EstimatorV2


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2026-02-28 18:25:36.097759: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772303136.343623      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772303136.402823      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772303136.955029      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772303136.955085      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772303136.955089      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[Sanity] ResNet load strict=False | missing: 0 | unexpected: 0

[Sanity] NIH sizes: {'stream_total': 86524, 'calib': 512, 'sessions': [28841, 28841, 28842], 'test': 25596}


[Sanity] Loaded index_sklearn.joblib (kneighbors available).

[Sanity] Memory loaded: {'N': 30000, 'D': 12, 't_desc_dim': 7, 'sizeGB': 0.01169471, 'has_v_img': True, 'has_prototypes': True, 'has_base_conf': True, 'has_metadata': True, 'sklearn_index': True}

[Sanity] fmap/tokens: {'fmap': (16, 2048, 7, 7), 'xcls': (16, 1, 2048), 'ximg': (16, 49, 2048), 'H': 7, 'W': 7}
[Sanity] Baseline check max|resnet - cad(domain0)| = 0.0
[Sanity] memory dt: {'dt': (16, 7), 'sims_minmax': (-0.9195320010185242, -0.5889816284179688)}
[Sanity] Quantum: {'base': (16, 5), 'delta': (16, 5), 'g_minmax': (0.5396497249603271, 0.5406746864318848)}

[Stage] Train sessions + absorb + centers ...

Session 1/3 (domain 0) TRAIN
[sess0] step   10  loss=0.4114 bce=0.1427 mlcl=0.0000 mem=0.5373 g_mean=0.536  elapsed=88.4s
[sess0] step   20  loss=0.4709 bce=0.2011 mlcl=0.3472 mem=0.4702 g_mean=0.535  elapsed=177.1s
[sess0] step   30  loss=0.6895 bce=0.3694 mlcl=0.8601 mem=0.4681 g_mean=0.535  elapsed=266.8s
[sess0] ste

# New Eval pipeline
This version keeps the strong ResNet baseline, adds CaD, memory, and quantum correction, then tests performance on both NIH and unseen CheXpert validation images to check generalization. 


In [ ]:
# ============================================================
# NIH Domain-Incremental Learning (CaD-aligned) + Quantum + Memory
# ✅ FIXED to preserve high AUROC baseline
# PLUS: After NIH eval, compute per-class AUC/PR-AUC on CheXpert VALID (unseen in NIH stream)
#   CheXpert valid CSV: /kaggle/input/chexpert/valid.csv
#   CheXpert image base: /kaggle/input/chexpert
# ============================================================

import os, re, json, math, time, random, inspect, sys, subprocess
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# -------------------------
# (0) Repro
# -------------------------
SEED = 7
def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# (1) Paths (yours)
# -------------------------
MEM_DIR = "/kaggle/input/datasets/zarinn/memory/MEMORY"

RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

# NEW: CheXpert valid (for post-CaD evaluation)
CHEX_BASE  = "/kaggle/input/chexpert"
CHEX_VALID = "/kaggle/input/chexpert/valid.csv"

UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(UNIFIED_LABELS)

OUT_DIR = "/kaggle/working/outputs_cad_aligned_fixed_auc"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# (2) Speed knobs
# -------------------------
BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

CALIBRATION_N = 512
EPOCHS_CALIB  = 1

# Domain sessions (simulate T sessions on NIH stream)
N_SESSIONS    = 3
EPOCHS_PER_SESSION = 1
MAX_STEPS_PER_SESSION = 80
LOG_EVERY     = 10

TEST_MAX_BATCHES = None

# Memory knobs
K_RETRIEVE     = 8
TAU_MEM        = 10.0
LAMBDA_MEMPROB = 0.5

# CaD loss knob (Eq.17)
LAMBDA_MLCL    = 0.1
TEMP_MLCL      = 0.2

# Quantum residual adapter
DQ_MAIN   = 4
VQC_REPS  = 1
N_OBS     = 2
LR_CLASSICAL = 1e-3
LR_QUANTUM   = 1e-2
DELTA_LOGIT_SCALE = 0.35

# CaD bottleneck d'
CAD_DPRIME = 256

# Whether to fine-tune domain FC heads (usually keep False to preserve baseline)
TRAIN_DOMAIN_FC = False

print("\n[Sanity] Config:",
      {"BATCH_SIZE":BATCH_SIZE, "N_SESSIONS":N_SESSIONS,
       "CAD_DPRIME":CAD_DPRIME, "DQ_MAIN":DQ_MAIN,
       "K_RETRIEVE":K_RETRIEVE, "LAMBDA_MLCL":LAMBDA_MLCL, "TEMP_MLCL":TEMP_MLCL,
       "TRAIN_DOMAIN_FC": TRAIN_DOMAIN_FC})

# ============================================================
# (3) Qiskit (EstimatorQNN) — robust to missing qiskit_aer
#     Tries Aer; if not available, falls back to StatevectorEstimator (no Aer needed).
# ============================================================
import qiskit
import qiskit_machine_learning
print("Qiskit version:", getattr(qiskit, "__version__", "unknown"))
print("Qiskit-ML version:", getattr(qiskit_machine_learning, "__version__", "unknown"))

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

HAS_AER = False
ESTIMATOR = None
EST_BACKEND = None
_PM = None

# Try to import/install Aer (optional)
try:
    import qiskit_aer  # noqa
except Exception:
    # Try install (Kaggle usually allows pip installs)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "qiskit-aer"])
        import qiskit_aer  # noqa
    except Exception as e:
        print("[Warn] Could not install/import qiskit-aer. Will use StatevectorEstimator. Reason:", repr(e))

try:
    import qiskit_aer
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
    from qiskit_aer import AerSimulator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    ESTIMATOR = AerEstimatorV2()
    EST_BACKEND = "qiskit_aer.primitives.EstimatorV2"
    HAS_AER = True

    _AER_BACKEND = AerSimulator()
    _PM = generate_preset_pass_manager(optimization_level=1, backend=_AER_BACKEND)
except Exception:
    from qiskit.primitives import StatevectorEstimator
    ESTIMATOR = StatevectorEstimator()
    EST_BACKEND = "qiskit.primitives.StatevectorEstimator"
    HAS_AER = False
    _PM = None

# Sanity: ensure it's not EstimatorV1 signature
sig = inspect.signature(ESTIMATOR.run)
params = list(sig.parameters.keys())
if ("observables" in params) and ("circuits" in params):
    raise RuntimeError(f"Estimator looks like V1 ({EST_BACKEND}). Need V2-style estimator for EstimatorQNN.")
print("Using estimator:", EST_BACKEND, "| HAS_AER:", HAS_AER)

def build_qnn(n_qubits, reps=1, entanglement="circular", n_obs=2):
    fm = zz_feature_map(feature_dimension=n_qubits, reps=2, entanglement=entanglement)
    ans = real_amplitudes(num_qubits=n_qubits, reps=reps, entanglement=entanglement)

    qc = QuantumCircuit(n_qubits)
    qc = qc.compose(fm, inplace=False, wrap=False)
    qc = qc.compose(ans, inplace=False, wrap=False)
    qc = qc.decompose(reps=15)

    if HAS_AER and _PM is not None:
        qc = _PM.run(qc)

    obs = []
    n_obs = int(min(max(1, n_obs), n_qubits))
    for i in range(n_obs):
        pauli = ["I"] * n_qubits
        pauli[i] = "Z"
        obs.append(SparsePauliOp("".join(pauli)))

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm.parameters,
        weight_params=ans.parameters,
        observables=obs,
        input_gradients=True,
        estimator=ESTIMATOR,
    )
    return qnn, len(fm.parameters), qnn.num_weights, len(obs)

# ============================================================
# (4) Encoders: BiomedBERT + ResNet50 (frozen)
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torchvision
from torchvision import transforms

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

def _strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p): kk = kk[len(p):]
        out[kk] = v
    return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu")
    if isinstance(ck, dict) and "state_dict" in ck: sd = ck["state_dict"]
    elif isinstance(ck, dict) and "model" in ck: sd = ck["model"]
    elif isinstance(ck, dict) and "net" in ck: sd = ck["net"]
    else: sd = ck
    return _strip_prefix(sd)

def load_biomedbert():
    tok_dir = os.path.dirname(BIOMED_CKPT)
    try:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

    model = AutoModel.from_pretrained(TEXT_MODEL_NAME)
    if os.path.exists(BIOMED_CKPT):
        sd = load_state_dict_any(BIOMED_CKPT)
        sd = {k:v for k,v in sd.items() if not k.startswith("cls.")}
        model.load_state_dict(sd, strict=False)

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return tokenizer, model

tokenizer, text_model = load_biomedbert()
DT_DIM = int(text_model.config.hidden_size)

@torch.no_grad()
def text_embed(text_list, max_len=64):
    enc = tokenizer(text_list, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    out = text_model(**enc, return_dict=True)
    t = out.last_hidden_state[:, 0, :]  # CLS
    return F.normalize(t, dim=1)

def load_resnet50_full_and_spatial():
    m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(2048, C)

    sd = load_state_dict_any(RESNET_CKPT)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print("[Sanity] ResNet load strict=False | missing:", len(missing), "| unexpected:", len(unexpected))

    m.eval().to(DEVICE)
    for p in m.parameters():
        p.requires_grad = False

    @torch.no_grad()
    def spatial(x):
        b = m
        x = b.conv1(x); x = b.bn1(x); x = b.relu(x); x = b.maxpool(x)
        x = b.layer1(x); x = b.layer2(x); x = b.layer3(x); x = b.layer4(x)  # [B,2048,7,7]
        return x

    @torch.no_grad()
    def logits(x):
        return m(x)  # uses ckpt head

    return m, spatial, logits

resnet_full, resnet_spatial, cnn_logits_from_resnet = load_resnet50_full_and_spatial()

img_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def pool2048_from_fmap(fmap):
    v = F.adaptive_avg_pool2d(fmap, (1,1)).flatten(1)
    return F.normalize(v, dim=1)

# ============================================================
# (5) NIH dataset + session split
# ============================================================
def read_list(path):
    with open(path, "r") as f:
        return [x.strip() for x in f if x.strip()]

nih_df = pd.read_csv(NIH_CSV)
nih_df = nih_df.rename(columns={"Image Index":"image", "Finding Labels":"labels"})
nih_row = {r["image"]: r for _, r in nih_df.iterrows()}

def labels_to_multi_hot(lbl_str):
    parts = set([p.strip() for p in str(lbl_str).split("|")])
    y = np.zeros((C,), dtype=np.float32)
    for i, name in enumerate(UNIFIED_LABELS):
        if name == "Pleural Effusion":
            if "Effusion" in parts: y[i] = 1.0
        else:
            if name in parts: y[i] = 1.0
    return y

def make_text_descriptor_nih(r):
    view = r.get("View Position", r.get("ViewPosition", ""))
    age  = r.get("Patient Age", r.get("PatientAge", ""))
    sex  = r.get("Patient Gender", r.get("PatientGender", ""))
    return f"Chest X-ray. Dataset: NIH. View: {view}. Patient age: {age}. Sex: {sex}."

class NIHListDataset(Dataset):
    def __init__(self, names):
        self.items = list(names)
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        name = self.items[idx]
        img = Image.open(os.path.join(NIH_IMG, name)).convert("RGB")
        x = img_tfms(img)
        r = nih_row.get(name, {})
        y = labels_to_multi_hot(r.get("labels", ""))
        txt = make_text_descriptor_nih(r)
        return x, torch.tensor(y, dtype=torch.float32), txt, name

stream_list = read_list(NIH_STREAM)
test_list   = read_list(NIH_TEST)

calib_list = stream_list[:CALIBRATION_N]

def split_sessions(lst, n_sessions):
    n = len(lst)
    splits = []
    for s in range(n_sessions):
        a = (n*s)//n_sessions
        b = (n*(s+1))//n_sessions
        splits.append(lst[a:b])
    return splits

session_lists = split_sessions(stream_list, N_SESSIONS)

calib_loader = DataLoader(NIHListDataset(calib_list), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(NIHListDataset(test_list),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("\n[Sanity] NIH sizes:", {"stream_total":len(stream_list),
                             "calib":len(calib_list),
                             "sessions":[len(x) for x in session_lists],
                             "test":len(test_list)})

# ============================================================
# (6) Memory bundle (ALL files) + retrieval
# ============================================================
import faiss
try:
    import joblib
except Exception:
    joblib = None

class MemoryBundle:
    def __init__(self, mem_dir):
        self.mem_dir = mem_dir
        self.path_embeddings = os.path.join(mem_dir, "embeddings.npy")
        self.path_labels     = os.path.join(mem_dir, "labels.npy")
        self.path_tdesc      = os.path.join(mem_dir, "t_desc.npy")
        self.path_vimg       = os.path.join(mem_dir, "v_img.npy")
        self.path_protos     = os.path.join(mem_dir, "prototypes.npy")
        self.path_protolbl   = os.path.join(mem_dir, "prototypes_labels.json")
        self.path_baseconf   = os.path.join(mem_dir, "base_conf.npy")
        self.path_meta       = os.path.join(mem_dir, "metadata.jsonl")
        self.path_idx_skl    = os.path.join(mem_dir, "index_sklearn.joblib")

        assert os.path.exists(self.path_embeddings), "Missing embeddings.npy"
        assert os.path.exists(self.path_labels), "Missing labels.npy"

        E = np.load(self.path_embeddings)
        Y = np.load(self.path_labels)

        if E.dtype != np.float32: E = E.astype(np.float32, copy=False)
        E = np.ascontiguousarray(E)
        self.E = E
        self.N, self.D = self.E.shape

        if Y.ndim == 1:
            Y_oh = np.zeros((Y.shape[0], C), dtype=np.float32)
            Y_oh[np.arange(Y.shape[0]), Y.astype(np.int64)] = 1.0
            Y = Y_oh
        if Y.dtype != np.float32: Y = Y.astype(np.float32, copy=False)
        self.Y = np.ascontiguousarray(Y)

        self.t_desc = None
        if os.path.exists(self.path_tdesc):
            tdesc = np.load(self.path_tdesc)
            if tdesc.dtype != np.float32: tdesc = tdesc.astype(np.float32, copy=False)
            self.t_desc = np.ascontiguousarray(tdesc)
        self.t_desc_dim = int(self.t_desc.shape[1]) if self.t_desc is not None else 0

        self.v_img = None
        if os.path.exists(self.path_vimg):
            vimg = np.load(self.path_vimg)
            if vimg.dtype != np.float32: vimg = vimg.astype(np.float32, copy=False)
            self.v_img = np.ascontiguousarray(vimg)

        self.prototypes = None
        self.prototypes_labels = None
        if os.path.exists(self.path_protos):
            P = np.load(self.path_protos)
            if P.dtype != np.float32: P = P.astype(np.float32, copy=False)
            self.prototypes = np.ascontiguousarray(P)
        if os.path.exists(self.path_protolbl):
            with open(self.path_protolbl, "r") as f:
                self.prototypes_labels = json.load(f)

        self.base_conf = None
        if os.path.exists(self.path_baseconf):
            bc = np.load(self.path_baseconf).astype(np.float32, copy=False).reshape(-1)
            if bc.shape[0] == self.N:
                self.base_conf = np.clip(bc, 0.0, 1.0)

        self.metadata_available = os.path.exists(self.path_meta)

        self.index_sklearn = None
        if os.path.exists(self.path_idx_skl) and joblib is not None:
            try:
                obj = joblib.load(self.path_idx_skl)
                if hasattr(obj, "kneighbors"):
                    self.index_sklearn = obj
                    print("[Sanity] Loaded index_sklearn.joblib (kneighbors available).")
            except Exception as e:
                print("[Warn] Could not load index_sklearn.joblib; using FAISS. Reason:", repr(e))

        # FAISS cosine via normalized IP
        E2 = self.E.copy()
        faiss.normalize_L2(E2)
        self.faiss_index = faiss.IndexFlatIP(self.D)
        self.faiss_index.add(E2)
        self._E_norm = E2

        self._Y_t = torch.tensor(self.Y, device=DEVICE, dtype=torch.float32)
        self._td_t = torch.tensor(self.t_desc, device=DEVICE, dtype=torch.float32) if self.t_desc is not None else None

    def memory_gb(self):
        total = 0
        for fn in ["embeddings.npy","labels.npy","t_desc.npy","v_img.npy","prototypes.npy","base_conf.npy","metadata.jsonl","index_sklearn.joblib"]:
            p = os.path.join(self.mem_dir, fn)
            if os.path.exists(p):
                total += os.path.getsize(p)
        return total / 1e9

    @torch.no_grad()
    def query_key(self, v, t):
        dv, dt = v.shape[1], t.shape[1]
        if self.D == dv + dt:
            q = torch.cat([v, t], dim=1)
        elif self.D == dv:
            q = v
        elif self.D == dt:
            q = t
        else:
            torch.manual_seed(SEED)
            W = torch.randn(dv + dt, self.D, device=v.device) / math.sqrt(dv + dt)
            q = torch.cat([v, t], dim=1) @ W
        return F.normalize(q, dim=1)

    @torch.no_grad()
    def retrieve(self, q, k=16):
        qn = F.normalize(q, dim=1)
        q_np = np.ascontiguousarray(qn.detach().cpu().numpy().astype(np.float32, copy=False))

        if self.index_sklearn is not None:
            try:
                dists, idxs = self.index_sklearn.kneighbors(q_np, n_neighbors=min(k, self.N), return_distance=True)
                sims = -dists.astype(np.float32, copy=False)
                return idxs.astype(np.int64, copy=False), sims
            except Exception:
                pass

        sims, idxs = self.faiss_index.search(q_np, min(k, self.N))
        return idxs.astype(np.int64, copy=False), sims.astype(np.float32, copy=False)

    @torch.no_grad()
    def retrieve_dt(self, idxs):
        if self._td_t is None:
            return torch.zeros((idxs.shape[0], 1), device=DEVICE, dtype=torch.float32)
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        dt = self._td_t[idxs_t].mean(dim=1)
        return F.normalize(dt, dim=1)

    @torch.no_grad()
    def pmem_from_neighbors(self, idxs, sims, tau=10.0, eps=1e-8):
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        sims_t = torch.tensor(sims, device=DEVICE, dtype=torch.float32)

        neigh_y = self._Y_t[idxs_t]  # [B,k,C]
        w = torch.softmax(tau * sims_t, dim=1)  # [B,k]

        if self.base_conf is not None:
            conf = torch.tensor(self.base_conf, device=DEVICE, dtype=torch.float32)[idxs_t]  # [B,k]
            w = w * conf
            w = w / (w.sum(dim=1, keepdim=True) + 1e-12)

        w = w.unsqueeze(-1)
        p = (w * neigh_y).sum(dim=1)
        return torch.clamp(p, eps, 1 - eps)

mem = MemoryBundle(MEM_DIR)
dt_cond_dim = mem.t_desc_dim if mem.t_desc_dim > 0 else 1

print("\n[Sanity] Memory loaded:",
      {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
       "has_v_img":mem.v_img is not None, "has_prototypes":mem.prototypes is not None,
       "has_base_conf":mem.base_conf is not None, "has_metadata":mem.metadata_available,
       "sklearn_index":mem.index_sklearn is not None})

# ============================================================
# (7) CaD tokenization + C/D adapters (FIXED)
# ============================================================
def fmap_to_tokens(fmap):
    B, d, H, W = fmap.shape
    x_img = fmap.permute(0,2,3,1).reshape(B, H*W, d)  # [B,49,2048]
    x_cls = fmap.mean(dim=(2,3), keepdim=False).unsqueeze(1)  # [B,1,2048]
    return x_cls, x_img, H, W

class CAdapter(nn.Module):
    def __init__(self, d=2048, dprime=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in):
        M_mid = F.relu(self.down(self.ln1(M_in)))
        M_til = self.up(self.ln2(M_mid))
        M_out = F.relu(M_til + M_in)
        return M_out

@torch.no_grad()
def absorb_cadapter_(cadapter_running: CAdapter, cadapter_current: CAdapter, t: int):
    for (n1,p1), (n2,p2) in zip(cadapter_running.named_parameters(), cadapter_current.named_parameters()):
        assert n1 == n2
        p1.data.mul_((t-1)/t).add_(p2.data, alpha=(1.0/t))

class DAdapter(nn.Module):
    def __init__(self, d=2048, dprime=256, k=3):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2 = nn.LayerNorm(dprime)
        self.conv = nn.Conv2d(dprime, dprime, kernel_size=k, padding=k//2)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.conv.weight)
        if self.conv.bias is not None: nn.init.zeros_(self.conv.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in, H=7, W=7):
        M_mid = F.relu(self.down(self.ln1(M_in)))
        x_cls = M_mid[:, :1, :]
        x_img = M_mid[:, 1:, :]
        x_grid = x_img.reshape(x_img.size(0), H, W, x_img.size(2)).permute(0,3,1,2).contiguous()
        x_grid2 = self.conv(x_grid)
        x_img2 = x_grid2.permute(0,2,3,1).reshape(x_img.size(0), H*W, x_img.size(2)).contiguous()
        M_til_mid = F.relu(torch.cat([x_cls, x_img2], dim=1))
        M_til = self.up(self.ln2(M_til_mid))
        M_out = F.relu(M_til + M_in)
        return M_out

class DAdapterBank(nn.Module):
    def __init__(self, n_domains, d=2048, dprime=256):
        super().__init__()
        self.adapters = nn.ModuleList([DAdapter(d=d, dprime=dprime) for _ in range(n_domains)])

    def forward(self, M_in, domain_id, H=7, W=7):
        return self.adapters[domain_id](M_in, H=H, W=W)

    def freeze_domain(self, domain_id):
        for p in self.adapters[domain_id].parameters():
            p.requires_grad = False

class DomainFC(nn.Module):
    def __init__(self, d=2048, n_classes=5):
        super().__init__()
        self.fc = nn.Linear(d, n_classes)
    def forward(self, cls_token):
        return self.fc(cls_token)

class CaDModel(nn.Module):
    def __init__(self, n_sessions, d=2048, dprime=256, n_classes=5, init_fc_from_resnet=True):
        super().__init__()
        self.c_running = CAdapter(d=d, dprime=dprime)
        self.c_current = CAdapter(d=d, dprime=dprime)

        self.d_bank = DAdapterBank(n_domains=n_sessions, d=d, dprime=dprime)
        self.fc_bank = nn.ModuleList([DomainFC(d=d, n_classes=n_classes) for _ in range(n_sessions)])

        if init_fc_from_resnet:
            base_sd = resnet_full.fc.state_dict()
            for di in range(n_sessions):
                self.fc_bank[di].fc.load_state_dict(base_sd, strict=True)

        self.domain_centers = [None for _ in range(n_sessions)]

    def forward_tokens(self, fmap, domain_id):
        x_cls, x_img, H, W = fmap_to_tokens(fmap)
        M_in = torch.cat([x_cls, x_img], dim=1)
        M_c = self.c_running(M_in)
        M_d = self.d_bank(M_in, domain_id=domain_id, H=H, W=W)
        M_out = M_c + M_d - M_in   # ✅ critical fix
        cls = M_out[:, :1, :].squeeze(1)
        img_tokens = M_out[:, 1:, :]
        return cls, img_tokens

    def logits_for_domain(self, fmap, domain_id):
        cls, img_tokens = self.forward_tokens(fmap, domain_id)
        logits = self.fc_bank[domain_id](cls)
        return logits, cls, img_tokens

cad = CaDModel(n_sessions=N_SESSIONS, d=2048, dprime=CAD_DPRIME, n_classes=C, init_fc_from_resnet=True).to(DEVICE)

for di in range(N_SESSIONS):
    for p in cad.fc_bank[di].parameters():
        p.requires_grad = TRAIN_DOMAIN_FC

# ============================================================
# (8) Quantum residual adapter
# ============================================================
class NonLinearProjector(nn.Module):
    def __init__(self, in_dim, dq):
        super().__init__()
        h = max(64, 2*dq)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Linear(h, dq),
        )
    def forward(self, z):
        return self.net(z)

class QuantumResidualAdapter(nn.Module):
    def __init__(self, dv=2048, dt=DT_DIM, dt_cond_dim=1, dq=4, n_classes=5, reps=1, n_obs=2):
        super().__init__()
        self.dq = dq
        self.dt_cond_dim = max(int(dt_cond_dim), 1)

        self.proj = NonLinearProjector(dv + dt, dq)
        self.dt_to_angles = nn.Sequential(
            nn.Linear(self.dt_cond_dim, max(8, 2*dq)),
            nn.GELU(),
            nn.Linear(max(8, 2*dq), dq),
        )
        self.dt_gate = nn.Sequential(
            nn.Linear(self.dt_cond_dim, 8),
            nn.GELU(),
            nn.Linear(8, 1),
        )

        qnn, _, n_w, nobs = build_qnn(dq, reps=reps, entanglement="circular", n_obs=n_obs)
        init_w = 0.01 * np.random.randn(n_w).astype(np.float32)
        self.q = TorchConnector(qnn, initial_weights=init_w)

        self.delta_head = nn.Sequential(
            nn.Linear(nobs, max(16, 2*dq)),
            nn.GELU(),
            nn.Linear(max(16, 2*dq), n_classes),
        )
        self._nobs = nobs

    def forward_delta(self, v, t, dt):
        z = torch.cat([v, t], dim=1)
        u = self.proj(z)
        u = math.pi * torch.tanh(u)

        mod = math.pi * torch.tanh(self.dt_to_angles(dt))
        u_mod = u + 0.25 * mod

        g = torch.sigmoid(self.dt_gate(dt))
        q_out = self.q(u_mod)
        delta = self.delta_head(q_out)
        return delta, g, q_out

    def forward_logits(self, base_logits, v, t, dt):
        delta, g, _ = self.forward_delta(v, t, dt)
        logits = base_logits + (g * DELTA_LOGIT_SCALE) * delta
        return logits, delta, g

qadapter = QuantumResidualAdapter(dq=DQ_MAIN, n_classes=C, dt_cond_dim=dt_cond_dim, reps=VQC_REPS, n_obs=N_OBS).to(DEVICE)

def split_params_for_opt(model):
    q_params, c_params = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "q." in n:
            q_params.append(p)
        else:
            c_params.append(p)
    return c_params, q_params

def make_opt(model, extra_params=None):
    c_params, q_params = split_params_for_opt(model)
    if extra_params is not None:
        c_params = list(c_params) + list(extra_params)
    return torch.optim.Adam([
        {"params": c_params, "lr": LR_CLASSICAL},
        {"params": q_params, "lr": LR_QUANTUM},
    ])

# ============================================================
# (9) Loss: BCE + Multi-label contrastive + memprob
# ============================================================
crit_bce = nn.BCEWithLogitsLoss()
crit_mem = nn.BCELoss()

def label_similarity_r(y_i, y_j, eps=1e-9):
    yi = (y_i > 0.5).float()
    yj = (y_j > 0.5).float()
    inter = (yi*yj).sum(dim=-1)
    uni   = ((yi + yj) > 0.0).float().sum(dim=-1)
    return inter / (uni + eps)

def mlcl_loss(features, labels, temp=0.2, eps=1e-9):
    B = features.size(0)
    f = F.normalize(features, dim=1)
    sim = (f @ f.t()) / temp
    sim = sim - torch.eye(B, device=sim.device) * 1e9

    yi = labels.unsqueeze(1).expand(B,B,-1)
    yj = labels.unsqueeze(0).expand(B,B,-1)
    r = label_similarity_r(yi, yj)
    r = r * (1.0 - torch.eye(B, device=r.device))

    logp = F.log_softmax(sim, dim=1)
    loss_i = -(r * logp).sum(dim=1) / (r.sum(dim=1) + eps)
    return loss_i.mean()

# ============================================================
# (10) Optimizer per session (train c_current + D_s (+ FC_s if enabled) + qadapter)
# ============================================================
def make_session_optimizer(session_id):
    for p in cad.c_running.parameters():
        p.requires_grad = False
    for p in cad.c_current.parameters():
        p.requires_grad = True

    for di in range(N_SESSIONS):
        for p in cad.d_bank.adapters[di].parameters():
            p.requires_grad = (di == session_id)
        for p in cad.fc_bank[di].parameters():
            p.requires_grad = (TRAIN_DOMAIN_FC and (di == session_id))

    train_params = (
        list(cad.c_current.parameters()) +
        list(cad.d_bank.adapters[session_id].parameters()) +
        (list(cad.fc_bank[session_id].parameters()) if TRAIN_DOMAIN_FC else [])
    )
    opt = make_opt(qadapter, extra_params=train_params)
    return opt

def forward_cad_train(fmap, session_id):
    x_cls, x_img, H, W = fmap_to_tokens(fmap)
    M_in = torch.cat([x_cls, x_img], dim=1)

    M_c = cad.c_current(M_in)
    M_d = cad.d_bank(M_in, domain_id=session_id, H=H, W=W)
    M_out = M_c + M_d - M_in   # ✅ critical fix

    cls = M_out[:, :1, :].squeeze(1)
    img_tokens = M_out[:, 1:, :]
    logits = cad.fc_bank[session_id](cls)
    return logits, cls, img_tokens

# ============================================================
# (11) Sanity checks
# ============================================================
@torch.no_grad()
def sanity_check_shapes():
    x,y,txt,_ = next(iter(calib_loader))
    x = x.to(DEVICE)
    fmap = resnet_spatial(x)

    base_logits = cnn_logits_from_resnet(x)
    cad_logits0, _, _ = cad.logits_for_domain(fmap, 0)
    diff = (base_logits - cad_logits0).abs().max().item()
    print("\n[Sanity] Baseline check max|resnet - cad(domain0)| =", diff)

    v = pool2048_from_fmap(fmap)
    t = text_embed(list(txt))
    qkey = mem.query_key(v,t)
    idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
    dt = mem.retrieve_dt(idxs)
    qlogits, delta, g = qadapter.forward_logits(base_logits, v, t, dt)
    print("[Sanity] Quantum g range:", (float(g.min()), float(g.max())), "| delta:", tuple(delta.shape))

sanity_check_shapes()

# ============================================================
# (12) Training loop: sessions + absorb + centers
# ============================================================
def compute_domain_center(session_id, loader):
    cad.eval()
    feats = []
    with torch.no_grad():
        for x,y,txt,_ in loader:
            x = x.to(DEVICE)
            fmap = resnet_spatial(x)
            _, img_tokens = cad.forward_tokens(fmap, domain_id=session_id)
            f = img_tokens.mean(dim=1)
            feats.append(f.cpu())
    Fm = torch.cat(feats, dim=0).mean(dim=0).numpy().astype(np.float32)
    return Fm

def train_sessions():
    history = []
    for s in range(N_SESSIONS):
        print(f"\n==============================")
        print(f"Session {s+1}/{N_SESSIONS} (domain {s}) TRAIN")
        print("==============================")

        ds = NIHListDataset(session_lists[s])
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

        opt = make_session_optimizer(s)
        cad.train()
        qadapter.train()

        step = 0
        t0 = time.time()

        for ep in range(EPOCHS_PER_SESSION):
            for x,y,txt,_ in loader:
                x = x.to(DEVICE); y = y.to(DEVICE)
                fmap = resnet_spatial(x)

                v = pool2048_from_fmap(fmap)
                t = text_embed(list(txt))
                qkey = mem.query_key(v,t)
                idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
                dt = mem.retrieve_dt(idxs)
                p_mem = mem.pmem_from_neighbors(idxs, sims, tau=TAU_MEM)

                cad_logits, cls_feat, _ = forward_cad_train(fmap, session_id=s)
                logits, _, g = qadapter.forward_logits(cad_logits, v, t, dt)

                loss_bce = crit_bce(logits, y)
                loss_mlcl = mlcl_loss(cls_feat, y, temp=TEMP_MLCL)
                p_hat = torch.sigmoid(logits)
                loss_mem = crit_mem(p_hat, p_mem)

                loss = loss_bce + LAMBDA_MLCL * loss_mlcl + LAMBDA_MEMPROB * loss_mem

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

                step += 1
                if step % LOG_EVERY == 0:
                    print(f"[sess{s}] step {step:4d}  loss={loss.item():.4f} "
                          f"bce={loss_bce.item():.4f} mlcl={loss_mlcl.item():.4f} mem={loss_mem.item():.4f} "
                          f"g_mean={float(g.mean().detach().cpu()):.3f}  elapsed={time.time()-t0:.1f}s")

                if step >= MAX_STEPS_PER_SESSION:
                    break
            if step >= MAX_STEPS_PER_SESSION:
                break

        cad.eval()
        with torch.no_grad():
            absorb_cadapter_(cad.c_running, cad.c_current, t=s+1)
            cad.c_current.load_state_dict(cad.c_running.state_dict())

        cad.d_bank.freeze_domain(s)
        if TRAIN_DOMAIN_FC:
            for p in cad.fc_bank[s].parameters():
                p.requires_grad = False

        center_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        center = compute_domain_center(s, center_loader)
        cad.domain_centers[s] = center
        print(f"[sess{s}] stored domain center shape:", center.shape)

        history.append({"session": s, "steps": step, "train_sec": time.time()-t0})

    return pd.DataFrame(history)

print("\n[Stage] Train sessions + absorb + centers ...")
train_log = train_sessions()
train_log.to_csv(os.path.join(OUT_DIR, "train_sessions_log.csv"), index=False)
print("[Stage] Saved:", os.path.join(OUT_DIR, "train_sessions_log.csv"))

# ============================================================
# (13) Evaluation helpers
# ============================================================
def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p_pred >= lo) & (p_pred < hi) if i < n_bins-1 else (p_pred >= lo) & (p_pred <= hi)
        if m.sum() == 0:
            continue
        conf = p_pred[m].mean()
        acc  = y_true[m].mean()
        ece += (m.sum()/len(y_true)) * abs(acc - conf)
    return float(ece)

def brier_binary(y_true, p_pred):
    y_true = y_true.astype(np.float32)
    p_pred = p_pred.astype(np.float32)
    return float(np.mean((p_pred - y_true)**2))

def per_class_metrics(Y, P, labels):
    rows = []
    for j, lbl in enumerate(labels):
        yt, yp = Y[:, j], P[:, j]
        if len(np.unique(yt)) < 2:
            aucv, apv = np.nan, np.nan
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v = f1_score(yt.astype(int), yhat, zero_division=0)
        ecev = ece_binary(yt.astype(np.float32), yp.astype(np.float32), n_bins=15)
        brv  = brier_binary(yt, yp)
        rows.append([lbl, aucv, apv, f1v, ecev, brv])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5","ECE","Brier"])
    macro_auc = float(np.nanmean(df["ROC_AUC"].values))
    macro_ap  = float(np.nanmean(df["PR_AUC"].values))
    macro_f1  = float(np.nanmean(df["F1@0.5"].values))
    macro_ece = float(np.nanmean(df["ECE"].values))
    macro_br  = float(np.nanmean(df["Brier"].values))
    return df, {"macro_auc":macro_auc,"macro_ap":macro_ap,"macro_f1":macro_f1,"macro_ece":macro_ece,"macro_brier":macro_br}

@torch.no_grad()
def collect_preds_A_cnn_only(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        logits = cnn_logits_from_resnet(x)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_B_quantum_no_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        dt = torch.zeros((x.size(0), dt_cond_dim), device=DEVICE, dtype=torch.float32)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_C_quantum_memory(loader, max_batches=None):
    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))

        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        base_logits = cnn_logits_from_resnet(x)
        logits, _, _ = qadapter.forward_logits(base_logits, v, t, dt)
        p = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0)

@torch.no_grad()
def collect_preds_D_cad_quantum_memory(loader, max_batches=None):
    """
    Domain-ID routing uses NIH-learned domain centers (cad.domain_centers).
    This is what you want for "after CaD" evaluation.
    """
    cad.eval(); qadapter.eval()
    centers = [c for c in cad.domain_centers if c is not None]
    T = len(centers)
    A = torch.tensor(np.stack(centers, axis=0), device=DEVICE, dtype=torch.float32)  # [T,d]

    Ys, Ps = [], []
    for bi, (x,y,txt,_) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)

        fmap = resnet_spatial(x)
        v = pool2048_from_fmap(fmap)
        t = text_embed(list(txt))
        qkey = mem.query_key(v,t)
        idxs, sims = mem.retrieve(qkey, k=K_RETRIEVE)
        dt = mem.retrieve_dt(idxs)

        logits_all = []
        feat_all = []
        for di in range(T):
            logits_di, cls_di, img_tokens_di = cad.logits_for_domain(fmap, di)
            fi = img_tokens_di.mean(dim=1)  # [B,d]
            logits_all.append(logits_di.unsqueeze(1))  # [B,1,C]
            feat_all.append(fi.unsqueeze(1))           # [B,1,d]
        logits_all = torch.cat(logits_all, dim=1)  # [B,T,C]
        feat_all   = torch.cat(feat_all, dim=1)    # [B,T,d]

        dist = ((feat_all - A.unsqueeze(0))**2).sum(dim=2)  # [B,T]
        dom_id = torch.argmin(dist, dim=1)

        Bsz = x.size(0)
        picked = logits_all[torch.arange(Bsz, device=DEVICE), dom_id, :]  # [B,C]

        logits, _, _ = qadapter.forward_logits(picked, v, t, dt)
        p = torch.sigmoid(logits)

        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())

    return np.concatenate(Ys,0), np.concatenate(Ps,0)

# ============================================================
# (14) NIH evaluation (A)(B)(C)(D)
# ============================================================
print("\n==============================")
print("(A) NIH: CNN-only (no CaD, no quantum, no memory)")
print("==============================")
Y0, P0 = collect_preds_A_cnn_only(test_loader, max_batches=TEST_MAX_BATCHES)
df0, m0 = per_class_metrics(Y0, P0, UNIFIED_LABELS)
print(df0); print("[NIH-A] Macro:", m0)
df0.to_csv(os.path.join(OUT_DIR, "NIH_per_class_A_cnn_only.csv"), index=False)

print("\n==============================")
print("(B) NIH: Quantum adapter (dt=0, no retrieval)")
print("==============================")
Y1, P1 = collect_preds_B_quantum_no_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df1, m1 = per_class_metrics(Y1, P1, UNIFIED_LABELS)
print(df1); print("[NIH-B] Macro:", m1)
df1.to_csv(os.path.join(OUT_DIR, "NIH_per_class_B_quantum_no_memory.csv"), index=False)

print("\n==============================")
print("(C) NIH: Quantum adapter + Memory retrieval")
print("==============================")
Y2, P2 = collect_preds_C_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df2, m2 = per_class_metrics(Y2, P2, UNIFIED_LABELS)
print(df2); print("[NIH-C] Macro:", m2)
df2.to_csv(os.path.join(OUT_DIR, "NIH_per_class_C_quantum_memory.csv"), index=False)

print("\n==============================")
print("(D) NIH: CaD + Domain-ID + Quantum + Memory")
print("==============================")
Y3, P3 = collect_preds_D_cad_quantum_memory(test_loader, max_batches=TEST_MAX_BATCHES)
df3, m3 = per_class_metrics(Y3, P3, UNIFIED_LABELS)
print(df3); print("[NIH-D] Macro:", m3)
df3.to_csv(os.path.join(OUT_DIR, "NIH_per_class_D_cad_quantum_memory.csv"), index=False)

# ============================================================
# (15) NEW: CheXpert VALID dataset + loader
# ============================================================
def parse_chex_patient_id(path_str):
    m = re.search(r"patient(\d+)", str(path_str))
    return m.group(1) if m else "unknown"

def detect_chex_image_root(chex_base, sample_rel_paths):
    candidates = [
        chex_base,
        os.path.join(chex_base, "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0-small", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0"),
    ]

    def try_join(root, rel):
        rel = str(rel)
        rel2 = rel.lstrip("./")
        rel3 = rel.replace("CheXpert-v1.0-small/", "")
        rel4 = rel.replace("CheXpert-v1.0/", "")
        tries = [
            os.path.join(root, rel),
            os.path.join(root, rel2),
            os.path.join(root, rel3),
            os.path.join(root, rel4),
            os.path.join(root, "CheXpert-v1.0-small", rel),
            os.path.join(root, "CheXpert-v1.0-small", rel3),
        ]
        for p in tries:
            if os.path.exists(p):
                return p, tries
        return None, tries

    best_root, best_found = None, -1
    for root in candidates:
        found = 0
        for rel in sample_rel_paths:
            p, _ = try_join(root, rel)
            if p is not None:
                found += 1
        if found > best_found:
            best_found = found
            best_root = root

    if best_found <= 0:
        rel0 = sample_rel_paths[0]
        p, tries = try_join(chex_base, rel0)
        raise RuntimeError(
            f"❌ Could not locate CheXpert images.\n"
            f"chex_base={chex_base}\n"
            f"sample rel={rel0}\n"
            f"Tried:\n  - " + "\n  - ".join(tries[:12])
        )

    print(f"✅ Detected CheXpert image root: {best_root} (matched {best_found}/{len(sample_rel_paths)} samples)")

    def resolver(rel):
        p, tries = try_join(best_root, rel)
        if p is None:
            raise FileNotFoundError(
                f"❌ CheX image not found.\nroot={best_root}\nrel={rel}\nTried:\n  - " + "\n  - ".join(tries[:12])
            )
        return p
    return resolver

def build_chex_valid_df(csv_path):
    df = pd.read_csv(csv_path)

    img_col = None
    for c in ["Path","path","image_path","Image"]:
        if c in df.columns:
            img_col = c
            break
    if img_col is None:
        raise RuntimeError(f"❌ CheXpert valid.csv missing image path col. Columns={list(df.columns)[:30]}")

    pleural_col = None
    for c in ["Pleural Effusion","Pleural_Effusion"]:
        if c in df.columns:
            pleural_col = c
            break

    needed = ["Atelectasis","Cardiomegaly","Consolidation","Edema"]
    for c in needed:
        if c not in df.columns:
            raise RuntimeError(f"❌ CheXpert valid.csv missing label col: {c}")
    if pleural_col is None:
        raise RuntimeError("❌ CheXpert valid.csv missing Pleural Effusion column.")

    # filter frontal if possible
    for frontal_col in ["Frontal/Lateral","Frontal_Lateral"]:
        if frontal_col in df.columns:
            df = df[df[frontal_col].astype(str).str.lower().str.contains("frontal")].copy()
            break

    df = df.rename(columns={img_col:"image", pleural_col:"Pleural Effusion"}).copy()

    # uncertain policy: -1 -> 0, NaN -> 0
    for c in UNIFIED_LABELS:
        df[c] = df[c].replace(-1, 0).fillna(0).astype(np.float32)

    df["patient_id"] = df["image"].apply(parse_chex_patient_id).astype(str)
    df = df[["image","patient_id"] + UNIFIED_LABELS].reset_index(drop=True)
    return df

class CheXValidDataset(Dataset):
    def __init__(self, df, resolver):
        self.df = df.reset_index(drop=True)
        self.resolver = resolver
        self.images = self.df["image"].tolist()
        self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        rel = self.images[idx]
        p = self.resolver(rel)
        img = Image.open(p).convert("RGB")
        x = img_tfms(img)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)

        # simple descriptor for BiomedBERT (keeps pipeline consistent)
        txt = "Chest X-ray. Dataset: CheXpert. View: frontal."
        return x, y, txt, rel

print("\n[Stage] Load CheXpert VALID and evaluate AFTER CaD ...")
chex_valid_df = build_chex_valid_df(CHEX_VALID)
print("[CheX] valid rows:", len(chex_valid_df), "| patients:", chex_valid_df["patient_id"].nunique())

chex_resolver = detect_chex_image_root(CHEX_BASE, chex_valid_df["image"].head(50).tolist())
chex_valid_ds = CheXValidDataset(chex_valid_df, chex_resolver)
chex_valid_loader = DataLoader(chex_valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ============================================================
# (16) NEW: CheXpert VALID per-class AUC after CaD (use D pipeline)
# ============================================================
print("\n==============================")
print("(E) CheXpert VALID: AFTER CaD + Domain-ID + Quantum + Memory  ✅ requested")
print("==============================")
Yc, Pc = collect_preds_D_cad_quantum_memory(chex_valid_loader, max_batches=TEST_MAX_BATCHES)
dfc, mc = per_class_metrics(Yc, Pc, UNIFIED_LABELS)
print(dfc); print("[CheX-E] Macro:", mc)

dfc.to_csv(os.path.join(OUT_DIR, "CheXpertValid_per_class_E_after_CaD.csv"), index=False)

summary = {
    "estimator_backend": EST_BACKEND,
    "HAS_AER": HAS_AER,
    "config": {
        "N_SESSIONS": N_SESSIONS,
        "CAD_DPRIME": CAD_DPRIME,
        "DQ_MAIN":DQ_MAIN, "VQC_REPS":VQC_REPS, "N_OBS":N_OBS,
        "DELTA_LOGIT_SCALE":DELTA_LOGIT_SCALE,
        "K_RETRIEVE":K_RETRIEVE,"TAU_MEM":TAU_MEM,"LAMBDA_MEMPROB":LAMBDA_MEMPROB,
        "LAMBDA_MLCL":LAMBDA_MLCL,"TEMP_MLCL":TEMP_MLCL,
        "TRAIN_DOMAIN_FC": TRAIN_DOMAIN_FC
    },
    "memory": {"N":mem.N,"D":mem.D,"t_desc_dim":mem.t_desc_dim,"sizeGB":mem.memory_gb(),
               "has_v_img":mem.v_img is not None,"has_prototypes":mem.prototypes is not None,
               "has_base_conf":mem.base_conf is not None,"has_metadata":mem.metadata_available,
               "sklearn_index":mem.index_sklearn is not None},
    "NIH_metrics_A": m0,
    "NIH_metrics_B": m1,
    "NIH_metrics_C": m2,
    "NIH_metrics_D": m3,
    "CheXpertValid_metrics_E_after_CaD": mc
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Saved outputs to:", OUT_DIR)
print("  - train_sessions_log.csv")
print("  - NIH_per_class_A_cnn_only.csv")
print("  - NIH_per_class_B_quantum_no_memory.csv")
print("  - NIH_per_class_C_quantum_memory.csv")
print("  - NIH_per_class_D_cad_quantum_memory.csv")
print("  - CheXpertValid_per_class_E_after_CaD.csv")
print("  - summary.json")

# 23rd March,2026

# This model uses five small 3-qubit quantum branches, one for each disease. CaD learns across sessions, memory retrieves similar cases, and quantum outputs gently correct the CNN prediction. It also prunes weak entanglement, encourages branch diversity, and uses domain-aware initialization. 


In [12]:
# ============================================================
# NIH Domain-Incremental Learning
# CaD (Wang et al. 2025) + Parallel 5-Branch 3-Qubit QNNs
# ============================================================
# Novelties implemented:
#   1) Zero-init quantum weights (Grant et al. 2019)
#   2) Session-adaptive entanglement pruning
#   3) Observable diversity regularization
#   4) Meta-initialization network (domain center -> angles)
#   5) 5 parallel 3-qubit branches (one per class label)
#   6) FC head retained + quantum as additive correction
# ============================================================
# Metrics reported:
#   Per-class AUROC, F1, PR-AUC, ECE, Brier
#   Macro AUC with 95% Bootstrap CI
#   p-value (Wilcoxon signed-rank vs baseline)
#   Cohen's d, CV(%)
#   Mean/Std gradient magnitude per session
#   Normalized gradient vs single 3-qubit reference
#   Parameter count (quantum vs classical split)
#   Memory (GB), Training time (s)
# ============================================================

import os, json, math, time, random, inspect
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import wilcoxon, ttest_rel

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# ── (0) Reproducibility ──────────────────────────────────────
SEED = 7

def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── (1) Paths ─────────────────────────────────────────────────
MEM_DIR     = "/kaggle/input/datasets/zarinn/memory/MEMORY"
RESNET_CKPT = "/kaggle/input/resnet/pytorch/default/1/best_resnet50_chex_to_nih.pt"
BIOMED_CKPT = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"
NIH_IMG     = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV     = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM  = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

UNIFIED_LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
C = len(UNIFIED_LABELS)  # 5

OUT_DIR = "/kaggle/working/outputs_parallel_quantum"
os.makedirs(OUT_DIR, exist_ok=True)

# ── (2) Hyperparameters ───────────────────────────────────────
BATCH_SIZE            = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS           = 0
CALIBRATION_N         = 512
N_SESSIONS            = 3
EPOCHS_PER_SESSION    = 1
MAX_STEPS_PER_SESSION = 80
LOG_EVERY             = 10
TEST_MAX_BATCHES      = None

# Memory
K_RETRIEVE     = 8
TAU_MEM        = 10.0
LAMBDA_MEMPROB = 0.5

# CaD (Wang et al. 2025)
LAMBDA_MLCL     = 0.1
TEMP_MLCL       = 0.2
CAD_DPRIME      = 256
TRAIN_DOMAIN_FC = False

# Quantum — 5 branches × 3 qubits (one branch per class label)
N_BRANCHES   = 5
N_QUBITS     = 3
VQC_REPS     = 1
N_OBS        = 3       # Pauli-Z observables per branch (one per qubit)
DELTA_SCALE  = 0.35    # scale of additive quantum correction

# Novelty hyperparameters
LAMBDA_DIV       = 0.05    # observable diversity regularization weight
PRUNE_THRESHOLD  = 1e-5    # gradient variance below this -> reduce entanglement
META_INIT_SCALE  = 0.1     # output scale of MetaInitNet
BARREN_LOG_EVERY = 10      # log gradient variance every N steps

# Learning rates
LR_CLASSICAL = 1e-3
LR_QUANTUM   = 1e-2

print("\n[Config]", {
    "N_BRANCHES": N_BRANCHES, "N_QUBITS": N_QUBITS,
    "N_SESSIONS": N_SESSIONS, "LAMBDA_DIV": LAMBDA_DIV,
    "PRUNE_THRESHOLD": PRUNE_THRESHOLD,
    "DELTA_SCALE": DELTA_SCALE,
})

# ── (3) Qiskit setup ──────────────────────────────────────────
import qiskit
import qiskit_machine_learning
from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

print("Qiskit:", getattr(qiskit, "__version__", "?"))
print("Qiskit-ML:", getattr(qiskit_machine_learning, "__version__", "?"))

ESTIMATOR   = None
EST_BACKEND = None
try:
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
    ESTIMATOR   = AerEstimatorV2()
    EST_BACKEND = "qiskit_aer.primitives.EstimatorV2"
except Exception:
    try:
        from qiskit.primitives import StatevectorEstimator
        ESTIMATOR   = StatevectorEstimator()
        EST_BACKEND = "qiskit.primitives.StatevectorEstimator"
    except Exception as e:
        raise ImportError("No V2 Estimator found. Install qiskit-aer.") from e

sig    = inspect.signature(ESTIMATOR.run)
params = list(sig.parameters.keys())
if "observables" in params and "circuits" in params:
    raise RuntimeError(f"Estimator looks like V1 ({EST_BACKEND}). Need V2.")
print("Estimator backend:", EST_BACKEND)

_AER_BACKEND = AerSimulator()
_PM = generate_preset_pass_manager(optimization_level=1, backend=_AER_BACKEND)


def build_qnn(n_qubits=3, reps=1, entanglement="circular", n_obs=3):
    """Build a 3-qubit EstimatorQNN.
    Zero-initialized weights per Grant et al. (2019) arXiv:1903.05076.
    Local Pauli-Z observables per Cerezo et al. (2021) Nat. Commun. 12, 1791.
    Returns (TorchConnector, n_weights, n_obs_actual).
    """
    fm  = zz_feature_map(feature_dimension=n_qubits, reps=1,
                         entanglement=entanglement)
    ans = real_amplitudes(num_qubits=n_qubits, reps=reps,
                          entanglement=entanglement)

    qc = QuantumCircuit(n_qubits)
    qc = qc.compose(fm,  inplace=False)
    qc = qc.compose(ans, inplace=False)
    qc = qc.decompose(reps=10)
    qc = _PM.run(qc)

    n_obs_actual = min(n_obs, n_qubits)
    obs = []
    for i in range(n_obs_actual):
        pauli    = ["I"] * n_qubits
        pauli[i] = "Z"
        obs.append(SparsePauliOp("".join(pauli)))

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm.parameters,
        weight_params=ans.parameters,
        observables=obs,
        input_gradients=True,   # needed for hybrid backprop
        estimator=ESTIMATOR,
    )
    # ✅ Zero initialization (Grant et al. 2019: identity block init)
    init_w    = np.zeros(qnn.num_weights, dtype=np.float32)
    connector = TorchConnector(qnn, initial_weights=init_w)
    return connector, qnn.num_weights, n_obs_actual


# ── (4) Encoders: BiomedBERT + ResNet-50 (frozen) ────────────
from transformers import AutoTokenizer, AutoModel
import torchvision
from torchvision import transforms

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"


def _strip_prefix(sd, prefixes=("module.", "model.", "backbone.", "net.")):
    out = {}
    for k, v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p): kk = kk[len(p):]
        out[kk] = v
    return out


def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu")
    if isinstance(ck, dict) and "state_dict" in ck: sd = ck["state_dict"]
    elif isinstance(ck, dict) and "model"      in ck: sd = ck["model"]
    elif isinstance(ck, dict) and "net"        in ck: sd = ck["net"]
    else: sd = ck
    return _strip_prefix(sd)


def load_biomedbert():
    tok_dir = os.path.dirname(BIOMED_CKPT)
    try:    tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    except: tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
    model = AutoModel.from_pretrained(TEXT_MODEL_NAME)
    if os.path.exists(BIOMED_CKPT):
        sd = load_state_dict_any(BIOMED_CKPT)
        sd = {k: v for k, v in sd.items() if not k.startswith("cls.")}
        model.load_state_dict(sd, strict=False)
    model.eval().to(DEVICE)
    for p in model.parameters(): p.requires_grad = False
    return tokenizer, model


tokenizer, text_model = load_biomedbert()
DT_DIM = int(text_model.config.hidden_size)


@torch.no_grad()
def text_embed(text_list, max_len=64):
    enc = tokenizer(text_list, padding=True, truncation=True,
                    max_length=max_len, return_tensors="pt").to(DEVICE)
    out = text_model(**enc, return_dict=True)
    return F.normalize(out.last_hidden_state[:, 0, :], dim=1)


def load_resnet50():
    m    = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(2048, C)
    sd   = load_state_dict_any(RESNET_CKPT)
    miss, unexp = m.load_state_dict(sd, strict=False)
    print(f"[ResNet] missing={len(miss)}  unexpected={len(unexp)}")
    m.eval().to(DEVICE)
    for p in m.parameters(): p.requires_grad = False

    @torch.no_grad()
    def spatial(x):
        b = m
        x = b.conv1(x); x = b.bn1(x); x = b.relu(x); x = b.maxpool(x)
        x = b.layer1(x); x = b.layer2(x); x = b.layer3(x); x = b.layer4(x)
        return x  # [B, 2048, 7, 7]

    @torch.no_grad()
    def logits_fn(x): return m(x)

    return m, spatial, logits_fn


resnet_full, resnet_spatial, cnn_logits_from_resnet = load_resnet50()

img_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def pool2048(fmap):
    v = F.adaptive_avg_pool2d(fmap, (1, 1)).flatten(1)
    return F.normalize(v, dim=1)


# ── (5) Dataset ───────────────────────────────────────────────
def read_list(path):
    with open(path) as f:
        return [x.strip() for x in f if x.strip()]


nih_df  = pd.read_csv(NIH_CSV)
nih_df  = nih_df.rename(columns={"Image Index": "image", "Finding Labels": "labels"})
nih_row = {r["image"]: r for _, r in nih_df.iterrows()}


def labels_to_multi_hot(lbl_str):
    parts = set(p.strip() for p in str(lbl_str).split("|"))
    y     = np.zeros(C, dtype=np.float32)
    for i, name in enumerate(UNIFIED_LABELS):
        key = "Effusion" if name == "Pleural Effusion" else name
        if key in parts: y[i] = 1.0
    return y


def make_text_descriptor(r):
    view = r.get("View Position", r.get("ViewPosition", ""))
    age  = r.get("Patient Age",   r.get("PatientAge",   ""))
    sex  = r.get("Patient Gender", r.get("PatientGender", ""))
    return f"Chest X-ray. View: {view}. Patient age: {age}. Sex: {sex}."


class NIHDataset(Dataset):
    def __init__(self, names):
        self.items = list(names)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        name = self.items[idx]
        img  = Image.open(os.path.join(NIH_IMG, name)).convert("RGB")
        x    = img_tfms(img)
        r    = nih_row.get(name, {})
        y    = labels_to_multi_hot(r.get("labels", ""))
        txt  = make_text_descriptor(r)
        return x, torch.tensor(y, dtype=torch.float32), txt, name


stream_list = read_list(NIH_STREAM)
test_list   = read_list(NIH_TEST)
calib_list  = stream_list[:CALIBRATION_N]


def split_sessions(lst, n):
    total  = len(lst)
    splits = []
    for s in range(n):
        a = (total * s)     // n
        b = (total * (s+1)) // n
        splits.append(lst[a:b])
    return splits


session_lists = split_sessions(stream_list, N_SESSIONS)
calib_loader  = DataLoader(NIHDataset(calib_list), batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=NUM_WORKERS)
test_loader   = DataLoader(NIHDataset(test_list),  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS)

print("\n[Dataset]", {
    "stream": len(stream_list),
    "sessions": [len(x) for x in session_lists],
    "test": len(test_list)
})

# ── (6) Memory bundle ─────────────────────────────────────────
import faiss
try:    import joblib
except: joblib = None


class MemoryBundle:
    def __init__(self, mem_dir):
        self.mem_dir = mem_dir

        E = np.load(os.path.join(mem_dir, "embeddings.npy"))
        Y = np.load(os.path.join(mem_dir, "labels.npy"))
        if E.dtype != np.float32: E = E.astype(np.float32)
        E = np.ascontiguousarray(E)
        self.E = E; self.N, self.D = E.shape

        if Y.ndim == 1:
            Y_oh = np.zeros((len(Y), C), dtype=np.float32)
            Y_oh[np.arange(len(Y)), Y.astype(np.int64)] = 1.0
            Y = Y_oh
        self.Y = np.ascontiguousarray(Y.astype(np.float32))

        tdesc_path = os.path.join(mem_dir, "t_desc.npy")
        self.t_desc = None
        if os.path.exists(tdesc_path):
            td          = np.load(tdesc_path).astype(np.float32)
            self.t_desc = np.ascontiguousarray(td)
        self.t_desc_dim = self.t_desc.shape[1] if self.t_desc is not None else 0

        bc_path        = os.path.join(mem_dir, "base_conf.npy")
        self.base_conf = None
        if os.path.exists(bc_path):
            bc = np.load(bc_path).astype(np.float32).reshape(-1)
            if bc.shape[0] == self.N:
                self.base_conf = np.clip(bc, 0.0, 1.0)

        E2 = E.copy(); faiss.normalize_L2(E2)
        self.faiss_index = faiss.IndexFlatIP(self.D)
        self.faiss_index.add(E2)

        self._Y_t  = torch.tensor(self.Y,     device=DEVICE, dtype=torch.float32)
        self._td_t = (torch.tensor(self.t_desc, device=DEVICE, dtype=torch.float32)
                      if self.t_desc is not None else None)

    def memory_gb(self):
        total = 0
        for fn in ["embeddings.npy", "labels.npy", "t_desc.npy",
                   "v_img.npy", "prototypes.npy", "base_conf.npy"]:
            p = os.path.join(self.mem_dir, fn)
            if os.path.exists(p): total += os.path.getsize(p)
        return total / 1e9

    @torch.no_grad()
    def query_key(self, v, t):
        dv, dt = v.shape[1], t.shape[1]
        if   self.D == dv + dt: q = torch.cat([v, t], dim=1)
        elif self.D == dv:      q = v
        elif self.D == dt:      q = t
        else:
            torch.manual_seed(SEED)
            W = torch.randn(dv + dt, self.D, device=v.device) / math.sqrt(dv + dt)
            q = torch.cat([v, t], dim=1) @ W
        return F.normalize(q, dim=1)

    @torch.no_grad()
    def retrieve(self, q, k=16):
        qn  = F.normalize(q, dim=1).detach().cpu().numpy().astype(np.float32)
        qn  = np.ascontiguousarray(qn)
        sims, idxs = self.faiss_index.search(qn, min(k, self.N))
        return idxs.astype(np.int64), sims.astype(np.float32)

    @torch.no_grad()
    def retrieve_dt(self, idxs):
        if self._td_t is None:
            return torch.zeros(idxs.shape[0], 1, device=DEVICE)
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        dt     = self._td_t[idxs_t].mean(dim=1)
        return F.normalize(dt, dim=1)

    @torch.no_grad()
    def pmem_from_neighbors(self, idxs, sims, tau=10.0, eps=1e-8):
        idxs_t = torch.tensor(idxs, device=DEVICE, dtype=torch.long)
        sims_t = torch.tensor(sims, device=DEVICE, dtype=torch.float32)
        neigh_y = self._Y_t[idxs_t]
        w       = torch.softmax(tau * sims_t, dim=1)
        if self.base_conf is not None:
            conf = torch.tensor(self.base_conf, device=DEVICE)[idxs_t]
            w    = w * conf
            w    = w / (w.sum(dim=1, keepdim=True) + 1e-12)
        p = (w.unsqueeze(-1) * neigh_y).sum(dim=1)
        return torch.clamp(p, eps, 1 - eps)


mem         = MemoryBundle(MEM_DIR)
dt_cond_dim = mem.t_desc_dim if mem.t_desc_dim > 0 else 1
print("\n[Memory]", {"N": mem.N, "D": mem.D, "t_desc_dim": mem.t_desc_dim,
                     "sizeGB": round(mem.memory_gb(), 3)})

# ── (7) CaD adapters (Wang et al. 2025 — teammates' code) ────
def fmap_to_tokens(fmap):
    B, d, H, W = fmap.shape
    x_img = fmap.permute(0, 2, 3, 1).reshape(B, H * W, d)
    x_cls = fmap.mean(dim=(2, 3)).unsqueeze(1)
    return x_cls, x_img, H, W


class CAdapter(nn.Module):
    """Cross-domain Invariant Feature Absorption (CIFA). Wang et al. 2025 Eq.(2-4)."""
    def __init__(self, d=2048, dprime=256):
        super().__init__()
        self.ln1  = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2  = nn.LayerNorm(dprime)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in):
        M_mid = F.relu(self.down(self.ln1(M_in)))
        M_til = self.up(self.ln2(M_mid))
        return F.relu(M_til + M_in)


@torch.no_grad()
def absorb_cadapter_(running: CAdapter, current: CAdapter, t: int):
    """Wang et al. 2025 Eq.(5-8): running <- (1/t)*((t-1)*running + current)."""
    for (_, p1), (_, p2) in zip(running.named_parameters(),
                                 current.named_parameters()):
        p1.data.mul_((t - 1) / t).add_(p2.data, alpha=1.0 / t)


class DAdapter(nn.Module):
    """Domain-Specific Feature Retention (DSFR). Wang et al. 2025 Eq.(9-15)."""
    def __init__(self, d=2048, dprime=256, k=3):
        super().__init__()
        self.ln1  = nn.LayerNorm(d)
        self.down = nn.Linear(d, dprime)
        self.ln2  = nn.LayerNorm(dprime)
        self.conv = nn.Conv2d(dprime, dprime, kernel_size=k, padding=k // 2)
        self.up   = nn.Linear(dprime, d)
        nn.init.zeros_(self.conv.weight)
        if self.conv.bias is not None: nn.init.zeros_(self.conv.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, M_in, H=7, W=7):
        M_mid   = F.relu(self.down(self.ln1(M_in)))
        x_cls   = M_mid[:, :1, :]
        x_img   = M_mid[:, 1:, :]
        x_grid  = x_img.reshape(x_img.size(0), H, W,
                                 x_img.size(2)).permute(0, 3, 1, 2).contiguous()
        x_grid2 = self.conv(x_grid)
        x_img2  = x_grid2.permute(0, 2, 3, 1).reshape(
            x_img.size(0), H * W, x_img.size(2)).contiguous()
        M_mid2  = F.relu(torch.cat([x_cls, x_img2], dim=1))
        M_til   = self.up(self.ln2(M_mid2))
        return F.relu(M_til + M_in)


class DAdapterBank(nn.Module):
    def __init__(self, n, d=2048, dprime=256):
        super().__init__()
        self.adapters = nn.ModuleList([DAdapter(d, dprime) for _ in range(n)])

    def forward(self, M_in, domain_id, H=7, W=7):
        return self.adapters[domain_id](M_in, H=H, W=W)

    def freeze(self, domain_id):
        for p in self.adapters[domain_id].parameters(): p.requires_grad = False


class DomainFC(nn.Module):
    def __init__(self, d=2048, nc=5):
        super().__init__()
        self.fc = nn.Linear(d, nc)

    def forward(self, x): return self.fc(x)


class CaDModel(nn.Module):
    def __init__(self, n_sessions, d=2048, dprime=256, nc=5):
        super().__init__()
        self.c_running = CAdapter(d, dprime)
        self.c_current = CAdapter(d, dprime)
        self.d_bank    = DAdapterBank(n_sessions, d, dprime)
        self.fc_bank   = nn.ModuleList([DomainFC(d, nc) for _ in range(n_sessions)])
        base_sd        = resnet_full.fc.state_dict()
        for di in range(n_sessions):
            self.fc_bank[di].fc.load_state_dict(base_sd, strict=True)
        self.domain_centers = [None] * n_sessions

    def forward_tokens(self, fmap, domain_id):
        x_cls, x_img, H, W = fmap_to_tokens(fmap)
        M_in  = torch.cat([x_cls, x_img], dim=1)
        M_c   = self.c_running(M_in)
        M_d   = self.d_bank(M_in, domain_id, H, W)
        M_out = M_c + M_d - M_in   # Eq.1 Wang et al.: no double-counting shortcut
        return M_out[:, :1, :].squeeze(1), M_out[:, 1:, :]

    def logits_for_domain(self, fmap, domain_id):
        cls, img = self.forward_tokens(fmap, domain_id)
        return self.fc_bank[domain_id](cls), cls, img


cad = CaDModel(N_SESSIONS, d=2048, dprime=CAD_DPRIME, nc=C).to(DEVICE)
for di in range(N_SESSIONS):
    for p in cad.fc_bank[di].parameters():
        p.requires_grad = TRAIN_DOMAIN_FC


def forward_cad_train(fmap, session_id):
    x_cls, x_img, H, W = fmap_to_tokens(fmap)
    M_in  = torch.cat([x_cls, x_img], dim=1)
    M_c   = cad.c_current(M_in)
    M_d   = cad.d_bank(M_in, session_id, H, W)
    M_out = M_c + M_d - M_in
    cls   = M_out[:, :1, :].squeeze(1)
    return cad.fc_bank[session_id](cls), cls, M_out[:, 1:, :]


# ── (8) MetaInitNet: domain center → initial quantum angles ──
class MetaInitNet(nn.Module):
    """
    Novel contribution: domain-context-conditioned quantum parameter initialization.
    Maps domain center (2048-dim) to initial rotation angles for all branches.
    Output is near-zero to stay close to identity initialization (Grant et al. 2019).
    Differentiated from Q-MAML (arXiv:2501.05906) by being domain-incremental-aware.
    """
    def __init__(self, domain_dim: int, total_q_params: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(domain_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, total_q_params),
        )
        # Near-zero init to keep output close to zero → near-identity start
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.05)
                nn.init.zeros_(m.bias)

    def forward(self, domain_center):
        # Output in [-META_INIT_SCALE, META_INIT_SCALE] ≈ near zero
        return META_INIT_SCALE * torch.tanh(self.net(domain_center))


# ── (9) ParallelQuantumAdapter ────────────────────────────────
class ParallelQuantumAdapter(nn.Module):
    """
    5 parallel 3-qubit QNN branches (one per class label).

    Novel contributions:
      (a) Zero initialization — Grant et al. 2019 (arXiv:1903.05076)
          Keeps each branch near identity at session start, avoiding barren plateaus
          from the first gradient step.

      (b) Observable diversity regularization
          Penalizes cosine similarity between branch expectation-value vectors.
          Prevents branch collapse (branches becoming identical) which would reduce
          the parallel structure to a single wider circuit — the exact regime that
          induces barren plateaus (McClean et al. 2018, Nat. Commun. 9, 4812).
          No quantum prior art for this regularization.

      (c) Session-adaptive entanglement pruning
          After each domain session, measure gradient variance per branch.
          Branches whose ZZ-entanglement parameters show variance below threshold
          have entanglement topology reduced (circular → linear → sca).
          Connects domain-incremental session boundaries to circuit topology —
          a gap confirmed absent in the literature (no paper connects entanglement
          pruning to continual learning).

      (d) Meta-initialization network
          Warm-starts branch weights from domain center vector before each session.

    Architecture:
      CLS token [B, 2048] → thumbnail proj [B, 15]
      → 5 × [B, 3] slices → 5 QNN branches
      → concat [B, 15] → collapse head → delta [B, 5]
      Final: logits = base_logits + gate(dt) * DELTA_SCALE * delta
      (FC head from CaD is preserved — quantum is additive correction)
    """

    def __init__(self, d_in=2048, n_branches=5, n_qubits=3,
                 reps=1, n_obs=3, n_classes=5, dt_cond_dim=1):
        super().__init__()
        self.n_branches  = n_branches
        self.n_qubits    = n_qubits
        self.n_classes   = n_classes
        self.dt_cond_dim = max(dt_cond_dim, 1)

        # Thumbnail projector: CLS → angle inputs for all branches
        self.thumb_proj = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.GELU(),
            nn.Linear(128, n_branches * n_qubits),
        )

        # Build 5 parallel branches with zero init
        self.entanglement_per_branch = ["circular"] * n_branches
        self.branches                = nn.ModuleList()
        self._n_weights_per_branch   = []
        self._n_obs_per_branch       = []

        for b in range(n_branches):
            conn, nw, nobs = build_qnn(n_qubits, reps, "circular", n_obs)
            self.branches.append(conn)
            self._n_weights_per_branch.append(nw)
            self._n_obs_per_branch.append(nobs)

        total_q_out     = sum(self._n_obs_per_branch)  # n_branches * n_obs
        total_q_weights = sum(self._n_weights_per_branch)

        # MetaInitNet: maps domain center → all branch init angles
        self.meta_init = MetaInitNet(2048, total_q_weights)

        # dt gate: memory conditioning on the correction scale
        self.dt_gate = nn.Sequential(
            nn.Linear(self.dt_cond_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

        # Collapse head: quantum expectation values → delta logits
        self.collapse = nn.Sequential(
            nn.Linear(total_q_out, 32),
            nn.GELU(),
            nn.Linear(32, n_classes),
        )

        # Gradient tracking for pruning decision
        self.grad_history = {b: [] for b in range(n_branches)}

    def forward(self, base_logits, cls_token, dt):
        """
        base_logits : [B, C]      from CaD FC head (preserved)
        cls_token   : [B, 2048]   from CaD CLS token
        dt          : [B, dt_dim] from RAG memory retrieval
        Returns (logits [B,C], branch_outs list[tensor], gate [B,1])
        """
        # Thumbnail projection and angle encoding
        z      = self.thumb_proj(cls_token)         # [B, n_branches * n_qubits]
        z      = math.pi * torch.tanh(z)            # scale to (-π, π)
        slices = z.chunk(self.n_branches, dim=1)    # list of [B, n_qubits]

        # Parallel branch forward passes
        branch_outs = []
        for b in range(self.n_branches):
            q_out = self.branches[b](slices[b])     # [B, n_obs]
            branch_outs.append(q_out)

        q_concat = torch.cat(branch_outs, dim=1)    # [B, total_q_out]

        # Gate from memory signal
        gate  = torch.sigmoid(self.dt_gate(dt))     # [B, 1]

        # Delta correction
        delta  = self.collapse(q_concat)            # [B, C]

        # Additive correction — FC head from CaD is preserved
        logits = base_logits + gate * DELTA_SCALE * delta
        return logits, branch_outs, gate

    def diversity_loss(self, branch_outs):
        """
        Novel: Observable diversity regularization.
        Penalizes off-diagonal cosine similarity between branch output vectors.
        Prevents branch collapse → maintains gradient diversity across branches.
        No quantum prior art (QuEEn arXiv:2409.09103 explicitly lists this as future work).
        """
        stacked  = torch.stack(branch_outs, dim=1)   # [B, n_branches, n_obs]
        normed   = F.normalize(stacked, dim=-1)
        sim_mat  = torch.bmm(normed, normed.transpose(1, 2))  # [B, n_branches, n_branches]
        n        = self.n_branches
        mask     = ~torch.eye(n, dtype=torch.bool, device=sim_mat.device)
        off_diag = sim_mat[:, mask]                   # [B, n*(n-1)]
        return off_diag.abs().mean()

    def record_gradients(self):
        """Called after loss.backward() to log gradient magnitudes per branch."""
        for b in range(self.n_branches):
            grads = [p.grad.detach().abs().mean().item()
                     for p in self.branches[b].parameters()
                     if p.grad is not None]
            if grads:
                self.grad_history[b].append(float(np.mean(grads)))

    def gradient_variance_per_branch(self):
        """Return Var[∂L/∂θ] per branch — used for pruning decision."""
        return {b: float(np.var(h)) if len(h) > 1 else 0.0
                for b, h in self.grad_history.items()}

    def apply_meta_init(self, domain_center_tensor: torch.Tensor):
        """
        Warm-start branch weights from domain center via MetaInitNet.
        Novel: domain-geometry-conditioned VQC initialization for continual learning.
        """
        with torch.no_grad():
            init_angles = self.meta_init(
                domain_center_tensor.unsqueeze(0)).squeeze(0)  # [total_q_weights]
            offset = 0
            for b in range(self.n_branches):
                nw  = self._n_weights_per_branch[b]
                w_b = init_angles[offset: offset + nw].cpu().float().numpy()
                self.branches[b].weight.data = torch.tensor(w_b, dtype=torch.float32)
                offset += nw

    def prune_and_rebuild(self, reps=1, n_obs=3):
        """
        Session-adaptive entanglement pruning (Novel Idea 1).
        After each domain session: branches with Var[∂L/∂θ] < threshold
        have their entanglement topology reduced: circular → linear.
        This maintains at least 'linear' entanglement (minimum 1 ZZ gate pair)
        to preserve quantum advantage (product-state circuits are trivially classical).

        Connects two disconnected literatures:
          - Quantum circuit pruning (ATP arXiv:2503.21815, QAdaPrune arXiv:2408.13352)
          - Quantum continual learning (arXiv:2409.09729, QUARTA 2024)
        """
        variances   = self.gradient_variance_per_branch()
        pruning_log = {}
        ent_order   = ["circular", "linear"]  # sca as last resort if supported

        for b in range(self.n_branches):
            var     = variances[b]
            current = self.entanglement_per_branch[b]
            pruning_log[b] = {"var": round(var, 8), "before": current, "after": current}

            if var < PRUNE_THRESHOLD and current != "linear":
                new_ent = "linear"
                try:
                    conn, nw, nobs = build_qnn(self.n_qubits, reps, new_ent, n_obs)
                    self.branches[b]               = conn.to(DEVICE)
                    self._n_weights_per_branch[b]  = nw
                    self._n_obs_per_branch[b]      = nobs
                    self.entanglement_per_branch[b] = new_ent
                    pruning_log[b]["after"]        = new_ent
                    print(f"  [Prune] Branch {b}: {current} → {new_ent}  "
                          f"(var={var:.2e} < {PRUNE_THRESHOLD:.2e})")
                except Exception as e:
                    print(f"  [Prune] Branch {b} rebuild failed: {e}")
            else:
                print(f"  [Prune] Branch {b}: kept '{current}'  (var={var:.2e})")

        # Reset history after pruning (new topology = new baseline)
        self.grad_history = {b: [] for b in range(self.n_branches)}
        return pruning_log


# Instantiate
qadapter = ParallelQuantumAdapter(
    d_in=2048, n_branches=N_BRANCHES, n_qubits=N_QUBITS,
    reps=VQC_REPS, n_obs=N_OBS, n_classes=C, dt_cond_dim=dt_cond_dim
).to(DEVICE)

total_q_params = sum(qadapter._n_weights_per_branch)
total_q_obs    = sum(qadapter._n_obs_per_branch)
print(f"\n[QAdapter] {N_BRANCHES} branches × {N_QUBITS} qubits")
print(f"  Weights per branch : {qadapter._n_weights_per_branch}")
print(f"  Obs per branch     : {qadapter._n_obs_per_branch}")
print(f"  Total Q weights    : {total_q_params}")
print(f"  Total Q outputs    : {total_q_obs}")

# ── (10) Loss functions ───────────────────────────────────────
crit_bce = nn.BCEWithLogitsLoss()
crit_mem = nn.BCELoss()


def label_similarity(y_i, y_j, eps=1e-9):
    yi = (y_i > 0.5).float()
    yj = (y_j > 0.5).float()
    return ((yi * yj).sum(-1) /
            (((yi + yj) > 0).float().sum(-1) + eps))


def mlcl_loss(features, labels, temp=0.2, eps=1e-9):
    """Multi-Label Contrastive Loss — Wang et al. 2025 Eq.(19)."""
    B    = features.size(0)
    f    = F.normalize(features, dim=1)
    sim  = (f @ f.t()) / temp - torch.eye(B, device=f.device) * 1e9
    yi   = labels.unsqueeze(1).expand(B, B, -1)
    yj   = labels.unsqueeze(0).expand(B, B, -1)
    r    = label_similarity(yi, yj) * (1 - torch.eye(B, device=f.device))
    logp = F.log_softmax(sim, dim=1)
    return (-(r * logp).sum(1) / (r.sum(1) + eps)).mean()


# ── (11) Optimizer ────────────────────────────────────────────
def make_session_optimizer(session_id):
    # CaD: freeze running C, train current C + current D
    for p in cad.c_running.parameters(): p.requires_grad = False
    for p in cad.c_current.parameters(): p.requires_grad = True
    for di in range(N_SESSIONS):
        for p in cad.d_bank.adapters[di].parameters():
            p.requires_grad = (di == session_id)
        for p in cad.fc_bank[di].parameters():
            p.requires_grad = TRAIN_DOMAIN_FC and (di == session_id)

    cad_params = (list(cad.c_current.parameters()) +
                  list(cad.d_bank.adapters[session_id].parameters()) +
                  (list(cad.fc_bank[session_id].parameters()) if TRAIN_DOMAIN_FC else []))

    # Split quantum and classical params for different LRs
    q_params, c_params = [], []
    for name, p in qadapter.named_parameters():
        if not p.requires_grad: continue
        is_branch = any(f"branches.{b}." in name for b in range(N_BRANCHES))
        if is_branch: q_params.append(p)
        else:         c_params.append(p)

    c_params = c_params + cad_params
    return torch.optim.Adam([
        {"params": c_params, "lr": LR_CLASSICAL},
        {"params": q_params, "lr": LR_QUANTUM},
    ])


# ── (12) Reference gradient variance (for normalization) ─────
def measure_reference_grad_variance(n_samples=20):
    """
    Measure Var[∂L/∂θ] for a single 3-qubit circuit with random init.
    Used to normalize the reported gradient magnitudes.
    Reproduces the measurement protocol from McClean et al. 2018.
    """
    ref_conn, _, _ = build_qnn(N_QUBITS, VQC_REPS, "circular", N_OBS)
    # Override zero init with random for the reference baseline
    n_w = ref_conn.weight.numel()
    ref_conn.weight.data = torch.tensor(
        np.random.uniform(-math.pi, math.pi, n_w).astype(np.float32))
    ref_conn = ref_conn.to(DEVICE)
    grads = []
    for _ in range(n_samples):
        x = torch.rand(4, N_QUBITS, device=DEVICE) * math.pi
        try:
            out  = ref_conn(x)
            loss = out.sum()
            ref_conn.zero_grad()
            loss.backward()
            g = [p.grad.detach().abs().mean().item()
                 for p in ref_conn.parameters() if p.grad is not None]
            if g: grads.append(float(np.mean(g)))
        except Exception:
            pass
    if grads:
        return float(np.mean(grads)), float(np.var(grads))
    return 1.0, 1.0


print("\n[Measuring reference gradient (single 3-qubit random init) ...]")
ref_grad_mean, ref_grad_var = measure_reference_grad_variance(n_samples=20)
print(f"  Ref mean grad: {ref_grad_mean:.4e}  Ref var: {ref_grad_var:.4e}")

# ── (13) Domain center computation ───────────────────────────
def compute_domain_center(session_id, loader):
    cad.eval()
    feats = []
    with torch.no_grad():
        for x, y, txt, _ in loader:
            x    = x.to(DEVICE)
            fmap = resnet_spatial(x)
            _, img_tokens = cad.forward_tokens(fmap, session_id)
            feats.append(img_tokens.mean(dim=1).cpu())
    return torch.cat(feats, 0).mean(0).numpy().astype(np.float32)


# ── (14) Sanity check ────────────────────────────────────────
@torch.no_grad()
@torch.no_grad()
def sanity_check():
    x, y, txt, _ = next(iter(calib_loader))
    x = x.to(DEVICE)
    fmap = resnet_spatial(x)
    x_cls, x_img, H, W = fmap_to_tokens(fmap)
    print("\n[Sanity] fmap:", tuple(fmap.shape),
          "cls:", tuple(x_cls.shape), "img:", tuple(x_img.shape))

    # Baseline equality check at init
    base               = cnn_logits_from_resnet(x)
    cad_l0, cad_cls, _ = cad.logits_for_domain(fmap, 0)
    diff               = (base - cad_l0).abs().max().item()
    print(f"[Sanity] max|ResNet - CaD(d=0)| = {diff:.6f}  (should be ≈ 0)")

    # Memory retrieval — must come BEFORE quantum forward
    v          = pool2048(fmap)
    t          = text_embed(list(txt))
    qkey       = mem.query_key(v, t)
    idxs, sims = mem.retrieve(qkey, K_RETRIEVE)
    dt         = mem.retrieve_dt(idxs)
    print(f"[Sanity] dt: {tuple(dt.shape)}  sims: [{sims.min():.3f}, {sims.max():.3f}]")

    # Quick quantum forward — cls token [B,2048], not logits [B,5]
    logits, branch_outs, gate = qadapter(cad_l0, cad_cls, dt)
    div_l = qadapter.diversity_loss(branch_outs)
    print(f"[Sanity] logits: {tuple(logits.shape)}  gate: [{gate.min().item():.3f}, {gate.max().item():.3f}]")
    print(f"[Sanity] diversity_loss at init: {div_l.item():.4f}  (ideally > 0)")


sanity_check()

# ── (15) Training loop ────────────────────────────────────────
def train_sessions():
    history         = []
    all_grad_log    = []

    for s in range(N_SESSIONS):
        print(f"\n{'='*55}")
        print(f"  Session {s+1}/{N_SESSIONS}  (domain {s})")
        print(f"{'='*55}")

        ds     = NIHDataset(session_lists[s])
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS)
        opt    = make_session_optimizer(s)
        cad.train()
        qadapter.train()

        step             = 0
        t0               = time.time()
        session_grads    = []

        for ep in range(EPOCHS_PER_SESSION):
            for x, y, txt, _ in loader:
                x = x.to(DEVICE); y = y.to(DEVICE)
                fmap = resnet_spatial(x)

                # Memory retrieval
                v      = pool2048(fmap)
                t_emb  = text_embed(list(txt))
                qkey   = mem.query_key(v, t_emb)
                idxs, sims = mem.retrieve(qkey, K_RETRIEVE)
                dt     = mem.retrieve_dt(idxs)
                p_m    = mem.pmem_from_neighbors(idxs, sims, TAU_MEM)

                # CaD forward (c_current + D_s)
                base_logits, cls_feat, _ = forward_cad_train(fmap, s)

                # Parallel quantum forward
                logits, branch_outs, gate = qadapter(base_logits, cls_feat, dt)

                # Losses
                loss_bce  = crit_bce(logits, y)
                loss_mlcl = mlcl_loss(cls_feat, y, TEMP_MLCL)
                loss_mem  = crit_mem(torch.sigmoid(logits), p_m)
                loss_div  = qadapter.diversity_loss(branch_outs)

                loss = (loss_bce
                        + LAMBDA_MLCL    * loss_mlcl
                        + LAMBDA_MEMPROB * loss_mem
                        + LAMBDA_DIV     * loss_div)

                opt.zero_grad(set_to_none=True)
                loss.backward()

                # Record gradient magnitudes BEFORE optimizer step
                qadapter.record_gradients()

                # Collect for this step
                for b in range(N_BRANCHES):
                    h = qadapter.grad_history[b]
                    if h:
                        session_grads.append(h[-1])
                        all_grad_log.append({
                            "session": s, "step": step, "branch": b,
                            "grad_mean": h[-1],
                            "normalized": h[-1] / (ref_grad_mean + 1e-20),
                        })

                opt.step()
                step += 1

                if step % LOG_EVERY == 0:
                    gvars = qadapter.gradient_variance_per_branch()
                    avg_var = np.mean(list(gvars.values()))
                    print(f"  [s{s}] step={step:4d}  "
                          f"loss={loss.item():.4f}  "
                          f"bce={loss_bce.item():.4f}  "
                          f"div={loss_div.item():.4f}  "
                          f"gate={gate.mean().item():.3f}  "
                          f"avg_grad_var={avg_var:.2e}  "
                          f"t={time.time()-t0:.1f}s")

                if step >= MAX_STEPS_PER_SESSION: break
            if step >= MAX_STEPS_PER_SESSION: break

        # ── Post-session operations ──────────────────────────

        # 1) Session-adaptive entanglement pruning
        print(f"\n[Session {s}] Entanglement pruning ...")
        pruning_log = qadapter.prune_and_rebuild(VQC_REPS, N_OBS)

        # 2) Compute domain center
        cad.eval()
        center_loader = DataLoader(ds, batch_size=BATCH_SIZE,
                                   shuffle=False, num_workers=NUM_WORKERS)
        center = compute_domain_center(s, center_loader)
        cad.domain_centers[s] = center
        print(f"[Session {s}] Domain center: {center.shape}")

        # 3) MetaInitNet warm-start for next session
        center_t = torch.tensor(center, device=DEVICE, dtype=torch.float32)
        print(f"[Session {s}] Applying MetaInitNet warm-start ...")
        qadapter.apply_meta_init(center_t)

        # 4) Absorb C-adapter (Wang et al. 2025 Eq.5-8)
        with torch.no_grad():
            absorb_cadapter_(cad.c_running, cad.c_current, t=s + 1)
            cad.c_current.load_state_dict(cad.c_running.state_dict())

        # 5) Freeze D-adapter
        cad.d_bank.freeze(s)
        if TRAIN_DOMAIN_FC:
            for p in cad.fc_bank[s].parameters(): p.requires_grad = False

        # Session gradient summary
        mean_g = float(np.mean(session_grads)) if session_grads else 0.0
        std_g  = float(np.std(session_grads))  if session_grads else 0.0
        norm_g = mean_g / (ref_grad_mean + 1e-20)
        print(f"\n[Session {s}] Mean |grad|={mean_g:.4e}  "
              f"Std={std_g:.4e}  "
              f"Normalized={norm_g:.4f}")

        history.append({
            "session":        s,
            "steps":          step,
            "train_sec":      round(time.time() - t0, 2),
            "mean_grad_mag":  round(mean_g, 8),
            "std_grad_mag":   round(std_g,  8),
            "normalized_grad": round(norm_g, 6),
            "pruning_log":    str(pruning_log),
        })

    pd.DataFrame(all_grad_log).to_csv(
        os.path.join(OUT_DIR, "gradient_variance_log.csv"), index=False)
    return pd.DataFrame(history)


print("\n[Training ...]")
train_log = train_sessions()
train_log.to_csv(os.path.join(OUT_DIR, "train_log.csv"), index=False)
print(train_log.to_string())

# ── (16) Evaluation helpers ───────────────────────────────────
def ece_binary(yt, yp, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = ((yp >= lo) & (yp < hi)
             if i < n_bins - 1
             else (yp >= lo) & (yp <= hi))
        if m.sum() == 0: continue
        ece += (m.sum() / len(yt)) * abs(yt[m].mean() - yp[m].mean())
    return float(ece)


def brier_score(yt, yp):
    return float(np.mean((yp.astype(np.float32) - yt.astype(np.float32)) ** 2))


def bootstrap_macro_auc_ci(Y, P, n_boot=1000, alpha=0.05, seed=42):
    """Bootstrap 95% CI on macro AUROC."""
    rng  = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx  = rng.choice(len(Y), len(Y), replace=True)
        Yb   = Y[idx]; Pb = P[idx]
        try:
            macro = float(np.nanmean([
                roc_auc_score(Yb[:, j], Pb[:, j])
                for j in range(Y.shape[1])
                if len(np.unique(Yb[:, j])) > 1
            ]))
            aucs.append(macro)
        except Exception:
            pass
    aucs = sorted(aucs)
    lo   = float(np.percentile(aucs, 100 * alpha / 2))
    hi   = float(np.percentile(aucs, 100 * (1 - alpha / 2)))
    return lo, hi


def cohens_d(a, b):
    """Cohen's d effect size between two arrays."""
    a = np.array(a); b = np.array(b)
    pooled = np.sqrt((a.std() ** 2 + b.std() ** 2) / 2.0)
    return float((a.mean() - b.mean()) / (pooled + 1e-9))


def per_class_metrics(Y, P, labels):
    rows     = []
    auc_vals = []
    for j, lbl in enumerate(labels):
        yt   = Y[:, j]; yp = P[:, j]
        if len(np.unique(yt)) < 2:
            aucv = apv = float("nan")
        else:
            aucv = roc_auc_score(yt, yp)
            apv  = average_precision_score(yt, yp)
            auc_vals.append(aucv)
        yhat = (yp >= 0.5).astype(np.int32)
        f1v  = f1_score(yt.astype(int), yhat, zero_division=0)
        ecev = ece_binary(yt, yp)
        brv  = brier_score(yt, yp)
        rows.append([lbl, aucv, apv, f1v, ecev, brv])

    df       = pd.DataFrame(rows, columns=["Label", "ROC_AUC", "PR_AUC",
                                            "F1@0.5", "ECE", "Brier"])
    macro    = {k: float(np.nanmean(df[k])) for k in
                ["ROC_AUC", "PR_AUC", "F1@0.5", "ECE", "Brier"]}
    cv_pct   = (float(np.std(auc_vals) / (np.mean(auc_vals) + 1e-9) * 100)
                if auc_vals else 0.0)
    ci_lo, ci_hi = bootstrap_macro_auc_ci(Y, P, n_boot=500)
    macro.update({"cv_pct": round(cv_pct, 3),
                  "ci_95_lo": round(ci_lo, 4),
                  "ci_95_hi": round(ci_hi, 4)})
    return df, macro


# ── (17) Inference functions ──────────────────────────────────
@torch.no_grad()
def collect_A_cnn_only(loader, max_batches=None):
    """Baseline: frozen ResNet-50 with checkpoint head. No CaD, no quantum."""
    Ys, Ps = [], []
    for bi, (x, y, txt, _) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        p = torch.sigmoid(cnn_logits_from_resnet(x))
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys), np.concatenate(Ps)


@torch.no_grad()
def collect_B_quantum_no_memory(loader, max_batches=None):
    """CaD (domain 0) + parallel quantum (dt=0, no retrieval)."""
    cad.eval(); qadapter.eval()
    Ys, Ps = [], []
    for bi, (x, y, txt, _) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap        = resnet_spatial(x)
        dt_zero     = torch.zeros(x.size(0), dt_cond_dim, device=DEVICE)
        base, cls, _ = cad.logits_for_domain(fmap, 0)
        logits, _, _ = qadapter(base, cls, dt_zero)
        p            = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys), np.concatenate(Ps)


@torch.no_grad()
def collect_C_quantum_memory(loader, max_batches=None):
    """CaD (domain 0) + parallel quantum + memory retrieval."""
    cad.eval(); qadapter.eval()
    Ys, Ps = [], []
    for bi, (x, y, txt, _) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap         = resnet_spatial(x)
        v            = pool2048(fmap)
        t_emb        = text_embed(list(txt))
        qkey         = mem.query_key(v, t_emb)
        idxs, sims   = mem.retrieve(qkey, K_RETRIEVE)
        dt           = mem.retrieve_dt(idxs)
        base, cls, _ = cad.logits_for_domain(fmap, 0)
        logits, _, _ = qadapter(base, cls, dt)
        p            = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys), np.concatenate(Ps)


@torch.no_grad()
def collect_D_full(loader, max_batches=None):
    """Full pipeline: CaD domain routing + parallel quantum + memory (Eq.16 Wang et al.)."""
    cad.eval(); qadapter.eval()
    centers  = [c for c in cad.domain_centers if c is not None]
    T        = len(centers)
    A_centers = torch.tensor(np.stack(centers), device=DEVICE, dtype=torch.float32)

    Ys, Ps = [], []
    for bi, (x, y, txt, _) in enumerate(loader):
        if max_batches is not None and bi >= max_batches: break
        x = x.to(DEVICE); y = y.to(DEVICE)
        fmap       = resnet_spatial(x)
        v          = pool2048(fmap)
        t_emb      = text_embed(list(txt))
        qkey       = mem.query_key(v, t_emb)
        idxs, sims = mem.retrieve(qkey, K_RETRIEVE)
        dt         = mem.retrieve_dt(idxs)

        # Compute logits + CLS token for every domain
        logits_all, cls_all, feat_all = [], [], []
        for di in range(T):
            logits_di, cls_di, img_di = cad.logits_for_domain(fmap, di)
            logits_all.append(logits_di.unsqueeze(1))   # [B, 1, C]
            cls_all.append(cls_di.unsqueeze(1))          # [B, 1, 2048]
            feat_all.append(img_di.mean(1).unsqueeze(1)) # [B, 1, 2048]

        logits_all = torch.cat(logits_all, 1)  # [B, T, C]
        cls_all    = torch.cat(cls_all,    1)  # [B, T, 2048]
        feat_all   = torch.cat(feat_all,   1)  # [B, T, 2048]

        # Domain routing: nearest cluster center (Wang et al. 2025 Eq.16)
        dist        = ((feat_all - A_centers.unsqueeze(0)) ** 2).sum(2)  # [B, T]
        dom_id      = dist.argmin(1)                                      # [B]
        B_sz        = x.size(0)
        idx_b       = torch.arange(B_sz, device=DEVICE)
        base_logits = logits_all[idx_b, dom_id]   # [B, C]
        cls_token   = cls_all[idx_b,    dom_id]   # [B, 2048]

        logits, _, _ = qadapter(base_logits, cls_token, dt)
        p            = torch.sigmoid(logits)
        Ys.append(y.cpu().numpy()); Ps.append(p.cpu().numpy())
    return np.concatenate(Ys), np.concatenate(Ps)


# ── (18) Run all four evaluations ────────────────────────────
print("\n" + "="*55)
print("(A) CNN only — ResNet-50 baseline (no CaD, no quantum)")
Y0, P0   = collect_A_cnn_only(test_loader, TEST_MAX_BATCHES)
df0, m0  = per_class_metrics(Y0, P0, UNIFIED_LABELS)
print(df0.to_string(index=False)); print("[A] Macro:", m0)
df0.to_csv(os.path.join(OUT_DIR, "A_cnn_only.csv"), index=False)

print("\n" + "="*55)
print("(B) CaD + Parallel Quantum (no memory, dt=0)")
Y1, P1   = collect_B_quantum_no_memory(test_loader, TEST_MAX_BATCHES)
df1, m1  = per_class_metrics(Y1, P1, UNIFIED_LABELS)
print(df1.to_string(index=False)); print("[B] Macro:", m1)
df1.to_csv(os.path.join(OUT_DIR, "B_quantum_no_memory.csv"), index=False)

print("\n" + "="*55)
print("(C) CaD + Parallel Quantum + Memory retrieval")
Y2, P2   = collect_C_quantum_memory(test_loader, TEST_MAX_BATCHES)
df2, m2  = per_class_metrics(Y2, P2, UNIFIED_LABELS)
print(df2.to_string(index=False)); print("[C] Macro:", m2)
df2.to_csv(os.path.join(OUT_DIR, "C_quantum_memory.csv"), index=False)

print("\n" + "="*55)
print("(D) Full pipeline: CaD domain routing + Parallel QNN + Memory + All Novelties")
Y3, P3   = collect_D_full(test_loader, TEST_MAX_BATCHES)
df3, m3  = per_class_metrics(Y3, P3, UNIFIED_LABELS)
print(df3.to_string(index=False)); print("[D] Macro:", m3)
df3.to_csv(os.path.join(OUT_DIR, "D_full_pipeline.csv"), index=False)

# ── (19) Statistical comparison table ────────────────────────
def build_stats_table(all_Y, all_P, model_names, labels):
    """
    Computes per-model:
      Macro AUC, 95% CI (bootstrap), p-value (Wilcoxon vs A),
      Cohen's d (vs A), CV(%), Seed Stability (N/A for single run).
    """
    # Per-class AUC vectors for each model
    auc_vecs = []
    for Y, P in zip(all_Y, all_P):
        row = []
        for j in range(len(labels)):
            yt = Y[:, j]; yp = P[:, j]
            if len(np.unique(yt)) > 1:
                row.append(roc_auc_score(yt, yp))
        auc_vecs.append(row)

    baseline = auc_vecs[0]
    rows     = []
    for i, (name, Y, P) in enumerate(zip(model_names, all_Y, all_P)):
        aucs   = auc_vecs[i]
        macro  = float(np.nanmean(aucs))
        cv     = float(np.std(aucs) / (np.mean(aucs) + 1e-9) * 100)
        ci_lo, ci_hi = bootstrap_macro_auc_ci(Y, P, n_boot=500)

        if i == 0 or len(aucs) != len(baseline):
            pval = 1.0; cd = 0.0
        else:
            try:
                _, pval = wilcoxon(baseline, aucs)
            except Exception:
                try:    _, pval = ttest_rel(baseline, aucs)
                except: pval    = float("nan")
            cd = cohens_d(aucs, baseline)

        rows.append({
            "Model":            name,
            "Macro AUC":        round(macro, 4),
            "95% CI":           f"[{ci_lo:.4f}, {ci_hi:.4f}]",
            "p-value (vs A)":   "baseline" if i == 0 else
                                (f"{pval:.4f}" if not math.isnan(pval) else "N/A"),
            "Cohen's d":        round(cd, 4),
            "CV (%)":           round(cv, 2),
            "Seed Stability":   "single run",
        })
    return pd.DataFrame(rows)


stats_df = build_stats_table(
    [Y0, Y1, Y2, Y3], [P0, P1, P2, P3],
    ["A: CNN-only", "B: Quantum (no mem)", "C: Quantum+Mem", "D: Full pipeline"],
    UNIFIED_LABELS,
)
print("\n" + "="*55)
print("Statistical comparison")
print(stats_df.to_string(index=False))
stats_df.to_csv(os.path.join(OUT_DIR, "stats_comparison.csv"), index=False)

# ── (20) Gradient variance summary table ─────────────────────
grad_df = pd.DataFrame([
    {
        "Session":                    int(r["session"]) + 1,
        "Steps":                      int(r["steps"]),
        "Train time (s)":             r["train_sec"],
        "Mean grad magnitude":        r["mean_grad_mag"],
        "Std Dev":                    r["std_grad_mag"],
        "Normalized (vs 3-q ref)":    r["normalized_grad"],
    }
    for r in train_log.to_dict("records")
])
print("\n" + "="*55)
print("Gradient variance summary across sessions")
print(grad_df.to_string(index=False))
grad_df.to_csv(os.path.join(OUT_DIR, "gradient_summary.csv"), index=False)

# ── (21) Parameter and memory table ──────────────────────────
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def tensor_mem_gb(model):
    return sum(p.numel() * p.element_size()
               for p in model.parameters()) / 1e9


cad_total, _ = count_params(cad)
qa_total,  _ = count_params(qadapter)
q_only       = sum(qadapter._n_weights_per_branch)
c_only       = qa_total - q_only

param_df = pd.DataFrame([
    {"Component": "CaD model (all adapters + FC)",  "Params": cad_total},
    {"Component": "QAdapter total",                  "Params": qa_total},
    {"Component": "  — quantum branch weights only", "Params": q_only},
    {"Component": "  — classical adapter weights",   "Params": c_only},
])
print("\n" + "="*55)
print("Parameter counts")
print(param_df.to_string(index=False))
param_df.to_csv(os.path.join(OUT_DIR, "parameter_counts.csv"), index=False)

mem_gb_rag = mem.memory_gb()
mem_gb_cad = tensor_mem_gb(cad)
mem_gb_qa  = tensor_mem_gb(qadapter)
print(f"\nMemory: RAG={mem_gb_rag:.3f} GB  CaD={mem_gb_cad:.4f} GB  QAdapter={mem_gb_qa:.4f} GB")

# ── (22) Full summary JSON ────────────────────────────────────
summary = {
    "pipeline": {
        "n_branches":   N_BRANCHES,
        "n_qubits":     N_QUBITS,
        "vqc_reps":     VQC_REPS,
        "n_obs":        N_OBS,
        "n_sessions":   N_SESSIONS,
        "novelties": {
            "zero_init":       "Grant et al. 2019 arXiv:1903.05076",
            "diversity_reg":   "Novel — no quantum prior art (QuEEn arXiv:2409.09103 lists as future work)",
            "entangle_pruning":"Novel — gap between pruning lit and CL lit confirmed",
            "meta_init":       "Extends Q-MAML to domain-incremental setting",
        },
        "cost_function": "Local Pauli-Z (Cerezo et al. 2021 Nat.Commun. 12, 1791)",
    },
    "estimator_backend":   EST_BACKEND,
    "reference_gradient":  {"mean": ref_grad_mean, "var": ref_grad_var},
    "parameters": {
        "cad_total": cad_total, "qadapter_total": qa_total,
        "quantum_only": q_only, "classical_only": c_only,
    },
    "memory_gb": {"rag": mem_gb_rag, "cad": mem_gb_cad, "qadapter": mem_gb_qa},
    "metrics": {
        "A_cnn_only":          m0,
        "B_quantum_no_memory": m1,
        "C_quantum_memory":    m2,
        "D_full_pipeline":     m3,
    },
    "stats_table":    stats_df.to_dict("records"),
    "grad_summary":   grad_df.to_dict("records"),
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

# ── (23) Final print ──────────────────────────────────────────
print(f"\n{'='*55}")
print("FINAL RESULTS SUMMARY")
print(f"{'='*55}")
print(f"  (A) CNN-only           macro AUC = {m0['ROC_AUC']:.4f}  F1 = {m0['F1@0.5']:.4f}")
print(f"  (B) Quantum (no mem)   macro AUC = {m1['ROC_AUC']:.4f}  F1 = {m1['F1@0.5']:.4f}")
print(f"  (C) Quantum + Memory   macro AUC = {m2['ROC_AUC']:.4f}  F1 = {m2['F1@0.5']:.4f}")
print(f"  (D) Full pipeline      macro AUC = {m3['ROC_AUC']:.4f}  F1 = {m3['F1@0.5']:.4f}")
print(f"\n  95% CI (D): [{m3['ci_95_lo']:.4f}, {m3['ci_95_hi']:.4f}]")
print(f"  CV% (D): {m3['cv_pct']:.2f}%")
print(f"\n  Ref grad (random 3-q): {ref_grad_mean:.4e}")
row_D = stats_df[stats_df["Model"] == "D: Full pipeline"].iloc[0]
print(f"  p-value D vs A: {row_D['p-value (vs A)']}")
cohen_key = "Cohen's d"
print(f"  Cohen's d D vs A: {row_D[cohen_key]}")
print(f"\n  Q weights: {q_only}  |  Classical adapter: {c_only}")
print(f"  RAG memory: {mem_gb_rag:.3f} GB")
print(f"\n✅ All outputs saved to: {OUT_DIR}")
print("  train_log.csv")
print("  gradient_variance_log.csv")
print("  gradient_summary.csv")
print("  A_cnn_only.csv")
print("  B_quantum_no_memory.csv")
print("  C_quantum_memory.csv")
print("  D_full_pipeline.csv")
print("  stats_comparison.csv")
print("  parameter_counts.csv")
print("  summary.json")

Device: cuda

[Config] {'N_BRANCHES': 5, 'N_QUBITS': 3, 'N_SESSIONS': 3, 'LAMBDA_DIV': 0.05, 'PRUNE_THRESHOLD': 1e-05, 'DELTA_SCALE': 0.35}
Qiskit: 2.3.1
Qiskit-ML: 0.9.0
Estimator backend: qiskit_aer.primitives.EstimatorV2
[ResNet] missing=0  unexpected=0



[Dataset] {'stream': 86524, 'sessions': [28841, 28841, 28842], 'test': 25596}

[Memory] {'N': 30000, 'D': 12, 't_desc_dim': 7, 'sizeGB': 0.004}

[QAdapter] 5 branches × 3 qubits
  Weights per branch : [6, 6, 6, 6, 6]
  Obs per branch     : [3, 3, 3, 3, 3]
  Total Q weights    : 30
  Total Q outputs    : 15

[Measuring reference gradient (single 3-qubit random init) ...]
  Ref mean grad: 6.2991e-01  Ref var: 4.9177e-02

[Sanity] fmap: (16, 2048, 7, 7) cls: (16, 1, 2048) img: (16, 49, 2048)
[Sanity] max|ResNet - CaD(d=0)| = 0.000000  (should be ≈ 0)
[Sanity] dt: (16, 7)  sims: [0.074, 0.336]
[Sanity] logits: (16, 5)  gate: [0.519, 0.519]
[Sanity] diversity_loss at init: 0.5111  (ideally > 0)

[Training ...]

  Session 1/3  (domain 0)
  [s0] step=  10  loss=0.4338  bce=0.1412  div=0.4803  gate=0.523  avg_grad_var=6.15e-05  t=214.2s
  [s0] step=  20  loss=0.4992  bce=0.2021  div=0.4623  gate=0.523  avg_grad_var=6.41e-05  t=428.7s
  [s0] step=  30  loss=0.7166  bce=0.3810  div=0.3825  gate